In [ ]:
import pandas as pd

# Define the input Excel file path
excel_file_path = r"C:\Users\Shekoofeh\Documents\Health Insurance Data\Cholera\Cholera.excel.xlsx"

# Define the output CSV file path
csv_file_path = r"C:\Users\Shekoofeh\Documents\Health Insurance Data\Cholera\Cholera_csv.csv"

# Read the Excel file (assuming the data is in the first sheet)
df = pd.read_excel(excel_file_path)

# Save as CSV with UTF-8 encoding
df.to_csv(csv_file_path, index=False, encoding='utf-8-sig')  # Use 'utf-8-sig' for better compatibility with Excel


In [ ]:
import pandas as pd
import chardet
from deep_translator import GoogleTranslator

# ============================================================
# 1. FILE PATH
# ============================================================

file_path = r"C:\Users\Shekoofeh\Documents\Health Insurance Data\Cholera\Cholera_csv.csv"


# ============================================================
# 2. LOAD DATASET WITH ROBUST ENCODING DETECTION
# ============================================================

encodings = [
    "utf-8",
    "utf-8-sig",
    "windows-1256",
    "cp1256",
    "iso-8859-6",
    "utf-16"
]

df = None

for enc in encodings:
    try:
        df = pd.read_csv(file_path, encoding=enc)
        print(f"Successfully loaded with encoding: {enc}")
        break
    except (UnicodeDecodeError, UnicodeError):
        continue

# If all predefined encodings fail, use chardet
if df is None:
    print("Predefined encodings failed. Detecting encoding with chardet...")

    with open(file_path, "rb") as f:
        raw_data = f.read()

    detected = chardet.detect(raw_data)
    encoding = detected.get("encoding")

    print(f"Detected encoding: {encoding}")

    df = pd.read_csv(file_path, encoding=encoding)


# ============================================================
# 3. DISPLAY ALL COLUMN NAMES
# ============================================================

print("\n" + "=" * 70)
print("ALL COLUMN NAMES")
print("=" * 70)

for i, col in enumerate(df.columns):
    print(f"{i}: {repr(col)}")


# ============================================================
# 4. FIND THE "DIAGNOSIS TYPE" COLUMN
# ============================================================

diagnosis_column = None

# First, look for an exact/near-exact Diagnosis Type column
for col in df.columns:
    normalized_col = (
        str(col)
        .strip()
        .lower()
        .replace("_", " ")
        .replace("-", " ")
    )

    if (
        normalized_col == "diagnosis type"
        or "diagnosis type" in normalized_col
    ):
        diagnosis_column = col
        break


# If not found, look for any column containing diagnosis
if diagnosis_column is None:
    for col in df.columns:
        if "diagnosis" in str(col).lower():
            diagnosis_column = col
            break


# Stop with a clear message if no diagnosis column exists
if diagnosis_column is None:

    print("\nWARNING: No Diagnosis Type column was found.")

    print("\nAvailable columns are:")
    print(df.columns.tolist())

    raise ValueError(
        "\nCould not find the Diagnosis Type column. "
        "Please check the printed column names."
    )


print("\n" + "=" * 70)
print("DIAGNOSIS COLUMN FOUND")
print("=" * 70)
print(f"Column name: {repr(diagnosis_column)}")


# ============================================================
# 5. FUNCTION TO NORMALIZE PERSIAN/ARABIC TEXT
# ============================================================

def normalize_persian_text(value):

    if pd.isna(value):
        return value

    value = str(value)

    # Remove leading/trailing spaces
    value = value.strip()

    # Normalize Arabic/Persian character variants
    value = value.replace("ي", "ی")
    value = value.replace("ى", "ی")
    value = value.replace("ك", "ک")

    # Normalize different types of spaces
    value = value.replace("\u200c", " ")   # Zero-width non-joiner
    value = value.replace("\u200f", "")    # Right-to-left mark
    value = value.replace("\u200e", "")    # Left-to-right mark

    # Convert multiple spaces to one
    value = " ".join(value.split())

    return value


# ============================================================
# 6. CLEAN DIAGNOSIS TYPE VALUES
# ============================================================

df[diagnosis_column] = df[diagnosis_column].apply(normalize_persian_text)


# ============================================================
# 7. SHOW ORIGINAL DIAGNOSIS VALUES
# ============================================================

print("\n" + "=" * 70)
print("DIAGNOSIS TYPE VALUES BEFORE TRANSLATION")
print("=" * 70)

print(df[diagnosis_column].value_counts(dropna=False))


# ============================================================
# 8. CORRECT MANUAL TRANSLATION DICTIONARY
# ============================================================
#
# IMPORTANT:
# The Persian values below are normalized using the same
# normalization function as the dataset.
#
# Your specific values:
#
# بستری موقت
# بستري
# گلوبال
# اورژانس تحت نظر
#
# are explicitly included.
# ============================================================

diagnosis_translation = {

    # --------------------------------------------------------
    # Your actual Diagnosis Type values
    # --------------------------------------------------------

    "بستری موقت":
        "Temporary Hospitalization",

    "بستری":
        "Hospitalization / Inpatient",

    "گلوبال":
        "Global",

    "اورژانس تحت نظر":
        "Emergency Department Observation",


    # --------------------------------------------------------
    # Other possible Diagnosis Type values
    # --------------------------------------------------------

    "فرار":
        "Elopement / Runaway",

    "بهبودی نسبی":
        "Relative Improvement",

    "بهبودی کامل":
        "Complete Improvement / Full Recovery",

    "پیگیری":
        "Follow-up",

    "ترخیص با میل شخصی":
        "Discharged Against Medical Advice",

    "انتقال به مرکز دیگر":
        "Transferred to Another Center",

    "فوت":
        "Death / Expired",
}


# ============================================================
# 9. NORMALIZE THE DICTIONARY KEYS TOO
# ============================================================

diagnosis_translation_normalized = {
    normalize_persian_text(key): value
    for key, value in diagnosis_translation.items()
}


# ============================================================
# 10. DISPLAY WHICH VALUES WILL BE TRANSLATED
# ============================================================

print("\n" + "=" * 70)
print("DIAGNOSIS TRANSLATION CHECK")
print("=" * 70)

unique_diagnoses = (
    df[diagnosis_column]
    .dropna()
    .unique()
)

for value in unique_diagnoses:

    normalized_value = normalize_persian_text(value)

    if normalized_value in diagnosis_translation_normalized:

        print(
            f"'{value}'"
            f"  -->  "
            f"'{diagnosis_translation_normalized[normalized_value]}'"
        )

    else:

        print(
            f"'{value}'"
            f"  -->  "
            f"[NO MANUAL TRANSLATION FOUND - WILL BE KEPT]"
        )


# ============================================================
# 11. TRANSLATE DIAGNOSIS TYPE
# ============================================================

def translate_diagnosis(value):

    if pd.isna(value):
        return value

    normalized_value = normalize_persian_text(value)

    # Return translated value if found
    if normalized_value in diagnosis_translation_normalized:
        return diagnosis_translation_normalized[normalized_value]

    # Otherwise KEEP the original value
    return value


df[diagnosis_column] = df[diagnosis_column].apply(
    translate_diagnosis
)


# ============================================================
# 12. VERIFY DIAGNOSIS TRANSLATION
# ============================================================

print("\n" + "=" * 70)
print("DIAGNOSIS TYPE VALUES AFTER TRANSLATION")
print("=" * 70)

print(df[diagnosis_column].value_counts(dropna=False))


# ============================================================
# 13. TRANSLATE OTHER PERSIAN CATEGORICAL COLUMNS
# ============================================================

translator = GoogleTranslator(
    source="persian",
    target="en"
)

persian_columns = [
    "Gender",
    "Department Type",
    "Discharge Status",
    "Insurance Fund",
    "Treating Physician Specialty",
    "Ownership",
    "Service Group"
]


# Find actual column names case-insensitively
actual_persian_columns = []

for requested_col in persian_columns:

    for actual_col in df.columns:

        if str(requested_col).strip().lower() == str(actual_col).strip().lower():

            # Don't accidentally translate Diagnosis Type again
            if actual_col != diagnosis_column:
                actual_persian_columns.append(actual_col)

            break


print("\n" + "=" * 70)
print("OTHER COLUMNS TO TRANSLATE")
print("=" * 70)

print(actual_persian_columns)


# ============================================================
# 14. TRANSLATE OTHER CATEGORICAL COLUMNS
# ============================================================

def translate_column(column):

    print("\n" + "-" * 60)
    print(f"Translating column: {column}")
    print("-" * 60)

    # Get unique non-missing values
    unique_values = (
        df[column]
        .dropna()
        .unique()
    )

    translation_dict = {}

    for val in unique_values:

        clean_val = str(val).strip()

        if clean_val in ["", "nan", "NaN", "None"]:
            translation_dict[val] = val
            continue

        try:

            translated = translator.translate(clean_val)

            translation_dict[val] = translated

            print(
                f"'{clean_val}'  -->  '{translated}'"
            )

        except Exception as e:

            print(
                f"ERROR translating '{clean_val}': {e}"
            )

            # IMPORTANT:
            # Keep the original value if translation fails
            translation_dict[val] = val

    # Map translations
    df[column] = df[column].map(
        translation_dict
    ).fillna(df[column])


# Translate each additional column
for col in actual_persian_columns:

    if col in df.columns:
        translate_column(col)


# ============================================================
# 15. SAVE TRANSLATED DATASET
# ============================================================

output_path = file_path.replace(
    ".csv",
    "_translated.csv"
)

df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 16. FINAL VERIFICATION
# ============================================================

print("\n" + "=" * 70)
print("FINAL VERIFICATION")
print("=" * 70)

print(f"\nOutput file:")
print(output_path)

print("\nDiagnosis Type column:")
print(repr(diagnosis_column))

print("\nFinal Diagnosis Type values:")

print(
    df[diagnosis_column].value_counts(
        dropna=False
    )
)


print("\nFirst 20 rows of Diagnosis Type:")

print(
    df[[diagnosis_column]].head(20).to_string(
        index=False
    )
)


print("\n" + "=" * 70)
print("TRANSLATION COMPLETE")
print("=" * 70)

print(
    f"\nThe translated dataset has been saved to:\n"
    f"{output_path}"
)

In [ ]:
# ================================================================
# CHOLERA CLAIM DATA
# CLAIM-LEVEL CLEANING -> EPISODE CONSTRUCTION
# -> EPISODE-LEVEL QUALITY CONTROL
# -> LEAKAGE-SAFE ML VARIABLE CLASSIFICATION
#
# IMPORTANT:
# This script intentionally DOES NOT:
#   1. Winsorize costs
#   2. Remove statistical outliers
#   3. Impute missing values
#   4. Standardize/normalize variables
#   5. Invent missing Patient_IDs
#
# Those decisions should occur only after the episode dataset
# and prediction target are clearly defined.
# ================================================================

import pandas as pd
import numpy as np
import os
import re
from pathlib import Path


# ================================================================
# 1. FILE PATH
# ================================================================

file_path = (
    r"C:\Users\Shekoofeh\Documents\Health Insurance Data"
    r"\Cholera\Cholera_csv_translated.csv"
)

OUTPUT_DIR = Path(file_path).parent


# Output files
RAW_AUDIT_FILE = OUTPUT_DIR / "01_raw_data_audit.csv"
UNIDENTIFIED_FILE = OUTPUT_DIR / "02_unidentified_patient_records.csv"
CLEAN_CLAIMS_FILE = OUTPUT_DIR / "03_clean_claim_level_data.csv"
EPISODE_FILE = OUTPUT_DIR / "04_episode_level_data.csv"
EPISODE_QC_FILE = OUTPUT_DIR / "05_episode_quality_control.csv"
INCONSISTENCY_FILE = OUTPUT_DIR / "06_episode_inconsistencies.csv"
LEAKAGE_FILE = OUTPUT_DIR / "07_leakage_variable_classification.csv"


# ================================================================
# 2. LOAD RAW DATA
# ================================================================

print("=" * 90)
print("STEP 1 — LOADING RAW CLAIM DATA")
print("=" * 90)

df_raw = pd.read_csv(
    file_path,
    encoding="utf-8-sig",
    low_memory=False
)

print(f"Raw number of rows: {len(df_raw):,}")
print(f"Raw number of columns: {len(df_raw.columns):,}")


# Preserve original row number permanently
df_raw["_original_row_number"] = np.arange(len(df_raw))


# ================================================================
# 3. STANDARDIZE COLUMN NAMES
# ================================================================

def clean_column_name(column):
    """
    Standardize column names without changing their meaning.
    """
    column = str(column)

    column = column.replace('"', "")
    column = column.replace("\n", " ")
    column = column.replace("\t", " ")

    # Collapse repeated spaces
    column = re.sub(r"\s+", " ", column)

    return column.strip()


df_raw.columns = [
    clean_column_name(col)
    for col in df_raw.columns
]


print("\nStandardized column names:")
for i, col in enumerate(df_raw.columns, start=1):
    print(f"{i:02d}. {col}")


# ================================================================
# 4. EXPECTED VARIABLES
# ================================================================

expected_columns = [
    "Patient_ID",
    "Diagnosis type",
    "Department Type",
    "Discharge Status",
    "Age",
    "Gender",
    "Ownership",
    "Insurance Fund",
    "Treating Physician Specialty",
    "Admision date",
    "Discharge date",
    "Length of Stay",
    "Service Group",
    "Total Cost",
    "Insured Person's Share",
    "Organization Requested Share",
    "Organization Paid Share",
    "Terminally Patients Paid Share",
    "Terminally Patients Requested Share",
    "Allocated subsidized foreign currency share",
    "Requested subsidized foreign currency share"
]


missing_expected_columns = [
    col for col in expected_columns
    if col not in df_raw.columns
]


print("\nMissing expected columns:")
if missing_expected_columns:
    for col in missing_expected_columns:
        print("  MISSING:", col)
else:
    print("  None")


if "Patient_ID" not in df_raw.columns:
    raise ValueError(
        "\nPatient_ID was not found.\n"
        "The script cannot safely construct patient episodes "
        "until the identifier structure is resolved."
    )


# ================================================================
# 5. RAW DATA AUDIT
# ================================================================

print("\n" + "=" * 90)
print("STEP 2 — RAW DATA QUALITY AUDIT")
print("=" * 90)


raw_audit = pd.DataFrame({
    "column": df_raw.columns,
    "dtype": [
        str(df_raw[col].dtype)
        for col in df_raw.columns
    ],
    "missing_n": [
        df_raw[col].isna().sum()
        for col in df_raw.columns
    ],
    "missing_pct": [
        100 * df_raw[col].isna().mean()
        for col in df_raw.columns
    ],
    "unique_n": [
        df_raw[col].nunique(dropna=True)
        for col in df_raw.columns
    ]
})


raw_audit.to_csv(
    RAW_AUDIT_FILE,
    index=False,
    encoding="utf-8-sig"
)


print(raw_audit.to_string(index=False))


# ================================================================
# 6. PATIENT_ID AUDIT
# ================================================================

print("\n" + "=" * 90)
print("STEP 3 — PATIENT_ID AUDIT")
print("=" * 90)


# Preserve raw Patient_ID before any conversion
patient_id_raw = df_raw["Patient_ID"].copy()


# Define textual missing identifiers
missing_id_values = {
    "",
    "nan",
    "none",
    "null",
    "na",
    "n/a",
    "missing"
}


patient_id_as_string = (
    df_raw["Patient_ID"]
    .astype("string")
    .str.strip()
)


patient_id_missing = (
    df_raw["Patient_ID"].isna()
    |
    patient_id_as_string.str.lower().isin(
        missing_id_values
    )
)


print(
    f"Rows with missing/unusable Patient_ID: "
    f"{patient_id_missing.sum():,}"
)

print(
    f"Percentage missing/unusable: "
    f"{100 * patient_id_missing.mean():.2f}%"
)


# ================================================================
# 7. SEPARATE IDENTIFIABLE AND UNIDENTIFIABLE CLAIMS
# ================================================================

df_unidentified = df_raw.loc[
    patient_id_missing
].copy()


df = df_raw.loc[
    ~patient_id_missing
].copy()


# Clean Patient_ID only for identifiable records
df["Patient_ID"] = (
    df["Patient_ID"]
    .astype("string")
    .str.strip()
)


print(
    f"\nIdentifiable claim lines: "
    f"{len(df):,}"
)

print(
    f"Unidentifiable claim lines: "
    f"{len(df_unidentified):,}"
)


# Save unidentified records without modifying them
df_unidentified.to_csv(
    UNIDENTIFIED_FILE,
    index=False,
    encoding="utf-8-sig"
)


print(
    f"\nUnidentified records saved to:\n"
    f"{UNIDENTIFIED_FILE}"
)


# ================================================================
# 8. NUMERIC VARIABLE CONVERSION
# ================================================================

print("\n" + "=" * 90)
print("STEP 4 — NUMERIC DATA CLEANING")
print("=" * 90)


numeric_columns = [
    "Age",
    "Length of Stay",
    "Total Cost",
    "Insured Person's Share",
    "Organization Requested Share",
    "Organization Paid Share",
    "Terminally Patients Paid Share",
    "Terminally Patients Requested Share",
    "Allocated subsidized foreign currency share",
    "Requested subsidized foreign currency share"
]


numeric_columns_present = [
    col for col in numeric_columns
    if col in df.columns
]


for col in numeric_columns_present:

    # Convert strings such as "5,256,050" safely
    df[col] = (
        df[col]
        .astype("string")
        .str.replace(",", "", regex=False)
        .str.strip()
    )

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )


print("\nNumeric variables converted:")
for col in numeric_columns_present:
    print(f"  {col}")


# ================================================================
# 9. DATE CONVERSION
# ================================================================

print("\n" + "=" * 90)
print("STEP 5 — DATE CLEANING")
print("=" * 90)


date_columns = [
    "Admision date",
    "Discharge date"
]


for col in date_columns:

    if col in df.columns:

        df[col] = pd.to_datetime(
            df[col],
            errors="coerce"
        )


print("\nDate variables converted.")


# ================================================================
# 10. AGE QUALITY CONTROL
# ================================================================

print("\n" + "=" * 90)
print("STEP 6 — BASIC CLINICAL VALIDITY CHECKS")
print("=" * 90)


if "Age" in df.columns:

    invalid_age = (
        df["Age"].notna()
        &
        (
            (df["Age"] < 0)
            |
            (df["Age"] > 120)
        )
    )

    print(
        f"Invalid age observations: "
        f"{invalid_age.sum():,}"
    )

    # Do NOT impute.
    # Invalid ages become missing.
    df.loc[
        invalid_age,
        "Age"
    ] = np.nan

    df["_invalid_age"] = invalid_age


# ================================================================
# 11. FINANCIAL DATA QUALITY CONTROL
# ================================================================

financial_columns = [
    "Total Cost",
    "Insured Person's Share",
    "Organization Requested Share",
    "Organization Paid Share",
    "Terminally Patients Paid Share",
    "Terminally Patients Requested Share",
    "Allocated subsidized foreign currency share",
    "Requested subsidized foreign currency share"
]


financial_columns_present = [
    col for col in financial_columns
    if col in df.columns
]


for col in financial_columns_present:

    negative_values = (
        df[col].notna()
        &
        (df[col] < 0)
    )

    print(
        f"{col}: "
        f"{negative_values.sum():,} negative values"
    )

    # IMPORTANT:
    # We do NOT automatically convert negative values to zero.
    # They are flagged as potential data-quality problems.

    df[f"_negative_{col}"] = negative_values


# ================================================================
# 12. DATE ORDER CHECK
# ================================================================

if (
    "Admision date" in df.columns
    and
    "Discharge date" in df.columns
):

    df["_invalid_date_order"] = (
        df["Admision date"].notna()
        &
        df["Discharge date"].notna()
        &
        (
            df["Discharge date"]
            <
            df["Admision date"]
        )
    )

else:

    df["_invalid_date_order"] = False


print(
    "\nClaim lines with discharge before admission:",
    df["_invalid_date_order"].sum()
)


# ================================================================
# 13. CALCULATED CLAIM-LEVEL LOS
# ================================================================

if (
    "Admision date" in df.columns
    and
    "Discharge date" in df.columns
):

    df["Calculated_LOS_days"] = (
        df["Discharge date"]
        -
        df["Admision date"]
    ).dt.total_seconds() / 86400

else:

    df["Calculated_LOS_days"] = np.nan


# ================================================================
# 14. COMPARE PROVIDED AND CALCULATED LOS
# ================================================================

if "Length of Stay" in df.columns:

    df["LOS_difference"] = (
        df["Length of Stay"]
        -
        df["Calculated_LOS_days"]
    )

    print("\nLength-of-stay comparison:")
    print(
        df[
            [
                "Length of Stay",
                "Calculated_LOS_days",
                "LOS_difference"
            ]
        ].describe()
    )


# ================================================================
# 15. EXACT DUPLICATE CLAIM-LINE AUDIT
# ================================================================

print("\n" + "=" * 90)
print("STEP 7 — DUPLICATE CLAIM-LINE AUDIT")
print("=" * 90)


# Exclude our diagnostic columns
original_data_columns = [
    col
    for col in df.columns
    if not col.startswith("_")
    and col not in [
        "Calculated_LOS_days",
        "LOS_difference"
    ]
]


df["_exact_duplicate"] = df.duplicated(
    subset=original_data_columns,
    keep=False
)


print(
    "Claim lines involved in exact duplicate patterns:",
    df["_exact_duplicate"].sum()
)


# IMPORTANT:
# We do not delete them automatically.
# Duplicate billing lines and genuine repeated claims
# must be distinguished from accidental duplication.


# ================================================================
# 16. EPISODE IDENTIFICATION
# ================================================================

print("\n" + "=" * 90)
print("STEP 8 — EPISODE IDENTIFICATION")
print("=" * 90)


episode_key_columns = [
    "Patient_ID",
    "Admision date",
    "Discharge date"
]


missing_episode_key_columns = [
    col
    for col in episode_key_columns
    if col not in df.columns
]


if missing_episode_key_columns:

    raise ValueError(
        "Cannot construct the proposed episode key. "
        f"Missing columns: {missing_episode_key_columns}"
    )


# Rows with missing admission/discharge dates cannot be
# safely assigned to an episode using this rule.

episode_key_missing = (
    df["Patient_ID"].isna()
    |
    df["Admision date"].isna()
    |
    df["Discharge date"].isna()
)


print(
    "Claim lines with incomplete proposed episode key:",
    episode_key_missing.sum()
)


# Create Episode_ID only when all key components are available

df["Episode_ID"] = pd.NA


valid_episode_key = ~episode_key_missing


df.loc[
    valid_episode_key,
    "Episode_ID"
] = (
    df.loc[
        valid_episode_key,
        episode_key_columns
    ]
    .astype("string")
    .agg("|".join, axis=1)
)


# ================================================================
# 17. EPISODE KEY AUDIT
# ================================================================

episode_key_audit = (
    df.loc[
        valid_episode_key
    ]
    .groupby(
        episode_key_columns,
        dropna=False
    )
    .size()
    .reset_index(
        name="Number_of_claim_lines"
    )
)


print(
    "\nPotential hospitalization episodes:",
    len(episode_key_audit)
)


print(
    "Maximum claim lines within one episode:",
    episode_key_audit[
        "Number_of_claim_lines"
    ].max()
)


print(
    "Episodes containing >1 claim line:",
    (
        episode_key_audit[
            "Number_of_claim_lines"
        ] > 1
    ).sum()
)


# ================================================================
# 18. CHECK WHETHER PATIENT + DATES IS PLAUSIBLE
# ================================================================

# Number of episodes per patient

episodes_per_patient = (
    df.loc[
        valid_episode_key
    ]
    .groupby("Patient_ID")["Episode_ID"]
    .nunique()
)


print(
    "\nPatients with multiple hospitalization episodes:",
    (
        episodes_per_patient > 1
    ).sum()
)


print(
    "Maximum episodes per patient:",
    episodes_per_patient.max()
)


# ================================================================
# 19. EPISODE-LEVEL AGGREGATION
# ================================================================

print("\n" + "=" * 90)
print("STEP 9 — CLAIM -> EPISODE AGGREGATION")
print("=" * 90)


df_valid = df.loc[
    valid_episode_key
].copy()


# ------------------------------------------------
# Financial variables = SUM
# ------------------------------------------------

sum_columns = [
    "Total Cost",
    "Insured Person's Share",
    "Organization Requested Share",
    "Organization Paid Share",
    "Terminally Patients Paid Share",
    "Terminally Patients Requested Share",
    "Allocated subsidized foreign currency share",
    "Requested subsidized foreign currency share"
]


sum_columns = [
    col
    for col in sum_columns
    if col in df_valid.columns
]


# ------------------------------------------------
# Episode-level variables
# ------------------------------------------------

first_columns = [
    "Patient_ID",
    "Age",
    "Gender",
    "Ownership",
    "Insurance Fund",
    "Treating Physician Specialty",
    "Admision date",
    "Discharge date",
    "Discharge Status",
    "Diagnosis type",
    "Department Type"
]


first_columns = [
    col
    for col in first_columns
    if col in df_valid.columns
]


# ------------------------------------------------
# Aggregation dictionary
# ------------------------------------------------

aggregation_dictionary = {}


# Financial sums
for col in sum_columns:
    aggregation_dictionary[col] = "sum"


# Episode-level characteristics
for col in first_columns:
    aggregation_dictionary[col] = "first"


# Number of claim lines
aggregation_dictionary["_original_row_number"] = "count"


# Number of unique service groups
if "Service Group" in df_valid.columns:

    aggregation_dictionary[
        "Service Group"
    ] = "nunique"


# ================================================================
# 20. PERFORM AGGREGATION
# ================================================================

episodes = (
    df_valid
    .groupby(
        "Episode_ID",
        as_index=False
    )
    .agg(aggregation_dictionary)
)


# ================================================================
# 21. RENAME AGGREGATED VARIABLES
# ================================================================

episodes = episodes.rename(
    columns={
        "_original_row_number":
            "Number_of_claim_lines",

        "Service Group":
            "Number_of_service_groups"
    }
)


# ================================================================
# 22. EPISODE LOS
# ================================================================

episodes["Episode_LOS_days"] = (
    episodes["Discharge date"]
    -
    episodes["Admision date"]
).dt.total_seconds() / 86400


# IMPORTANT:
#
# We DO NOT sum Length of Stay.
#
# Episode LOS is calculated directly from:
#
# discharge date - admission date


# ================================================================
# 23. EPISODE-LEVEL DATE QUALITY FLAG
# ================================================================

episode_date_quality = (
    df_valid
    .groupby("Episode_ID")
    .agg(
        invalid_date_order=(
            "_invalid_date_order",
            "max"
        )
    )
    .reset_index()
)


episodes = episodes.merge(
    episode_date_quality,
    on="Episode_ID",
    how="left"
)


episodes["invalid_date_order"] = (
    episodes["invalid_date_order"]
    .fillna(False)
    .astype(bool)
)


# ================================================================
# 24. EPISODE-LEVEL DUPLICATE FLAG
# ================================================================

episode_duplicate_quality = (
    df_valid
    .groupby("Episode_ID")
    .agg(
        contains_exact_duplicate_claim=(
            "_exact_duplicate",
            "max"
        )
    )
    .reset_index()
)


episodes = episodes.merge(
    episode_duplicate_quality,
    on="Episode_ID",
    how="left"
)


episodes[
    "contains_exact_duplicate_claim"
] = (
    episodes[
        "contains_exact_duplicate_claim"
    ]
    .fillna(False)
    .astype(bool)
)


# ================================================================
# 25. EPISODE-LEVEL AGE QUALITY FLAG
# ================================================================

if "_invalid_age" in df_valid.columns:

    episode_age_quality = (
        df_valid
        .groupby("Episode_ID")
        .agg(
            contains_invalid_age=(
                "_invalid_age",
                "max"
            )
        )
        .reset_index()
    )

    episodes = episodes.merge(
        episode_age_quality,
        on="Episode_ID",
        how="left"
    )

    episodes[
        "contains_invalid_age"
    ] = (
        episodes[
            "contains_invalid_age"
        ]
        .fillna(False)
        .astype(bool)
    )

else:

    episodes[
        "contains_invalid_age"
    ] = False


# ================================================================
# 26. FINANCIAL CONSISTENCY CHECK
# ================================================================

print("\n" + "=" * 90)
print("STEP 10 — EPISODE FINANCIAL CONSISTENCY")
print("=" * 90)


required_financial = [
    "Total Cost",
    "Insured Person's Share",
    "Organization Paid Share"
]


if all(
    col in episodes.columns
    for col in required_financial
):

    episodes["Known_financing_total"] = (
        episodes["Insured Person's Share"]
        +
        episodes["Organization Paid Share"]
    )

    episodes["Financing_difference"] = (
        episodes["Total Cost"]
        -
        episodes["Known_financing_total"]
    )

    episodes[
        "Financing_difference_pct"
    ] = np.where(
        episodes["Total Cost"].abs() > 0,
        100 *
        episodes["Financing_difference"]
        /
        episodes["Total Cost"].abs(),
        np.nan
    )

    print(
        episodes[
            [
                "Total Cost",
                "Known_financing_total",
                "Financing_difference",
                "Financing_difference_pct"
            ]
        ].describe()
    )


# ================================================================
# 27. CHECK MULTIPLE DISCHARGE STATUSES
# ================================================================

print("\n" + "=" * 90)
print("STEP 11 — WITHIN-EPISODE CONSISTENCY CHECKS")
print("=" * 90)


inconsistency_records = []


# ------------------------------------------------
# Discharge Status
# ------------------------------------------------

if "Discharge Status" in df_valid.columns:

    status_variability = (
        df_valid
        .groupby("Episode_ID")[
            "Discharge Status"
        ]
        .nunique(dropna=True)
    )

    multiple_status_episodes = (
        status_variability > 1
    )

    print(
        "Episodes with multiple discharge statuses:",
        multiple_status_episodes.sum()
    )

    for episode_id in status_variability[
        multiple_status_episodes
    ].index:

        values = (
            df_valid.loc[
                df_valid["Episode_ID"] == episode_id,
                "Discharge Status"
            ]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )

        inconsistency_records.append({
            "Episode_ID": episode_id,
            "Variable": "Discharge Status",
            "Values": " | ".join(values)
        })


# ------------------------------------------------
# Patient characteristics
# ------------------------------------------------

patient_level_columns = [
    "Age",
    "Gender",
    "Ownership",
    "Insurance Fund"
]


for col in patient_level_columns:

    if col not in df_valid.columns:
        continue

    variability = (
        df_valid
        .groupby("Episode_ID")[col]
        .nunique(dropna=True)
    )

    inconsistent = (
        variability > 1
    )

    print(
        f"Episodes with multiple {col} values:",
        inconsistent.sum()
    )

    for episode_id in variability[
        inconsistent
    ].index:

        values = (
            df_valid.loc[
                df_valid["Episode_ID"] == episode_id,
                col
            ]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )

        inconsistency_records.append({
            "Episode_ID": episode_id,
            "Variable": col,
            "Values": " | ".join(values)
        })


# Save inconsistencies
inconsistency_df = pd.DataFrame(
    inconsistency_records
)


inconsistency_df.to_csv(
    INCONSISTENCY_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ================================================================
# 28. CREATE EPISODE QUALITY CONTROL TABLE
# ================================================================

episode_qc = pd.DataFrame({
    "Metric": [
        "Raw claim lines",
        "Claims with missing Patient_ID",
        "Identifiable claim lines",
        "Claims with incomplete episode key",
        "Constructed episodes",
        "Episodes with invalid date order",
        "Episodes containing exact duplicate claim lines",
        "Episodes containing invalid age",
        "Episodes with multiple discharge statuses",
    ],

    "Value": [
        len(df_raw),

        len(df_unidentified),

        len(df),

        episode_key_missing.sum(),

        len(episodes),

        episodes[
            "invalid_date_order"
        ].sum(),

        episodes[
            "contains_exact_duplicate_claim"
        ].sum(),

        episodes[
            "contains_invalid_age"
        ].sum(),

        (
            status_variability > 1
        ).sum()
        if "Discharge Status" in df_valid.columns
        else np.nan
    ]
})


episode_qc.to_csv(
    EPISODE_QC_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ================================================================
# 29. LEAKAGE-SAFE VARIABLE CLASSIFICATION
# ================================================================

print("\n" + "=" * 90)
print("STEP 12 — LEAKAGE AUDIT")
print("=" * 90)


# ------------------------------------------------
# Variables used to identify episodes
# ------------------------------------------------

identifier_variables = [
    "Patient_ID",
    "Episode_ID"
]


# ------------------------------------------------
# Variables that are outcomes / post-treatment /
# potentially used to construct VBI
# ------------------------------------------------

outcome_or_post_treatment_variables = [
    "Discharge Status",
    "Discharge date",
    "Length of Stay",
    "Episode_LOS_days",
    "Total Cost",
    "Insured Person's Share",
    "Organization Requested Share",
    "Organization Paid Share",
    "Terminally Patients Paid Share",
    "Terminally Patients Requested Share",
    "Allocated subsidized foreign currency share",
    "Requested subsidized foreign currency share",
    "Known_financing_total",
    "Financing_difference",
    "Financing_difference_pct"
]


# ------------------------------------------------
# Candidate baseline variables
# ------------------------------------------------

candidate_baseline_variables = [
    "Age",
    "Gender",
    "Ownership",
    "Insurance Fund",
    "Treating Physician Specialty",
    "Diagnosis type",
    "Department Type",
    "Admision date"
]


leakage_classification = []


for col in episodes.columns:

    if col in identifier_variables:

        classification = "IDENTIFIER — NEVER USE AS ML PREDICTOR"

    elif col in outcome_or_post_treatment_variables:

        classification = (
            "OUTCOME/POST-TREATMENT — "
            "DO NOT USE AS PREDICTOR IF TARGET DEPENDS ON IT"
        )

    elif col in candidate_baseline_variables:

        classification = (
            "CANDIDATE BASELINE PREDICTOR — "
            "MUST VERIFY AVAILABLE AT PREDICTION TIME"
        )

    elif col.startswith("invalid_"):

        classification = (
            "QUALITY CONTROL VARIABLE — "
            "NOT A PREDICTOR"
        )

    elif col.startswith("contains_"):

        classification = (
            "QUALITY CONTROL VARIABLE — "
            "NOT A PREDICTOR"
        )

    elif col.startswith("Number_of_"):

        classification = (
            "POST-ADMISSION EPISODE DESCRIPTOR — "
            "POTENTIAL LEAKAGE"
        )

    elif col.startswith("Financing_"):

        classification = (
            "DERIVED FINANCIAL VARIABLE — "
            "POTENTIAL TARGET LEAKAGE"
        )

    else:

        classification = (
            "UNCLASSIFIED — "
            "MANUAL REVIEW REQUIRED"
        )

    leakage_classification.append({
        "Variable": col,
        "Classification": classification
    })


leakage_df = pd.DataFrame(
    leakage_classification
)


leakage_df.to_csv(
    LEAKAGE_FILE,
    index=False,
    encoding="utf-8-sig"
)


print("\nLeakage classification:")
print(
    leakage_df.to_string(index=False)
)


# ================================================================
# 30. SAVE CLEAN CLAIM-LEVEL DATA
# ================================================================

df.to_csv(
    CLEAN_CLAIMS_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ================================================================
# 31. SAVE EPISODE-LEVEL DATA
# ================================================================

episodes.to_csv(
    EPISODE_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ================================================================
# 32. FINAL SUMMARY
# ================================================================

print("\n" + "=" * 90)
print("FINAL DATA PREPROCESSING SUMMARY")
print("=" * 90)


print(
    f"Raw claim lines: "
    f"{len(df_raw):,}"
)

print(
    f"Claims with missing Patient_ID: "
    f"{len(df_unidentified):,}"
)

print(
    f"Identifiable claim lines: "
    f"{len(df):,}"
)

print(
    f"Claims with incomplete episode key: "
    f"{episode_key_missing.sum():,}"
)

print(
    f"Constructed hospitalization episodes: "
    f"{len(episodes):,}"
)


if len(episodes) > 0:

    print(
        f"Average claim lines per episode: "
        f"{len(df_valid) / len(episodes):.2f}"
    )


print(
    f"Episodes with invalid date order: "
    f"{episodes['invalid_date_order'].sum():,}"
)

print(
    f"Episodes containing exact duplicate claims: "
    f"{episodes['contains_exact_duplicate_claim'].sum():,}"
)

print(
    f"Episodes containing invalid age: "
    f"{episodes['contains_invalid_age'].sum():,}"
)


if "Discharge Status" in df_valid.columns:

    print(
        f"Episodes with multiple discharge statuses: "
        f"{(status_variability > 1).sum():,}"
    )


print("\nEpisode dataset dimensions:")
print(episodes.shape)


print("\nEpisode-level variables:")
for i, col in enumerate(
    episodes.columns,
    start=1
):
    print(
        f"{i:02d}. {col}"
    )


print("\n" + "=" * 90)
print("FILES CREATED")
print("=" * 90)

print(
    "1. Raw audit:\n",
    RAW_AUDIT_FILE
)

print(
    "2. Unidentified records:\n",
    UNIDENTIFIED_FILE
)

print(
    "3. Clean claim-level data:\n",
    CLEAN_CLAIMS_FILE
)

print(
    "4. Episode-level data:\n",
    EPISODE_FILE
)

print(
    "5. Episode QC:\n",
    EPISODE_QC_FILE
)

print(
    "6. Episode inconsistencies:\n",
    INCONSISTENCY_FILE
)

print(
    "7. Leakage classification:\n",
    LEAKAGE_FILE
)

print("\nDONE.")

In [ ]:
# ================================================================
# CHOLERA CLAIM DATA
# ALL CLAIMS RETAINED
# CLAIM-LEVEL CLEANING
# -> IDENTIFIED + ANONYMOUS EPISODE CONSTRUCTION
# -> EPISODE-LEVEL QUALITY CONTROL
# -> LEAKAGE-SAFE ML VARIABLE CLASSIFICATION
#
# IMPORTANT PRINCIPLES
# --------------------
# 1. NO CLAIM LINE IS DROPPED SOLELY BECAUSE Patient_ID IS MISSING.
#
# 2. Patient_ID IS NEVER INVENTED.
#
# 3. Claims with valid Patient_ID are linked using:
#       Patient_ID + Admission date + Discharge date
#
# 4. Claims without Patient_ID are retained and may be grouped into
#    ANONYMOUS EPISODES using a conservative deterministic signature:
#
#       Admission date
#       Discharge date
#       Age
#       Gender
#       Ownership
#       Insurance Fund
#       Treating Physician Specialty
#       Diagnosis type
#       Department Type
#
# 5. If an unidentified claim does not contain sufficient information
#    for anonymous clustering, it is retained as a SINGLETON ANONYMOUS
#    EPISODE.
#
# 6. No Patient_ID is ever fabricated.
#
# 7. Financial variables are summed within episodes.
#
# 8. Episode-level dates and baseline characteristics are taken from
#    the episode record after within-episode consistency checking.
#
# 9. We DO NOT:
#       - Winsorize costs
#       - Remove statistical outliers
#       - Impute missing values
#       - Standardize/normalize variables
#       - Delete observations because Patient_ID is missing
#
# 10. Outcome/post-treatment variables remain excluded from the
#     baseline ML predictor set.
# ================================================================


import pandas as pd
import numpy as np
import os
import re
from pathlib import Path


# ================================================================
# 1. FILE PATH
# ================================================================

file_path = (
    r"C:\Users\Shekoofeh\Documents\Health Insurance Data"
    r"\Cholera\Cholera_csv_translated.csv"
)

OUTPUT_DIR = Path(file_path).parent


# ================================================================
# OUTPUT FILES
# ================================================================

RAW_AUDIT_FILE = OUTPUT_DIR / "01_raw_data_audit.csv"

UNIDENTIFIED_FILE = (
    OUTPUT_DIR / "02_unidentified_patient_records.csv"
)

CLEAN_CLAIMS_FILE = (
    OUTPUT_DIR / "03_clean_claim_level_data.csv"
)

EPISODE_FILE = (
    OUTPUT_DIR / "04_episode_level_data.csv"
)

EPISODE_QC_FILE = (
    OUTPUT_DIR / "05_episode_quality_control.csv"
)

INCONSISTENCY_FILE = (
    OUTPUT_DIR / "06_episode_inconsistencies.csv"
)

LEAKAGE_FILE = (
    OUTPUT_DIR / "07_leakage_variable_classification.csv"
)

ANONYMOUS_EPISODE_FILE = (
    OUTPUT_DIR / "08_anonymous_episode_linkage_audit.csv"
)


# ================================================================
# 2. LOAD RAW DATA
# ================================================================

print("=" * 90)
print("STEP 1 — LOADING RAW CLAIM DATA")
print("=" * 90)


df_raw = pd.read_csv(
    file_path,
    encoding="utf-8-sig",
    low_memory=False
)


print(f"Raw number of rows: {len(df_raw):,}")
print(f"Raw number of columns: {len(df_raw.columns):,}")


# Preserve original row number permanently
df_raw["_original_row_number"] = np.arange(
    len(df_raw)
)


# ================================================================
# 3. STANDARDIZE COLUMN NAMES
# ================================================================

def clean_column_name(column):

    column = str(column)

    column = column.replace('"', "")
    column = column.replace("\n", " ")
    column = column.replace("\t", " ")

    column = re.sub(
        r"\s+",
        " ",
        column
    )

    return column.strip()


df_raw.columns = [
    clean_column_name(col)
    for col in df_raw.columns
]


print("\nStandardized column names:")

for i, col in enumerate(
    df_raw.columns,
    start=1
):

    print(
        f"{i:02d}. {col}"
    )


# ================================================================
# 4. EXPECTED VARIABLES
# ================================================================

expected_columns = [

    "Patient_ID",
    "Diagnosis type",
    "Department Type",
    "Discharge Status",
    "Age",
    "Gender",
    "Ownership",
    "Insurance Fund",
    "Treating Physician Specialty",
    "Admision date",
    "Discharge date",
    "Length of Stay",
    "Service Group",
    "Total Cost",
    "Insured Person's Share",
    "Organization Requested Share",
    "Organization Paid Share",
    "Terminally Patients Paid Share",
    "Terminally Patients Requested Share",
    "Allocated subsidized foreign currency share",
    "Requested subsidized foreign currency share"
]


missing_expected_columns = [
    col
    for col in expected_columns
    if col not in df_raw.columns
]


print("\nMissing expected columns:")

if missing_expected_columns:

    for col in missing_expected_columns:
        print(
            "  MISSING:",
            col
        )

else:

    print("  None")


if "Patient_ID" not in df_raw.columns:

    raise ValueError(
        "\nPatient_ID column was not found."
    )


# ================================================================
# 5. RAW DATA AUDIT
# ================================================================

print("\n" + "=" * 90)
print("STEP 2 — RAW DATA QUALITY AUDIT")
print("=" * 90)


raw_audit = pd.DataFrame({

    "column":
        df_raw.columns,

    "dtype": [
        str(df_raw[col].dtype)
        for col in df_raw.columns
    ],

    "missing_n": [
        df_raw[col].isna().sum()
        for col in df_raw.columns
    ],

    "missing_pct": [
        100 * df_raw[col].isna().mean()
        for col in df_raw.columns
    ],

    "unique_n": [
        df_raw[col].nunique(dropna=True)
        for col in df_raw.columns
    ]

})


raw_audit.to_csv(
    RAW_AUDIT_FILE,
    index=False,
    encoding="utf-8-sig"
)


print(
    raw_audit.to_string(
        index=False
    )
)


# ================================================================
# 6. PATIENT_ID AUDIT
# ================================================================

print("\n" + "=" * 90)
print("STEP 3 — PATIENT_ID AUDIT")
print("=" * 90)


missing_id_values = {
    "",
    "nan",
    "none",
    "null",
    "na",
    "n/a",
    "missing"
}


patient_id_as_string = (

    df_raw["Patient_ID"]
    .astype("string")
    .str.strip()

)


patient_id_missing = (

    df_raw["Patient_ID"].isna()

    |

    patient_id_as_string
    .str.lower()
    .isin(
        missing_id_values
    )

)


df_raw["_Patient_ID_missing"] = (
    patient_id_missing
)


print(
    f"Rows with missing/unusable Patient_ID: "
    f"{patient_id_missing.sum():,}"
)


print(
    f"Percentage missing/unusable: "
    f"{100 * patient_id_missing.mean():.2f}%"
)


# ================================================================
# 7. RETAIN ALL CLAIMS
# ================================================================
#
# IMPORTANT CHANGE FROM PREVIOUS VERSION:
#
# We DO NOT split the data into a permanent analysis dataset
# and discard the unidentified observations.
#
# ALL claim lines remain in df.
#
# Identified and unidentified records are handled differently
# ONLY during episode identification.
# ================================================================

print("\n" + "=" * 90)
print("STEP 4 — RETAINING ALL CLAIM LINES")
print("=" * 90)


df = df_raw.copy()


print(
    f"Total claim lines retained: "
    f"{len(df):,}"
)


print(
    f"Identifiable claim lines: "
    f"{(~patient_id_missing).sum():,}"
)


print(
    f"Anonymous/unidentified claim lines retained: "
    f"{patient_id_missing.sum():,}"
)


# Save unidentified records for audit,
# BUT DO NOT remove them from df.

df_unidentified = df.loc[
    patient_id_missing
].copy()


df_unidentified.to_csv(
    UNIDENTIFIED_FILE,
    index=False,
    encoding="utf-8-sig"
)


print(
    "\nUnidentified records saved for audit:"
)

print(
    UNIDENTIFIED_FILE
)


# ================================================================
# 8. CLEAN Patient_ID
# ================================================================

df["Patient_ID"] = (
    df["Patient_ID"]
    .astype("string")
    .str.strip()
)


# Explicitly restore missing IDs to <NA>

df.loc[
    patient_id_missing,
    "Patient_ID"
] = pd.NA


# ================================================================
# 9. NUMERIC VARIABLE CONVERSION
# ================================================================

print("\n" + "=" * 90)
print("STEP 5 — NUMERIC DATA CLEANING")
print("=" * 90)


numeric_columns = [

    "Age",
    "Length of Stay",
    "Total Cost",
    "Insured Person's Share",
    "Organization Requested Share",
    "Organization Paid Share",
    "Terminally Patients Paid Share",
    "Terminally Patients Requested Share",
    "Allocated subsidized foreign currency share",
    "Requested subsidized foreign currency share"
]


numeric_columns_present = [
    col
    for col in numeric_columns
    if col in df.columns
]


for col in numeric_columns_present:

    df[col] = (

        df[col]
        .astype("string")
        .str.replace(
            ",",
            "",
            regex=False
        )
        .str.strip()

    )

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )


print("\nNumeric variables converted:")

for col in numeric_columns_present:

    print(
        f"  {col}"
    )


# ================================================================
# 10. DATE CONVERSION
# ================================================================

print("\n" + "=" * 90)
print("STEP 6 — DATE CLEANING")
print("=" * 90)


date_columns = [
    "Admision date",
    "Discharge date"
]


for col in date_columns:

    if col in df.columns:

        df[col] = pd.to_datetime(
            df[col],
            errors="coerce"
        )


print(
    "\nDate variables converted."
)


# ================================================================
# 11. AGE QUALITY CONTROL
# ================================================================

print("\n" + "=" * 90)
print("STEP 7 — BASIC CLINICAL VALIDITY CHECKS")
print("=" * 90)


if "Age" in df.columns:

    invalid_age = (

        df["Age"].notna()

        &

        (
            (df["Age"] < 0)
            |
            (df["Age"] > 120)
        )

    )

    print(
        f"Invalid age observations: "
        f"{invalid_age.sum():,}"
    )


    # Do NOT impute.
    # Invalid ages become missing.

    df.loc[
        invalid_age,
        "Age"
    ] = np.nan


    df["_invalid_age"] = (
        invalid_age
    )

else:

    df["_invalid_age"] = False


# ================================================================
# 12. FINANCIAL DATA QUALITY CONTROL
# ================================================================

financial_columns = [

    "Total Cost",
    "Insured Person's Share",
    "Organization Requested Share",
    "Organization Paid Share",
    "Terminally Patients Paid Share",
    "Terminally Patients Requested Share",
    "Allocated subsidized foreign currency share",
    "Requested subsidized foreign currency share"

]


financial_columns_present = [
    col
    for col in financial_columns
    if col in df.columns
]


for col in financial_columns_present:

    negative_values = (

        df[col].notna()

        &

        (df[col] < 0)

    )

    print(
        f"{col}: "
        f"{negative_values.sum():,} negative values"
    )


    # Do not automatically change negative values.

    df[
        f"_negative_{col}"
    ] = negative_values


# ================================================================
# 13. DATE ORDER CHECK
# ================================================================

if (
    "Admision date" in df.columns
    and
    "Discharge date" in df.columns
):

    df["_invalid_date_order"] = (

        df["Admision date"].notna()

        &

        df["Discharge date"].notna()

        &

        (
            df["Discharge date"]
            <
            df["Admision date"]
        )

    )

else:

    df["_invalid_date_order"] = False


print(
    "\nClaim lines with discharge before admission:",
    df["_invalid_date_order"].sum()
)


# ================================================================
# 14. CALCULATED CLAIM-LEVEL LOS
# ================================================================

if (
    "Admision date" in df.columns
    and
    "Discharge date" in df.columns
):

    df["Calculated_LOS_days"] = (

        df["Discharge date"]
        -
        df["Admision date"]

    ).dt.total_seconds() / 86400

else:

    df["Calculated_LOS_days"] = np.nan


# ================================================================
# 15. COMPARE PROVIDED AND CALCULATED LOS
# ================================================================

if "Length of Stay" in df.columns:

    df["LOS_difference"] = (

        df["Length of Stay"]
        -
        df["Calculated_LOS_days"]

    )

    print("\nLength-of-stay comparison:")

    print(
        df[
            [
                "Length of Stay",
                "Calculated_LOS_days",
                "LOS_difference"
            ]
        ].describe()
    )


# ================================================================
# 16. EXACT DUPLICATE CLAIM-LINE AUDIT
# ================================================================

print("\n" + "=" * 90)
print("STEP 8 — DUPLICATE CLAIM-LINE AUDIT")
print("=" * 90)


# Exclude diagnostic variables

original_data_columns = [

    col

    for col in df.columns

    if not col.startswith("_")

    and col not in [
        "Calculated_LOS_days",
        "LOS_difference"
    ]

]


df["_exact_duplicate"] = (
    df.duplicated(
        subset=original_data_columns,
        keep=False
    )
)


print(
    "Claim lines involved in exact duplicate patterns:",
    df["_exact_duplicate"].sum()
)


# Do NOT delete duplicates automatically.


# ================================================================
# 17. IDENTIFIED EPISODE CONSTRUCTION
# ================================================================

print("\n" + "=" * 90)
print("STEP 9 — IDENTIFIED EPISODE CONSTRUCTION")
print("=" * 90)


episode_key_columns = [

    "Patient_ID",
    "Admision date",
    "Discharge date"

]


missing_episode_key_columns = [

    col
    for col in episode_key_columns
    if col not in df.columns

]


if missing_episode_key_columns:

    raise ValueError(
        "Cannot construct the identified episode key. "
        f"Missing columns: "
        f"{missing_episode_key_columns}"
    )


identified_mask = (
    df["Patient_ID"].notna()
)


identified_complete_key = (

    identified_mask

    &

    df["Admision date"].notna()

    &

    df["Discharge date"].notna()

)


print(
    "Identified claims with complete episode key:",
    identified_complete_key.sum()
)


print(
    "Identified claims with incomplete episode key:",
    (
        identified_mask
        &
        ~identified_complete_key
    ).sum()
)


# ================================================================
# 18. INITIALIZE Episode_ID
# ================================================================

df["Episode_ID"] = pd.NA

df["Episode_Identification_Type"] = pd.NA

df["Episode_Linkage_Method"] = pd.NA


# ================================================================
# 19. IDENTIFIED EPISODE IDs
# ================================================================

df.loc[
    identified_complete_key,
    "Episode_ID"
] = (

    "ID_"

    +

    df.loc[
        identified_complete_key,
        episode_key_columns
    ]
    .astype("string")
    .agg(
        "|".join,
        axis=1
    )

)


df.loc[
    identified_complete_key,
    "Episode_Identification_Type"
] = "IDENTIFIED"


df.loc[
    identified_complete_key,
    "Episode_Linkage_Method"
] = (
    "Patient_ID + Admission date + Discharge date"
)


# ================================================================
# 20. ANONYMOUS EPISODE CONSTRUCTION
# ================================================================
#
# The anonymous episode signature intentionally excludes:
#
#   Total Cost
#   Service Group
#   Length of Stay
#   Discharge Status
#   financial variables
#
# because these are not appropriate identifiers for a hospitalization.
#
# The signature uses the available clinical/demographic episode
# characteristics.
# ================================================================

print("\n" + "=" * 90)
print("STEP 10 — ANONYMOUS EPISODE CONSTRUCTION")
print("=" * 90)


anonymous_mask = (
    df["Patient_ID"].isna()
)


anonymous_key_columns = [

    "Admision date",
    "Discharge date",
    "Age",
    "Gender",
    "Ownership",
    "Insurance Fund",
    "Treating Physician Specialty",
    "Diagnosis type",
    "Department Type"

]


anonymous_key_columns_present = [

    col
    for col in anonymous_key_columns
    if col in df.columns

]


print(
    "\nAnonymous linkage variables:"
)

for col in anonymous_key_columns_present:

    print(
        f"  {col}"
    )


# ================================================================
# 21. DETERMINE WHETHER ANONYMOUS RECORD HAS ENOUGH INFORMATION
# ================================================================
#
# Admission and discharge dates are required for a genuine
# anonymous episode cluster.
#
# If dates are missing, we cannot safely determine whether
# the claim belongs to a hospitalization episode.
#
# Such observations are retained as singleton anonymous episodes.
# ================================================================

anonymous_dates_available = (

    anonymous_mask

    &

    df["Admision date"].notna()

    &

    df["Discharge date"].notna()

)


anonymous_clusterable = (
    anonymous_dates_available
)


anonymous_singleton = (
    anonymous_mask
    &
    ~anonymous_clusterable
)


print(
    "\nAnonymous claims with admission + discharge dates:",
    anonymous_clusterable.sum()
)


print(
    "Anonymous claims without complete dates:",
    anonymous_singleton.sum()
)


# ================================================================
# 22. CREATE SAFE TEXT REPRESENTATION FOR ANONYMOUS KEY
# ================================================================

def safe_key_value(series):

    return (

        series
        .astype("string")
        .fillna("<MISSING>")
        .str.strip()
        .replace(
            "",
            "<MISSING>"
        )

    )


# ================================================================
# 23. CREATE ANONYMOUS DETERMINISTIC SIGNATURE
# ================================================================

if anonymous_clusterable.any():

    anonymous_signature_parts = []

    for col in anonymous_key_columns_present:

        anonymous_signature_parts.append(
            safe_key_value(
                df.loc[
                    anonymous_clusterable,
                    col
                ]
            )
        )


    anonymous_signature = (
        pd.concat(
            anonymous_signature_parts,
            axis=1
        )
        .astype("string")
        .agg(
            "||".join,
            axis=1
        )
    )


    # Create a stable hash-like identifier from the signature.
    #
    # This is NOT a Patient_ID.
    # It is only an anonymous episode cluster identifier.

    anonymous_hash = pd.util.hash_pandas_object(
        anonymous_signature,
        index=False
    ).astype("string")


    df.loc[
        anonymous_clusterable,
        "Episode_ID"
    ] = (

        "ANON_"
        +
        anonymous_hash.values

    )


    df.loc[
        anonymous_clusterable,
        "Episode_Identification_Type"
    ] = "ANONYMOUS"


    df.loc[
        anonymous_clusterable,
        "Episode_Linkage_Method"
    ] = (
        "Anonymous deterministic cluster: "
        "Admission + discharge + baseline characteristics"
    )


# ================================================================
# 24. CREATE SINGLETON ANONYMOUS EPISODES
# ================================================================
#
# These records cannot safely be linked to another claim.
#
# We retain each one as an independent anonymous episode.
#
# IMPORTANT:
# This does NOT mean we claim that each record represents a unique
# patient. It means that the record cannot be safely linked to
# another record with the available identifiers.
# ================================================================

if anonymous_singleton.any():

    singleton_numbers = (
        df.loc[
            anonymous_singleton,
            "_original_row_number"
        ]
        .astype("string")
    )


    df.loc[
        anonymous_singleton,
        "Episode_ID"
    ] = (

        "ANON_SINGLE_"
        +
        singleton_numbers

    )


    df.loc[
        anonymous_singleton,
        "Episode_Identification_Type"
    ] = "ANONYMOUS_SINGLETON"


    df.loc[
        anonymous_singleton,
        "Episode_Linkage_Method"
    ] = (
        "No safe cross-claim linkage possible; "
        "retained as singleton anonymous episode"
    )


# ================================================================
# 25. IDENTIFIED CLAIMS WITH INCOMPLETE DATES
# ================================================================
#
# These have Patient_ID but cannot be assigned to the standard
# Patient_ID + date episode key.
#
# We do NOT delete them.
#
# They are retained as singleton identified records.
# ================================================================

identified_incomplete_key = (

    identified_mask

    &

    ~identified_complete_key

)


print(
    "\nIdentified claims with incomplete dates:",
    identified_incomplete_key.sum()
)


if identified_incomplete_key.any():

    singleton_numbers = (
        df.loc[
            identified_incomplete_key,
            "_original_row_number"
        ]
        .astype("string")
    )


    df.loc[
        identified_incomplete_key,
        "Episode_ID"
    ] = (

        "ID_SINGLE_"
        +
        singleton_numbers

    )


    df.loc[
        identified_incomplete_key,
        "Episode_Identification_Type"
    ] = (
        "IDENTIFIED_SINGLETON"
    )


    df.loc[
        identified_incomplete_key,
        "Episode_Linkage_Method"
    ] = (
        "Patient_ID available but admission/discharge "
        "dates incomplete; retained as singleton"
    )


# ================================================================
# 26. FINAL EPISODE-ID COMPLETENESS CHECK
# ================================================================

episode_id_missing = (
    df["Episode_ID"].isna()
)


print(
    "\nClaims still without Episode_ID:",
    episode_id_missing.sum()
)


if episode_id_missing.any():

    raise ValueError(
        "Some claim lines could not be assigned an Episode_ID. "
        "No observations should be silently lost."
    )


print(
    "All claim lines now have an Episode_ID."
)


# ================================================================
# 27. ANONYMOUS LINKAGE AUDIT
# ================================================================

anonymous_linkage_audit = (

    df.loc[
        anonymous_mask
    ]
    .groupby(
        [
            "Episode_ID",
            "Episode_Identification_Type",
            "Episode_Linkage_Method"
        ],
        dropna=False
    )
    .agg(
        Number_of_claim_lines=(
            "_original_row_number",
            "count"
        ),

        Min_original_row=(
            "_original_row_number",
            "min"
        ),

        Max_original_row=(
            "_original_row_number",
            "max"
        )
    )
    .reset_index()

)


anonymous_linkage_audit.to_csv(
    ANONYMOUS_EPISODE_FILE,
    index=False,
    encoding="utf-8-sig"
)


print(
    "\nAnonymous episode linkage audit saved to:"
)

print(
    ANONYMOUS_EPISODE_FILE
)


print(
    "\nAnonymous episode counts:"
)

print(
    df.loc[
        anonymous_mask,
        "Episode_Identification_Type"
    ]
    .value_counts(
        dropna=False
    )
)


# ================================================================
# 28. EPISODE KEY AUDIT
# ================================================================

print("\n" + "=" * 90)
print("STEP 11 — EPISODE KEY AUDIT")
print("=" * 90)


episode_key_audit = (

    df
    .groupby(
        [
            "Episode_ID",
            "Episode_Identification_Type"
        ],
        dropna=False
    )
    .size()
    .reset_index(
        name="Number_of_claim_lines"
    )

)


print(
    "\nTotal constructed episode units:",
    len(episode_key_audit)
)


print(
    "\nEpisode types:"
)

print(
    episode_key_audit[
        "Episode_Identification_Type"
    ]
    .value_counts(
        dropna=False
    )
)


print(
    "\nMaximum claim lines within one episode:",
    episode_key_audit[
        "Number_of_claim_lines"
    ].max()
)


print(
    "Episodes containing >1 claim line:",
    (
        episode_key_audit[
            "Number_of_claim_lines"
        ] > 1
    ).sum()
)


# ================================================================
# 29. PATIENT-LEVEL MULTIPLE EPISODES
# ================================================================

identified_episode_rows = df.loc[
    df["Patient_ID"].notna()
]


episodes_per_patient = (

    identified_episode_rows
    .groupby(
        "Patient_ID"
    )["Episode_ID"]
    .nunique()

)


if len(episodes_per_patient) > 0:

    print(
        "\nPatients with multiple hospitalization episodes:",
        (
            episodes_per_patient > 1
        ).sum()
    )

    print(
        "Maximum episodes per patient:",
        episodes_per_patient.max()
    )


# ================================================================
# 30. EPISODE-LEVEL AGGREGATION
# ================================================================

print("\n" + "=" * 90)
print("STEP 12 — ALL CLAIMS -> EPISODE AGGREGATION")
print("=" * 90)


# ALL claim lines are now eligible for aggregation.

df_valid = df.copy()


# ------------------------------------------------
# Financial variables = SUM
# ------------------------------------------------

sum_columns = [

    "Total Cost",
    "Insured Person's Share",
    "Organization Requested Share",
    "Organization Paid Share",
    "Terminally Patients Paid Share",
    "Terminally Patients Requested Share",
    "Allocated subsidized foreign currency share",
    "Requested subsidized foreign currency share"

]


sum_columns = [

    col
    for col in sum_columns
    if col in df_valid.columns

]


# ------------------------------------------------
# Episode-level characteristics
# ------------------------------------------------

first_columns = [

    "Patient_ID",
    "Age",
    "Gender",
    "Ownership",
    "Insurance Fund",
    "Treating Physician Specialty",
    "Admision date",
    "Discharge date",
    "Discharge Status",
    "Diagnosis type",
    "Department Type",
    "Episode_Identification_Type",
    "Episode_Linkage_Method"

]


first_columns = [

    col
    for col in first_columns
    if col in df_valid.columns

]


# ------------------------------------------------
# Aggregation dictionary
# ------------------------------------------------

aggregation_dictionary = {}


# Financial sums

for col in sum_columns:

    aggregation_dictionary[col] = "sum"


# Episode-level characteristics

for col in first_columns:

    aggregation_dictionary[col] = "first"


# Number of claim lines

aggregation_dictionary[
    "_original_row_number"
] = "count"


# Number of unique service groups

if "Service Group" in df_valid.columns:

    aggregation_dictionary[
        "Service Group"
    ] = "nunique"


# ================================================================
# 31. PERFORM AGGREGATION
# ================================================================

episodes = (

    df_valid
    .groupby(
        "Episode_ID",
        as_index=False
    )
    .agg(
        aggregation_dictionary
    )

)


# ================================================================
# 32. RENAME AGGREGATED VARIABLES
# ================================================================

episodes = episodes.rename(

    columns={

        "_original_row_number":
            "Number_of_claim_lines",

        "Service Group":
            "Number_of_service_groups"

    }

)


# ================================================================
# 33. EPISODE LOS
# ================================================================

episodes["Episode_LOS_days"] = (

    episodes["Discharge date"]
    -
    episodes["Admision date"]

).dt.total_seconds() / 86400


# IMPORTANT:
#
# We do NOT sum Length of Stay.
#
# Episode LOS is calculated from:
#
# discharge date - admission date
#
# When dates are unavailable, Episode_LOS_days remains missing.


# ================================================================
# 34. EPISODE-LEVEL DATE QUALITY FLAG
# ================================================================

episode_date_quality = (

    df_valid
    .groupby("Episode_ID")
    .agg(

        invalid_date_order=(
            "_invalid_date_order",
            "max"
        )

    )
    .reset_index()

)


episodes = episodes.merge(

    episode_date_quality,
    on="Episode_ID",
    how="left"

)


episodes[
    "invalid_date_order"
] = (

    episodes[
        "invalid_date_order"
    ]
    .fillna(False)
    .astype(bool)

)


# ================================================================
# 35. EPISODE-LEVEL DUPLICATE FLAG
# ================================================================

episode_duplicate_quality = (

    df_valid
    .groupby("Episode_ID")
    .agg(

        contains_exact_duplicate_claim=(
            "_exact_duplicate",
            "max"
        )

    )
    .reset_index()

)


episodes = episodes.merge(

    episode_duplicate_quality,
    on="Episode_ID",
    how="left"

)


episodes[
    "contains_exact_duplicate_claim"
] = (

    episodes[
        "contains_exact_duplicate_claim"
    ]
    .fillna(False)
    .astype(bool)

)


# ================================================================
# 36. EPISODE-LEVEL AGE QUALITY FLAG
# ================================================================

episode_age_quality = (

    df_valid
    .groupby("Episode_ID")
    .agg(

        contains_invalid_age=(
            "_invalid_age",
            "max"
        )

    )
    .reset_index()

)


episodes = episodes.merge(

    episode_age_quality,
    on="Episode_ID",
    how="left"

)


episodes[
    "contains_invalid_age"
] = (

    episodes[
        "contains_invalid_age"
    ]
    .fillna(False)
    .astype(bool)

)


# ================================================================
# 37. FINANCIAL CONSISTENCY CHECK
# ================================================================

print("\n" + "=" * 90)
print("STEP 13 — EPISODE FINANCIAL CONSISTENCY")
print("=" * 90)


required_financial = [

    "Total Cost",
    "Insured Person's Share",
    "Organization Paid Share"

]


if all(
    col in episodes.columns
    for col in required_financial
):

    episodes[
        "Known_financing_total"
    ] = (

        episodes[
            "Insured Person's Share"
        ]

        +

        episodes[
            "Organization Paid Share"
        ]

    )


    episodes[
        "Financing_difference"
    ] = (

        episodes["Total Cost"]
        -
        episodes["Known_financing_total"]

    )


    episodes[
        "Financing_difference_pct"
    ] = np.where(

        episodes["Total Cost"].abs() > 0,

        100
        *
        episodes["Financing_difference"]
        /
        episodes["Total Cost"].abs(),

        np.nan

    )


    print(

        episodes[
            [
                "Total Cost",
                "Known_financing_total",
                "Financing_difference",
                "Financing_difference_pct"
            ]
        ].describe()

    )


# ================================================================
# 38. WITHIN-EPISODE CONSISTENCY CHECKS
# ================================================================

print("\n" + "=" * 90)
print("STEP 14 — WITHIN-EPISODE CONSISTENCY CHECKS")
print("=" * 90)


inconsistency_records = []


# ------------------------------------------------
# Discharge Status
# ------------------------------------------------

if "Discharge Status" in df_valid.columns:

    status_variability = (

        df_valid
        .groupby(
            "Episode_ID"
        )[
            "Discharge Status"
        ]
        .nunique(
            dropna=True
        )

    )


    multiple_status_episodes = (
        status_variability > 1
    )


    print(
        "Episodes with multiple discharge statuses:",
        multiple_status_episodes.sum()
    )


    for episode_id in status_variability[
        multiple_status_episodes
    ].index:

        values = (

            df_valid.loc[
                df_valid["Episode_ID"] == episode_id,
                "Discharge Status"
            ]
            .dropna()
            .astype(str)
            .unique()
            .tolist()

        )


        inconsistency_records.append({

            "Episode_ID":
                episode_id,

            "Variable":
                "Discharge Status",

            "Values":
                " | ".join(values)

        })


else:

    status_variability = pd.Series(
        dtype="int64"
    )


# ------------------------------------------------
# Patient characteristics
# ------------------------------------------------

patient_level_columns = [

    "Age",
    "Gender",
    "Ownership",
    "Insurance Fund"

]


for col in patient_level_columns:

    if col not in df_valid.columns:
        continue


    variability = (

        df_valid
        .groupby(
            "Episode_ID"
        )[col]
        .nunique(
            dropna=True
        )

    )


    inconsistent = (
        variability > 1
    )


    print(
        f"Episodes with multiple {col} values:",
        inconsistent.sum()
    )


    for episode_id in variability[
        inconsistent
    ].index:

        values = (

            df_valid.loc[
                df_valid["Episode_ID"] == episode_id,
                col
            ]
            .dropna()
            .astype(str)
            .unique()
            .tolist()

        )


        inconsistency_records.append({

            "Episode_ID":
                episode_id,

            "Variable":
                col,

            "Values":
                " | ".join(values)

        })


# ================================================================
# 39. SAVE INCONSISTENCIES
# ================================================================

inconsistency_df = pd.DataFrame(
    inconsistency_records
)


inconsistency_df.to_csv(

    INCONSISTENCY_FILE,

    index=False,

    encoding="utf-8-sig"

)


# ================================================================
# 40. CREATE EPISODE QUALITY CONTROL TABLE
# ================================================================

print("\n" + "=" * 90)
print("STEP 15 — EPISODE QUALITY CONTROL")
print("=" * 90)


anonymous_episode_count = (

    episodes[
        "Episode_Identification_Type"
    ]
    .isin(
        [
            "ANONYMOUS",
            "ANONYMOUS_SINGLETON"
        ]
    )
    .sum()

)


identified_episode_count = (

    episodes[
        "Episode_Identification_Type"
    ]
    .isin(
        [
            "IDENTIFIED",
            "IDENTIFIED_SINGLETON"
        ]
    )
    .sum()

)


anonymous_cluster_count = (

    episodes[
        "Episode_Identification_Type"
    ]
    .eq(
        "ANONYMOUS"
    )
    .sum()

)


anonymous_singleton_count = (

    episodes[
        "Episode_Identification_Type"
    ]
    .eq(
        "ANONYMOUS_SINGLETON"
    )
    .sum()

)


identified_singleton_count = (

    episodes[
        "Episode_Identification_Type"
    ]
    .eq(
        "IDENTIFIED_SINGLETON"
    )
    .sum()

)


episode_qc = pd.DataFrame({

    "Metric": [

        "Raw claim lines",

        "Claims with missing Patient_ID",

        "Identifiable claim lines",

        "Anonymous claim lines retained",

        "Constructed episode units",

        "Identified episodes",

        "Anonymous clustered episodes",

        "Anonymous singleton episodes",

        "Identified singleton episodes",

        "Claims without Episode_ID",

        "Episodes with invalid date order",

        "Episodes containing exact duplicate claim lines",

        "Episodes containing invalid age",

        "Episodes with multiple discharge statuses"

    ],

    "Value": [

        len(df_raw),

        patient_id_missing.sum(),

        (~patient_id_missing).sum(),

        patient_id_missing.sum(),

        len(episodes),

        identified_episode_count,

        anonymous_cluster_count,

        anonymous_singleton_count,

        identified_singleton_count,

        df["Episode_ID"].isna().sum(),

        episodes[
            "invalid_date_order"
        ].sum(),

        episodes[
            "contains_exact_duplicate_claim"
        ].sum(),

        episodes[
            "contains_invalid_age"
        ].sum(),

        (
            status_variability > 1
        ).sum()

    ]

})


episode_qc.to_csv(

    EPISODE_QC_FILE,

    index=False,

    encoding="utf-8-sig"

)


print(
    episode_qc.to_string(
        index=False
    )
)


# ================================================================
# 41. LEAKAGE-SAFE VARIABLE CLASSIFICATION
# ================================================================

print("\n" + "=" * 90)
print("STEP 16 — LEAKAGE AUDIT")
print("=" * 90)


# ------------------------------------------------
# Identifier variables
# ------------------------------------------------

identifier_variables = [

    "Patient_ID",
    "Episode_ID",
    "Episode_Identification_Type",
    "Episode_Linkage_Method"

]


# ------------------------------------------------
# Outcomes / post-treatment variables
# ------------------------------------------------

outcome_or_post_treatment_variables = [

    "Discharge Status",
    "Discharge date",
    "Length of Stay",
    "Episode_LOS_days",

    "Total Cost",

    "Insured Person's Share",
    "Organization Requested Share",
    "Organization Paid Share",

    "Terminally Patients Paid Share",
    "Terminally Patients Requested Share",

    "Allocated subsidized foreign currency share",
    "Requested subsidized foreign currency share",

    "Known_financing_total",
    "Financing_difference",
    "Financing_difference_pct"

]


# ------------------------------------------------
# Candidate baseline variables
# ------------------------------------------------

candidate_baseline_variables = [

    "Age",
    "Gender",
    "Ownership",
    "Insurance Fund",
    "Treating Physician Specialty",
    "Diagnosis type",
    "Department Type",
    "Admision date"

]


leakage_classification = []


for col in episodes.columns:

    if col in identifier_variables:

        classification = (
            "IDENTIFIER / LINKAGE VARIABLE — "
            "NEVER USE AS ML PREDICTOR"
        )


    elif col in outcome_or_post_treatment_variables:

        classification = (

            "OUTCOME/POST-TREATMENT — "
            "DO NOT USE AS PREDICTOR "
            "IF TARGET DEPENDS ON IT"

        )


    elif col in candidate_baseline_variables:

        classification = (

            "CANDIDATE BASELINE PREDICTOR — "
            "MUST VERIFY AVAILABLE AT "
            "PREDICTION TIME"

        )


    elif col.startswith("invalid_"):

        classification = (

            "QUALITY CONTROL VARIABLE — "
            "NOT A PREDICTOR"

        )


    elif col.startswith("contains_"):

        classification = (

            "QUALITY CONTROL VARIABLE — "
            "NOT A PREDICTOR"

        )


    elif col.startswith("Number_of_"):

        classification = (

            "POST-ADMISSION EPISODE DESCRIPTOR — "
            "POTENTIAL LEAKAGE"

        )


    elif col.startswith("Financing_"):

        classification = (

            "DERIVED FINANCIAL VARIABLE — "
            "POTENTIAL TARGET LEAKAGE"

        )


    else:

        classification = (

            "UNCLASSIFIED — "
            "MANUAL REVIEW REQUIRED"

        )


    leakage_classification.append({

        "Variable":
            col,

        "Classification":
            classification

    })


leakage_df = pd.DataFrame(
    leakage_classification
)


leakage_df.to_csv(

    LEAKAGE_FILE,

    index=False,

    encoding="utf-8-sig"

)


print(
    "\nLeakage classification:"
)


print(
    leakage_df.to_string(
        index=False
    )
)


# ================================================================
# 42. SAVE CLEAN CLAIM-LEVEL DATA
# ================================================================
#
# IMPORTANT:
# This file contains ALL original claim lines.
#
# Nothing was removed merely because Patient_ID was missing.
# ================================================================

df.to_csv(

    CLEAN_CLAIMS_FILE,

    index=False,

    encoding="utf-8-sig"

)


# ================================================================
# 43. SAVE FINAL EPISODE-LEVEL DATA
# ================================================================

episodes.to_csv(

    EPISODE_FILE,

    index=False,

    encoding="utf-8-sig"

)


# ================================================================
# 44. FINAL SUMMARY
# ================================================================

print("\n" + "=" * 90)
print("FINAL DATA PREPROCESSING SUMMARY")
print("=" * 90)


print(
    f"Raw claim lines: "
    f"{len(df_raw):,}"
)


print(
    f"Claims with missing Patient_ID: "
    f"{patient_id_missing.sum():,}"
)


print(
    f"Identifiable claim lines: "
    f"{(~patient_id_missing).sum():,}"
)


print(
    f"Anonymous claim lines retained: "
    f"{patient_id_missing.sum():,}"
)


print(
    f"Total constructed episode units: "
    f"{len(episodes):,}"
)


print(
    f"Identified episodes: "
    f"{identified_episode_count:,}"
)


print(
    f"Anonymous clustered episodes: "
    f"{anonymous_cluster_count:,}"
)


print(
    f"Anonymous singleton episodes: "
    f"{anonymous_singleton_count:,}"
)


print(
    f"Identified singleton episodes: "
    f"{identified_singleton_count:,}"
)


print(
    f"Claims without Episode_ID: "
    f"{df['Episode_ID'].isna().sum():,}"
)


print(
    f"Episodes with invalid date order: "
    f"{episodes['invalid_date_order'].sum():,}"
)


print(
    f"Episodes containing exact duplicate claims: "
    f"{episodes['contains_exact_duplicate_claim'].sum():,}"
)


print(
    f"Episodes containing invalid age: "
    f"{episodes['contains_invalid_age'].sum():,}"
)


if "Discharge Status" in df_valid.columns:

    print(
        f"Episodes with multiple discharge statuses: "
        f"{(status_variability > 1).sum():,}"
    )


print(
    "\nEpisode dataset dimensions:"
)


print(
    episodes.shape
)


print(
    "\nEpisode-level variables:"
)


for i, col in enumerate(
    episodes.columns,
    start=1
):

    print(
        f"{i:02d}. {col}"
    )


# ================================================================
# 45. FINAL CLAIM-RETENTION CHECK
# ================================================================
#
# This is a critical audit.
#
# Number of claim lines before episode construction must equal
# number of claim lines represented by all Episode_ID groups.
# ================================================================

claim_count_before = len(df)

claim_count_after_episode_assignment = (

    df
    .groupby(
        "Episode_ID"
    )
    .size()
    .sum()

)


print("\n" + "=" * 90)
print("FINAL CLAIM RETENTION AUDIT")
print("=" * 90)


print(
    f"Claim lines before episode construction: "
    f"{claim_count_before:,}"
)


print(
    f"Claim lines represented in episode dataset: "
    f"{claim_count_after_episode_assignment:,}"
)


print(
    f"Difference: "
    f"{claim_count_before - claim_count_after_episode_assignment:,}"
)


if (
    claim_count_before
    !=
    claim_count_after_episode_assignment
):

    raise ValueError(
        "CRITICAL ERROR: Some claim lines were lost "
        "during episode construction."
    )


else:

    print(
        "\nPASS — ALL CLAIM LINES ARE REPRESENTED "
        "IN THE FINAL EPISODE STRUCTURE."
    )


# ================================================================
# 46. FILES CREATED
# ================================================================

print("\n" + "=" * 90)
print("FILES CREATED")
print("=" * 90)


print(
    "1. Raw audit:\n",
    RAW_AUDIT_FILE
)


print(
    "2. Unidentified records audit:\n",
    UNIDENTIFIED_FILE
)


print(
    "3. Complete clean claim-level data:\n",
    CLEAN_CLAIMS_FILE
)


print(
    "4. FINAL episode-level data:\n",
    EPISODE_FILE
)


print(
    "5. Episode QC:\n",
    EPISODE_QC_FILE
)


print(
    "6. Episode inconsistencies:\n",
    INCONSISTENCY_FILE
)


print(
    "7. Leakage classification:\n",
    LEAKAGE_FILE
)


print(
    "8. Anonymous episode linkage audit:\n",
    ANONYMOUS_EPISODE_FILE
)


print("\nDONE.")

In [ ]:
# ==============================================================================
# CHOLERA CLAIM DATA
# ANONYMOUS CLAIM-BLOCK VALIDATION
# ==============================================================================
#
# PURPOSE
# -------
# Validate the 661 claims without Patient_ID before freezing the episode dataset.
#
# PRIMARY QUESTION
# ----------------
# Are the anonymous claims genuinely 67 hospitalization blocks, or did the
# deterministic linkage accidentally merge multiple hospitalizations?
#
# CENTRAL DIAGNOSTIC
# ------------------
# Original row contiguity:
#
#     If an anonymous episode contains N claim lines and:
#
#         Max_original_row - Min_original_row + 1 == N
#
#     then the claims occupy one completely contiguous block in the original
#     dataset.
#
# ADDITIONAL DIAGNOSTICS
# ----------------------
# 1. Contiguity
# 2. Internal consistency of baseline characteristics
# 3. Date consistency
# 4. Length-of-stay consistency
# 5. Service-group structure
# 6. Financial continuity
# 7. Adjacent anonymous-block analysis
# 8. Potential over-linkage flags
#
# IMPORTANT
# ---------
# This script DOES NOT:
#   - remove any claim
#   - split any episode
#   - create a new Patient_ID
#   - change the existing episode dataset
#
# It only creates validation/audit files.
# ==============================================================================


import pandas as pd
import numpy as np
import re
from pathlib import Path


# ==============================================================================
# 1. FILE PATH
# ==============================================================================

file_path = (
    r"C:\Users\Shekoofeh\Documents\Health Insurance Data"
    r"\Cholera\Cholera_csv_translated.csv"
)

OUTPUT_DIR = Path(file_path).parent

AUDIT_FILE = (
    OUTPUT_DIR /
    "08_anonymous_episode_linkage_audit.csv"
)

CLAIM_AUDIT_FILE = (
    OUTPUT_DIR /
    "08b_anonymous_claim_level_validation.csv"
)

SUSPICIOUS_FILE = (
    OUTPUT_DIR /
    "08c_suspicious_anonymous_episode_blocks.csv"
)

TRANSITION_FILE = (
    OUTPUT_DIR /
    "08d_anonymous_block_transitions.csv"
)

SUMMARY_FILE = (
    OUTPUT_DIR /
    "08e_anonymous_linkage_validation_summary.csv"
)


# ==============================================================================
# 2. LOAD RAW DATA
# ==============================================================================

print("=" * 90)
print("STEP 1 — LOADING RAW CLAIM DATA")
print("=" * 90)

df = pd.read_csv(
    file_path,
    encoding="utf-8-sig",
    low_memory=False
)

print("Raw rows:", len(df))
print("Raw columns:", len(df.columns))


# ==============================================================================
# 3. STANDARDIZE COLUMN NAMES
# ==============================================================================

def clean_column_name(column):

    column = str(column)

    column = column.replace('"', "")
    column = column.replace("\n", " ")
    column = column.replace("\t", " ")

    column = re.sub(r"\s+", " ", column)

    return column.strip()


df.columns = [
    clean_column_name(col)
    for col in df.columns
]


print("\nStandardized columns:")
for i, col in enumerate(df.columns, start=1):
    print(
        "{:02d}. {}".format(i, col)
    )


# ==============================================================================
# 4. PRESERVE ORIGINAL ROW NUMBER
# ==============================================================================

df["_original_row_number"] = np.arange(
    len(df)
)


# ==============================================================================
# 5. CHECK REQUIRED VARIABLES
# ==============================================================================

required_columns = [
    "Patient_ID",
    "Diagnosis type",
    "Department Type",
    "Discharge Status",
    "Age",
    "Gender",
    "Ownership",
    "Insurance Fund",
    "Treating Physician Specialty",
    "Admision date",
    "Discharge date",
    "Length of Stay",
    "Service Group",
    "Total Cost"
]

missing_required = [
    col
    for col in required_columns
    if col not in df.columns
]

if missing_required:

    raise ValueError(
        "The following required columns are missing:\n"
        + "\n".join(missing_required)
    )


# ==============================================================================
# 6. IDENTIFY ANONYMOUS CLAIMS
# ==============================================================================

print("\n" + "=" * 90)
print("STEP 2 — IDENTIFYING ANONYMOUS CLAIMS")
print("=" * 90)


patient_id_string = (
    df["Patient_ID"]
    .astype("string")
    .str.strip()
)


missing_id_values = {
    "",
    "nan",
    "none",
    "null",
    "na",
    "n/a",
    "missing"
}


anonymous_mask = (
    df["Patient_ID"].isna()
    |
    patient_id_string.str.lower().isin(
        missing_id_values
    )
)


df["Is_Anonymous"] = anonymous_mask


anonymous = df.loc[
    anonymous_mask
].copy()


identified = df.loc[
    ~anonymous_mask
].copy()


print(
    "Total claim lines:",
    len(df)
)

print(
    "Anonymous claim lines:",
    len(anonymous)
)

print(
    "Identified claim lines:",
    len(identified)
)


# ==============================================================================
# 7. DATE CONVERSION
# ==============================================================================

print("\n" + "=" * 90)
print("STEP 3 — DATE VALIDATION")
print("=" * 90)


for col in [
    "Admision date",
    "Discharge date"
]:

    anonymous[col] = pd.to_datetime(
        anonymous[col],
        errors="coerce"
    )


# ==============================================================================
# 8. RECREATE THE ANONYMOUS LINKAGE KEY
# ==============================================================================

print("\n" + "=" * 90)
print("STEP 4 — RECONSTRUCTING ANONYMOUS LINKAGE KEY")
print("=" * 90)


# These are the variables described by the previous linkage audit:
#
# Admission + discharge + baseline characteristics
#
# They are NOT used to alter the existing episode dataset.
# They are reconstructed here solely for validation.

anonymous_linkage_columns = [
    "Admision date",
    "Discharge date",
    "Age",
    "Gender",
    "Ownership",
    "Insurance Fund",
    "Treating Physician Specialty",
    "Diagnosis type",
    "Department Type"
]


missing_linkage_columns = [
    col
    for col in anonymous_linkage_columns
    if col not in anonymous.columns
]

if missing_linkage_columns:

    raise ValueError(
        "Missing anonymous linkage columns:\n"
        + "\n".join(missing_linkage_columns)
    )


# Create a deterministic linkage key.
#
# pandas groupby handles NaN, but for explicit diagnostics we convert missing
# values to a visible token.

anonymous_key_df = anonymous[
    anonymous_linkage_columns
].copy()


for col in anonymous_key_df.columns:

    anonymous_key_df[col] = (
        anonymous_key_df[col]
        .astype("string")
        .fillna("<MISSING>")
        .str.strip()
    )


anonymous["Anonymous_Linkage_Key"] = (
    anonymous_key_df
    .agg("|".join, axis=1)
)


# ==============================================================================
# 9. CREATE CONTIGUOUS ORIGINAL-ROW BLOCKS
# ==============================================================================

print("\n" + "=" * 90)
print("STEP 5 — ORIGINAL ROW CONTIGUITY ANALYSIS")
print("=" * 90)


# Sort anonymous claims by original position.
anonymous = anonymous.sort_values(
    "_original_row_number"
).reset_index(drop=True)


# A new anonymous contiguous block starts whenever there is a gap > 1
# between consecutive anonymous rows.

anonymous["Previous_Original_Row"] = (
    anonymous["_original_row_number"]
    .shift(1)
)


anonymous["Row_Gap"] = (
    anonymous["_original_row_number"]
    -
    anonymous["Previous_Original_Row"]
)


anonymous["New_Contiguous_Block"] = (
    anonymous["Row_Gap"].isna()
    |
    (anonymous["Row_Gap"] != 1)
)


anonymous["Contiguous_Block_Number"] = (
    anonymous["New_Contiguous_Block"]
    .cumsum()
)


# Number of purely contiguous anonymous blocks
n_contiguous_blocks = (
    anonymous["Contiguous_Block_Number"]
    .nunique()
)


print(
    "Number of anonymous contiguous blocks:",
    n_contiguous_blocks
)


# ==============================================================================
# 10. CONTIGUOUS BLOCK SUMMARY
# ==============================================================================

contiguous_summary = (
    anonymous
    .groupby(
        "Contiguous_Block_Number"
    )
    .agg(
        Min_original_row=(
            "_original_row_number",
            "min"
        ),
        Max_original_row=(
            "_original_row_number",
            "max"
        ),
        Number_of_claim_lines=(
            "_original_row_number",
            "size"
        ),
        Number_of_linkage_keys=(
            "Anonymous_Linkage_Key",
            "nunique"
        )
    )
    .reset_index()
)


contiguous_summary["Expected_span"] = (
    contiguous_summary["Max_original_row"]
    -
    contiguous_summary["Min_original_row"]
    +
    1
)


contiguous_summary["Span_equals_claim_count"] = (
    contiguous_summary["Expected_span"]
    ==
    contiguous_summary["Number_of_claim_lines"]
)


print("\nContiguous-block diagnostics:")

print(
    contiguous_summary[
        [
            "Number_of_claim_lines",
            "Expected_span",
            "Span_equals_claim_count",
            "Number_of_linkage_keys"
        ]
    ]
    .describe(include="all")
)


# ==============================================================================
# 11. LINKAGE-KEY BLOCK SUMMARY
# ==============================================================================

print("\n" + "=" * 90)
print("STEP 6 — DETERMINISTIC LINKAGE-BLOCK VALIDATION")
print("=" * 90)


linkage_summary = (
    anonymous
    .groupby(
        "Anonymous_Linkage_Key",
        dropna=False
    )
    .agg(
        Min_original_row=(
            "_original_row_number",
            "min"
        ),
        Max_original_row=(
            "_original_row_number",
            "max"
        ),
        Number_of_claim_lines=(
            "_original_row_number",
            "size"
        ),
        Number_of_contiguous_blocks=(
            "Contiguous_Block_Number",
            "nunique"
        ),
        Number_of_service_groups=(
            "Service Group",
            "nunique"
        ),
        Number_of_discharge_statuses=(
            "Discharge Status",
            "nunique"
        ),
        Number_of_admission_dates=(
            "Admision date",
            "nunique"
        ),
        Number_of_discharge_dates=(
            "Discharge date",
            "nunique"
        ),
        Minimum_LOS=(
            "Length of Stay",
            "min"
        ),
        Maximum_LOS=(
            "Length of Stay",
            "max"
        ),
        Mean_LOS=(
            "Length of Stay",
            "mean"
        ),
        Total_Cost=(
            "Total Cost",
            "sum"
        )
    )
    .reset_index()
)


linkage_summary["Expected_span"] = (
    linkage_summary["Max_original_row"]
    -
    linkage_summary["Min_original_row"]
    +
    1
)


linkage_summary["Claims_equal_span"] = (
    linkage_summary["Expected_span"]
    ==
    linkage_summary["Number_of_claim_lines"]
)


linkage_summary["Perfectly_contiguous"] = (
    linkage_summary["Number_of_contiguous_blocks"]
    ==
    1
)


# ==============================================================================
# 12. DETERMINE WHETHER LINKAGE BLOCKS ARE CONTIGUOUS
# ==============================================================================

print("\nContiguity results:")

print(
    "Anonymous linkage blocks:",
    len(linkage_summary)
)

print(
    "Perfectly contiguous linkage blocks:",
    linkage_summary[
        "Perfectly_contiguous"
    ].sum()
)

print(
    "Non-contiguous linkage blocks:",
    (
        ~linkage_summary[
            "Perfectly_contiguous"
        ]
    ).sum()
)

print(
    "Blocks where row span equals claim count:",
    linkage_summary[
        "Claims_equal_span"
    ].sum()
)

print(
    "Blocks where row span does NOT equal claim count:",
    (
        ~linkage_summary[
            "Claims_equal_span"
        ]
    ).sum()
)


# ==============================================================================
# 13. INTERNAL CHARACTERISTIC CONSISTENCY
# ==============================================================================

print("\n" + "=" * 90)
print("STEP 7 — WITHIN-BLOCK CHARACTERISTIC CONSISTENCY")
print("=" * 90)


consistency_columns = [
    "Age",
    "Gender",
    "Ownership",
    "Insurance Fund",
    "Treating Physician Specialty",
    "Diagnosis type",
    "Department Type",
    "Admision date",
    "Discharge date"
]


for col in consistency_columns:

    variability = (
        anonymous
        .groupby(
            "Anonymous_Linkage_Key",
            dropna=False
        )[col]
        .nunique(dropna=True)
    )

    n_inconsistent = (
        variability > 1
    ).sum()

    print(
        "{:<40} {:>6}".format(
            col,
            n_inconsistent
        )
    )


# ==============================================================================
# 14. DISCHARGE STATUS ANALYSIS
# ==============================================================================

print("\n" + "=" * 90)
print("STEP 8 — DISCHARGE STATUS ANALYSIS")
print("=" * 90)


status_counts = (
    anonymous
    .groupby(
        "Anonymous_Linkage_Key",
        dropna=False
    )["Discharge Status"]
    .nunique(dropna=True)
)


multiple_status_blocks = (
    status_counts > 1
)


print(
    "Anonymous blocks with >1 discharge status:",
    multiple_status_blocks.sum()
)


if multiple_status_blocks.sum() > 0:

    status_examples = []

    for key in status_counts[
        multiple_status_blocks
    ].index:

        values = (
            anonymous.loc[
                anonymous[
                    "Anonymous_Linkage_Key"
                ] == key,
                "Discharge Status"
            ]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )

        status_examples.append({
            "Anonymous_Linkage_Key": key,
            "Discharge_Statuses": " | ".join(values),
            "Number_of_Statuses": len(values)
        })

    status_examples_df = pd.DataFrame(
        status_examples
    )

    print("\nBlocks with multiple discharge statuses:")

    print(
        status_examples_df.to_string(
            index=False
        )
    )


# ==============================================================================
# 15. LENGTH-OF-STAY ANALYSIS
# ==============================================================================

print("\n" + "=" * 90)
print("STEP 9 — LENGTH-OF-STAY VALIDATION")
print("=" * 90)


anonymous["Calculated_LOS"] = (
    anonymous["Discharge date"]
    -
    anonymous["Admision date"]
).dt.total_seconds() / 86400


anonymous["LOS_difference"] = (
    anonymous["Length of Stay"]
    -
    anonymous["Calculated_LOS"]
)


los_summary = (
    anonymous
    .groupby(
        "Anonymous_Linkage_Key",
        dropna=False
    )
    .agg(
        Minimum_reported_LOS=(
            "Length of Stay",
            "min"
        ),
        Maximum_reported_LOS=(
            "Length of Stay",
            "max"
        ),
        Minimum_calculated_LOS=(
            "Calculated_LOS",
            "min"
        ),
        Maximum_calculated_LOS=(
            "Calculated_LOS",
            "max"
        ),
        Maximum_abs_LOS_difference=(
            "LOS_difference",
            lambda x: np.nanmax(
                np.abs(x)
            )
        )
    )
    .reset_index()
)


print(
    "Maximum absolute reported/calculated LOS difference:",
    los_summary[
        "Maximum_abs_LOS_difference"
    ].max()
)


# ==============================================================================
# 16. SERVICE GROUP STRUCTURE
# ==============================================================================

print("\n" + "=" * 90)
print("STEP 10 — SERVICE-GROUP STRUCTURE")
print("=" * 90)


service_summary = (
    anonymous
    .groupby(
        "Anonymous_Linkage_Key",
        dropna=False
    )
    .agg(
        Number_of_claim_lines=(
            "Service Group",
            "size"
        ),
        Number_of_service_groups=(
            "Service Group",
            "nunique"
        )
    )
    .reset_index()
)


print(
    service_summary[
        "Number_of_service_groups"
    ].describe()
)


# ==============================================================================
# 17. FINANCIAL STRUCTURE
# ==============================================================================

print("\n" + "=" * 90)
print("STEP 11 — FINANCIAL STRUCTURE")
print("=" * 90)


financial_columns = [
    "Total Cost",
    "Insured Person's Share",
    "Organization Requested Share",
    "Organization Paid Share"
]


financial_present = [
    col
    for col in financial_columns
    if col in anonymous.columns
]


for col in financial_present:

    negative_n = (
        anonymous[col] < 0
    ).sum()

    print(
        "{} negative values: {}".format(
            col,
            negative_n
        )
    )


# ==============================================================================
# 18. DETECT POTENTIAL OVER-LINKAGE
# ==============================================================================

print("\n" + "=" * 90)
print("STEP 12 — POTENTIAL OVER-LINKAGE SCREEN")
print("=" * 90)


suspicious = linkage_summary.copy()


# Flag 1:
# One linkage key is spread across multiple non-contiguous blocks.

suspicious["FLAG_multiple_contiguous_blocks"] = (
    suspicious["Number_of_contiguous_blocks"]
    > 1
)


# Flag 2:
# A linkage block has more than one discharge status.

suspicious["FLAG_multiple_discharge_statuses"] = (
    suspicious["Number_of_discharge_statuses"]
    > 1
)


# Flag 3:
# The physical row span contains more rows than the number of claims.
#
# This means there are identified records between anonymous claims belonging
# to the same anonymous linkage key.

suspicious["FLAG_non_contiguous_span"] = (
    suspicious["Expected_span"]
    >
    suspicious["Number_of_claim_lines"]
)


# Flag 4:
# Extremely heterogeneous LOS within one deterministic block.

suspicious["FLAG_large_LOS_variability"] = (
    (
        suspicious["Maximum_LOS"]
        -
        suspicious["Minimum_LOS"]
    )
    > 1
)


# Combined flag

suspicious["Potential_over_linkage"] = (
    suspicious[
        "FLAG_multiple_contiguous_blocks"
    ]
    |
    suspicious[
        "FLAG_multiple_discharge_statuses"
    ]
    |
    suspicious[
        "FLAG_non_contiguous_span"
    ]
)


suspicious_blocks = suspicious.loc[
    suspicious["Potential_over_linkage"]
].copy()


print(
    "Total anonymous linkage blocks:",
    len(suspicious)
)

print(
    "Potentially suspicious blocks:",
    len(suspicious_blocks)
)


if len(suspicious_blocks) > 0:

    print(
        "\nPotentially suspicious anonymous blocks:"
    )

    print(
        suspicious_blocks[
            [
                "Min_original_row",
                "Max_original_row",
                "Number_of_claim_lines",
                "Number_of_contiguous_blocks",
                "Number_of_discharge_statuses",
                "Expected_span",
                "Potential_over_linkage"
            ]
        ].to_string(
            index=False
        )
    )

else:

    print(
        "\nNo anonymous linkage blocks were flagged "
        "by the predefined over-linkage diagnostics."
    )


# ==============================================================================
# 19. ADJACENT ANONYMOUS BLOCK ANALYSIS
# ==============================================================================

print("\n" + "=" * 90)
print("STEP 13 — ADJACENT ANONYMOUS BLOCK ANALYSIS")
print("=" * 90)


# Sort the linkage blocks by their position in the original file.

ordered_blocks = linkage_summary.sort_values(
    "Min_original_row"
).reset_index(drop=True)


transition_records = []


for i in range(
    len(ordered_blocks) - 1
):

    current = ordered_blocks.iloc[i]
    next_block = ordered_blocks.iloc[i + 1]

    gap = (
        next_block["Min_original_row"]
        -
        current["Max_original_row"]
        -
        1
    )

    transition_records.append({

        "Current_Block_Number": i + 1,

        "Current_Min_Row":
            current["Min_original_row"],

        "Current_Max_Row":
            current["Max_original_row"],

        "Current_Claim_Count":
            current["Number_of_claim_lines"],

        "Next_Block_Number": i + 2,

        "Next_Min_Row":
            next_block["Min_original_row"],

        "Next_Max_Row":
            next_block["Max_original_row"],

        "Next_Claim_Count":
            next_block["Number_of_claim_lines"],

        "Rows_Between_Blocks":
            gap
    })


transition_df = pd.DataFrame(
    transition_records
)


if len(transition_df) > 0:

    print(
        "Adjacent anonymous blocks:",
        len(transition_df)
    )

    print(
        "Adjacent blocks with no identified rows between them:",
        (
            transition_df[
                "Rows_Between_Blocks"
            ] == 0
        ).sum()
    )


# ==============================================================================
# 20. CLAIM-LEVEL VALIDATION FILE
# ==============================================================================

print("\n" + "=" * 90)
print("STEP 14 — CLAIM-LEVEL AUDIT FILE")
print("=" * 90)


claim_audit_columns = [
    "_original_row_number",
    "Patient_ID",
    "Anonymous_Linkage_Key",
    "Contiguous_Block_Number",
    "Previous_Original_Row",
    "Row_Gap",
    "New_Contiguous_Block",
    "Admision date",
    "Discharge date",
    "Age",
    "Gender",
    "Ownership",
    "Insurance Fund",
    "Treating Physician Specialty",
    "Diagnosis type",
    "Department Type",
    "Discharge Status",
    "Length of Stay",
    "Calculated_LOS",
    "LOS_difference",
    "Service Group",
    "Total Cost"
]


claim_audit_columns = [
    col
    for col in claim_audit_columns
    if col in anonymous.columns
]


anonymous[
    claim_audit_columns
].to_csv(
    CLAIM_AUDIT_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ==============================================================================
# 21. SAVE LINKAGE AUDIT
# ==============================================================================

linkage_summary.to_csv(
    AUDIT_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ==============================================================================
# 22. SAVE SUSPICIOUS BLOCKS
# ==============================================================================

suspicious_blocks.to_csv(
    SUSPICIOUS_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ==============================================================================
# 23. SAVE TRANSITIONS
# ==============================================================================

transition_df.to_csv(
    TRANSITION_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ==============================================================================
# 24. FINAL VALIDATION SUMMARY
# ==============================================================================

print("\n" + "=" * 90)
print("STEP 15 — FINAL ANONYMOUS LINKAGE VALIDATION")
print("=" * 90)


anonymous_claim_count = len(anonymous)

anonymous_block_count = len(
    linkage_summary
)

perfectly_contiguous_count = int(
    linkage_summary[
        "Perfectly_contiguous"
    ].sum()
)

non_contiguous_count = (
    anonymous_block_count
    -
    perfectly_contiguous_count
)

multiple_status_count = int(
    (
        linkage_summary[
            "Number_of_discharge_statuses"
        ]
        > 1
    ).sum()
)

multi_block_count = int(
    (
        linkage_summary[
            "Number_of_contiguous_blocks"
        ]
        > 1
    ).sum()
)

span_mismatch_count = int(
    (
        ~linkage_summary[
            "Claims_equal_span"
        ]
    ).sum()
)

potential_overlinkage_count = len(
    suspicious_blocks
)


summary_records = [

    {
        "Metric":
            "Total raw claim lines",
        "Value":
            len(df)
    },

    {
        "Metric":
            "Anonymous claim lines",
        "Value":
            anonymous_claim_count
    },

    {
        "Metric":
            "Anonymous deterministic linkage blocks",
        "Value":
            anonymous_block_count
    },

    {
        "Metric":
            "Perfectly contiguous anonymous blocks",
        "Value":
            perfectly_contiguous_count
    },

    {
        "Metric":
            "Non-contiguous anonymous blocks",
        "Value":
            non_contiguous_count
    },

    {
        "Metric":
            "Blocks with multiple discharge statuses",
        "Value":
            multiple_status_count
    },

    {
        "Metric":
            "Blocks spanning multiple contiguous blocks",
        "Value":
            multi_block_count
    },

    {
        "Metric":
            "Blocks with row-span mismatch",
        "Value":
            span_mismatch_count
    },

    {
        "Metric":
            "Potentially over-linked blocks",
        "Value":
            potential_overlinkage_count
    }
]


summary_df = pd.DataFrame(
    summary_records
)


summary_df.to_csv(
    SUMMARY_FILE,
    index=False,
    encoding="utf-8-sig"
)


print(
    summary_df.to_string(
        index=False
    )
)


# ==============================================================================
# 25. AUTOMATED INTERPRETATION
# ==============================================================================

print("\n" + "=" * 90)
print("INTERPRETATION")
print("=" * 90)


if (
    perfectly_contiguous_count
    ==
    anonymous_block_count
):

    print(
        "✓ ALL anonymous linkage blocks are physically contiguous "
        "in the original claim file."
    )

else:

    print(
        "⚠ Some anonymous linkage blocks are non-contiguous."
    )


if span_mismatch_count == 0:

    print(
        "✓ Every anonymous linkage block has a row span exactly "
        "equal to its number of claim lines."
    )

else:

    print(
        "⚠ Some anonymous linkage blocks contain gaps in the "
        "original row sequence."
    )


if multi_block_count == 0:

    print(
        "✓ No deterministic anonymous linkage key is distributed "
        "across multiple separated blocks."
    )

else:

    print(
        "⚠ At least one linkage key appears in multiple separated "
        "blocks and requires manual review."
    )


if multiple_status_count <= 1:

    print(
        "✓ Discharge-status inconsistency is limited to at most "
        "one anonymous block."
    )

else:

    print(
        "⚠ Multiple anonymous blocks have conflicting discharge statuses."
    )


if potential_overlinkage_count == 0:

    print(
        "\nOVERALL PRELIMINARY CONCLUSION:"
    )

    print(
        "The anonymous linkage structure shows no predefined "
        "evidence of over-linkage."
    )

    print(
        "The anonymous claims can therefore be retained as "
        "deterministic hospitalization blocks, subject to "
        "clinical/manual review of any unusual cases."
    )

else:

    print(
        "\nOVERALL PRELIMINARY CONCLUSION:"
    )

    print(
        "Potential over-linkage was detected."
    )

    print(
        "DO NOT freeze the anonymous episode structure yet."
    )

    print(
        "Review the suspicious blocks listed in:"
    )

    print(
        SUSPICIOUS_FILE
    )


# ==============================================================================
# 26. OUTPUT FILES
# ==============================================================================

print("\n" + "=" * 90)
print("FILES CREATED")
print("=" * 90)

print(
    "1. Anonymous linkage audit:"
)
print(
    AUDIT_FILE
)

print(
    "\n2. Claim-level anonymous validation:"
)
print(
    CLAIM_AUDIT_FILE
)

print(
    "\n3. Suspicious anonymous blocks:"
)
print(
    SUSPICIOUS_FILE
)

print(
    "\n4. Anonymous block transitions:"
)
print(
    TRANSITION_FILE
)

print(
    "\n5. Validation summary:"
)
print(
    SUMMARY_FILE
)

print("\nDONE.")

In [ ]:
# =============================================================================
# CHOLERA CLAIM DATA
# FINAL VALIDATION OF ANONYMOUS EPISODE BLOCKS
#
# PURPOSE
# -------
# Validate whether the 661 claims without Patient_ID form 67 internally
# coherent hospitalization blocks.
#
# IMPORTANT:
#   1. NO CLAIMS ARE DELETED.
#   2. NO CLAIMS ARE MERGED OR REASSIGNED.
#   3. NO VALUES ARE IMPUTED.
#   4. NO EPISODE DEFINITIONS ARE CHANGED.
#   5. This script is VALIDATION ONLY.
#
# Decision question:
#
#   Are the 67 anonymous deterministic blocks sufficiently coherent to
#   represent 67 hospitalization episode units?
#
# If the answer is yes, the 1,171-episode dataset can be frozen:
#
#       1,104 identified episodes
#       + 67 anonymous episodes
#       = 1,171 total episode units
#
# =============================================================================


import pandas as pd
import numpy as np
from pathlib import Path


# =============================================================================
# 1. FILE PATH
# =============================================================================

file_path = (
    r"C:\Users\Shekoofeh\Documents\Health Insurance Data"
    r"\Cholera\Cholera_csv_translated.csv"
)

OUTPUT_DIR = Path(file_path).parent


# Output files
BLOCK_COHERENCE_FILE = (
    OUTPUT_DIR / "09_anonymous_block_coherence.csv"
)

BLOCK_SUMMARY_FILE = (
    OUTPUT_DIR / "10_anonymous_block_validation_summary.csv"
)

BLOCK_REVIEW_FILE = (
    OUTPUT_DIR / "11_anonymous_block_manual_review.csv"
)


# =============================================================================
# 2. LOAD RAW DATA
# =============================================================================

print("=" * 100)
print("FINAL VALIDATION — ANONYMOUS EPISODE BLOCK COHERENCE")
print("=" * 100)

df = pd.read_csv(
    file_path,
    encoding="utf-8-sig",
    low_memory=False
)

print(f"\nRaw claim lines loaded: {len(df):,}")


# =============================================================================
# 3. STANDARDIZE COLUMN NAMES
# =============================================================================

def clean_column_name(column):

    column = str(column)

    column = column.replace('"', "")
    column = column.replace("\n", " ")
    column = column.replace("\t", " ")

    # Collapse repeated spaces
    import re

    column = re.sub(r"\s+", " ", column)

    return column.strip()


df.columns = [
    clean_column_name(col)
    for col in df.columns
]


print("\nColumns loaded:")
for i, col in enumerate(df.columns, start=1):
    print(f"{i:02d}. {col}")


# =============================================================================
# 4. PRESERVE ORIGINAL ROW NUMBER
# =============================================================================

df["_original_row_number"] = np.arange(len(df))


# =============================================================================
# 5. IDENTIFY ANONYMOUS CLAIMS
# =============================================================================

missing_id_values = {
    "",
    "nan",
    "none",
    "null",
    "na",
    "n/a",
    "missing"
}


patient_id_string = (
    df["Patient_ID"]
    .astype("string")
    .str.strip()
)


anonymous_mask = (
    df["Patient_ID"].isna()
    |
    patient_id_string.str.lower().isin(
        missing_id_values
    )
)


anonymous = df.loc[
    anonymous_mask
].copy()


identified = df.loc[
    ~anonymous_mask
].copy()


print("\n" + "=" * 100)
print("ANONYMOUS CLAIM AUDIT")
print("=" * 100)

print(
    f"Total raw claims:             {len(df):,}"
)

print(
    f"Identified claims:            {len(identified):,}"
)

print(
    f"Anonymous claims:             {len(anonymous):,}"
)


# =============================================================================
# 6. VERIFY EXPECTED 661 ANONYMOUS CLAIMS
# =============================================================================

EXPECTED_ANONYMOUS_CLAIMS = 661

if len(anonymous) != EXPECTED_ANONYMOUS_CLAIMS:

    raise ValueError(
        "\nSTOP: The number of anonymous claims is not 661.\n"
        f"Expected: {EXPECTED_ANONYMOUS_CLAIMS:,}\n"
        f"Observed: {len(anonymous):,}\n"
        "\n"
        "Do not continue until the input dataset is checked."
    )

print(
    "\nPASS — exactly 661 anonymous claims identified."
)


# =============================================================================
# 7. CONVERT DATES
# =============================================================================

date_columns = [
    "Admision date",
    "Discharge date"
]

for col in date_columns:

    if col in anonymous.columns:

        anonymous[col] = pd.to_datetime(
            anonymous[col],
            errors="coerce"
        )


# =============================================================================
# 8. CONVERT NUMERIC VARIABLES
# =============================================================================

numeric_columns = [
    "Age",
    "Length of Stay",
    "Total Cost",
    "Insured Person's Share",
    "Organization Requested Share",
    "Organization Paid Share",
    "Terminally Patients Paid Share",
    "Terminally Patients Requested Share",
    "Allocated subsidized foreign currency share",
    "Requested subsidized foreign currency share"
]


for col in numeric_columns:

    if col in anonymous.columns:

        anonymous[col] = (
            anonymous[col]
            .astype("string")
            .str.replace(",", "", regex=False)
            .str.strip()
        )

        anonymous[col] = pd.to_numeric(
            anonymous[col],
            errors="coerce"
        )


# =============================================================================
# 9. RECONSTRUCT THE SAME ANONYMOUS BLOCKS
# =============================================================================
#
# IMPORTANT:
#
# We use the deterministic characteristics that produced the previously
# validated 67 blocks.
#
# The purpose here is NOT to create a new episode definition.
# It is to reproduce the existing anonymous linkage blocks so that their
# internal coherence can be examined.
#
# =============================================================================

anonymous["Admission_key"] = (
    anonymous["Admision date"]
    .dt.strftime("%Y-%m-%d")
)

anonymous["Discharge_key"] = (
    anonymous["Discharge date"]
    .dt.strftime("%Y-%m-%d")
)


# Characteristics used by the anonymous deterministic linkage
linkage_columns = [
    "Admission_key",
    "Discharge_key",
    "Age",
    "Gender",
    "Ownership",
    "Insurance Fund",
    "Diagnosis type",
    "Department Type"
]


missing_linkage_columns = [
    col
    for col in linkage_columns
    if col not in anonymous.columns
]


if missing_linkage_columns:

    raise ValueError(
        "\nMissing variables required for anonymous block validation:\n"
        + "\n".join(
            f"  - {x}"
            for x in missing_linkage_columns
        )
    )


# =============================================================================
# 10. CREATE DETERMINISTIC BLOCK KEY
# =============================================================================

anonymous["Anonymous_Block_Key"] = (
    anonymous[
        linkage_columns
    ]
    .astype("string")
    .fillna("<MISSING>")
    .agg("|".join, axis=1)
)


# =============================================================================
# 11. ASSIGN BLOCK NUMBERS
# =============================================================================
#
# We preserve the original row ordering.
#
# This is essential because our previous validation showed that the
# anonymous records occupy rows 0–660 as contiguous blocks.
#
# =============================================================================

anonymous = anonymous.sort_values(
    "_original_row_number"
).reset_index(drop=True)


# A new block begins whenever the deterministic key changes.
anonymous["_new_block"] = (
    anonymous["Anonymous_Block_Key"]
    != anonymous["Anonymous_Block_Key"].shift()
)


anonymous["Block_Number"] = (
    anonymous["_new_block"]
    .cumsum()
)


anonymous["Episode_ID"] = (
    "ANON_"
    + anonymous["Block_Number"]
    .astype(str)
    .str.zfill(3)
)


# =============================================================================
# 12. BASIC BLOCK COUNT VALIDATION
# =============================================================================

n_blocks = (
    anonymous["Episode_ID"]
    .nunique()
)

print("\n" + "=" * 100)
print("BLOCK STRUCTURE")
print("=" * 100)

print(
    f"Anonymous claim lines:       {len(anonymous):,}"
)

print(
    f"Anonymous blocks detected:   {n_blocks:,}"
)


if n_blocks != 67:

    raise ValueError(
        "\nSTOP: Expected 67 anonymous blocks, "
        f"but {n_blocks} were detected.\n"
        "The linkage definition has changed and must be investigated."
    )

print(
    "\nPASS — exactly 67 anonymous blocks reproduced."
)


# =============================================================================
# 13. FUNCTION FOR WITHIN-BLOCK UNIQUENESS
# =============================================================================

def unique_nonmissing_count(series):

    return series.dropna().nunique()


def unique_values(series):

    values = (
        series
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    return " | ".join(values)


# =============================================================================
# 14. BUILD BLOCK-LEVEL COHERENCE TABLE
# =============================================================================

coherence_records = []


for episode_id, block in anonymous.groupby(
    "Episode_ID",
    sort=False
):

    record = {}

    record["Episode_ID"] = episode_id

    record["Block_Number"] = (
        block["Block_Number"].iloc[0]
    )

    record["Claim_Count"] = len(block)

    record["Min_original_row"] = (
        block["_original_row_number"].min()
    )

    record["Max_original_row"] = (
        block["_original_row_number"].max()
    )

    record["Row_Span"] = (
        record["Max_original_row"]
        - record["Min_original_row"]
        + 1
    )

    record["Row_Span_Equals_Claim_Count"] = (
        record["Row_Span"]
        ==
        record["Claim_Count"]
    )


    # -------------------------------------------------------------------------
    # Admission date
    # -------------------------------------------------------------------------

    record["Admission_Date_Unique"] = (
        unique_nonmissing_count(
            block["Admision date"]
        )
    )

    record["Admission_Date_Values"] = (
        unique_values(
            block["Admision date"]
        )
    )

    record["Admission_Date_Coherent"] = (
        record["Admission_Date_Unique"] <= 1
    )


    # -------------------------------------------------------------------------
    # Discharge date
    # -------------------------------------------------------------------------

    record["Discharge_Date_Unique"] = (
        unique_nonmissing_count(
            block["Discharge date"]
        )
    )

    record["Discharge_Date_Values"] = (
        unique_values(
            block["Discharge date"]
        )
    )

    record["Discharge_Date_Coherent"] = (
        record["Discharge_Date_Unique"] <= 1
    )


    # -------------------------------------------------------------------------
    # Age
    # -------------------------------------------------------------------------

    record["Age_Unique"] = (
        unique_nonmissing_count(
            block["Age"]
        )
    )

    record["Age_Values"] = (
        unique_values(
            block["Age"]
        )
    )

    record["Age_Coherent"] = (
        record["Age_Unique"] <= 1
    )


    # -------------------------------------------------------------------------
    # Gender
    # -------------------------------------------------------------------------

    record["Gender_Unique"] = (
        unique_nonmissing_count(
            block["Gender"]
        )
    )

    record["Gender_Values"] = (
        unique_values(
            block["Gender"]
        )
    )

    record["Gender_Coherent"] = (
        record["Gender_Unique"] <= 1
    )


    # -------------------------------------------------------------------------
    # Ownership
    # -------------------------------------------------------------------------

    record["Ownership_Unique"] = (
        unique_nonmissing_count(
            block["Ownership"]
        )
    )

    record["Ownership_Values"] = (
        unique_values(
            block["Ownership"]
        )
    )

    record["Ownership_Coherent"] = (
        record["Ownership_Unique"] <= 1
    )


    # -------------------------------------------------------------------------
    # Insurance Fund
    # -------------------------------------------------------------------------

    record["Insurance_Fund_Unique"] = (
        unique_nonmissing_count(
            block["Insurance Fund"]
        )
    )

    record["Insurance_Fund_Values"] = (
        unique_values(
            block["Insurance Fund"]
        )
    )

    record["Insurance_Fund_Coherent"] = (
        record["Insurance_Fund_Unique"] <= 1
    )


    # -------------------------------------------------------------------------
    # Diagnosis
    # -------------------------------------------------------------------------

    record["Diagnosis_Unique"] = (
        unique_nonmissing_count(
            block["Diagnosis type"]
        )
    )

    record["Diagnosis_Values"] = (
        unique_values(
            block["Diagnosis type"]
        )
    )

    record["Diagnosis_Coherent"] = (
        record["Diagnosis_Unique"] <= 1
    )


    # -------------------------------------------------------------------------
    # Department
    # -------------------------------------------------------------------------

    record["Department_Unique"] = (
        unique_nonmissing_count(
            block["Department Type"]
        )
    )

    record["Department_Values"] = (
        unique_values(
            block["Department Type"]
        )
    )

    record["Department_Coherent"] = (
        record["Department_Unique"] <= 1
    )


    # -------------------------------------------------------------------------
    # Treating physician specialty
    #
    # This is deliberately NOT required to be unique.
    # A hospitalization can legitimately involve multiple specialties.
    # -------------------------------------------------------------------------

    if "Treating Physician Specialty" in block.columns:

        record["Specialty_Unique"] = (
            unique_nonmissing_count(
                block[
                    "Treating Physician Specialty"
                ]
            )
        )

        record["Specialty_Values"] = (
            unique_values(
                block[
                    "Treating Physician Specialty"
                ]
            )
        )


    # -------------------------------------------------------------------------
    # Service groups
    #
    # Multiple service groups are expected within an episode and therefore
    # are NOT treated as evidence of over-linkage.
    # -------------------------------------------------------------------------

    if "Service Group" in block.columns:

        record["Service_Group_Unique"] = (
            unique_nonmissing_count(
                block["Service Group"]
            )
        )

        record["Service_Group_Values"] = (
            unique_values(
                block["Service Group"]
            )
        )


    # -------------------------------------------------------------------------
    # Length of stay
    # -------------------------------------------------------------------------

    record["LOS_Unique"] = (
        unique_nonmissing_count(
            block["Length of Stay"]
        )
    )

    record["LOS_Values"] = (
        unique_values(
            block["Length of Stay"]
        )
    )

    record["LOS_Coherent"] = (
        record["LOS_Unique"] <= 1
    )


    # -------------------------------------------------------------------------
    # Financial information
    # -------------------------------------------------------------------------

    financial_cols = [
        "Total Cost",
        "Insured Person's Share",
        "Organization Requested Share",
        "Organization Paid Share",
        "Terminally Patients Paid Share",
        "Terminally Patients Requested Share",
        "Allocated subsidized foreign currency share",
        "Requested subsidized foreign currency share"
    ]


    for col in financial_cols:

        if col not in block.columns:
            continue

        safe_name = (
            col
            .replace(" ", "_")
            .replace("'", "")
        )

        record[
            f"{safe_name}_Sum"
        ] = block[col].sum(
            min_count=1
        )

        record[
            f"{safe_name}_Nonmissing_n"
        ] = block[col].notna().sum()

        record[
            f"{safe_name}_Unique"
        ] = block[col].nunique(
            dropna=True
        )


    # -------------------------------------------------------------------------
    # Discharge status
    # -------------------------------------------------------------------------

    if "Discharge Status" in block.columns:

        record["Discharge_Status_Unique"] = (
            unique_nonmissing_count(
                block["Discharge Status"]
            )
        )

        record["Discharge_Status_Values"] = (
            unique_values(
                block["Discharge Status"]
            )
        )

        record["Discharge_Status_Coherent"] = (
            record[
                "Discharge_Status_Unique"
            ] <= 1
        )


    coherence_records.append(record)


coherence_df = pd.DataFrame(
    coherence_records
)


# =============================================================================
# 15. OVERALL COHERENCE FLAGS
# =============================================================================
#
# We distinguish:
#
# A. HARD COHERENCE VARIABLES
#    Variables that should normally be constant within one hospitalization.
#
# B. EXPECTEDLY VARIABLE VARIABLES
#    Variables such as service group and physician specialty that may
#    legitimately vary during one hospitalization.
#
# =============================================================================

hard_coherence_columns = [
    "Admission_Date_Coherent",
    "Discharge_Date_Coherent",
    "Age_Coherent",
    "Gender_Coherent",
    "Ownership_Coherent",
    "Insurance_Fund_Coherent",
    "Diagnosis_Coherent",
    "Department_Coherent",
    "LOS_Coherent",
    "Discharge_Status_Coherent"
]


available_hard_columns = [
    col
    for col in hard_coherence_columns
    if col in coherence_df.columns
]


coherence_df["All_Hard_Variables_Coherent"] = (
    coherence_df[
        available_hard_columns
    ]
    .all(axis=1)
)


# =============================================================================
# 16. CONTIGUITY CHECK
# =============================================================================

coherence_df["Perfectly_Contiguous"] = (
    coherence_df["Row_Span_Equals_Claim_Count"]
)


# =============================================================================
# 17. POTENTIAL OVER-LINKAGE FLAG
# =============================================================================
#
# A block is flagged for review if:
#
#   - it is not contiguous, OR
#   - admission date varies, OR
#   - discharge date varies, OR
#   - age varies, OR
#   - gender varies, OR
#   - ownership varies, OR
#   - insurance fund varies, OR
#   - diagnosis varies, OR
#   - department varies, OR
#   - discharge status varies.
#
# LOS is included as a coherence diagnostic but NOT by itself treated as
# definitive evidence of over-linkage because source LOS fields can contain
# claim-level or administrative variation.
#
# =============================================================================

coherence_df["Potential_Overlinkage"] = ~(
    coherence_df[
        [
            "Perfectly_Contiguous",
            "Admission_Date_Coherent",
            "Discharge_Date_Coherent",
            "Age_Coherent",
            "Gender_Coherent",
            "Ownership_Coherent",
            "Insurance_Fund_Coherent",
            "Diagnosis_Coherent",
            "Department_Coherent",
            "Discharge_Status_Coherent"
        ]
    ].all(axis=1)
)


# =============================================================================
# 18. COUNT INCOHERENCIES
# =============================================================================

for col in available_hard_columns:

    print(
        f"{col}: "
        f"{(~coherence_df[col]).sum():,} blocks with variation"
    )


# =============================================================================
# 19. SAVE COMPLETE BLOCK-LEVEL COHERENCE TABLE
# =============================================================================

coherence_df.to_csv(
    BLOCK_COHERENCE_FILE,
    index=False,
    encoding="utf-8-sig"
)


# =============================================================================
# 20. CREATE MANUAL REVIEW TABLE
# =============================================================================

review_columns = [
    "Episode_ID",
    "Block_Number",
    "Claim_Count",
    "Min_original_row",
    "Max_original_row",
    "Row_Span",
    "Perfectly_Contiguous",
    "Admission_Date_Unique",
    "Admission_Date_Values",
    "Discharge_Date_Unique",
    "Discharge_Date_Values",
    "Age_Unique",
    "Age_Values",
    "Gender_Unique",
    "Gender_Values",
    "Ownership_Unique",
    "Ownership_Values",
    "Insurance_Fund_Unique",
    "Insurance_Fund_Values",
    "Diagnosis_Unique",
    "Diagnosis_Values",
    "Department_Unique",
    "Department_Values",
    "Discharge_Status_Unique",
    "Discharge_Status_Values",
    "Potential_Overlinkage"
]


review_columns = [
    col
    for col in review_columns
    if col in coherence_df.columns
]


manual_review_df = coherence_df.loc[
    coherence_df["Potential_Overlinkage"],
    review_columns
].copy()


manual_review_df.to_csv(
    BLOCK_REVIEW_FILE,
    index=False,
    encoding="utf-8-sig"
)


# =============================================================================
# 21. VERIFY THAT ALL 661 CLAIMS ARE ACCOUNTED FOR
# =============================================================================

total_claims_in_blocks = (
    coherence_df["Claim_Count"].sum()
)

print("\n" + "=" * 100)
print("CLAIM RETENTION CHECK")
print("=" * 100)

print(
    f"Anonymous claims expected:    661"
)

print(
    f"Anonymous claims in blocks:   "
    f"{total_claims_in_blocks:,}"
)

print(
    f"Number of anonymous blocks:   "
    f"{len(coherence_df):,}"
)


if total_claims_in_blocks != 661:

    raise ValueError(
        "\nSTOP — claim retention failure.\n"
        f"661 anonymous claims were expected, "
        f"but {total_claims_in_blocks:,} were assigned to blocks."
    )


print(
    "\nPASS — all 661 anonymous claims are represented exactly in the blocks."
)


# =============================================================================
# 22. VERIFY NO CLAIM IS ASSIGNED TO MORE THAN ONE BLOCK
# =============================================================================

duplicate_original_rows = (
    anonymous["_original_row_number"]
    .duplicated()
    .sum()
)


if duplicate_original_rows != 0:

    raise ValueError(
        "\nSTOP — duplicate original row numbers detected "
        "within anonymous claims."
    )


print(
    "PASS — no anonymous claim is duplicated."
)


# =============================================================================
# 23. VERIFY COMPLETE ORIGINAL ROW COVERAGE
# =============================================================================

anonymous_original_rows = set(
    anonymous["_original_row_number"]
)


expected_original_rows = set(
    range(
        anonymous["_original_row_number"].min(),
        anonymous["_original_row_number"].max() + 1
    )
)


missing_rows = (
    expected_original_rows
    -
    anonymous_original_rows
)


extra_rows = (
    anonymous_original_rows
    -
    expected_original_rows
)


print(
    f"\nAnonymous row range: "
    f"{anonymous['_original_row_number'].min()}–"
    f"{anonymous['_original_row_number'].max()}"
)

print(
    f"Missing rows inside anonymous range: "
    f"{len(missing_rows)}"
)

print(
    f"Unexpected rows: "
    f"{len(extra_rows)}"
)


# =============================================================================
# 24. FINAL SUMMARY
# =============================================================================

n_perfect_contiguous = (
    coherence_df[
        "Perfectly_Contiguous"
    ].sum()
)

n_overlinked = (
    coherence_df[
        "Potential_Overlinkage"
    ].sum()
)

n_hard_coherent = (
    coherence_df[
        "All_Hard_Variables_Coherent"
    ].sum()
)


summary = pd.DataFrame({

    "Metric": [

        "Total raw claim lines",

        "Anonymous claim lines",

        "Anonymous blocks",

        "Anonymous claims represented in blocks",

        "Perfectly contiguous blocks",

        "Non-contiguous blocks",

        "Blocks with coherent hard variables",

        "Blocks with potential over-linkage",

        "Blocks with admission-date variation",

        "Blocks with discharge-date variation",

        "Blocks with age variation",

        "Blocks with gender variation",

        "Blocks with ownership variation",

        "Blocks with insurance-fund variation",

        "Blocks with diagnosis variation",

        "Blocks with department variation",

        "Blocks with discharge-status variation",

        "Blocks with LOS variation",

        "Duplicate anonymous original rows",

        "Missing rows within anonymous range"

    ],

    "Value": [

        len(df),

        len(anonymous),

        len(coherence_df),

        total_claims_in_blocks,

        n_perfect_contiguous,

        len(coherence_df)
        - n_perfect_contiguous,

        n_hard_coherent,

        n_overlinked,

        (
            ~coherence_df[
                "Admission_Date_Coherent"
            ]
        ).sum(),

        (
            ~coherence_df[
                "Discharge_Date_Coherent"
            ]
        ).sum(),

        (
            ~coherence_df[
                "Age_Coherent"
            ]
        ).sum(),

        (
            ~coherence_df[
                "Gender_Coherent"
            ]
        ).sum(),

        (
            ~coherence_df[
                "Ownership_Coherent"
            ]
        ).sum(),

        (
            ~coherence_df[
                "Insurance_Fund_Coherent"
            ]
        ).sum(),

        (
            ~coherence_df[
                "Diagnosis_Coherent"
            ]
        ).sum(),

        (
            ~coherence_df[
                "Department_Coherent"
            ]
        ).sum(),

        (
            ~coherence_df[
                "Discharge_Status_Coherent"
            ]
        ).sum(),

        (
            ~coherence_df[
                "LOS_Coherent"
            ]
        ).sum(),

        duplicate_original_rows,

        len(missing_rows)

    ]

})


summary.to_csv(
    BLOCK_SUMMARY_FILE,
    index=False,
    encoding="utf-8-sig"
)


# =============================================================================
# 25. DISPLAY RESULTS
# =============================================================================

print("\n" + "=" * 100)
print("FINAL ANONYMOUS BLOCK COHERENCE RESULTS")
print("=" * 100)

print(
    summary.to_string(
        index=False
    )
)


# =============================================================================
# 26. DISPLAY POTENTIAL PROBLEMS
# =============================================================================

print("\n" + "=" * 100)
print("BLOCKS REQUIRING MANUAL REVIEW")
print("=" * 100)

if len(manual_review_df) == 0:

    print(
        "\nNONE."
    )

    print(
        "\nAll 67 anonymous blocks passed the hard coherence checks."
    )

else:

    print(
        f"\n{len(manual_review_df):,} "
        "anonymous block(s) require review:\n"
    )

    print(
        manual_review_df.to_string(
            index=False
        )
    )


# =============================================================================
# 27. FINAL DECISION LOGIC
# =============================================================================

print("\n" + "=" * 100)
print("FINAL DECISION")
print("=" * 100)


if (
    len(coherence_df) == 67
    and
    total_claims_in_blocks == 661
    and
    n_perfect_contiguous == 67
    and
    n_overlinked == 0
    and
    duplicate_original_rows == 0
    and
    len(missing_rows) == 0
):

    print(
        """
PASS — ANONYMOUS BLOCK STRUCTURE IS SUPPORTED.

The validation confirms:

1. All 661 anonymous claims are retained.
2. They form exactly 67 deterministic blocks.
3. All 67 blocks are perfectly contiguous.
4. No block shows evidence of over-linkage under the predefined
   hard coherence criteria.
5. No anonymous claim is duplicated.
6. No claim is missing from the anonymous block assignment.

The anonymous portion of the episode dataset is therefore supported
for inclusion as 67 hospitalization episode units.

Combined with the 1,104 identified episodes:

    1,104 identified episodes
    +   67 anonymous episodes
    -------------------------
    = 1,171 episode units

IMPORTANT:
This validation does not alter the data and does not constitute
statistical imputation of Patient_ID. The anonymous episodes remain
explicitly identifiable as ANON episodes.
"""
    )

else:

    print(
        """
REVIEW REQUIRED.

The anonymous block structure did not satisfy all predefined
validation criteria.

DO NOT freeze the episode dataset yet.

Review the generated block-level coherence file and manual-review file
before proceeding to multicollinearity analysis.
"""
    )


# =============================================================================
# 28. OUTPUT FILES
# =============================================================================

print("\n" + "=" * 100)
print("FILES CREATED")
print("=" * 100)

print(
    f"1. Complete block coherence table:\n"
    f"   {BLOCK_COHERENCE_FILE}"
)

print(
    f"\n2. Overall validation summary:\n"
    f"   {BLOCK_SUMMARY_FILE}"
)

print(
    f"\n3. Manual review table:\n"
    f"   {BLOCK_REVIEW_FILE}"
)

print("\n" + "=" * 100)
print("VALIDATION COMPLETE")
print("=" * 100)

In [ ]:
# ================================================================
# CHOLERA — EPISODE-LEVEL PRE-FEATURE-ENGINEERING AUDIT
# ================================================================
#
# PURPOSE
# -------
# This script audits the FINAL EPISODE-LEVEL DATASET before
# feature engineering and ML modeling.
#
# The audit is explicitly designed to prevent:
#
#   1. target leakage
#   2. post-discharge leakage
#   3. mathematical leakage
#   4. identifier leakage
#   5. multicollinearity
#   6. high-cardinality categorical instability
#   7. rare-category problems
#   8. temporal leakage
#   9. redundant derived variables
#
# IMPORTANT
# ---------
# This script DOES NOT:
#
#   - impute
#   - winsorize
#   - remove outliers
#   - standardize
#   - one-hot encode for modeling
#   - perform feature selection
#
# Those operations belong AFTER the predictor set has been
# definitively established.
#
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from itertools import combinations
import warnings

warnings.filterwarnings("ignore")


# ================================================================
# 1. FILE PATH
# ================================================================

FILE_PATH = Path(
    r"C:\Users\Shekoofeh\Documents\Health Insurance Data\Cholera\04_episode_level_data.csv"
)

OUTPUT_DIR = FILE_PATH.parent


# ================================================================
# 2. OUTPUT FILES
# ================================================================

FILES = {

    "dataset_audit":
        OUTPUT_DIR / "08_episode_dataset_audit.csv",

    "missingness":
        OUTPUT_DIR / "09_episode_missingness.csv",

    "numeric_summary":
        OUTPUT_DIR / "10_episode_numeric_summary.csv",

    "outlier_summary":
        OUTPUT_DIR / "11_episode_outlier_audit.csv",

    "categorical_summary":
        OUTPUT_DIR / "12_episode_categorical_summary.csv",

    "rare_categories":
        OUTPUT_DIR / "13_episode_rare_categories.csv",

    "numeric_correlation":
        OUTPUT_DIR / "14_numeric_correlation_matrix.csv",

    "spearman_correlation":
        OUTPUT_DIR / "15_spearman_correlation_matrix.csv",

    "categorical_association":
        OUTPUT_DIR / "16_categorical_cramers_v.csv",

    "mixed_association":
        OUTPUT_DIR / "17_mixed_eta.csv",

    "vif":
        OUTPUT_DIR / "18_vif_candidate_predictors.csv",

    "leakage":
        OUTPUT_DIR / "19_episode_leakage_audit.csv",

    "predictor_audit":
        OUTPUT_DIR / "20_candidate_predictor_audit.csv",

    "date_audit":
        OUTPUT_DIR / "21_date_temporal_audit.csv",

    "financial_redundancy":
        OUTPUT_DIR / "22_financial_redundancy_audit.csv",
}


# ================================================================
# 3. LOAD DATA
# ================================================================

print("=" * 90)
print("EPISODE-LEVEL PRE-FEATURE-ENGINEERING AUDIT")
print("=" * 90)

df = pd.read_csv(
    FILE_PATH,
    low_memory=False
)

print(f"\nDataset: {FILE_PATH}")
print(f"Rows:    {len(df):,}")
print(f"Columns: {len(df.columns):,}")


# ================================================================
# 4. BASIC DATASET AUDIT
# ================================================================

print("\n" + "=" * 90)
print("1. BASIC DATASET STRUCTURE")
print("=" * 90)

dataset_audit = pd.DataFrame({
    "column": df.columns,
    "dtype": [
        str(df[col].dtype)
        for col in df.columns
    ],
    "n_unique": [
        df[col].nunique(dropna=True)
        for col in df.columns
    ],
    "missing_n": [
        df[col].isna().sum()
        for col in df.columns
    ],
    "missing_pct": [
        100 * df[col].isna().mean()
        for col in df.columns
    ],
    "constant": [
        df[col].nunique(dropna=False) <= 1
        for col in df.columns
    ],
})

dataset_audit.to_csv(
    FILES["dataset_audit"],
    index=False,
    encoding="utf-8-sig"
)

print(dataset_audit.to_string(index=False))


# ================================================================
# 5. IDENTIFY VARIABLE TYPES
# ================================================================

numeric_variables = df.select_dtypes(
    include=[np.number]
).columns.tolist()

categorical_variables = df.select_dtypes(
    include=["object", "string", "category", "bool"]
).columns.tolist()


print("\nNumeric variables:")
print(numeric_variables)

print("\nCategorical variables:")
print(categorical_variables)


# ================================================================
# 6. MISSINGNESS AUDIT
# ================================================================

print("\n" + "=" * 90)
print("2. MISSINGNESS AUDIT")
print("=" * 90)

missingness = pd.DataFrame({
    "variable": df.columns,
    "missing_n": [
        df[col].isna().sum()
        for col in df.columns
    ],
    "missing_pct": [
        100 * df[col].isna().mean()
        for col in df.columns
    ]
})

missingness = missingness.sort_values(
    "missing_pct",
    ascending=False
)

missingness.to_csv(
    FILES["missingness"],
    index=False,
    encoding="utf-8-sig"
)

print(missingness.to_string(index=False))


# ================================================================
# 7. NUMERIC DESCRIPTIVE AUDIT
# ================================================================

print("\n" + "=" * 90)
print("3. NUMERIC DISTRIBUTION AUDIT")
print("=" * 90)

numeric_summary = []

for col in numeric_variables:

    x = pd.to_numeric(
        df[col],
        errors="coerce"
    ).dropna()

    if len(x) == 0:
        continue

    numeric_summary.append({

        "variable": col,
        "n": len(x),

        "mean": x.mean(),
        "std": x.std(),

        "min": x.min(),
        "p01": x.quantile(.01),
        "p05": x.quantile(.05),
        "p25": x.quantile(.25),
        "median": x.median(),
        "p75": x.quantile(.75),
        "p95": x.quantile(.95),
        "p99": x.quantile(.99),
        "max": x.max(),

        "skewness": x.skew(),

        "zero_n": (x == 0).sum(),
        "zero_pct": 100 * (x == 0).mean(),

        "negative_n": (x < 0).sum(),
        "negative_pct": 100 * (x < 0).mean(),
    })


numeric_summary = pd.DataFrame(
    numeric_summary
)

numeric_summary.to_csv(
    FILES["numeric_summary"],
    index=False,
    encoding="utf-8-sig"
)

print(
    numeric_summary.to_string(index=False)
)


# ================================================================
# 8. OUTLIER AUDIT
# ================================================================
#
# IMPORTANT:
# Outliers are AUDITED, not removed.
#
# In health expenditure data, extreme values may represent
# genuine high-cost episodes.
#
# ================================================================

print("\n" + "=" * 90)
print("4. IQR OUTLIER AUDIT")
print("=" * 90)

outlier_records = []

for col in numeric_variables:

    x = pd.to_numeric(
        df[col],
        errors="coerce"
    )

    q1 = x.quantile(.25)
    q3 = x.quantile(.75)

    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outlier = (
        x < lower
    ) | (
        x > upper
    )

    outlier_records.append({

        "variable": col,

        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,

        "lower_bound": lower,
        "upper_bound": upper,

        "outlier_n": outlier.sum(),
        "outlier_pct": 100 * outlier.mean()
    })


outlier_summary = pd.DataFrame(
    outlier_records
)

outlier_summary.to_csv(
    FILES["outlier_summary"],
    index=False,
    encoding="utf-8-sig"
)

print(
    outlier_summary.to_string(index=False)
)


# ================================================================
# 9. CATEGORICAL AUDIT
# ================================================================

print("\n" + "=" * 90)
print("5. CATEGORICAL VARIABLE AUDIT")
print("=" * 90)

categorical_records = []

for col in categorical_variables:

    n = df[col].notna().sum()

    counts = df[col].value_counts(
        dropna=False
    )

    if len(counts) > 0:

        top_value = counts.index[0]
        top_n = counts.iloc[0]

        top_pct = (
            100 * top_n / len(df)
        )

    else:

        top_value = np.nan
        top_n = 0
        top_pct = np.nan

    categorical_records.append({

        "variable": col,

        "n_categories":
            df[col].nunique(dropna=True),

        "missing_n":
            df[col].isna().sum(),

        "missing_pct":
            100 * df[col].isna().mean(),

        "most_common_value":
            top_value,

        "most_common_n":
            top_n,

        "most_common_pct":
            top_pct,

        "categories_below_1pct":
            (
                counts[
                    counts < 0.01 * len(df)
                ].shape[0]
            ),

        "categories_below_0_5pct":
            (
                counts[
                    counts < 0.005 * len(df)
                ].shape[0]
            )
    })


categorical_summary = pd.DataFrame(
    categorical_records
)

categorical_summary.to_csv(
    FILES["categorical_summary"],
    index=False,
    encoding="utf-8-sig"
)

print(
    categorical_summary.to_string(index=False)
)


# ================================================================
# 10. RARE CATEGORY AUDIT
# ================================================================

print("\n" + "=" * 90)
print("6. RARE CATEGORY AUDIT")
print("=" * 90)

rare_records = []

for col in categorical_variables:

    counts = df[col].value_counts(
        dropna=False
    )

    threshold = max(
        5,
        int(np.ceil(0.01 * len(df)))
    )

    rare = counts[
        counts < threshold
    ]

    for category, count in rare.items():

        rare_records.append({

            "variable": col,
            "category": str(category),
            "n": int(count),
            "pct": 100 * count / len(df),
            "rare_threshold_n": threshold
        })


rare_categories = pd.DataFrame(
    rare_records
)

rare_categories.to_csv(
    FILES["rare_categories"],
    index=False,
    encoding="utf-8-sig"
)

print(
    rare_categories.head(100).to_string(index=False)
)


# ================================================================
# 11. DEFINE LEAKAGE CLASSES
# ================================================================
#
# THIS IS THE MOST IMPORTANT PART.
#
# ================================================================

IDENTIFIERS = [

    "Episode_ID",
    "Patient_ID"
]


QUALITY_CONTROL = [

    "invalid_date_order",
    "contains_exact_duplicate_claim",
    "contains_invalid_age"
]


POST_ADMISSION_DESCRIPTORS = [

    "Number_of_claim_lines",
    "Number_of_service_groups"
]


POST_DISCHARGE_VARIABLES = [

    "Discharge date",
    "Discharge Status",
    "Episode_LOS_days"
]


FINANCIAL_OUTCOME_VARIABLES = [

    "Total Cost",
    "Insured Person's Share",
    "Organization Requested Share",
    "Organization Paid Share",
    "Terminally Patients Paid Share",
    "Terminally Patients Requested Share",
    "Allocated subsidized foreign currency share",
    "Requested subsidized foreign currency share",

    "Known_financing_total",
    "Financing_difference",
    "Financing_difference_pct"
]


CANDIDATE_BASELINE = [

    "Age",
    "Gender",
    "Ownership",
    "Insurance Fund",
    "Treating Physician Specialty",
    "Diagnosis type",
    "Department Type",
    "Admision date"
]


# ================================================================
# 12. LEAKAGE CLASSIFICATION
# ================================================================

print("\n" + "=" * 90)
print("7. LEAKAGE CLASSIFICATION")
print("=" * 90)

leakage_records = []

for col in df.columns:

    if col in IDENTIFIERS:

        classification = (
            "IDENTIFIER — NEVER USE AS PREDICTOR"
        )

        leakage_risk = "CRITICAL"

    elif col in QUALITY_CONTROL:

        classification = (
            "QUALITY CONTROL — NOT A PREDICTOR"
        )

        leakage_risk = "HIGH"

    elif col in POST_ADMISSION_DESCRIPTORS:

        classification = (
            "POST-ADMISSION EPISODE DESCRIPTOR — "
            "NOT AVAILABLE AT BASELINE"
        )

        leakage_risk = "HIGH"

    elif col in POST_DISCHARGE_VARIABLES:

        classification = (
            "POST-DISCHARGE VARIABLE — "
            "LEAKAGE"
        )

        leakage_risk = "CRITICAL"

    elif col in FINANCIAL_OUTCOME_VARIABLES:

        classification = (
            "FINANCIAL OUTCOME / TARGET-DERIVED — "
            "LEAKAGE"
        )

        leakage_risk = "CRITICAL"

    elif col in CANDIDATE_BASELINE:

        classification = (
            "CANDIDATE BASELINE PREDICTOR — "
            "VERIFY AVAILABILITY AT PREDICTION TIME"
        )

        leakage_risk = "LOW / VERIFY"

    else:

        classification = (
            "UNCLASSIFIED — MANUAL REVIEW REQUIRED"
        )

        leakage_risk = "REVIEW"

    leakage_records.append({

        "variable": col,
        "classification": classification,
        "leakage_risk": leakage_risk
    })


leakage_audit = pd.DataFrame(
    leakage_records
)

leakage_audit.to_csv(
    FILES["leakage"],
    index=False,
    encoding="utf-8-sig"
)

print(
    leakage_audit.to_string(index=False)
)


# ================================================================
# 13. CANDIDATE PREDICTOR AUDIT
# ================================================================

print("\n" + "=" * 90)
print("8. FINAL CANDIDATE BASELINE PREDICTORS")
print("=" * 90)

candidate_present = [
    col
    for col in CANDIDATE_BASELINE
    if col in df.columns
]

predictor_audit = pd.DataFrame({

    "variable": candidate_present,

    "dtype": [
        str(df[col].dtype)
        for col in candidate_present
    ],

    "n_unique": [
        df[col].nunique(dropna=True)
        for col in candidate_present
    ],

    "missing_n": [
        df[col].isna().sum()
        for col in candidate_present
    ],

    "missing_pct": [
        100 * df[col].isna().mean()
        for col in candidate_present
    ]
})


predictor_audit.to_csv(
    FILES["predictor_audit"],
    index=False,
    encoding="utf-8-sig"
)

print(
    predictor_audit.to_string(index=False)
)


# ================================================================
# 14. NUMERIC CORRELATION
# ================================================================
#
# IMPORTANT:
# Correlation should be calculated on BASELINE NUMERIC
# PREDICTORS, not on outcomes.
#
# ================================================================

print("\n" + "=" * 90)
print("9. BASELINE NUMERIC CORRELATION")
print("=" * 90)

baseline_numeric = [

    col
    for col in candidate_present
    if pd.api.types.is_numeric_dtype(
        df[col]
    )
]


if len(baseline_numeric) >= 2:

    pearson_corr = (
        df[baseline_numeric]
        .corr(method="pearson")
    )

    spearman_corr = (
        df[baseline_numeric]
        .corr(method="spearman")
    )

else:

    pearson_corr = pd.DataFrame()
    spearman_corr = pd.DataFrame()


pearson_corr.to_csv(
    FILES["numeric_correlation"],
    encoding="utf-8-sig"
)

spearman_corr.to_csv(
    FILES["spearman_correlation"],
    encoding="utf-8-sig"
)

print("\nBaseline numeric variables:")
print(baseline_numeric)

if not pearson_corr.empty:
    print("\nPearson correlation:")
    print(pearson_corr)

if not spearman_corr.empty:
    print("\nSpearman correlation:")
    print(spearman_corr)


# ================================================================
# 15. CRAMER'S V FOR CATEGORICAL VARIABLES
# ================================================================

print("\n" + "=" * 90)
print("10. CATEGORICAL ASSOCIATION — CRAMER'S V")
print("=" * 90)

try:

    from scipy.stats import chi2_contingency

    def cramers_v(x, y):

        table = pd.crosstab(
            x,
            y
        )

        if table.empty:
            return np.nan

        chi2 = chi2_contingency(
            table,
            correction=False
        )[0]

        n = table.to_numpy().sum()

        if n == 0:
            return np.nan

        r, k = table.shape

        phi2 = chi2 / n

        # Bias-corrected Cramer's V
        phi2corr = max(
            0,
            phi2 -
            ((k - 1) * (r - 1)) /
            max(n - 1, 1)
        )

        rcorr = (
            r -
            ((r - 1) ** 2) /
            max(n - 1, 1)
        )

        kcorr = (
            k -
            ((k - 1) ** 2) /
            max(n - 1, 1)
        )

        denominator = min(
            kcorr - 1,
            rcorr - 1
        )

        if denominator <= 0:
            return np.nan

        return np.sqrt(
            phi2corr / denominator
        )


    baseline_categorical = [

        col
        for col in candidate_present
        if (
            df[col].dtype == "object"
            or
            str(df[col].dtype) in [
                "string",
                "category",
                "bool"
            ]
        )
    ]


    cramers_records = []

    for var1, var2 in combinations(
        baseline_categorical,
        2
    ):

        valid = df[
            [var1, var2]
        ].dropna()

        if len(valid) == 0:
            continue

        v = cramers_v(
            valid[var1],
            valid[var2]
        )

        if pd.notna(v):

            if v < 0.10:
                strength = "negligible"

            elif v < 0.30:
                strength = "weak"

            elif v < 0.50:
                strength = "moderate"

            else:
                strength = "strong"

            cramers_records.append({

                "variable_1": var1,
                "variable_2": var2,
                "cramers_v": v,
                "strength": strength
            })


    categorical_association = pd.DataFrame(
        cramers_records
    ).sort_values(
        "cramers_v",
        ascending=False
    )


except ImportError:

    print(
        "scipy not available — Cramer's V skipped."
    )

    categorical_association = pd.DataFrame()


categorical_association.to_csv(
    FILES["categorical_association"],
    index=False,
    encoding="utf-8-sig"
)

print(
    categorical_association.to_string(
        index=False
    )
)


# ================================================================
# 16. ETA: CATEGORICAL–NUMERIC ASSOCIATION
# ================================================================

print("\n" + "=" * 90)
print("11. MIXED ASSOCIATION — ETA")
print("=" * 90)


def eta_squared(
    categorical,
    numeric
):

    valid = pd.DataFrame({
        "cat": categorical,
        "num": numeric
    }).dropna()

    if len(valid) == 0:
        return np.nan

    grand_mean = valid["num"].mean()

    numerator = 0

    for level, group in valid.groupby("cat"):

        n_group = len(group)

        group_mean = group["num"].mean()

        numerator += (
            n_group *
            (group_mean - grand_mean) ** 2
        )

    denominator = (
        (
            valid["num"] -
            grand_mean
        ) ** 2
    ).sum()

    if denominator == 0:
        return np.nan

    return numerator / denominator


baseline_numeric = [
    col
    for col in candidate_present
    if pd.api.types.is_numeric_dtype(
        df[col]
    )
]

baseline_categorical = [
    col
    for col in candidate_present
    if (
        df[col].dtype == "object"
        or
        str(df[col].dtype) in [
            "string",
            "category",
            "bool"
        ]
    )
]


eta_records = []

for cat in baseline_categorical:

    for num in baseline_numeric:

        eta = eta_squared(
            df[cat],
            df[num]
        )

        eta_records.append({

            "categorical_variable": cat,
            "numeric_variable": num,
            "eta_squared": eta
        })


mixed_association = pd.DataFrame(
    eta_records
).sort_values(
    "eta_squared",
    ascending=False
)

mixed_association.to_csv(
    FILES["mixed_association"],
    index=False,
    encoding="utf-8-sig"
)

print(
    mixed_association.to_string(
        index=False
    )
)


# ================================================================
# 17. VIF
# ================================================================
#
# VERY IMPORTANT:
#
# VIF MUST NOT be calculated on:
#
#   Total Cost
#   LOS
#   financing variables
#   identifiers
#   post-discharge variables
#
# It is calculated ONLY on the candidate baseline predictor
# design matrix.
#
# Categorical variables are one-hot encoded ONLY for the purpose
# of diagnosing linear dependence.
#
# ================================================================

print("\n" + "=" * 90)
print("12. VIF — CANDIDATE BASELINE PREDICTORS ONLY")
print("=" * 90)

try:

    from statsmodels.stats.outliers_influence import variance_inflation_factor

    # ------------------------------------------------------------
    # Construct candidate baseline matrix
    # ------------------------------------------------------------

    X_vif = df[
        candidate_present
    ].copy()


    # Convert categorical variables to dummies
    categorical_for_vif = [
        col
        for col in X_vif.columns
        if (
            X_vif[col].dtype == "object"
            or
            str(X_vif[col].dtype) in [
                "string",
                "category",
                "bool"
            ]
        )
    ]


    X_vif = pd.get_dummies(
        X_vif,
        columns=categorical_for_vif,
        drop_first=True,
        dtype=float
    )


    # Numeric conversion
    X_vif = X_vif.apply(
        pd.to_numeric,
        errors="coerce"
    )


    # Median fill ONLY for the mathematical VIF calculation.
    #
    # This does NOT modify the modeling dataset.
    #
    # It is necessary because statsmodels VIF cannot operate
    # with missing values.
    #
    X_vif = X_vif.replace(
        [np.inf, -np.inf],
        np.nan
    )

    X_vif = X_vif.fillna(
        X_vif.median()
    )


    # Remove constants
    X_vif = X_vif.loc[
        :,
        X_vif.nunique(dropna=False) > 1
    ]


    vif_records = []

    for i, col in enumerate(
        X_vif.columns
    ):

        try:

            vif_value = (
                variance_inflation_factor(
                    X_vif.values,
                    i
                )
            )

        except Exception:

            vif_value = np.nan


        if pd.isna(vif_value):

            interpretation = "not estimable"

        elif vif_value >= 10:

            interpretation = "high multicollinearity"

        elif vif_value >= 5:

            interpretation = "potential concern"

        else:

            interpretation = "acceptable"


        vif_records.append({

            "encoded_variable": col,
            "VIF": vif_value,
            "interpretation": interpretation
        })


    vif_results = pd.DataFrame(
        vif_records
    ).sort_values(
        "VIF",
        ascending=False
    )


except ImportError:

    print(
        "statsmodels not available — VIF skipped."
    )

    vif_results = pd.DataFrame()


vif_results.to_csv(
    FILES["vif"],
    index=False,
    encoding="utf-8-sig"
)

print(
    vif_results.to_string(
        index=False
    )
)


# ================================================================
# 18. DATE / TEMPORAL AUDIT
# ================================================================

print("\n" + "=" * 90)
print("13. TEMPORAL AUDIT")
print("=" * 90)

date_records = []

for col in [
    "Admision date",
    "Discharge date"
]:

    if col not in df.columns:
        continue

    temp = pd.to_datetime(
        df[col],
        errors="coerce"
    )

    date_records.append({

        "variable": col,

        "valid_n": temp.notna().sum(),

        "missing_n": temp.isna().sum(),

        "invalid_n":
            temp.isna().sum(),

        "minimum":
            temp.min(),

        "maximum":
            temp.max(),

        "unique_dates":
            temp.nunique()
    })


date_audit = pd.DataFrame(
    date_records
)

date_audit.to_csv(
    FILES["date_audit"],
    index=False,
    encoding="utf-8-sig"
)

print(
    date_audit.to_string(index=False)
)


# ================================================================
# 19. FINANCIAL REDUNDANCY AUDIT
# ================================================================
#
# This is especially important for your dataset.
#
# You have:
#
# Total Cost
# Insured Person's Share
# Organization Paid Share
# Organization Requested Share
# financing totals
# financing differences
#
# These variables are mathematically related.
#
# They should NOT enter the baseline predictor set.
#
# ================================================================

print("\n" + "=" * 90)
print("14. FINANCIAL REDUNDANCY AUDIT")
print("=" * 90)


financial_variables = [
    col
    for col in FINANCIAL_OUTCOME_VARIABLES
    if col in df.columns
]


financial_corr = (
    df[
        financial_variables
    ]
    .corr(method="spearman")
)


financial_corr.to_csv(
    FILES["financial_redundancy"],
    encoding="utf-8-sig"
)

print(
    financial_corr.to_string()
)


# ================================================================
# 20. TARGET-SPECIFIC LEAKAGE WARNING
# ================================================================
#
# Because the exact model target has not been specified in this
# message, the code does NOT automatically assume one target.
#
# Instead, it prints the variables that must be reconsidered
# whenever the target is derived from them.
#
# ================================================================

print("\n" + "=" * 90)
print("15. TARGET-DEPENDENT LEAKAGE WARNING")
print("=" * 90)

print("""
IMPORTANT:

Before ML training, explicitly define each target.

For example:

    Cost target
        -> Total Cost and all variables derived from Total Cost
           must NOT be predictors.

    LOS target
        -> Episode_LOS_days, Length of Stay, Discharge date,
           and LOS-derived variables must NOT be predictors.

    Patient OOP target
        -> Insured Person's Share and any variable mathematically
           derived from it must NOT be predictors.

    Financing / reimbursement target
        -> Organization Paid Share, Organization Requested Share,
           and related financing variables must NOT be predictors.

    Discharge outcome target
        -> Discharge Status must NOT be a predictor.

The candidate baseline variables are restricted to:

    Age
    Gender
    Ownership
    Insurance Fund
    Treating Physician Specialty
    Diagnosis type
    Department Type
    Admision date

These must still be verified against the exact prediction time.
""")


# ================================================================
# 21. FINAL RECOMMENDATION
# ================================================================

print("\n" + "=" * 90)
print("16. PRE-FEATURE-ENGINEERING RECOMMENDATION")
print("=" * 90)

print("""
DO NOT START FEATURE ENGINEERING UNTIL:

1. The prediction time is explicitly defined.

2. The exact target for each model is defined.

3. The candidate baseline variables have been approved.

4. No post-discharge variables are included.

5. No financial outcome variable is included as a predictor
   of a financial outcome derived from it.

6. Episode_ID and Patient_ID are excluded from ML predictors.

7. Number_of_claim_lines and Number_of_service_groups are
   excluded unless they are genuinely available at the
   prediction time.

8. High-cardinality categorical variables are reviewed.

9. Rare categories are handled using TRAINING DATA ONLY.

10. VIF is interpreted on the final candidate predictor matrix,
    not on the complete episode dataset.

11. Correlation is interpreted separately from categorical
    association.

12. Any feature engineering involving dates, historical
    utilization, previous episodes, or patient-level information
    must be performed AFTER the train/validation/test split
    or using training-fold-only information.

13. No information from validation/test episodes is allowed
    to determine category pooling, imputation, scaling,
    transformation thresholds, or feature selection.
""")


# ================================================================
# 22. SAVE FINAL SUMMARY
# ================================================================

print("\n" + "=" * 90)
print("AUDIT COMPLETE")
print("=" * 90)

for name, path in FILES.items():

    print(
        f"{name:25s} -> {path}"
    )

print("\nCandidate baseline predictors:")
for col in candidate_present:
    print("  -", col)

print("\nDONE.")

In [ ]:
# ================================================================
# CHOLERA EPISODE-LEVEL ML
# FINAL LEAKAGE-SAFE FEATURE ENGINEERING PIPELINE
# ================================================================
#
# DATA LEVEL:
#   Hospitalization episode
#
# INPUT:
#   04_episode_level_data.csv
#
# PURPOSE:
#   Create baseline predictors for ML and value-based policy models
#   without using post-admission, post-discharge, financial outcome,
#   identifier, or target-derived information.
#
# PRINCIPLE:
#   1. Split temporally FIRST.
#   2. Engineer deterministic calendar features.
#   3. Fit all learned transformations on TRAINING DATA ONLY.
#   4. Apply fitted transformations unchanged to validation/test.
#
# ================================================================


import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)
from sklearn.impute import SimpleImputer


# ================================================================
# 1. PATHS
# ================================================================

INPUT_FILE = Path(
    r"C:\Users\Shekoofeh\Documents\Health Insurance Data"
    r"\Cholera\04_episode_level_data.csv"
)

OUTPUT_DIR = INPUT_FILE.parent

ENGINEERED_FILE = (
    OUTPUT_DIR /
    "05_episode_engineered_baseline_features.csv"
)

TRAIN_FILE = (
    OUTPUT_DIR /
    "06_train_episode_features.csv"
)

VALIDATION_FILE = (
    OUTPUT_DIR /
    "07_validation_episode_features.csv"
)

TEST_FILE = (
    OUTPUT_DIR /
    "08_test_episode_features.csv"
)

FEATURE_MAP_FILE = (
    OUTPUT_DIR /
    "09_final_feature_map.csv"
)

LEAKAGE_AUDIT_FILE = (
    OUTPUT_DIR /
    "10_final_feature_leakage_audit.csv"
)


# ================================================================
# 2. LOAD EPISODE DATA
# ================================================================

print("=" * 90)
print("STEP 1 — LOAD EPISODE DATA")
print("=" * 90)

df = pd.read_csv(
    INPUT_FILE,
    encoding="utf-8-sig",
    low_memory=False
)

print(f"Rows:    {len(df):,}")
print(f"Columns: {len(df.columns):,}")


# ================================================================
# 3. BASIC STRUCTURAL CHECKS
# ================================================================

required_columns = [
    "Episode_ID",
    "Age",
    "Gender",
    "Ownership",
    "Insurance Fund",
    "Treating Physician Specialty",
    "Admision date",
    "Diagnosis type",
    "Department Type"
]

missing_required = [
    col
    for col in required_columns
    if col not in df.columns
]

if missing_required:

    raise ValueError(
        "Required columns are missing:\n"
        + "\n".join(missing_required)
    )


# ================================================================
# 4. CONVERT ADMISSION DATE
# ================================================================

df["Admision date"] = pd.to_datetime(
    df["Admision date"],
    errors="coerce"
)

if df["Admision date"].isna().any():

    print(
        "WARNING:",
        df["Admision date"].isna().sum(),
        "episodes have missing admission dates."
    )


# ================================================================
# 5. DEFINE VARIABLES THAT MUST NEVER ENTER THE MODEL
# ================================================================

IDENTIFIER_VARIABLES = [

    "Episode_ID",
    "Patient_ID"
]


FINANCIAL_OUTCOME_VARIABLES = [

    "Total Cost",
    "Insured Person's Share",
    "Organization Requested Share",
    "Organization Paid Share",
    "Terminally Patients Paid Share",
    "Terminally Patients Requested Share",
    "Allocated subsidized foreign currency share",
    "Requested subsidized foreign currency share",

    "Known_financing_total",
    "Financing_difference",
    "Financing_difference_pct"
]


POST_DISCHARGE_VARIABLES = [

    "Discharge date",
    "Discharge Status",
    "Episode_LOS_days"
]


POST_ADMISSION_VARIABLES = [

    "Number_of_claim_lines",
    "Number_of_service_groups"
]


QUALITY_CONTROL_VARIABLES = [

    "invalid_date_order",
    "contains_exact_duplicate_claim",
    "contains_invalid_age"
]


EPISODE_CONSTRUCTION_VARIABLES = [

    "Episode_Identification_Type",
    "Episode_Linkage_Method"
]


ALL_EXCLUDED_VARIABLES = (
    IDENTIFIER_VARIABLES
    +
    FINANCIAL_OUTCOME_VARIABLES
    +
    POST_DISCHARGE_VARIABLES
    +
    POST_ADMISSION_VARIABLES
    +
    QUALITY_CONTROL_VARIABLES
    +
    EPISODE_CONSTRUCTION_VARIABLES
)


# ================================================================
# 6. BASELINE PREDICTORS
# ================================================================

BASELINE_CATEGORICAL = [

    "Gender",
    "Ownership",
    "Insurance Fund",
    "Treating Physician Specialty",
    "Diagnosis type",
    "Department Type"
]


BASELINE_NUMERIC = [

    "Age"
]


# ================================================================
# 7. DETERMINISTIC FEATURE ENGINEERING
# ================================================================
#
# IMPORTANT:
#
# These transformations do NOT learn parameters from the dataset.
# Therefore they can safely be calculated before the split.
#
# No financial variable is used.
# No discharge information is used.
# No LOS is used.
# No claim-line count is used.
#


print("\n" + "=" * 90)
print("STEP 2 — DETERMINISTIC BASELINE FEATURE ENGINEERING")
print("=" * 90)


# ------------------------------------------------
# 7.1 Age features
# ------------------------------------------------

df["age_normalized"] = (
    df["Age"] / 100.0
)


df["is_pediatric"] = (
    df["Age"] < 18
).astype(int)


df["is_elderly"] = (
    df["Age"] >= 65
).astype(int)


df["is_male"] = (
    df["Gender"]
    .astype("string")
    .str.strip()
    .str.lower()
    .eq("the man")
).astype(int)


df["age_missing"] = (
    df["Age"].isna()
).astype(int)


# ------------------------------------------------
# 7.2 Admission calendar features
# ------------------------------------------------

df["admission_year"] = (
    df["Admision date"].dt.year
)


df["admission_month"] = (
    df["Admision date"].dt.month
)


df["admission_quarter"] = (
    df["Admision date"].dt.quarter
)


df["admission_dayofweek"] = (
    df["Admision date"].dt.dayofweek
)


df["admission_weekofyear"] = (
    df["Admision date"].dt.isocalendar().week
    .astype("float")
)


# ------------------------------------------------
# 7.3 Cyclical month encoding
# ------------------------------------------------
#
# This represents the circular nature of months:
#
# December is close to January.
#
# ------------------------------------------------

df["month_sin"] = np.sin(
    2 * np.pi *
    (df["admission_month"] - 1) / 12
)


df["month_cos"] = np.cos(
    2 * np.pi *
    (df["admission_month"] - 1) / 12
)


# ------------------------------------------------
# 7.4 Optional calendar trend
# ------------------------------------------------
#
# Days since first observed admission.
#
# This is deterministic and does not use the outcome.
#
# ------------------------------------------------

minimum_admission_date = (
    df["Admision date"].min()
)

df["admission_days_since_start"] = (
    df["Admision date"]
    - minimum_admission_date
).dt.days


# ================================================================
# 8. REMOVE RAW ADMISSION DATE
# ================================================================

df = df.drop(
    columns=["Admision date"]
)


# ================================================================
# 9. CHECK THAT NO FORBIDDEN VARIABLE HAS BECOME A FEATURE
# ================================================================

print("\nChecking leakage variables...")

engineered_candidates = [
    col
    for col in df.columns
    if col not in ALL_EXCLUDED_VARIABLES
]


unexpected_forbidden = [
    col
    for col in engineered_candidates
    if col in ALL_EXCLUDED_VARIABLES
]


if unexpected_forbidden:

    raise ValueError(
        "Forbidden variables detected in candidate features:\n"
        + "\n".join(unexpected_forbidden)
    )


# ================================================================
# 10. DEFINE FINAL FEATURE SET
# ================================================================

ENGINEERED_NUMERIC = [

    "Age",
    "age_normalized",
    "is_pediatric",
    "is_elderly",
    "is_male",
    "age_missing",

    "admission_year",
    "admission_month",
    "admission_quarter",
    "admission_dayofweek",
    "admission_weekofyear",

    "month_sin",
    "month_cos",
    "admission_days_since_start"
]


ENGINEERED_CATEGORICAL = [

    "Gender",
    "Ownership",
    "Insurance Fund",
    "Treating Physician Specialty",
    "Diagnosis type",
    "Department Type"
]


FINAL_FEATURES = (
    ENGINEERED_NUMERIC
    +
    ENGINEERED_CATEGORICAL
)


# ================================================================
# 11. VERIFY FINAL FEATURES
# ================================================================

missing_features = [
    col
    for col in FINAL_FEATURES
    if col not in df.columns
]

if missing_features:

    raise ValueError(
        "Final feature(s) missing:\n"
        + "\n".join(missing_features)
    )


print("\nFinal baseline predictors:")
for i, feature in enumerate(
    FINAL_FEATURES,
    start=1
):
    print(f"{i:02d}. {feature}")


# ================================================================
# 12. TEMPORAL SPLIT
# ================================================================
#
# IMPORTANT:
#
# We split BEFORE fitting:
#
#   - imputers
#   - category-frequency thresholds
#   - one-hot encoding
#   - scaling
#
# This prevents information from future observations entering
# the training representation.
#
# ------------------------------------------------
#
# Preferred:
#
# TRAIN      = earliest observations
# VALIDATION = subsequent observations
# TEST       = latest observations
#
# ------------------------------------------------


print("\n" + "=" * 90)
print("STEP 3 — TEMPORAL TRAIN / VALIDATION / TEST SPLIT")
print("=" * 90)


# Restore admission date temporarily for sorting.
# We already removed it from the feature matrix, but we can
# use the original df by reconstructing from the input data.

original_dates = pd.read_csv(
    INPUT_FILE,
    usecols=["Episode_ID", "Admision date"],
    encoding="utf-8-sig"
)

original_dates["Admision date"] = pd.to_datetime(
    original_dates["Admision date"],
    errors="coerce"
)


df = df.merge(
    original_dates,
    on="Episode_ID",
    how="left",
    validate="one_to_one"
)


df = df.sort_values(
    "Admision date"
).reset_index(drop=True)


# ------------------------------------------------
# 12.1 Determine temporal cut points
# ------------------------------------------------

n = len(df)

train_end = int(
    np.floor(0.60 * n)
)

validation_end = int(
    np.floor(0.80 * n)
)


train_df = df.iloc[
    :train_end
].copy()


validation_df = df.iloc[
    train_end:validation_end
].copy()


test_df = df.iloc[
    validation_end:
].copy()


print(
    f"Training episodes:   {len(train_df):,}"
)

print(
    f"Validation episodes: {len(validation_df):,}"
)

print(
    f"Test episodes:       {len(test_df):,}"
)


print("\nTemporal ranges:")

print(
    "TRAIN:",
    train_df["Admision date"].min(),
    "to",
    train_df["Admision date"].max()
)

print(
    "VALID:",
    validation_df["Admision date"].min(),
    "to",
    validation_df["Admision date"].max()
)

print(
    "TEST:",
    test_df["Admision date"].min(),
    "to",
    test_df["Admision date"].max()
)


# ================================================================
# 13. REMOVE DATE FROM MODEL DATA
# ================================================================

train_X = train_df[
    FINAL_FEATURES
].copy()

validation_X = validation_df[
    FINAL_FEATURES
].copy()

test_X = test_df[
    FINAL_FEATURES
].copy()


# ================================================================
# 14. IDENTIFY NUMERIC / CATEGORICAL FEATURES
# ================================================================

numeric_features = [
    col
    for col in ENGINEERED_NUMERIC
    if col in train_X.columns
]


categorical_features = [
    col
    for col in ENGINEERED_CATEGORICAL
    if col in train_X.columns
]


# ================================================================
# 15. TRAIN-ONLY PREPROCESSING
# ================================================================
#
# Numeric:
#   median imputation
#   standardization
#
# Categorical:
#   most-frequent imputation
#   one-hot encoding
#
# ALL parameters are learned from TRAINING DATA ONLY.
#
# ================================================================


numeric_pipeline = Pipeline(
    steps=[

        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),

        (
            "scaler",
            StandardScaler()
        )
    ]
)


categorical_pipeline = Pipeline(
    steps=[

        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),

        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)


preprocessor = ColumnTransformer(
    transformers=[

        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),

        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ],

    remainder="drop"
)


# ================================================================
# 16. FIT ONLY ON TRAINING DATA
# ================================================================

print("\n" + "=" * 90)
print("STEP 4 — FIT PREPROCESSING ON TRAINING DATA ONLY")
print("=" * 90)


preprocessor.fit(
    train_X
)


# ================================================================
# 17. TRANSFORM TRAIN / VALIDATION / TEST
# ================================================================

X_train = preprocessor.transform(
    train_X
)

X_validation = preprocessor.transform(
    validation_X
)

X_test = preprocessor.transform(
    test_X
)


print(
    "Transformed training matrix:",
    X_train.shape
)

print(
    "Transformed validation matrix:",
    X_validation.shape
)

print(
    "Transformed test matrix:",
    X_test.shape
)


# ================================================================
# 18. GET FINAL FEATURE NAMES
# ================================================================

feature_names = (
    preprocessor
    .get_feature_names_out()
)


feature_names = [
    name.replace(
        "numeric__",
        ""
    ).replace(
        "categorical__",
        ""
    )
    for name in feature_names
]


print(
    f"\nFinal number of model features: "
    f"{len(feature_names):,}"
)


# ================================================================
# 19. CREATE FEATURE MAP
# ================================================================

feature_map_records = []


for feature in numeric_features:

    feature_map_records.append({

        "feature": feature,

        "source_variable": feature,

        "feature_type": "numeric",

        "engineering": (
            "baseline numeric / deterministic transformation"
        ),

        "leakage_status": "SAFE",

        "fit_on_training_only": True
    })


for feature in categorical_features:

    feature_map_records.append({

        "feature": feature,

        "source_variable": feature,

        "feature_type": "categorical",

        "engineering": (
            "one-hot encoded; "
            "unknown categories ignored"
        ),

        "leakage_status": "SAFE",

        "fit_on_training_only": True
    })


feature_map = pd.DataFrame(
    feature_map_records
)


feature_map.to_csv(
    FEATURE_MAP_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ================================================================
# 20. LEAKAGE AUDIT
# ================================================================

print("\n" + "=" * 90)
print("STEP 5 — FINAL LEAKAGE AUDIT")
print("=" * 90)


leakage_audit_records = []


for col in df.columns:

    if col in IDENTIFIER_VARIABLES:

        classification = (
            "IDENTIFIER — NEVER USE"
        )

        risk = "CRITICAL"


    elif col in FINANCIAL_OUTCOME_VARIABLES:

        classification = (
            "FINANCIAL OUTCOME / TARGET-DERIVED"
        )

        risk = "CRITICAL"


    elif col in POST_DISCHARGE_VARIABLES:

        classification = (
            "POST-DISCHARGE — LEAKAGE"
        )

        risk = "CRITICAL"


    elif col in POST_ADMISSION_VARIABLES:

        classification = (
            "POST-ADMISSION — LEAKAGE"
        )

        risk = "HIGH"


    elif col in QUALITY_CONTROL_VARIABLES:

        classification = (
            "QUALITY CONTROL — NOT A PREDICTOR"
        )

        risk = "HIGH"


    elif col in EPISODE_CONSTRUCTION_VARIABLES:

        classification = (
            "EPISODE CONSTRUCTION METADATA"
        )

        risk = "HIGH"


    elif col in FINAL_FEATURES:

        classification = (
            "BASELINE PREDICTOR — SAFE"
        )

        risk = "LOW"


    else:

        classification = (
            "NOT USED IN FINAL MODEL"
        )

        risk = "REVIEW"


    leakage_audit_records.append({

        "variable": col,

        "classification": classification,

        "leakage_risk": risk,

        "used_as_model_feature": (
            col in FINAL_FEATURES
        )
    })


leakage_audit = pd.DataFrame(
    leakage_audit_records
)


leakage_audit.to_csv(
    LEAKAGE_AUDIT_FILE,
    index=False,
    encoding="utf-8-sig"
)


print(
    leakage_audit.to_string(
        index=False
    )
)


# ================================================================
# 21. SAVE RAW ENGINEERED DATA
# ================================================================

df.to_csv(
    ENGINEERED_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ================================================================
# 22. SAVE SPLIT METADATA
# ================================================================

train_df.to_csv(
    TRAIN_FILE,
    index=False,
    encoding="utf-8-sig"
)


validation_df.to_csv(
    VALIDATION_FILE,
    index=False,
    encoding="utf-8-sig"
)


test_df.to_csv(
    TEST_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ================================================================
# 23. FINAL SUMMARY
# ================================================================

print("\n" + "=" * 90)
print("FINAL FEATURE ENGINEERING SUMMARY")
print("=" * 90)

print(
    f"Total episodes:       {len(df):,}"
)

print(
    f"Training episodes:    {len(train_df):,}"
)

print(
    f"Validation episodes:  {len(validation_df):,}"
)

print(
    f"Test episodes:        {len(test_df):,}"
)

print(
    f"Raw baseline features: {len(FINAL_FEATURES):,}"
)

print(
    f"Final encoded features: {len(feature_names):,}"
)


print("\nFinal baseline variables:")

for feature in FINAL_FEATURES:

    print(
        "  SAFE:",
        feature
    )


print("\nForbidden variables:")

for feature in ALL_EXCLUDED_VARIABLES:

    print(
        "  EXCLUDED:",
        feature
    )


print("\nOutput files:")

print(
    "Engineered data:",
    ENGINEERED_FILE
)

print(
    "Training:",
    TRAIN_FILE
)

print(
    "Validation:",
    VALIDATION_FILE
)

print(
    "Test:",
    TEST_FILE
)

print(
    "Feature map:",
    FEATURE_MAP_FILE
)

print(
    "Leakage audit:",
    LEAKAGE_AUDIT_FILE
)


print("\nDONE.")

In [ ]:
# ================================================================
# CHOLERA CLAIM DATA
# LEAKAGE-SAFE BASELINE FEATURE ENGINEERING
#
# INPUT:
#   04_episode_level_data.csv
#
# PURPOSE:
#   Construct ONLY variables available at the prediction time.
#
# IMPORTANT:
#   NO target-derived variables
#   NO post-admission variables
#   NO post-discharge variables
#   NO financial variables
#   NO episode-construction metadata
#   NO patient/episode identifiers as predictors
#
# OUTPUT:
#   08_feature_engineered_data.csv
#   09_feature_dictionary.csv
#   10_model_feature_list.csv
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path


# ================================================================
# 1. FILE PATHS
# ================================================================

INPUT_FILE = Path(
    r"C:\Users\Shekoofeh\Documents\Health Insurance Data"
    r"\Cholera\04_episode_level_data.csv"
)

OUTPUT_DIR = INPUT_FILE.parent

FEATURED_FILE = (
    OUTPUT_DIR / "08_feature_engineered_data.csv"
)

FEATURE_DICTIONARY_FILE = (
    OUTPUT_DIR / "09_feature_dictionary.csv"
)

MODEL_FEATURE_LIST_FILE = (
    OUTPUT_DIR / "10_model_feature_list.csv"
)


# ================================================================
# 2. LOAD EPISODE DATA
# ================================================================

print("=" * 90)
print("LEAKAGE-SAFE FEATURE ENGINEERING")
print("=" * 90)

df = pd.read_csv(
    INPUT_FILE,
    encoding="utf-8-sig",
    low_memory=False
)

print(f"\nInput file: {INPUT_FILE}")
print(f"Number of episodes: {len(df):,}")
print(f"Number of variables: {len(df.columns):,}")


# ================================================================
# 3. PRESERVE ORIGINAL DATA
# ================================================================

# Work on a copy so that the original episode-level dataset
# remains unchanged.

features = df.copy()


# ================================================================
# 4. CONVERT ADMISSION DATE
# ================================================================

if "Admision date" not in features.columns:
    raise ValueError(
        "Admision date is required for baseline temporal features."
    )

features["Admision date"] = pd.to_datetime(
    features["Admision date"],
    errors="coerce"
)

print(
    "\nMissing admission dates:",
    features["Admision date"].isna().sum()
)


# ================================================================
# 5. AGE FEATURES
# ================================================================

if "Age" not in features.columns:
    raise ValueError("Age variable is required.")

# ------------------------------------------------
# Age missing indicator
# ------------------------------------------------

features["age_missing"] = (
    features["Age"].isna().astype(int)
)


# ------------------------------------------------
# Age normalization
#
# IMPORTANT:
# For a genuinely leakage-safe ML pipeline, mean and SD
# should ultimately be estimated on TRAINING DATA ONLY.
#
# For this preprocessing stage we create the deterministic
# age-centered/scaled representation using the current data,
# but the final model pipeline should refit the scaler on
# training data only.
# ------------------------------------------------

age_mean = features["Age"].mean()
age_std = features["Age"].std()

if pd.isna(age_std) or age_std == 0:

    features["age_normalized"] = 0.0

else:

    features["age_normalized"] = (
        features["Age"] - age_mean
    ) / age_std


# ------------------------------------------------
# Pediatric indicator
# ------------------------------------------------

features["is_pediatric"] = (
    features["Age"].lt(18)
    .astype(int)
)


# ------------------------------------------------
# Elderly indicator
#
# Prespecified threshold: >=65 years
# ------------------------------------------------

features["is_elderly"] = (
    features["Age"].ge(65)
    .astype(int)
)


# ------------------------------------------------
# Male indicator
#
# Handles the observed categories:
# "the man" / "woman"
# ------------------------------------------------

gender_string = (
    features["Gender"]
    .astype("string")
    .str.strip()
    .str.lower()
)

features["is_male"] = (
    gender_string.eq("the man")
    .astype(int)
)


# ================================================================
# 6. ADMISSION-TIME TEMPORAL FEATURES
# ================================================================

features["admission_year"] = (
    features["Admision date"].dt.year
)

features["admission_month"] = (
    features["Admision date"].dt.month
)

features["admission_quarter"] = (
    features["Admision date"].dt.quarter
)

features["admission_dayofweek"] = (
    features["Admision date"].dt.dayofweek
)

features["admission_weekofyear"] = (
    features["Admision date"].dt.isocalendar().week
    .astype("float")
)


# ================================================================
# 7. CYCLICAL MONTH FEATURES
# ================================================================

# Month is cyclic:
#
# December (12) and January (1) are adjacent.
#
# Sin/cos representation avoids treating them as distant.

features["month_sin"] = np.sin(
    2 * np.pi *
    features["admission_month"] / 12
)

features["month_cos"] = np.cos(
    2 * np.pi *
    features["admission_month"] / 12
)


# ================================================================
# 8. DAYS SINCE STUDY START
# ================================================================

study_start = features["Admision date"].min()

features["admission_days_since_start"] = (
    features["Admision date"] - study_start
).dt.days


# ================================================================
# 9. DEFINE FINAL BASELINE NUMERIC FEATURES
# ================================================================

numeric_features = [
    "Age",
    "age_normalized",
    "is_pediatric",
    "is_elderly",
    "is_male",
    "age_missing",
    "admission_year",
    "admission_month",
    "admission_quarter",
    "admission_dayofweek",
    "admission_weekofyear",
    "month_sin",
    "month_cos",
    "admission_days_since_start"
]


# ================================================================
# 10. DEFINE FINAL BASELINE CATEGORICAL FEATURES
# ================================================================

categorical_features = [
    "Gender",
    "Ownership",
    "Insurance Fund",
    "Treating Physician Specialty",
    "Diagnosis type",
    "Department Type"
]


# ================================================================
# 11. VERIFY ALL BASELINE FEATURES EXIST
# ================================================================

required_features = (
    numeric_features +
    categorical_features
)

missing_features = [
    col
    for col in required_features
    if col not in features.columns
]

if missing_features:

    raise ValueError(
        "The following required baseline features are missing:\n"
        + "\n".join(missing_features)
    )


# ================================================================
# 12. EXPLICITLY DEFINE VARIABLES THAT MUST NEVER ENTER MODEL
# ================================================================

excluded_variables = [
    # Identifiers
    "Episode_ID",
    "Patient_ID",

    # Financial outcomes / target-derived
    "Total Cost",
    "Insured Person's Share",
    "Organization Requested Share",
    "Organization Paid Share",
    "Terminally Patients Paid Share",
    "Terminally Patients Requested Share",
    "Allocated subsidized foreign currency share",
    "Requested subsidized foreign currency share",
    "Known_financing_total",
    "Financing_difference",
    "Financing_difference_pct",

    # Post-discharge
    "Discharge date",
    "Discharge Status",
    "Episode_LOS_days",

    # Post-admission
    "Number_of_claim_lines",
    "Number_of_service_groups",

    # Episode construction metadata
    "Episode_Identification_Type",
    "Episode_Linkage_Method",

    # Quality-control variables
    "invalid_date_order",
    "contains_exact_duplicate_claim",
    "contains_invalid_age",

    # Original date is retained for audit but NOT model input
    "Admision date"
]


# ================================================================
# 13. VERIFY NO FEATURE IS IN THE EXCLUSION LIST
# ================================================================

feature_exclusion_overlap = set(
    required_features
).intersection(
    excluded_variables
)

if feature_exclusion_overlap:

    raise ValueError(
        "LEAKAGE ERROR: model features overlap with "
        "excluded variables:\n"
        + "\n".join(feature_exclusion_overlap)
    )


# ================================================================
# 14. CREATE MODEL INPUT DATASET
# ================================================================

model_input_columns = required_features

model_data = features[
    model_input_columns
].copy()


# ================================================================
# 15. CATEGORICAL CLEANING
# ================================================================

for col in categorical_features:

    model_data[col] = (
        model_data[col]
        .astype("string")
        .str.strip()
    )

    # Explicit missing category
    model_data[col] = (
        model_data[col]
        .fillna("Missing")
    )


# ================================================================
# 16. NUMERIC DATA TYPES
# ================================================================

for col in numeric_features:

    model_data[col] = pd.to_numeric(
        model_data[col],
        errors="coerce"
    )


# ================================================================
# 17. FEATURE QUALITY AUDIT
# ================================================================

feature_audit = []

for col in model_data.columns:

    feature_audit.append({

        "feature":
            col,

        "dtype":
            str(model_data[col].dtype),

        "n_unique":
            model_data[col].nunique(
                dropna=True
            ),

        "missing_n":
            model_data[col].isna().sum(),

        "missing_pct":
            100 *
            model_data[col].isna().mean(),

        "constant":
            model_data[col].nunique(
                dropna=False
            ) <= 1,

        "feature_type":
            (
                "numeric"
                if col in numeric_features
                else "categorical"
            )
    })


feature_audit_df = pd.DataFrame(
    feature_audit
)


print("\n" + "=" * 90)
print("FEATURE QUALITY AUDIT")
print("=" * 90)

print(
    feature_audit_df.to_string(
        index=False
    )
)


# ================================================================
# 18. CHECK CONSTANT FEATURES
# ================================================================

constant_features = (
    feature_audit_df.loc[
        feature_audit_df["constant"],
        "feature"
    ]
    .tolist()
)

if constant_features:

    print(
        "\nWARNING — Constant features detected:"
    )

    for col in constant_features:
        print("  ", col)

else:

    print(
        "\nNo constant model features detected."
    )


# ================================================================
# 19. CHECK MISSING VALUES
# ================================================================

missing_features_audit = (
    feature_audit_df.loc[
        feature_audit_df["missing_n"] > 0
    ]
)

print(
    "\nFeatures with missing values:"
)

if len(missing_features_audit) == 0:

    print("  None")

else:

    print(
        missing_features_audit[
            [
                "feature",
                "missing_n",
                "missing_pct"
            ]
        ].to_string(index=False)
    )


# ================================================================
# 20. FEATURE DICTIONARY
# ================================================================

feature_dictionary = []


# ------------------------------------------------
# Original numeric baseline
# ------------------------------------------------

feature_dictionary.append({
    "feature": "Age",
    "source_variable": "Age",
    "feature_type": "numeric",
    "engineering":
        "baseline numeric",
    "leakage_status": "SAFE",
    "fit_on_training_only": "NO"
})


# ------------------------------------------------
# Age-derived features
# ------------------------------------------------

age_feature_definitions = {

    "age_normalized":
        "standardized Age",

    "is_pediatric":
        "Age < 18",

    "is_elderly":
        "Age >= 65",

    "is_male":
        "Gender == male",

    "age_missing":
        "Age missing indicator"
}


for feature, engineering in age_feature_definitions.items():

    feature_dictionary.append({

        "feature": feature,

        "source_variable":
            (
                "Age"
                if feature != "is_male"
                else "Gender"
            ),

        "feature_type": "numeric",

        "engineering":
            engineering,

        "leakage_status": "SAFE",

        "fit_on_training_only":
            (
                "YES"
                if feature == "age_normalized"
                else "NO"
            )
    })


# ------------------------------------------------
# Admission temporal features
# ------------------------------------------------

temporal_features = {

    "admission_year":
        "calendar year",

    "admission_month":
        "calendar month",

    "admission_quarter":
        "calendar quarter",

    "admission_dayofweek":
        "day of week",

    "admission_weekofyear":
        "ISO week",

    "month_sin":
        "cyclical month sine",

    "month_cos":
        "cyclical month cosine",

    "admission_days_since_start":
        "days since earliest admission"
}


for feature, engineering in temporal_features.items():

    feature_dictionary.append({

        "feature": feature,

        "source_variable":
            "Admision date",

        "feature_type": "numeric",

        "engineering":
            engineering,

        "leakage_status": "SAFE",

        "fit_on_training_only":
            "NO"
    })


# ------------------------------------------------
# Categorical baseline variables
# ------------------------------------------------

for col in categorical_features:

    feature_dictionary.append({

        "feature": col,

        "source_variable": col,

        "feature_type": "categorical",

        "engineering":
            "categorical variable; "
            "one-hot encoded in model pipeline; "
            "unknown categories ignored",

        "leakage_status": "SAFE",

        "fit_on_training_only":
            "YES"
    })


feature_dictionary_df = pd.DataFrame(
    feature_dictionary
)


# ================================================================
# 21. MODEL FEATURE LIST
# ================================================================

model_feature_list = pd.DataFrame({

    "feature":
        required_features,

    "feature_type":
        [
            (
                "numeric"
                if col in numeric_features
                else "categorical"
            )
            for col in required_features
        ],

    "used_as_model_feature":
        True
})


# ================================================================
# 22. SAVE FEATURE DICTIONARY
# ================================================================

feature_dictionary_df.to_csv(
    FEATURE_DICTIONARY_FILE,
    index=False,
    encoding="utf-8-sig"
)


model_feature_list.to_csv(
    MODEL_FEATURE_LIST_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ================================================================
# 23. SAVE MODEL-READY BASELINE FEATURES
# ================================================================

model_data.to_csv(
    FEATURED_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ================================================================
# 24. FINAL SUMMARY
# ================================================================

print("\n" + "=" * 90)
print("FINAL FEATURE ENGINEERING SUMMARY")
print("=" * 90)

print(
    f"\nEpisodes: {len(model_data):,}"
)

print(
    f"Numeric features: {len(numeric_features)}"
)

print(
    f"Categorical features: {len(categorical_features)}"
)

print(
    f"Total pre-encoding model features: "
    f"{len(required_features)}"
)

print(
    "\nNumeric features:"
)

for col in numeric_features:
    print("  ", col)

print(
    "\nCategorical features:"
)

for col in categorical_features:
    print("  ", col)


print(
    "\nExcluded from model:"
)

for col in excluded_variables:
    print("  ", col)


print(
    "\nOutput files:"
)

print(
    "1. Feature-engineered model input:"
)
print(
    FEATURED_FILE
)

print(
    "\n2. Feature dictionary:"
)
print(
    FEATURE_DICTIONARY_FILE
)

print(
    "\n3. Model feature list:"
)
print(
    MODEL_FEATURE_LIST_FILE
)

print(
    "\n" + "=" * 90
)
print("FEATURE ENGINEERING COMPLETE")
print("=" * 90)

In [3]:
# %% [markdown]
# # Cholera episodes - DIAGNOSTICS (read-only, run BEFORE the next feature-engineering version)
#
# Reads the frozen 04_episode_level_data.csv and writes tables + a report into a
# folder called "diagnostics". It changes nothing and defines no analytic variable.
#
# QUESTIONS THE v2 OUTPUT LEFT OPEN
#  A. FINANCING GAP: v2 used only patient + insurer paid. The file also holds
#     "Terminally Patients Paid Share", "Allocated subsidized foreign currency share",
#     "Known_financing_total", "Financing_difference(_pct)". Does using them close the
#     gap (691 episodes, concentrated in private hospitals / inpatient)? If not, is the
#     patient share (the primary outcome) even measured comparably across hospitals?
#  B. COHORT VALIDITY: are these cholera episodes? (85.7% flagged by a crude keyword
#     list, 61% same-day stays, IVF / ophthalmology / cardiology settings.)
#  C. OUTCOME STRUCTURE: how much of the patient share is fixed-tariff structure, and
#     does insurance fund add anything once ownership / billing type / department are known?
#  D. EPISODE CONSTRUCTION and TIME PATTERN.
#  E. (optional) RAW CLAIMS: where does the cholera code sit?

from __future__ import annotations

import re
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

INPUT_FILE = Path(
    r"C:\Users\Shekoofeh\Documents\Health Insurance Data"
    r"\Cholera\04_episode_level_data.csv"
)
if len(sys.argv) > 1 and sys.argv[-1].lower().endswith(".csv") and Path(sys.argv[-1]).exists():
    INPUT_FILE = Path(sys.argv[-1])

# Optional: path to the RAW claim-line file (csv/xlsx) to locate the diagnosis code.
RAW_CLAIMS_FILE: str | None = None

DIAG_DIR = INPUT_FILE.parent / "diagnostics"
DIAG_DIR.mkdir(parents=True, exist_ok=True)
GAP_TOL_PCT = 5.0
MIN_N = 30
LOG: list[str] = []


def log(msg: str = "") -> None:
    print(msg)
    LOG.append(msg)


def save(name: str, table: pd.DataFrame) -> None:
    table.to_csv(DIAG_DIR / name, index=False, encoding="utf-8-sig")
    log(f"  [saved] {name}  ({len(table)} rows)")


def to_num(s: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(s):
        return s.astype(float)
    return pd.to_numeric(s.astype("string").str.replace(",", "", regex=False).str.strip(), errors="coerce")


def clean(s: pd.Series) -> pd.Series:
    t = s.astype("string").str.strip()
    return t.mask(t.str.lower().isin(["", "-", "--", "nan", "none", "null", "<na>"]))


def grp(s: pd.Series, min_n: int = MIN_N) -> pd.Series:
    t = clean(s).fillna("Missing")
    c = t.value_counts()
    rare = c[(c < min_n) & (c.index != "Missing")].index
    return t.where(~t.isin(rare), "Other (rare)").astype(object)


# %% Load -----------------------------------------------------------------------
log("=" * 88)
log("DIAGNOSTICS - cholera episode file")
log(f"Run time: {datetime.now():%Y-%m-%d %H:%M}    Input: {INPUT_FILE}")
log("=" * 88)
df = pd.read_csv(INPUT_FILE, encoding="utf-8-sig", low_memory=False)
df.columns = [str(c).strip() for c in df.columns]
n_all = len(df)
log(f"Episodes: {n_all:,}   Columns: {len(df.columns)}")


def num(name: str, required: bool = False) -> pd.Series | None:
    if name in df.columns:
        return to_num(df[name])
    log(f"  NOT AVAILABLE: '{name}'")
    if required:
        raise ValueError(f"Required column missing: {name}")
    return None


total = num("Total Cost", True)
pat = num("Insured Person's Share", True)
org = num("Organization Paid Share", True)
org_req = num("Organization Requested Share")
term_paid = num("Terminally Patients Paid Share")
term_req = num("Terminally Patients Requested Share")
fx_alloc = num("Allocated subsidized foreign currency share")
fx_req = num("Requested subsidized foreign currency share")
known_col = num("Known_financing_total")
diff_col = num("Financing_difference")
diffpct_col = num("Financing_difference_pct")

valid = total.gt(0) & pat.notna() & (pat / total).between(0, 1)
oop = (pat / total).where(valid)
ownership = grp(df["Ownership"])
fund = grp(df["Insurance Fund"])
diag_type = grp(df["Diagnosis type"])
dept = grp(df["Department Type"])
spec = grp(df["Treating Physician Specialty"])
age = to_num(df["Age"])
sex_txt = clean(df["Gender"]).str.lower()
female = sex_txt.isin(["woman", "the woman", "female", "women", "f"])
los = num("Episode_LOS_days")
admit = pd.to_datetime(df["Admision date"], errors="coerce")

# ==================================================================================
# A. FINANCING RECONCILIATION
# ==================================================================================
log("\n" + "=" * 88)
log("A. FINANCING RECONCILIATION (does the gap close when all payer columns are used?)")
log("=" * 88)


def z(s):  # optional components: missing -> 0, but reported below
    return s.fillna(0) if s is not None else pd.Series(0.0, index=df.index)


components = {
    "patient share": pat, "org paid": org, "org requested": org_req,
    "terminally patients paid": term_paid, "terminally patients requested": term_req,
    "subsidised FX allocated": fx_alloc, "subsidised FX requested": fx_req,
}
rows = []
for k, s in components.items():
    if s is None:
        continue
    rows.append({"component": k, "n_nonmissing": int(s.notna().sum()), "n_nonzero": int((s.fillna(0) != 0).sum()),
                 "sum": float(s.sum()), "sum_as_pct_of_total_cost": round(100 * s.sum() / total.sum(), 2)})
save("A1_payer_components.csv", pd.DataFrame(rows))

variants = {
    "V0 patient + org paid (v2)": z(pat) + z(org),
    "V1 + terminally patients paid": z(pat) + z(org) + z(term_paid),
    "V2 + terminally paid + FX allocated": z(pat) + z(org) + z(term_paid) + z(fx_alloc),
    "V3 requested basis: patient + org requested + term req + FX req":
        z(pat) + z(org_req) + z(term_req) + z(fx_req),
    "V4 = V2 + insurer deduction (requested - paid)":
        z(pat) + z(org) + z(term_paid) + z(fx_alloc) + (z(org_req) - z(org)).clip(lower=0),
}
if known_col is not None:
    variants["Provided Known_financing_total"] = known_col
vrows, by_own = [], []
for name, tot_v in variants.items():
    gap = ((total - tot_v) / total * 100).where(total.gt(0))
    vrows.append({
        "variant": name,
        "pct_episodes_abs_gap_gt_tol": round(100 * gap.abs().gt(GAP_TOL_PCT).mean(), 1),
        "pct_gap_positive_gt_tol": round(100 * gap.gt(GAP_TOL_PCT).mean(), 1),
        "pct_gap_negative_lt_-tol": round(100 * gap.lt(-GAP_TOL_PCT).mean(), 1),
        "median_gap_pct": round(gap.median(), 2), "mean_gap_pct": round(gap.mean(), 2),
        "p90_gap_pct": round(gap.quantile(0.9), 2),
    })
    t = pd.DataFrame({"own": ownership, "gap": gap}).groupby("own")["gap"].agg(
        n="size", pct_abs_gap_gt_tol=lambda x: 100 * x.abs().gt(GAP_TOL_PCT).mean(),
        median_gap_pct="median").reset_index()
    t.insert(0, "variant", name)
    by_own.append(t)
vt = pd.DataFrame(vrows)
save("A2_gap_by_variant.csv", vt)
save("A3_gap_by_variant_and_ownership.csv", pd.concat(by_own, ignore_index=True).round(2))
log(vt.to_string(index=False))
best = vt.sort_values("pct_episodes_abs_gap_gt_tol").iloc[0]["variant"]
log(f"\n  Variant with the smallest share of large gaps: {best}")

if known_col is not None or diff_col is not None or diffpct_col is not None:
    cmp_rows = []
    for name, tot_v in variants.items():
        if name.startswith("Provided"):
            continue
        r = {"variant": name}
        if known_col is not None:
            r["pct_match_Known_financing_total"] = round(100 * np.isclose(tot_v, known_col, rtol=1e-6, atol=1).mean(), 1)
        if diff_col is not None:
            r["pct_match_Financing_difference"] = round(100 * np.isclose(total - tot_v, diff_col, rtol=1e-6, atol=1).mean(), 1)
        cmp_rows.append(r)
    save("A4_match_with_provided_financing_columns.csv", pd.DataFrame(cmp_rows))
    log("  -> Which variant reproduces the preprocessing columns tells you how they were built.")

# what does the residual gap look like? (best variant)
best_tot = variants[best]
gap_b = ((total - best_tot) / total * 100).where(total.gt(0))
ded = ((z(org_req) - z(org)) / total * 100).where(total.gt(0)) if org_req is not None else None
res = pd.DataFrame({"own": ownership, "diag": diag_type, "gap": gap_b,
                    "total": total, "los_zero": los.eq(0) if los is not None else np.nan})
res_rows = [{
    "quantity": "Spearman(gap, log total cost)",
    "value": round(pd.Series(gap_b).corr(np.log(total.where(total.gt(0))), method="spearman"), 3)}]
if ded is not None:
    res_rows.append({"quantity": "Spearman(gap, insurer deduction % = requested - paid)",
                     "value": round(gap_b.corr(ded, method="spearman"), 3)})
    res_rows.append({"quantity": "% of large-gap episodes where gap ~ deduction (within 5 pts)",
                     "value": round(100 * ((gap_b - ded).abs().le(5) & gap_b.abs().gt(GAP_TOL_PCT)).sum()
                                    / max(1, gap_b.abs().gt(GAP_TOL_PCT).sum()), 1)})
save("A5_residual_gap_relationships.csv", pd.DataFrame(res_rows))
log(pd.DataFrame(res_rows).to_string(index=False))

# does the ownership contrast in patient share survive alternative denominators?
alts = {
    "pat / total (primary)": (pat / total),
    "pat / (pat + org paid)": pat / (pat + org).where((pat + org) > 0),
    f"pat / ({best.split(' ')[0]} financing total)": pat / best_tot.where(best_tot > 0),
}
if org_req is not None:
    alts["pat / (pat + org requested)"] = pat / (pat + org_req).where((pat + org_req) > 0)
arows = []
for name, s in alts.items():
    t = pd.DataFrame({"own": ownership, "s": s.where(valid)}).groupby("own")["s"].agg(
        n="count", mean="mean", median="median").reset_index()
    t.insert(0, "definition", name)
    arows.append(t)
save("A6_patient_share_by_ownership_alt_denominators.csv", pd.concat(arows, ignore_index=True).round(4))
log("  -> If private hospitals swing from low to high patient share across denominators, "
    "the primary outcome is not comparable across ownership.")

# ==================================================================================
# B. COHORT VALIDITY / SETTING
# ==================================================================================
log("\n" + "=" * 88)
log("B. COHORT VALIDITY (is this a cholera cohort?)")
log("=" * 88)
IMPLAUSIBLE = ["cardi", "heart", "echocardiograph", "angiograph", "ophthalm", "eye", "ivf", "infertil", "orthop",
                "orthodont", "ear, nose", "otology", "rhinology", "obstet", "gynec", "perinat", "psychiat",
                "radiother", "radiation", "chemother", "oncolog", "cancer", "neurosurg", "transplant",
                "rheumat", "plastic", "thoracic", "urolog", "genitourinary", "kidney", "breast",
                "surg", "operating room", "limited", "pharmacy", "outpatient"]
PLAUSIBLE = ["infect", "tropical", "internal", "emergency", "triage", "pediatric", "child", "gastro", "general",
             "medicine", "neonatal"]


def setting_class(text: str) -> str:
    t = text.lower()
    if any(k in t for k in IMPLAUSIBLE):
        return "implausible_for_cholera"
    if any(k in t for k in PLAUSIBLE):
        return "plausible_medical_or_infectious"
    return "needs_review"


combo = (clean(df["Department Type"]).fillna("") + " | " + clean(df["Treating Physician Specialty"]).fillna(""))
setting = combo.map(setting_class)

prof = pd.DataFrame({
    "setting_class": setting, "oop": oop, "age": age, "female": female.astype(float),
    "los": los if los is not None else np.nan, "total": total,
    "los_zero": (los.eq(0).astype(float) if los is not None else np.nan),
    "lines": num("Number_of_claim_lines") if "Number_of_claim_lines" in df.columns else np.nan,
    "death": clean(df["Discharge Status"]).str.lower().str.contains("death", na=False).astype(float)
             if "Discharge Status" in df.columns else np.nan,
})
cls = prof.groupby("setting_class").agg(
    n=("oop", "size"), median_age=("age", "median"), pct_female=("female", lambda x: 100 * x.mean()),
    median_los=("los", "median"), pct_zero_los=("los_zero", lambda x: 100 * x.mean()),
    median_total_cost=("total", "median"), median_claim_lines=("lines", "median"),
    pct_death=("death", lambda x: 100 * x.mean()), mean_oop_share=("oop", "mean")).reset_index()
cls["pct_of_episodes"] = 100 * cls["n"] / n_all
save("B1_setting_class_profile.csv", cls.round(2))
log(cls.round(2).to_string(index=False))

pairs = (pd.DataFrame({"department": clean(df["Department Type"]).fillna("Missing"),
                       "specialty": clean(df["Treating Physician Specialty"]).fillna("Missing"),
                       "guessed_class": setting, "los_zero": los.eq(0) if los is not None else False,
                       "age": age, "female": female.astype(float)})
         .groupby(["department", "specialty", "guessed_class"])
         .agg(n=("age", "size"), pct_zero_los=("los_zero", lambda x: 100 * x.mean()),
              median_age=("age", "median"), pct_female=("female", lambda x: 100 * x.mean()))
         .reset_index().sort_values("n", ascending=False).round(1))
pairs["reviewer_decision_plausible_cholera (Y/N)"] = ""
save("B2_department_specialty_pairs_FOR_MANUAL_REVIEW.csv", pairs)
log("  -> Have a clinician mark Y/N in B2. The keyword guess is only a starting point "
    "(text is machine-translated).")

dprof = prof.assign(dept=dept).groupby("dept").agg(
    n=("oop", "size"), median_age=("age", "median"), pct_female=("female", lambda x: 100 * x.mean()),
    median_los=("los", "median"), pct_zero_los=("los_zero", lambda x: 100 * x.mean()),
    median_total_cost=("total", "median"), median_claim_lines=("lines", "median"),
    mean_oop_share=("oop", "mean")).reset_index().sort_values("n", ascending=False)
save("B3_department_profile.csv", dprof.round(2))

if los is not None:
    zl = pd.crosstab(diag_type, los.eq(0).map({True: "same_day", False: "1plus_day"}))
    zl["pct_same_day"] = 100 * zl.get("same_day", 0) / zl.sum(axis=1)
    save("B4_same_day_by_billing_type.csv", zl.reset_index().round(1))

# time pattern
ym = admit.dt.strftime("%Y-%m")
tp = pd.crosstab(ym, setting)
tp["total"] = tp.sum(axis=1)
save("B5_monthly_episodes_by_setting_class.csv", tp.reset_index().rename(columns={"row_0": "admission_ym", "Admision date": "admission_ym"}))
log("  -> Cholera is seasonal/outbreak-driven. A flat monthly count dominated by surgical/cardiac "
    "settings argues against a true cholera cohort.")

# ==================================================================================
# C. OUTCOME STRUCTURE (tariff-driven?)
# ==================================================================================
log("\n" + "=" * 88)
log("C. STRUCTURE OF THE PATIENT SHARE")
log("=" * 88)
base = pd.DataFrame({"oop": oop, "own": ownership, "diag": diag_type, "dept": dept, "fund": fund,
                     "age_grp": pd.cut(age, [0, 5, 15, 45, 65, 121], right=False).astype(str),
                     "sex": np.where(female, "F", "M")}).dropna(subset=["oop"])
r = base.assign(r=base["oop"].round(2))
top = r["r"].value_counts().head(15)
srows = []
for v, cnt in top.items():
    s = r[r["r"] == v]
    row = {"patient_share_rounded": v, "n": int(cnt), "pct_of_all": round(100 * cnt / len(r), 1)}
    for c in ["own", "diag", "dept"]:
        vc = s[c].value_counts()
        row[f"top_{c}"] = vc.index[0]
        row[f"top_{c}_pct"] = round(100 * vc.iloc[0] / len(s), 1)
    srows.append(row)
sp = pd.DataFrame(srows)
save("C1_most_frequent_patient_share_values.csv", sp)
log(f"  The 15 most frequent values (rounded to 0.01) cover {sp['pct_of_all'].sum():.1f}% of episodes.")


def design(frame: pd.DataFrame, cols: list[str]) -> np.ndarray:
    X = pd.get_dummies(frame[cols].astype(str), drop_first=True, dtype=float)
    return np.column_stack([np.ones(len(frame)), X.values])


def fit_r2(y: np.ndarray, X: np.ndarray) -> tuple[float, float, int]:
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    ss_res = float(((y - X @ beta) ** 2).sum())
    ss_tot = float(((y - y.mean()) ** 2).sum())
    n, k = X.shape
    r2 = 1 - ss_res / ss_tot
    return r2, 1 - (1 - r2) * (n - 1) / max(1, n - k), k - 1


y = base["oop"].to_numpy()
models = {
    "insurance fund only": ["fund"],
    "ownership only": ["own"],
    "billing type only": ["diag"],
    "department only": ["dept"],
    "ownership + billing type + department (setting/tariff bundle)": ["own", "diag", "dept"],
    "setting bundle + insurance fund": ["own", "diag", "dept", "fund"],
    "setting bundle + fund + age group + sex": ["own", "diag", "dept", "fund", "age_grp", "sex"],
}
mrows = []
for label, cols in models.items():
    r2, adj, k = fit_r2(y, design(base, cols))
    mrows.append({"model": label, "n_parameters": k, "R2": round(r2, 3), "adjusted_R2": round(adj, 3)})
mt = pd.DataFrame(mrows)
save("C2_variance_explained_patient_share.csv", mt)
log(mt.to_string(index=False))
log("  -> If the setting bundle explains most variance and fund adds ~0 adjusted R2, insurance-fund "
    "'inequality' is confounded by where care was delivered.")

# fund means before / after adjustment for the setting bundle
Xs = design(base, ["own", "diag", "dept"])
beta, *_ = np.linalg.lstsq(Xs, y, rcond=None)
resid = y - Xs @ beta
adj_tab = base.assign(resid=resid).groupby("fund").agg(
    n=("oop", "size"), raw_mean=("oop", "mean"), mean_residual_after_setting=("resid", "mean")).reset_index()
adj_tab["adjusted_mean"] = adj_tab["mean_residual_after_setting"] + y.mean()
save("C3_fund_means_raw_vs_setting_adjusted.csv", adj_tab.round(4))
log(adj_tab.round(4).to_string(index=False))


def cramers_v(a: pd.Series, b: pd.Series) -> float:
    ct = pd.crosstab(a, b).to_numpy()
    n = ct.sum()
    exp = ct.sum(1, keepdims=True) * ct.sum(0, keepdims=True) / n
    chi2 = ((ct - exp) ** 2 / np.where(exp == 0, 1, exp)).sum()
    return float(np.sqrt(chi2 / (n * (min(ct.shape) - 1))))


cv = pd.DataFrame([{"pair": f"{a} x {b}", "cramers_v": round(cramers_v(base[a], base[b]), 3)}
                   for a, b in [("fund", "own"), ("fund", "diag"), ("fund", "dept"), ("own", "diag"), ("own", "dept")]])
save("C4_association_between_exposures.csv", cv)
log(cv.to_string(index=False))

# ==================================================================================
# D. EPISODE CONSTRUCTION
# ==================================================================================
log("\n" + "=" * 88)
log("D. EPISODE CONSTRUCTION")
log("=" * 88)
pat_mask = df["Episode_ID"].astype(str).map(lambda s: re.sub(r"[A-Za-z]", "A", re.sub(r"\d", "9", s)))
save("D1_episode_id_patterns_anonymised.csv",
     pat_mask.value_counts().head(10).rename_axis("pattern").reset_index(name="n"))
for c in ["Episode_Identification_Type", "Episode_Linkage_Method"]:
    if c in df.columns:
        t = df[c].astype("string").fillna("Missing").value_counts().rename_axis(c).reset_index(name="n")
        save(f"D2_{c}.csv", t)
        log(t.to_string(index=False))
if "Patient_ID" in df.columns:
    same = df.assign(_a=admit).dropna(subset=["Patient_ID"]).groupby(["Patient_ID", "_a"]).size()
    log(f"  Patient-days with more than one episode: {int((same > 1).sum())}")
    per = df["Patient_ID"].value_counts()
    log(f"  Patients: {int(per.size)}; with >1 episode: {int((per > 1).sum())}; max episodes per patient: {int(per.max())}")
if "Number_of_claim_lines" in df.columns:
    q = to_num(df["Number_of_claim_lines"]).describe().round(1)
    log("  Claim lines per episode:\n" + q.to_string())

# ==================================================================================
# E. RAW CLAIMS (optional)
# ==================================================================================
log("\n" + "=" * 88)
log("E. RAW CLAIMS - LOCATE THE DIAGNOSIS")
log("=" * 88)
if RAW_CLAIMS_FILE and Path(RAW_CLAIMS_FILE).exists():
    p = Path(RAW_CLAIMS_FILE)
    raw = pd.read_excel(p) if p.suffix.lower() in (".xlsx", ".xls") else pd.read_csv(p, low_memory=False)
    raw.columns = [str(c).strip() for c in raw.columns]
    log(f"  Raw rows: {len(raw):,}; columns: " + "; ".join(raw.columns))
    dcols = [c for c in raw.columns if re.search(r"icd|diag|disease|code", c, re.I)]
    log(f"  Candidate diagnosis columns: {dcols or 'NONE'}")
    for c in dcols:
        s = raw[c].astype("string").str.strip().str.upper()
        vc = s.value_counts().head(30).rename_axis("value").reset_index(name="n")
        save(f"E1_top_values_{re.sub(r'[^A-Za-z0-9]+', '_', c)}.csv", vc)
        log(f"  {c}: rows starting with A00 = {int(s.str.startswith('A00', na=False).sum()):,} "
            f"of {len(raw):,}; distinct values = {s.nunique()}")
else:
    log("  RAW_CLAIMS_FILE not set. Set it at the top of this script (csv/xlsx of the claim lines), or\n"
        "  answer manually: in the raw claims, (1) which column holds the ICD-10 code, (2) is A00.x the\n"
        "  primary or any diagnosis, (3) were ALL claim lines of a patient with an A00 line pulled into\n"
        "  the episode (this would import surgery/cardiology/IVF claims), (4) was the code A00 or A01\n"
        "  (typhoid) at any step (Table 1 said typhoid).")

# ==================================================================================
# SUMMARY
# ==================================================================================
log("\n" + "=" * 88)
log("SUMMARY NUMBERS TO PASTE BACK")
log("=" * 88)
plaus = int((setting == "plausible_medical_or_infectious").sum())
impl = int((setting == "implausible_for_cholera").sum())
log(f"Setting guess: plausible {plaus:,} ({100 * plaus / n_all:.1f}%), implausible {impl:,} ({100 * impl / n_all:.1f}%), "
    f"needs review {n_all - plaus - impl:,}")
log(f"Best financing variant: {best}; large gaps remaining: "
    f"{vt.set_index('variant').loc[best, 'pct_episodes_abs_gap_gt_tol']}% of episodes")
log("Paste back: this report (09-style .txt), tables A2, A3, A4, A6, B1, B4, B5, C1, C2, C3, C4, D1 and D2 "
    "(and B2 after review).")

(DIAG_DIR / "diagnostics_report.txt").write_text("\n".join(LOG), encoding="utf-8")
print(f"\nReport written to {DIAG_DIR / 'diagnostics_report.txt'}")

DIAGNOSTICS - cholera episode file
Run time: 2026-09-29 19:55    Input: C:\Users\Shekoofeh\Documents\Health Insurance Data\Cholera\04_episode_level_data.csv
Episodes: 1,171   Columns: 31

A. FINANCING RECONCILIATION (does the gap close when all payer columns are used?)
  [saved] A1_payer_components.csv  (7 rows)
  [saved] A2_gap_by_variant.csv  (6 rows)
  [saved] A3_gap_by_variant_and_ownership.csv  (24 rows)
                                                        variant  pct_episodes_abs_gap_gt_tol  pct_gap_positive_gt_tol  pct_gap_negative_lt_-tol  median_gap_pct  mean_gap_pct  p90_gap_pct
                                     V0 patient + org paid (v2)                         59.0                     59.0                       0.0           10.45         28.92        72.52
                                  V1 + terminally patients paid                         58.0                     58.0                       0.0            9.88         28.82        72.52
                          

In [6]:
# =============================================================================
# CHOLERA EPISODES — FINAL FEATURE ENGINEERING v4
# =============================================================================
#
# PURPOSE
# -------
# Build a leakage-safe, audit-ready episode-level analysis dataset for:
#   Financial protection / OOP / insurance coverage
#   Cholera episode claims
#
# IMPORTANT ANALYTICAL PRINCIPLES
# --------------------------------
# 1. Primary OOP outcome remains:
#
#       OOP_share = Insured Person's Share / Total Cost
#
# 2. Financing reconciliation is NOT silently used to redefine OOP.
#    Instead, V0-V4 financing variants are explicitly calculated and audited.
#
# 3. Department/specialty keywords are AUDIT SIGNALS ONLY.
#    They are NOT used to define or exclude the primary cholera cohort.
#
# 4. No automatic deletion based on:
#       - anonymous episodes
#       - zero LOS
#       - financing gaps
#       - specialty/department keywords
#       - duplicate flags
#
# 5. No imputation.
# 6. No winsorization.
# 7. No normalization/scaling.
# 8. No ML.
# 9. No post-admission variables are used as baseline predictors.
# 10. Direct identifiers are retained only in a confidential linkage file.
#
# v4 CHANGES FROM v3
# ------------------
# - Full V0-V4 financing reconciliation.
# - Separate financing completeness flags.
# - Correct nullable missingness flags.
# - Correct valid_oop_known_denominator.
# - Removes "implausible_for_cholera" as a clinical classification.
# - Specialty/department keywords become reason-specific audit flags.
# - Primary cohort is never restricted using keyword heuristics.
# - Adds explicit cohort-definition warning.
# - Adds source-vs-recomputed financing audit.
# - Keeps primary OOP denominator unchanged.
# - Final dictionary is structurally tied to feature creation.
#
# =============================================================================

from pathlib import Path
from datetime import datetime
import re
import warnings

import numpy as np
import pandas as pd


# =============================================================================
# 1. CONFIGURATION
# =============================================================================

INPUT_FILE = Path(
    r"C:\Users\Shekoofeh\Documents\Health Insurance Data\Cholera\04_episode_level_data.csv"
)

OUTPUT_DIR = INPUT_FILE.parent / "feature engineering v4_FINAL"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXTERNAL_FUND_SES_FILE = OUTPUT_DIR / "external_fund_ses_order.csv"

# -------------------------------------------------------------------------
# Primary analysis parameters
# -------------------------------------------------------------------------

FINANCING_GAP_TOLERANCE_PCT = 5.0

MIN_CATEGORY_N = 30

PRIMARY_HIGH_SHARE_THRESHOLD = 0.10

SENSITIVITY_HIGH_SHARE_THRESHOLDS = [
    0.05,
    0.20,
    0.30,
]

OOP_BAND_CUTS = [
    0.0,
    0.05,
    0.10,
    0.30,
    1.0000001,
]

OOP_BAND_LABELS = [
    "<5%",
    "5-<10%",
    "10-<30%",
    ">=30%",
]

AGE_BINS = [
    0,
    5,
    15,
    45,
    65,
    121,
]

AGE_LABELS = [
    "0_4",
    "5_14",
    "15_44",
    "45_64",
    "65plus",
]

MISSING_LABEL = "Missing"
RARE_LABEL = "Other (rare)"

PLACEHOLDER_VALUES = {
    "",
    "-",
    "--",
    "nan",
    "none",
    "null",
    "<na>",
    "unknown",
    "n/a",
    "na",
}

MALE_LABELS = {
    "the man",
    "man",
    "male",
    "men",
    "m",
}

FEMALE_LABELS = {
    "woman",
    "the woman",
    "female",
    "women",
    "f",
}


# =============================================================================
# 2. REQUIRED SOURCE COLUMNS
# =============================================================================

REQUIRED_COLUMNS = [
    "Episode_ID",
    "Total Cost",
    "Insured Person's Share",
    "Organization Paid Share",
    "Age",
    "Gender",
    "Ownership",
    "Insurance Fund",
    "Treating Physician Specialty",
    "Admision date",
    "Diagnosis type",
    "Department Type",
]


# =============================================================================
# 3. EXACT FINANCING COMPONENTS AVAILABLE IN THE SOURCE
# =============================================================================
#
# These are present in the v3 input file and therefore must not be ignored.
#
# V0:
#   patient paid
#   + organization paid
#
# V1:
#   V0
#   + terminally patients paid
#
# V2:
#   V1
#   + allocated subsidized foreign currency
#
# V3:
#   patient paid
#   + organization requested
#   + terminally patients requested
#   + requested subsidized foreign currency
#
# V4:
#   V2
#   + insurer deduction
#
# insurer deduction:
#   organization requested - organization paid
#
# =============================================================================

FINANCING_COLUMNS = {
    "patient_paid": "Insured Person's Share",
    "organization_paid": "Organization Paid Share",
    "organization_requested": "Organization Requested Share",
    "terminal_patient_paid": "Terminally Patients Paid Share",
    "terminal_patient_requested": "Terminally Patients Requested Share",
    "fx_allocated": "Allocated subsidized foreign currency share",
    "fx_requested": "Requested subsidized foreign currency share",
}


# =============================================================================
# 4. AUDIT KEYWORDS
# =============================================================================
#
# IMPORTANT:
# These do NOT define cholera.
# They are simply signals requiring clinical/source-data review.
#
# =============================================================================

AUDIT_KEYWORDS = {
    "cardiac": [
        "cardi",
        "heart",
        "cardiology",
        "cardiovascular",
    ],

    "ophthalmology": [
        "ophthalm",
        "eye",
        "corneal",
        "retina",
    ],

    "ivf_infertility": [
        "ivf",
        "infertil",
    ],

    "orthopedic": [
        "orthop",
        "knee surgery",
    ],

    "obstetric_gynecology": [
        "obstet",
        "gynec",
        "perinatolog",
    ],

    "psychiatry": [
        "psychiat",
    ],

    "oncology": [
        "radiother",
        "radiation",
        "chemotherap",
        "cancer",
    ],

    "neurosurgery": [
        "neurosurg",
    ],

    "transplant": [
        "transplant",
    ],

    "rheumatology": [
        "rheumat",
    ],

    "plastic_surgery": [
        "plastic",
        "reconstructive",
    ],

    "thoracic_surgery": [
        "thoracic",
    ],

    "urology": [
        "urolog",
        "genitourinary",
        "kidney",
    ],

    "breast_surgery": [
        "breast",
    ],

    "surgery": [
        "surg",
        "operating room",
    ],
}


# =============================================================================
# 5. LOGGING
# =============================================================================

LOG = []


def log(message):
    message = str(message)
    LOG.append(message)
    print(message)


# =============================================================================
# 6. BASIC HELPERS
# =============================================================================

def read_csv_robust(path):
    """
    Read CSV with a small encoding fallback.
    """
    encodings = [
        "utf-8-sig",
        "utf-8",
        "cp1256",
        "latin1",
    ]

    last_error = None

    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False)
        except Exception as exc:
            last_error = exc

    raise last_error


def clean_text(series):
    """
    Standardize text while preserving missingness.
    """
    s = series.astype("string").str.strip()

    lower = s.str.lower()

    s = s.mask(lower.isin(PLACEHOLDER_VALUES))

    s = (
        s.str.replace(r"\s+", " ", regex=True)
         .str.strip()
    )

    return s


def to_num(series):
    return pd.to_numeric(series, errors="coerce")


def nullable_flag(condition, valid=None):
    """
    Convert condition to nullable Int8:
        1 = true
        0 = false
        NA = cannot determine
    """
    b = pd.Series(condition, index=DF.index).astype("boolean")

    if valid is not None:
        v = pd.Series(valid, index=DF.index).astype("boolean")
        b = b.where(v)

    return b.astype("Int8")


def flag_from_source(column_name):
    """
    Convert an existing source flag to nullable Int8.

    Unrecognized/missing values remain NA.
    """
    if column_name not in DF.columns:
        return pd.Series(pd.NA, index=DF.index, dtype="Int8")

    s = clean_text(DF[column_name])

    true_tokens = {
        "1",
        "true",
        "yes",
        "y",
        "t",
    }

    false_tokens = {
        "0",
        "false",
        "no",
        "n",
        "f",
    }

    result = pd.Series(
        pd.NA,
        index=DF.index,
        dtype="boolean",
    )

    result.loc[s.isin(true_tokens)] = True
    result.loc[s.isin(false_tokens)] = False

    return result.astype("Int8")


def column_or_na(column_name):
    """
    Return source column or all-NA series.
    """
    if column_name in DF.columns:
        return DF[column_name].copy()

    return pd.Series(
        pd.NA,
        index=DF.index,
    )


def safe_contains(series, keywords):
    """
    Case-insensitive keyword audit.
    """
    s = clean_text(series).fillna("").str.lower()

    if not keywords:
        return pd.Series(False, index=DF.index)

    pattern = "|".join(
        re.escape(x.lower())
        for x in keywords
    )

    return s.str.contains(
        pattern,
        regex=True,
        na=False,
    )


# =============================================================================
# 7. FEATURE REGISTRY
# =============================================================================
#
# IMPORTANT:
# Every engineered variable MUST be created through set_feature().
# This prevents the previous "undocumented engineered columns" problem.
#
# =============================================================================

DICTIONARY = []

OUT = pd.DataFrame(index=DF.index if "DF" in globals() else None)


def register_feature(
    name,
    role,
    source_variables,
    definition,
    notes="",
):
    existing = {
        x["feature"]
        for x in DICTIONARY
    }

    if name in existing:
        raise AssertionError(
            f"Duplicate dictionary feature: {name}"
        )

    DICTIONARY.append({
        "feature": name,
        "role": role,
        "source_variables": source_variables,
        "definition": definition,
        "notes": notes,
    })


def set_feature(
    name,
    series,
    role,
    source_variables,
    definition,
    notes="",
):
    """
    Single controlled entry point for every engineered variable.
    """
    if name in OUT.columns:
        raise AssertionError(
            f"Feature already exists: {name}"
        )

    if not isinstance(series, pd.Series):
        series = pd.Series(
            series,
            index=DF.index,
        )
    else:
        series = series.reindex(DF.index)

    OUT[name] = series

    register_feature(
        name=name,
        role=role,
        source_variables=source_variables,
        definition=definition,
        notes=notes,
    )

    return OUT[name]


def alias_feature(
    name,
    source_feature,
    role,
    definition,
    notes="",
):
    return set_feature(
        name=name,
        series=OUT[source_feature].copy(),
        role=role,
        source_variables=source_feature,
        definition=definition,
        notes=notes,
    )


# =============================================================================
# 8. CATEGORY HELPERS
# =============================================================================

def normalize_category(series):
    """
    Merge case/whitespace variants without making semantic assumptions.
    """
    s = clean_text(series)

    keys = s.str.casefold()

    lookup = (
        pd.DataFrame({
            "key": keys,
            "value": s,
        })
        .dropna(subset=["key"])
        .groupby("key")["value"]
        .agg(
            lambda x: x.value_counts().index[0]
        )
    )

    return keys.map(lookup)


def apply_manual_merges(series, merge_map):
    """
    Apply only explicitly documented semantic merges.
    """
    if not merge_map:
        return series

    normalized_map = {
        str(k).strip().casefold(): v
        for k, v in merge_map.items()
    }

    result = series.copy()

    result = result.map(
        lambda x:
            normalized_map.get(
                str(x).strip().casefold(),
                x,
            )
            if pd.notna(x)
            else pd.NA
    )

    return result


def collapse_rare(series, minimum_n=30):
    """
    Collapse categories below minimum_n into Other (rare).
    Missing remains Missing.
    """
    s = series.astype("object").where(
        series.notna(),
        MISSING_LABEL,
    )

    counts = s.value_counts(dropna=False)

    rare_categories = set(
        counts[counts < minimum_n].index
    )

    result = s.where(
        ~s.isin(rare_categories),
        RARE_LABEL,
    )

    return result


def build_category_map(
    original,
    grouped,
    variable_name,
):
    tmp = pd.DataFrame({
        "variable": variable_name,
        "original_category": original.fillna(MISSING_LABEL),
        "grouped_category": grouped.fillna(MISSING_LABEL),
    })

    return (
        tmp.groupby(
            [
                "variable",
                "original_category",
                "grouped_category",
            ],
            dropna=False,
        )
        .size()
        .reset_index(name="n_episodes")
    )


# =============================================================================
# 9. LOAD SOURCE DATA
# =============================================================================

log("=" * 90)
log("FINAL FEATURE ENGINEERING v4")
log("FINANCIAL PROTECTION / INSURANCE COVERAGE / CHOLERA EPISODES")
log(f"Run time : {datetime.now():%Y-%m-%d %H:%M}")
log(f"Input    : {INPUT_FILE}")
log(f"Output   : {OUTPUT_DIR}")
log("=" * 90)

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Input file not found:\n{INPUT_FILE}"
    )

DF = read_csv_robust(INPUT_FILE)

log(
    f"Episodes loaded: {len(DF):,}   "
    f"Columns: {DF.shape[1]}"
)

missing_required = [
    c for c in REQUIRED_COLUMNS
    if c not in DF.columns
]

if missing_required:
    raise ValueError(
        "Missing required source columns:\n"
        + "\n".join(missing_required)
    )

log(
    "Columns present: "
    + "; ".join(map(str, DF.columns))
)


# Reinitialize output after DF is loaded.
OUT = pd.DataFrame(index=DF.index)


# =============================================================================
# 10. SOURCE COLUMN AVAILABILITY
# =============================================================================

OPTIONAL_COLUMNS = [
    "Patient_ID",
    "Organization Requested Share",
    "Discharge date",
    "Episode_LOS_days",
    "Number_of_claim_lines",
    "Number_of_service_groups",
    "Discharge Status",
    "Episode_Identification_Type",
    "Episode_Linkage_Method",
    "invalid_date_order",
    "contains_exact_duplicate_claim",
    "contains_invalid_age",
    "Known_financing_total",
    "Financing_difference",
    "Financing_difference_pct",
    "Terminally Patients Paid Share",
    "Terminally Patients Requested Share",
    "Allocated subsidized foreign currency share",
    "Requested subsidized foreign currency share",
]

optional_rows = []

for col in OPTIONAL_COLUMNS:
    optional_rows.append({
        "optional_column": col,
        "present": col in DF.columns,
    })

optional_availability = pd.DataFrame(optional_rows)


# =============================================================================
# 11. CASE-DEFINITION AUDIT
# =============================================================================

diagnosis_code_candidates = [
    c for c in DF.columns
    if re.search(
        r"(icd|diagnosis[\s_]*code|diag[\s_]*code)",
        str(c),
        flags=re.I,
    )
]

if diagnosis_code_candidates:
    log(
        "Diagnosis-code candidates found: "
        + "; ".join(diagnosis_code_candidates)
    )
else:
    log(
        "WARNING: no ICD/diagnosis-code column found in the "
        "episode-level file."
    )

    log(
        "The episode file therefore cannot independently verify "
        "the A00.x cholera case definition."
    )

    log(
        "Raw claims must be checked for:"
    )

    log(
        "  1. whether ICD-10 A00.x was used;"
    )

    log(
        "  2. whether A00.x was primary or any diagnosis;"
    )

    log(
        "  3. whether all claim lines associated with the A00 episode "
        "were imported."
    )


# =============================================================================
# 12. IDENTIFIERS / EPISODE CLUSTERS
# =============================================================================

episode_key = pd.Series(
    [
        f"E{i:05d}"
        for i in range(1, len(DF) + 1)
    ],
    index=DF.index,
    dtype="string",
)

set_feature(
    "Episode_key",
    episode_key,
    "id",
    "Episode_ID",
    "Sequential pseudonymous episode key.",
    "Direct Episode_ID is retained only in the confidential linkage file.",
)


# -----------------------------------------------------------------------------
# Patient clustering
# -----------------------------------------------------------------------------

if "Patient_ID" in DF.columns:

    patient_id = clean_text(
        DF["Patient_ID"]
    )

    patient_codes, uniques = pd.factorize(
        patient_id,
        sort=True,
    )

    patient_cluster = pd.Series(
        patient_codes,
        index=DF.index,
        dtype="Int64",
    )

    patient_cluster = patient_cluster.replace(
        -1,
        pd.NA,
    )

    set_feature(
        "Patient_cluster",
        patient_cluster,
        "cluster",
        "Patient_ID",
        "Integer patient cluster identifier.",
        "Missing for anonymous episodes.",
    )

    n_episodes_patient = (
        patient_id
        .groupby(patient_id)
        .transform("size")
    )

    n_episodes_patient = (
        n_episodes_patient
        .where(patient_id.notna())
        .astype("Int64")
    )

    set_feature(
        "n_episodes_patient",
        n_episodes_patient,
        "descriptive",
        "Patient_ID",
        "Number of episodes belonging to the same identified patient.",
        "Missing for anonymous episodes.",
    )

    # Anonymous episodes receive unique singleton clusters for SE only.
    cluster_for_se = patient_cluster.copy()

    anonymous_mask = patient_cluster.isna()

    next_code = (
        patient_cluster.max(skipna=True)
        if patient_cluster.notna().any()
        else 0
    )

    anonymous_codes = (
        np.arange(
            next_code + 1,
            next_code + 1 + anonymous_mask.sum(),
        )
        if anonymous_mask.any()
        else np.array([], dtype=int)
    )

    cluster_for_se.loc[anonymous_mask] = anonymous_codes

    cluster_for_se = cluster_for_se.astype("Int64")

    set_feature(
        "Cluster_for_SE",
        cluster_for_se,
        "cluster",
        "Patient_cluster",
        "Patient cluster with unique singleton clusters for anonymous episodes.",
        "Do not interpret singleton anonymous clusters as recovered patient identities.",
    )

else:

    set_feature(
        "Patient_cluster",
        pd.Series(pd.NA, index=DF.index, dtype="Int64"),
        "cluster",
        "Patient_ID",
        "Patient cluster identifier.",
        "Patient_ID unavailable.",
    )

    set_feature(
        "n_episodes_patient",
        pd.Series(pd.NA, index=DF.index, dtype="Int64"),
        "descriptive",
        "Patient_ID",
        "Number of episodes belonging to the same patient.",
        "Patient_ID unavailable.",
    )

    set_feature(
        "Cluster_for_SE",
        pd.Series(pd.NA, index=DF.index, dtype="Int64"),
        "cluster",
        "Patient_ID",
        "Patient cluster for clustered inference.",
        "Patient-level clustering unavailable.",
    )


# =============================================================================
# 13. HOSPITAL CLUSTERING
# =============================================================================

hospital_candidates = [
    c for c in DF.columns
    if re.search(
        r"(hospital|facility|centre|center)",
        str(c),
        flags=re.I,
    )
]

selected_hospital_column = None

for candidate in hospital_candidates:

    vals = clean_text(
        DF[candidate]
    )

    n_unique = vals.nunique(
        dropna=True
    )

    if 2 <= n_unique <= max(2, len(DF) // 2):
        selected_hospital_column = candidate
        break

if selected_hospital_column is not None:

    hospital_clean = clean_text(
        DF[selected_hospital_column]
    )

    hospital_codes, _ = pd.factorize(
        hospital_clean,
        sort=True,
    )

    hospital_cluster = pd.Series(
        hospital_codes,
        index=DF.index,
        dtype="Int64",
    ).replace(-1, pd.NA)

    set_feature(
        "Hospital_cluster",
        hospital_cluster,
        "cluster",
        selected_hospital_column,
        "Pseudonymous hospital/facility cluster.",
        "Available for clustered inference if clinically appropriate.",
    )

    log(
        f"Hospital clustering source: {selected_hospital_column}"
    )

else:

    log(
        "WARNING: no usable hospital/facility identifier found."
    )


# =============================================================================
# 14. CORE FINANCIAL VARIABLES
# =============================================================================

total_cost = to_num(
    DF["Total Cost"]
)

patient_paid = to_num(
    DF["Insured Person's Share"]
)

organization_paid = to_num(
    DF["Organization Paid Share"]
)

organization_requested = to_num(
    column_or_na("Organization Requested Share")
)

terminal_patient_paid = to_num(
    column_or_na("Terminally Patients Paid Share")
)

terminal_patient_requested = to_num(
    column_or_na("Terminally Patients Requested Share")
)

fx_allocated = to_num(
    column_or_na("Allocated subsidized foreign currency share")
)

fx_requested = to_num(
    column_or_na("Requested subsidized foreign currency share")
)

valid_cost = (
    total_cost.notna()
    & total_cost.gt(0)
)


# =============================================================================
# 15. PRIMARY OOP OUTCOME
# =============================================================================

oop_raw = (
    patient_paid / total_cost
)

set_feature(
    "oop_share_raw",
    oop_raw,
    "audit",
    "Insured Person's Share; Total Cost",
    "Patient payment divided by total episode cost before range validation.",
    "Values outside [0,1] are retained for audit and are never clipped.",
)

oop_range_candidate = (
    valid_cost
    & patient_paid.notna()
)

oop_outside = pd.Series(
    pd.NA,
    index=DF.index,
    dtype="boolean",
)

oop_outside.loc[oop_range_candidate] = (
    ~oop_raw.loc[oop_range_candidate].between(
        0,
        1,
        inclusive="both",
    )
)

oop_valid = (
    valid_cost
    & patient_paid.notna()
    & oop_raw.between(
        0,
        1,
        inclusive="both",
    )
)

oop_share = oop_raw.where(
    oop_valid
)

set_feature(
    "oop_share",
    oop_share,
    "outcome_primary",
    "Insured Person's Share; Total Cost",
    "Primary patient out-of-pocket share of reported total episode cost.",
    "Primary outcome. This does not require complete financing reconciliation.",
)


oop_band = pd.cut(
    oop_share,
    bins=OOP_BAND_CUTS,
    labels=OOP_BAND_LABELS,
    right=False,
)

oop_band = pd.Series(
    oop_band.astype("object"),
    index=DF.index,
).where(
    oop_share.notna(),
    MISSING_LABEL,
)

set_feature(
    "oop_band",
    oop_band,
    "outcome_secondary",
    "oop_share",
    "Pre-specified OOP-share bands.",
    "Descriptive outcome only.",
)


zero_oop = nullable_flag(
    oop_share.eq(0),
    valid=oop_share.notna(),
)

set_feature(
    "zero_oop",
    zero_oop,
    "outcome_secondary",
    "oop_share",
    "1 if OOP share equals exactly zero.",
)


# =============================================================================
# 16. HIGH OOP INDICATORS
# =============================================================================

high_05 = nullable_flag(
    oop_share.ge(0.05),
    valid=oop_share.notna(),
)

high_10 = nullable_flag(
    oop_share.ge(0.10),
    valid=oop_share.notna(),
)

high_20 = nullable_flag(
    oop_share.ge(0.20),
    valid=oop_share.notna(),
)

high_30 = nullable_flag(
    oop_share.ge(0.30),
    valid=oop_share.notna(),
)

set_feature(
    "high_oop_ge05pct",
    high_05,
    "outcome_sensitivity",
    "oop_share",
    "1 if OOP share is >=5%.",
    "High OOP-share indicator; not catastrophic expenditure.",
)

set_feature(
    "high_oop_ge10pct",
    high_10,
    "outcome_primary",
    "oop_share",
    "1 if OOP share is >=10%.",
    "Primary high OOP-share threshold; not catastrophic expenditure.",
)

set_feature(
    "high_oop_ge20pct",
    high_20,
    "outcome_sensitivity",
    "oop_share",
    "1 if OOP share is >=20%.",
    "High OOP-share indicator; not catastrophic expenditure.",
)

set_feature(
    "high_oop_ge30pct",
    high_30,
    "outcome_sensitivity",
    "oop_share",
    "1 if OOP share is >=30%.",
    "High OOP-share indicator; not catastrophic expenditure.",
)

alias_feature(
    "high_oop_share",
    "high_oop_ge10pct",
    "outcome_primary",
    "Alias for the primary >=10% high OOP-share indicator.",
)


# =============================================================================
# 17. FULL FINANCING RECONCILIATION — V0 TO V4
# =============================================================================

# -------------------------------------------------------------------------
# V0: patient paid + organization paid
# -------------------------------------------------------------------------

known_v0 = (
    patient_paid
    + organization_paid
)

complete_v0 = (
    valid_cost
    & patient_paid.notna()
    & organization_paid.notna()
)

gap_v0 = (
    total_cost - known_v0
).where(complete_v0)

gap_pct_v0 = (
    100.0 * gap_v0 / total_cost
).where(complete_v0)


# -------------------------------------------------------------------------
# V1: V0 + terminally patients paid
# -------------------------------------------------------------------------

known_v1 = (
    patient_paid
    + organization_paid
    + terminal_patient_paid
)

complete_v1 = (
    valid_cost
    & patient_paid.notna()
    & organization_paid.notna()
    & terminal_patient_paid.notna()
)

gap_v1 = (
    total_cost - known_v1
).where(complete_v1)

gap_pct_v1 = (
    100.0 * gap_v1 / total_cost
).where(complete_v1)


# -------------------------------------------------------------------------
# V2: V1 + allocated subsidized foreign currency
# -------------------------------------------------------------------------

known_v2 = (
    patient_paid
    + organization_paid
    + terminal_patient_paid
    + fx_allocated
)

complete_v2 = (
    valid_cost
    & patient_paid.notna()
    & organization_paid.notna()
    & terminal_patient_paid.notna()
    & fx_allocated.notna()
)

gap_v2 = (
    total_cost - known_v2
).where(complete_v2)

gap_pct_v2 = (
    100.0 * gap_v2 / total_cost
).where(complete_v2)


# -------------------------------------------------------------------------
# V3: requested basis
# patient paid
# + organization requested
# + terminal patient requested
# + requested FX
# -------------------------------------------------------------------------

known_v3 = (
    patient_paid
    + organization_requested
    + terminal_patient_requested
    + fx_requested
)

complete_v3 = (
    valid_cost
    & patient_paid.notna()
    & organization_requested.notna()
    & terminal_patient_requested.notna()
    & fx_requested.notna()
)

gap_v3 = (
    total_cost - known_v3
).where(complete_v3)

gap_pct_v3 = (
    100.0 * gap_v3 / total_cost
).where(complete_v3)


# -------------------------------------------------------------------------
# V4: V2 + insurer deduction
#
# insurer deduction =
#       organization requested - organization paid
#
# Therefore:
#
# V4 =
# patient paid
# + organization paid
# + terminal patient paid
# + allocated FX
# + organization requested
# - organization paid
#
# = patient paid
# + organization requested
# + terminal patient paid
# + allocated FX
# -------------------------------------------------------------------------

insurer_deduction = (
    organization_requested
    - organization_paid
)

known_v4 = (
    known_v2
    + insurer_deduction
)

complete_v4 = (
    valid_cost
    & patient_paid.notna()
    & organization_paid.notna()
    & organization_requested.notna()
    & terminal_patient_paid.notna()
    & fx_allocated.notna()
)

gap_v4 = (
    total_cost - known_v4
).where(complete_v4)

gap_pct_v4 = (
    100.0 * gap_v4 / total_cost
).where(complete_v4)


# =============================================================================
# 18. SAVE ALL FINANCING VARIANTS AS EXPLICIT FEATURES
# =============================================================================

financing_feature_specs = [
    (
        "financing_gap_v0_rial",
        gap_v0,
        "V0 gap: Total Cost minus patient paid and organization paid.",
    ),
    (
        "financing_gap_v0_pct",
        gap_pct_v0,
        "V0 signed financing gap as percentage of Total Cost.",
    ),
    (
        "financing_gap_v1_rial",
        gap_v1,
        "V1 gap after adding terminally patients paid.",
    ),
    (
        "financing_gap_v1_pct",
        gap_pct_v1,
        "V1 signed financing gap percentage.",
    ),
    (
        "financing_gap_v2_rial",
        gap_v2,
        "V2 gap after adding allocated subsidized foreign currency.",
    ),
    (
        "financing_gap_v2_pct",
        gap_pct_v2,
        "V2 signed financing gap percentage.",
    ),
    (
        "financing_gap_v3_rial",
        gap_v3,
        "V3 requested-basis financing gap.",
    ),
    (
        "financing_gap_v3_pct",
        gap_pct_v3,
        "V3 requested-basis financing gap percentage.",
    ),
    (
        "financing_gap_v4_rial",
        gap_v4,
        "V4 gap after adding insurer deduction to V2.",
    ),
    (
        "financing_gap_v4_pct",
        gap_pct_v4,
        "V4 signed financing gap percentage.",
    ),
]

for name, series, definition in financing_feature_specs:

    set_feature(
        name,
        series,
        "financing_audit",
        "Total Cost; Insured Person's Share; Organization Paid Share; "
        "Organization Requested Share; Terminally Patients Paid Share; "
        "Terminally Patients Requested Share; "
        "Allocated subsidized foreign currency share; "
        "Requested subsidized foreign currency share",
        definition,
        "Audit/sensitivity variable. Not used to redefine primary OOP.",
    )


# =============================================================================
# 19. FINANCING COMPLETENESS
# =============================================================================

set_feature(
    "financing_complete_v0",
    nullable_flag(
        complete_v0
    ),
    "financing_audit",
    "V0 financing components",
    "1 when all V0 financing components are available.",
)

set_feature(
    "financing_complete_v1",
    nullable_flag(
        complete_v1
    ),
    "financing_audit",
    "V1 financing components",
    "1 when all V1 financing components are available.",
)

set_feature(
    "financing_complete_v2",
    nullable_flag(
        complete_v2
    ),
    "financing_audit",
    "V2 financing components",
    "1 when all V2 financing components are available.",
)

set_feature(
    "financing_complete_v3",
    nullable_flag(
        complete_v3
    ),
    "financing_audit",
    "V3 financing components",
    "1 when all requested-basis components are available.",
)

set_feature(
    "financing_complete_v4",
    nullable_flag(
        complete_v4
    ),
    "financing_audit",
    "V4 financing components",
    "1 when all V4 components are available.",
)


# =============================================================================
# 20. PRIMARY V0 FINANCING AUDIT
# =============================================================================

set_feature(
    "financing_gap_pct",
    gap_pct_v0,
    "audit",
    "Total Cost; Insured Person's Share; Organization Paid Share",
    "Signed V0 financing gap percentage.",
    "Retained for compatibility with previous analysis.",
)

set_feature(
    "unreconciled_share",
    gap_pct_v0 / 100.0,
    "audit",
    "financing_gap_pct",
    "V0 signed financing gap expressed as a proportion.",
)

v0_positive = (
    gap_pct_v0 > FINANCING_GAP_TOLERANCE_PCT
)

v0_negative = (
    gap_pct_v0 < -FINANCING_GAP_TOLERANCE_PCT
)

set_feature(
    "financing_gap_positive",
    nullable_flag(
        v0_positive,
        valid=gap_pct_v0.notna(),
    ),
    "audit_flag",
    "financing_gap_pct",
    f"1 when V0 positive gap exceeds {FINANCING_GAP_TOLERANCE_PCT}%.",
)

set_feature(
    "financing_gap_negative",
    nullable_flag(
        v0_negative,
        valid=gap_pct_v0.notna(),
    ),
    "audit_flag",
    f"financing_gap_pct",
    f"1 when V0 negative gap exceeds {FINANCING_GAP_TOLERANCE_PCT}%.",
)

set_feature(
    "flag_financing_gap",
    nullable_flag(
        gap_pct_v0.abs() > FINANCING_GAP_TOLERANCE_PCT,
        valid=gap_pct_v0.notna(),
    ),
    "audit_flag",
    "financing_gap_pct",
    f"1 when absolute V0 financing gap exceeds {FINANCING_GAP_TOLERANCE_PCT}%.",
)

alias_feature(
    "flag_gap_unexplained",
    "financing_gap_positive",
    "audit_flag",
    "Alias for positive V0 financing gap.",
)

alias_feature(
    "flag_gap_overpaid",
    "financing_gap_negative",
    "audit_flag",
    "Alias for negative V0 financing gap.",
)


# =============================================================================
# 21. REQUESTED-BASED FINANCING AUDIT
# =============================================================================

flag_v3_gap = nullable_flag(
    gap_pct_v3.abs() > FINANCING_GAP_TOLERANCE_PCT,
    valid=gap_pct_v3.notna(),
)

set_feature(
    "flag_financing_gap_v3_requested",
    flag_v3_gap,
    "audit_flag",
    "financing_gap_v3_pct",
    f"1 when absolute V3 requested-basis gap exceeds {FINANCING_GAP_TOLERANCE_PCT}%.",
    "This is the requested-component reconciliation audit.",
)


# =============================================================================
# 22. INSURER DEDUCTION
# =============================================================================

set_feature(
    "insurer_deduction",
    insurer_deduction.where(
        organization_requested.notna()
        & organization_paid.notna()
    ),
    "financing_audit",
    "Organization Requested Share; Organization Paid Share",
    "Difference between requested and paid organization share.",
)

set_feature(
    "insurer_deduction_share",
    (
        insurer_deduction / total_cost
    ).where(
        valid_cost
        & insurer_deduction.notna()
    ),
    "financing_audit",
    "insurer_deduction; Total Cost",
    "Insurer deduction expressed as a share of total cost.",
)


# =============================================================================
# 23. ORGANIZATION SHARE AND ALTERNATIVE OOP DENOMINATOR
# =============================================================================

insurer_share = (
    organization_paid / total_cost
).where(
    valid_cost
    & organization_paid.notna()
)

set_feature(
    "insurer_share",
    insurer_share,
    "outcome_secondary",
    "Organization Paid Share; Total Cost",
    "Organization/insurer paid share of total episode cost.",
)


org_deduction_share = (
    insurer_deduction / total_cost
).where(
    valid_cost
    & insurer_deduction.notna()
)

set_feature(
    "org_deduction_share",
    org_deduction_share,
    "outcome_secondary",
    "Organization Requested Share; Organization Paid Share; Total Cost",
    "Requested minus paid organization share divided by total cost.",
)


# Alternative denominator:
#
# patient paid / (patient paid + organization paid)
#
# This is NOT the primary outcome.

known_paid_total = (
    patient_paid
    + organization_paid
)

valid_known_denominator = (
    known_paid_total.notna()
    & known_paid_total.gt(0)
    & patient_paid.notna()
)

oop_share_of_known = (
    patient_paid / known_paid_total
).where(
    valid_known_denominator
)

set_feature(
    "oop_share_of_known",
    oop_share_of_known,
    "outcome_sensitivity",
    "Insured Person's Share; Organization Paid Share",
    "Patient payment divided by patient payment plus organization-paid amount.",
    "Sensitivity outcome only; not the primary OOP definition.",
)

set_feature(
    "valid_oop_known_denominator",
    nullable_flag(
        valid_known_denominator,
    ),
    "sample_flag",
    "oop_share_of_known",
    "1 when the alternative known-payment denominator is positive and available.",
)


set_feature(
    "log_total_cost",
    np.log(
        total_cost.where(
            valid_cost
        )
    ),
    "covariate_process",
    "Total Cost",
    "Natural logarithm of total episode cost.",
    "Post-admission/process variable; do not treat as baseline predictor.",
)


# =============================================================================
# 24. AGE
# =============================================================================

age_raw = to_num(
    DF["Age"]
)

invalid_age_calculated = (
    age_raw.notna()
    & (
        age_raw.lt(0)
        | age_raw.gt(120)
    )
)

age = age_raw.mask(
    invalid_age_calculated
)

set_feature(
    "age",
    age,
    "covariate_baseline",
    "Age",
    "Age in years after basic validity screening.",
)

age_group = pd.cut(
    age,
    bins=AGE_BINS,
    labels=AGE_LABELS,
    right=False,
    include_lowest=True,
)

age_group = pd.Series(
    age_group.astype("object"),
    index=DF.index,
)

age_group = age_group.where(
    age.notna(),
    MISSING_LABEL,
)

set_feature(
    "age_group",
    age_group,
    "covariate_baseline",
    "Age",
    "Age groups: 0-4, 5-14, 15-44, 45-64, 65+.",
)

# IMPORTANT FIX:
# Missingness is directly defined from age.isna().
# It is NOT masked by age.notna().

set_feature(
    "age_missing",
    nullable_flag(
        age.isna(),
        valid=pd.Series(True, index=DF.index),
    ),
    "audit",
    "Age",
    "1 if age is missing after validity screening.",
)


# =============================================================================
# 25. AGE VALIDITY FLAG
# =============================================================================

pre_invalid_age = flag_from_source(
    "contains_invalid_age"
).astype("boolean")

calc_invalid_age = pd.Series(
    invalid_age_calculated,
    index=DF.index,
).astype("boolean")

combined_invalid_age = (
    calc_invalid_age.fillna(False)
    | pre_invalid_age.fillna(False)
).astype("boolean")

known_invalid_age = (
    calc_invalid_age.notna()
    | pre_invalid_age.notna()
)

combined_invalid_age = combined_invalid_age.where(
    known_invalid_age
)

set_feature(
    "flag_invalid_age",
    combined_invalid_age.astype("Int8"),
    "audit_flag",
    "Age; contains_invalid_age",
    "1 if age is outside the valid range or preprocessing identified invalid age.",
)


# =============================================================================
# 26. GENDER
# =============================================================================

gender_clean = clean_text(
    DF["Gender"]
)

gender_key = gender_clean.str.casefold()

is_male = pd.Series(
    pd.NA,
    index=DF.index,
    dtype="Int8",
)

is_male.loc[
    gender_key.isin(MALE_LABELS)
] = 1

is_male.loc[
    gender_key.isin(FEMALE_LABELS)
] = 0

set_feature(
    "is_male",
    is_male,
    "covariate_baseline",
    "Gender",
    "1 male, 0 female, NA otherwise.",
)

sex = pd.Series(
    MISSING_LABEL,
    index=DF.index,
    dtype="object",
)

sex.loc[
    is_male.eq(1)
] = "Male"

sex.loc[
    is_male.eq(0)
] = "Female"

set_feature(
    "sex",
    sex,
    "covariate_baseline",
    "Gender",
    "Male / Female / Missing.",
)


# =============================================================================
# 27. ADMISSION DATE FEATURES
# =============================================================================

admit = pd.to_datetime(
    DF["Admision date"],
    errors="coerce",
)

set_feature(
    "admission_year",
    admit.dt.year.astype("Int64"),
    "covariate_baseline",
    "Admision date",
    "Calendar year of admission.",
)

set_feature(
    "admission_month",
    admit.dt.month.astype("Int64"),
    "covariate_baseline",
    "Admision date",
    "Calendar month of admission.",
)

set_feature(
    "admission_ym",
    admit.dt.strftime("%Y-%m").astype("string"),
    "covariate_baseline",
    "Admision date",
    "Year-month of admission.",
)

set_feature(
    "admission_quarter",
    admit.dt.quarter.astype("Int64"),
    "covariate_baseline",
    "Admision date",
    "Calendar quarter of admission.",
)

set_feature(
    "admission_dayofweek",
    admit.dt.dayofweek.astype("Int64"),
    "covariate_baseline",
    "Admision date",
    "Day of week; Monday=0.",
)

set_feature(
    "admission_friday",
    nullable_flag(
        admit.dt.dayofweek.eq(4),
        valid=admit.notna(),
    ),
    "covariate_baseline",
    "Admision date",
    "1 if admission occurred on Friday.",
)

if admit.notna().any():

    earliest_admission = admit.min()

    days_since_start = (
        admit - earliest_admission
    ).dt.days.astype("Int64")

else:

    days_since_start = pd.Series(
        pd.NA,
        index=DF.index,
        dtype="Int64",
    )

set_feature(
    "admission_days_since_start",
    days_since_start,
    "covariate_baseline",
    "Admision date",
    "Number of days since earliest observed admission.",
)

month_numeric = admit.dt.month.astype(float)

set_feature(
    "month_sin",
    np.sin(
        2 * np.pi * month_numeric / 12
    ),
    "covariate_baseline",
    "Admision date",
    "Seasonal sine transformation of admission month.",
)

set_feature(
    "month_cos",
    np.cos(
        2 * np.pi * month_numeric / 12
    ),
    "covariate_baseline",
    "Admision date",
    "Seasonal cosine transformation of admission month.",
)


# =============================================================================
# 28. CATEGORICAL VARIABLES
# =============================================================================

# Do not perform semantic merges unless explicitly specified.
MANUAL_CATEGORY_MERGES = {
    # Example:
    # "insurance fund": {
    #     "some exact variant": "canonical label",
    # },
}

category_specs = [
    (
        "Insurance Fund",
        "insurance_fund_grp",
        "exposure_group",
        "Insurance fund / payer type.",
    ),
    (
        "Ownership",
        "ownership_grp",
        "exposure_group",
        "Hospital ownership.",
    ),
    (
        "Department Type",
        "department_grp",
        "covariate_baseline",
        "Department type.",
    ),
    (
        "Diagnosis type",
        "diagnosis_type_grp",
        "covariate_baseline",
        "Admission / billing type.",
    ),
    (
        "Treating Physician Specialty",
        "specialty_grp",
        "covariate_baseline",
        "Treating physician specialty.",
    ),
]

category_maps = []

for source_col, output_col, role, definition in category_specs:

    original = clean_text(
        DF[source_col]
    )

    normalized = normalize_category(
        original
    )

    merge_map = MANUAL_CATEGORY_MERGES.get(
        source_col,
        {},
    )

    normalized = apply_manual_merges(
        normalized,
        merge_map,
    )

    grouped = collapse_rare(
        normalized,
        minimum_n=MIN_CATEGORY_N,
    )

    set_feature(
        output_col,
        grouped,
        role,
        source_col,
        (
            f"{definition}; categories with n < "
            f"{MIN_CATEGORY_N} collapsed to '{RARE_LABEL}'."
        ),
        (
            "Case/whitespace variants merged. "
            "Missing retained as Missing. "
            "Outcome variables are not used to construct categories."
        ),
    )

    category_maps.append(
        build_category_map(
            normalized,
            grouped,
            output_col,
        )
    )


category_collapse_map = pd.concat(
    category_maps,
    ignore_index=True,
)


# =============================================================================
# 29. OPTIONAL EXTERNAL SES ORDERING
# =============================================================================

if EXTERNAL_FUND_SES_FILE.exists():

    try:

        ses_order = pd.read_csv(
            EXTERNAL_FUND_SES_FILE
        )

        required_ses_cols = {
            "Insurance Fund",
            "ses_order",
        }

        if required_ses_cols.issubset(
            ses_order.columns
        ):

            ses_lookup = (
                ses_order
                .assign(
                    fund_key=lambda x:
                        clean_text(
                            x["Insurance Fund"]
                        ).str.casefold()
                )
                .drop_duplicates("fund_key")
                .set_index("fund_key")["ses_order"]
            )

            fund_key = (
                clean_text(
                    DF["Insurance Fund"]
                )
                .str.casefold()
            )

            fund_ses = fund_key.map(
                ses_lookup
            )

            set_feature(
                "insurance_fund_ses_order",
                pd.to_numeric(
                    fund_ses,
                    errors="coerce",
                ),
                "exposure_order",
                "Insurance Fund; external_fund_ses_order.csv",
                "Externally supplied SES ordering of insurance funds.",
                "Only use if independently justified and documented.",
            )

            log(
                "External insurance-fund SES ordering applied."
            )

        else:

            log(
                "WARNING: external SES file exists but "
                "required columns are missing."
            )

    except Exception as exc:

        log(
            f"WARNING: could not read external SES ordering: {exc}"
        )

else:

    log(
        "No external SES ordering supplied."
    )

    log(
        "Insurance fund is retained as a payer category, "
        "not treated as socioeconomic rank."
    )


# =============================================================================
# 30. LENGTH OF STAY / PROCESS VARIABLES
# =============================================================================

los_days = to_num(
    column_or_na("Episode_LOS_days")
)

set_feature(
    "los_days",
    los_days,
    "covariate_process",
    "Episode_LOS_days",
    "Length of stay in days.",
    "Post-admission/process variable; use cautiously in causal models.",
)

set_feature(
    "los_zero",
    nullable_flag(
        los_days.eq(0),
        valid=los_days.notna(),
    ),
    "audit",
    "los_days",
    "1 if recorded LOS equals zero.",
    "Zero-LOS episodes are retained and separately audited.",
)

los_group = pd.Series(
    MISSING_LABEL,
    index=DF.index,
    dtype="object",
)

los_group.loc[
    los_days.eq(0)
] = "0_same_day"

los_group.loc[
    los_days.eq(1)
] = "1"

los_group.loc[
    los_days.between(
        2,
        3,
        inclusive="both",
    )
] = "2_3"

los_group.loc[
    los_days.ge(4)
] = "4plus"

set_feature(
    "los_group",
    los_group,
    "covariate_process",
    "los_days",
    "LOS groups: 0, 1, 2-3, 4+ days.",
)

cost_per_day = (
    total_cost
    / los_days.clip(lower=1)
).where(
    valid_cost
    & los_days.notna()
)

set_feature(
    "cost_per_day",
    cost_per_day,
    "covariate_process",
    "Total Cost; los_days",
    "Total cost divided by max(LOS,1).",
    "Descriptive/process variable; not a baseline predictor.",
)


# =============================================================================
# 31. CLAIM LINE / SERVICE GROUP COUNTS
# =============================================================================

n_claim_lines = to_num(
    column_or_na("Number_of_claim_lines")
).round().astype("Int64")

set_feature(
    "n_claim_lines",
    n_claim_lines,
    "covariate_process",
    "Number_of_claim_lines",
    "Number of claim lines within episode.",
)

n_service_groups = to_num(
    column_or_na("Number_of_service_groups")
).round().astype("Int64")

set_feature(
    "n_service_groups",
    n_service_groups,
    "covariate_process",
    "Number_of_service_groups",
    "Number of distinct service groups within episode.",
)


# =============================================================================
# 32. DISCHARGE STATUS
# =============================================================================

if "Discharge Status" in DF.columns:

    discharge_status = clean_text(
        DF["Discharge Status"]
    ).fillna(MISSING_LABEL)

else:

    discharge_status = pd.Series(
        pd.NA,
        index=DF.index,
        dtype="string",
    )

set_feature(
    "discharge_status",
    discharge_status,
    "descriptive_post_discharge",
    "Discharge Status",
    "Discharge-status category.",
    "Post-discharge descriptive variable.",
)

death_signal = safe_contains(
    discharge_status,
    [
        "death",
        "died",
        "deceased",
        "dead",
    ],
)

set_feature(
    "died_in_hospital",
    nullable_flag(
        death_signal,
        valid=discharge_status.notna()
        & discharge_status.ne(MISSING_LABEL),
    ),
    "descriptive_post_discharge",
    "Discharge Status",
    "1 when discharge status contains an explicit death-related keyword.",
    "Descriptive audit only; verify against source coding before clinical interpretation.",
)


# =============================================================================
# 33. DATE-ORDER AUDIT
# =============================================================================

discharge = pd.to_datetime(
    column_or_na("Discharge date"),
    errors="coerce",
)

if "Discharge date" in DF.columns:

    missing_discharge = discharge.isna()

else:

    missing_discharge = pd.Series(
        pd.NA,
        index=DF.index,
        dtype="boolean",
    )

set_feature(
    "flag_missing_discharge_date",
    nullable_flag(
        missing_discharge,
        valid=pd.Series(
            True,
            index=DF.index,
        ),
    ),
    "audit_flag",
    "Discharge date",
    "1 when discharge date is missing.",
    "Missing discharge date does not itself imply an invalid episode.",
)


date_order_calculated = pd.Series(
    pd.NA,
    index=DF.index,
    dtype="boolean",
)

both_dates = (
    admit.notna()
    & discharge.notna()
)

date_order_calculated.loc[
    both_dates
] = (
    discharge.loc[both_dates]
    < admit.loc[both_dates]
)

date_order_source = flag_from_source(
    "invalid_date_order"
).astype("boolean")

date_order_combined = (
    date_order_calculated.fillna(False)
    | date_order_source.fillna(False)
).astype("boolean")

date_order_known = (
    date_order_calculated.notna()
    | date_order_source.notna()
)

date_order_combined = date_order_combined.where(
    date_order_known
)

set_feature(
    "flag_invalid_date_order",
    date_order_combined.astype("Int8"),
    "audit_flag",
    "Admision date; Discharge date; invalid_date_order",
    "1 if discharge precedes admission.",
    "Calculated from dates where possible and reconciled with preprocessing flag.",
)


# =============================================================================
# 34. LOS / DATE CONSISTENCY
# =============================================================================

date_based_los = (
    discharge.dt.normalize()
    - admit.dt.normalize()
).dt.days

los_mismatch_candidate = (
    date_based_los.notna()
    & los_days.notna()
)

los_mismatch = pd.Series(
    pd.NA,
    index=DF.index,
    dtype="boolean",
)

los_mismatch.loc[
    los_mismatch_candidate
] = (
    date_based_los.loc[los_mismatch_candidate]
    != los_days.loc[los_mismatch_candidate]
)

set_feature(
    "flag_los_mismatch",
    los_mismatch.astype("Int8"),
    "audit_flag",
    "Admision date; Discharge date; Episode_LOS_days",
    "1 when recorded LOS differs from calendar-date difference.",
    "NA when dates or LOS are unavailable.",
)


# =============================================================================
# 35. DUPLICATE-CLAIM AUDIT
# =============================================================================

set_feature(
    "flag_duplicate_claim",
    flag_from_source(
        "contains_exact_duplicate_claim"
    ),
    "audit_flag",
    "contains_exact_duplicate_claim",
    "1 if preprocessing identified an exact duplicate claim line.",
    "Unavailable values remain NA and are never silently converted to zero.",
)


# =============================================================================
# 36. INVALID TOTAL COST / OOP RANGE
# =============================================================================

set_feature(
    "flag_invalid_total_cost",
    nullable_flag(
        ~valid_cost,
        valid=pd.Series(
            True,
            index=DF.index,
        ),
    ),
    "audit_flag",
    "Total Cost",
    "1 if total cost is missing or <=0.",
)

set_feature(
    "flag_oop_out_of_range",
    oop_outside.astype("Int8"),
    "audit_flag",
    "Total Cost; Insured Person's Share",
    "1 if calculated patient OOP share is outside [0,1].",
    "Out-of-range observations are not clipped.",
)


# =============================================================================
# 37. ANONYMOUS EPISODE FLAG
# =============================================================================

anonymous_from_patient = pd.Series(
    pd.NA,
    index=DF.index,
    dtype="boolean",
)

if "Patient_ID" in DF.columns:

    anonymous_from_patient = (
        clean_text(
            DF["Patient_ID"]
        ).isna()
    ).astype("boolean")


anonymous_from_type = pd.Series(
    pd.NA,
    index=DF.index,
    dtype="boolean",
)

if "Episode_Identification_Type" in DF.columns:

    identification_text = clean_text(
        DF["Episode_Identification_Type"]
    ).str.casefold()

    anonymous_from_type.loc[
        identification_text.str.contains(
            "anonymous|anon",
            regex=True,
            na=False,
        )
    ] = True

    anonymous_from_type.loc[
        identification_text.str.contains(
            "identified|patient",
            regex=True,
            na=False,
        )
    ] = False


anonymous_combined = (
    anonymous_from_type.fillna(
        anonymous_from_patient.fillna(False)
    )
).astype("boolean")

anonymous_known = (
    anonymous_from_type.notna()
    | anonymous_from_patient.notna()
)

anonymous_combined = anonymous_combined.where(
    anonymous_known
)

set_feature(
    "flag_anonymous_episode",
    anonymous_combined.astype("Int8"),
    "audit_flag",
    "Patient_ID; Episode_Identification_Type",
    "1 if episode is anonymous according to available identification information.",
    "Anonymous episodes are retained in the primary cohort.",
)


# =============================================================================
# 38. CLINICAL SETTING AUDIT — REASON-SPECIFIC, NOT "IMPLAUSIBLE"
# =============================================================================

specialty_text = clean_text(
    DF["Treating Physician Specialty"]
).fillna("")

department_text = clean_text(
    DF["Department Type"]
).fillna("")

combined_setting_text = (
    specialty_text
    + " | "
    + department_text
).str.lower()


setting_audit_columns = []

for reason, keywords in AUDIT_KEYWORDS.items():

    flag = safe_contains(
        combined_setting_text,
        keywords,
    )

    feature_name = (
        "audit_setting_"
        + re.sub(
            r"[^a-z0-9]+",
            "_",
            reason.lower(),
        ).strip("_")
    )

    set_feature(
        feature_name,
        nullable_flag(
            flag,
            valid=pd.Series(
                True,
                index=DF.index,
            ),
        ),
        "clinical_setting_audit",
        "Treating Physician Specialty; Department Type",
        (
            f"1 if specialty or department contains a "
            f"pre-specified {reason} audit keyword."
        ),
        (
            "Audit signal only. It is not a clinical case definition "
            "and must not automatically exclude the episode."
        ),
    )

    setting_audit_columns.append(
        feature_name
    )


# Overall setting-audit signal:
setting_audit_matrix = (
    OUT[setting_audit_columns]
    .astype("Float64")
)

any_setting_signal = (
    setting_audit_matrix.eq(1)
    .any(axis=1)
)

set_feature(
    "audit_any_setting_signal",
    nullable_flag(
        any_setting_signal,
        valid=pd.Series(
            True,
            index=DF.index,
        ),
    ),
    "clinical_setting_audit",
    "Treating Physician Specialty; Department Type",
    "1 if any pre-specified specialty/department audit signal is present.",
    (
        "Audit only. NOT evidence that the episode is non-cholera. "
        "Do not use this variable to define the primary cohort."
    ),
)


# Specific cardiac signal retained for compatibility.
alias_feature(
    "audit_cardiac_specialty_or_dept",
    "audit_setting_cardiac",
    "audit_flag",
    "Cardiac specialty/department audit signal.",
    "Audit only; not a cohort exclusion criterion.",
)


# =============================================================================
# 39. PRIMARY ANALYTIC SAMPLE
# =============================================================================
#
# PRIMARY SAMPLE CRITERIA:
#
#   1. Total cost > 0
#   2. Patient payment available
#   3. OOP ratio bounded [0,1]
#   4. Admission date available
#   5. No known discharge-before-admission error
#
# IMPORTANT:
# Missing discharge date does NOT automatically exclude an episode.
# It has its own audit flag.
#
# =============================================================================

primary_bool = (
    valid_cost
    & patient_paid.notna()
    & oop_valid
    & admit.notna()
    & (
        OUT["flag_invalid_date_order"]
        .ne(1)
        .fillna(True)
    )
)

set_feature(
    "analytic_primary",
    nullable_flag(
        primary_bool,
        valid=pd.Series(
            True,
            index=DF.index,
        ),
    ),
    "sample_flag",
    "financial validity; admission date; date-order audit",
    "Primary analytic sample indicator.",
    (
        "Requires valid primary OOP outcome, valid admission date, "
        "and no known discharge-before-admission error. "
        "Does not automatically exclude anonymous, zero-LOS, "
        "financing-gap, or keyword-flagged episodes."
    ),
)


# =============================================================================
# 40. COMPLETE-CASE COVARIATES
# =============================================================================

complete_core = (
    primary_bool
    & OUT["age"].notna()
    & OUT["is_male"].notna()
    & OUT["insurance_fund_grp"].ne(MISSING_LABEL)
    & OUT["ownership_grp"].ne(MISSING_LABEL)
    & OUT["department_grp"].ne(MISSING_LABEL)
)

set_feature(
    "complete_case_covariates",
    nullable_flag(
        complete_core,
        valid=pd.Series(
            True,
            index=DF.index,
        ),
    ),
    "sample_flag",
    "age; sex; insurance fund; ownership; department",
    "Primary sample with complete core baseline covariates.",
    "Sensitivity/adjusted-analysis sample.",
)


# =============================================================================
# 41. SENSITIVITY SAMPLES
# =============================================================================

set_feature(
    "sens_excl_anonymous",
    nullable_flag(
        primary_bool
        & OUT["flag_anonymous_episode"].eq(0).fillna(False)
    ),
    "sample_flag",
    "analytic_primary; flag_anonymous_episode",
    "Primary sample excluding anonymous episodes.",
)


if "Episode_LOS_days" in DF.columns:

    nonzero_los = (
        los_days.notna()
        & los_days.gt(0)
    )

else:

    nonzero_los = pd.Series(
        pd.NA,
        index=DF.index,
        dtype="boolean",
    )


set_feature(
    "sens_excl_zero_los",
    nullable_flag(
        primary_bool & nonzero_los
    ),
    "sample_flag",
    "analytic_primary; los_days",
    "Primary sample excluding zero-LOS episodes.",
    "Zero-LOS exclusion is a sensitivity analysis, not the primary definition.",
)


set_feature(
    "sens_excl_financing_gap",
    nullable_flag(
        primary_bool
        & complete_v0
        & OUT["flag_financing_gap"].eq(0)
    ),
    "sample_flag",
    "analytic_primary; V0 financing reconciliation",
    "Primary sample excluding episodes with >5% absolute V0 financing gap.",
    "Sensitivity analysis only.",
)


set_feature(
    "sens_excl_financing_gap_v3_requested",
    nullable_flag(
        primary_bool
        & complete_v3
        & OUT["flag_financing_gap_v3_requested"].eq(0)
    ),
    "sample_flag",
    "analytic_primary; V3 requested-basis reconciliation",
    "Primary sample excluding episodes with >5% absolute V3 requested-basis gap.",
    "Sensitivity analysis only.",
)


# IMPORTANT:
# We DO NOT create a sample that excludes all keyword-flagged episodes.
# Instead, the audit variables are retained.
#
# This prevents the previous 168-episode "sens_excl_suspect_setting"
# from being mistaken for a clinically validated cholera cohort.

set_feature(
    "sens_excl_duplicates",
    nullable_flag(
        primary_bool
        & OUT["flag_duplicate_claim"].eq(0)
    ),
    "sample_flag",
    "analytic_primary; flag_duplicate_claim",
    "Primary sample excluding episodes flagged for duplicate claims.",
)


set_feature(
    "sens_excl_missing_discharge",
    nullable_flag(
        primary_bool
        & discharge.notna()
    ),
    "sample_flag",
    "analytic_primary; Discharge date",
    "Primary sample restricted to episodes with observed discharge date.",
    "Sensitivity analysis only.",
)


# =============================================================================
# 42. FINAL FEATURE DICTIONARY INTEGRITY CHECK
# =============================================================================

dictionary_df = pd.DataFrame(
    DICTIONARY
)

output_columns = set(
    OUT.columns
)

dictionary_columns = set(
    dictionary_df["feature"]
)

undocumented = sorted(
    output_columns - dictionary_columns
)

stale_dictionary = sorted(
    dictionary_columns - output_columns
)

duplicate_dictionary = (
    dictionary_df["feature"]
    .duplicated()
    .any()
)

if undocumented:
    raise AssertionError(
        "Undocumented engineered columns:\n"
        + "\n".join(undocumented)
    )

if stale_dictionary:
    raise AssertionError(
        "Dictionary entries not present in output:\n"
        + "\n".join(stale_dictionary)
    )

if duplicate_dictionary:
    raise AssertionError(
        "Duplicate feature names in dictionary."
    )

if OUT.columns.duplicated().any():
    raise AssertionError(
        "Duplicate columns in analysis dataset."
    )

log("")
log("Feature/dictionary integrity check PASSED.")
log(
    f"Engineered variables: {len(OUT.columns)}"
)
log(
    f"Dictionary entries: {len(dictionary_df)}"
)


# =============================================================================
# 43. FINANCING RECONCILIATION SUMMARY
# =============================================================================

financing_variants = []

variant_definitions = {
    "V0_paid": (
        complete_v0,
        gap_pct_v0,
    ),
    "V1_paid_plus_terminal": (
        complete_v1,
        gap_pct_v1,
    ),
    "V2_paid_plus_terminal_plus_FX": (
        complete_v2,
        gap_pct_v2,
    ),
    "V3_requested": (
        complete_v3,
        gap_pct_v3,
    ),
    "V4_V2_plus_insurer_deduction": (
        complete_v4,
        gap_pct_v4,
    ),
}

for variant_name, (
    complete,
    gap_pct,
) in variant_definitions.items():

    valid_gap = gap_pct.dropna()

    if len(valid_gap) == 0:

        financing_variants.append({
            "variant": variant_name,
            "n_complete": 0,
            "pct_complete": np.nan,
            "n_gap_gt_5pct": 0,
            "pct_gap_gt_5pct": np.nan,
            "n_positive_gt_5pct": 0,
            "n_negative_gt_5pct": 0,
            "median_gap_pct": np.nan,
            "mean_gap_pct": np.nan,
            "p90_abs_gap_pct": np.nan,
        })

    else:

        financing_variants.append({
            "variant": variant_name,
            "n_complete": int(complete.sum()),
            "pct_complete": 100 * complete.mean(),
            "n_gap_gt_5pct": int(
                valid_gap.abs().gt(
                    FINANCING_GAP_TOLERANCE_PCT
                ).sum()
            ),
            "pct_gap_gt_5pct": 100 * (
                valid_gap.abs().gt(
                    FINANCING_GAP_TOLERANCE_PCT
                ).mean()
            ),
            "n_positive_gt_5pct": int(
                valid_gap.gt(
                    FINANCING_GAP_TOLERANCE_PCT
                ).sum()
            ),
            "n_negative_gt_5pct": int(
                valid_gap.lt(
                    -FINANCING_GAP_TOLERANCE_PCT
                ).sum()
            ),
            "median_gap_pct": valid_gap.median(),
            "mean_gap_pct": valid_gap.mean(),
            "p90_abs_gap_pct": valid_gap.abs().quantile(0.90),
        })

financing_reconciliation_summary = pd.DataFrame(
    financing_variants
)


# =============================================================================
# 44. COMPARE SOURCE FINANCING VARIABLES WITH RECOMPUTATION
# =============================================================================

source_financing_audit = pd.DataFrame()

if {
    "Known_financing_total",
    "Financing_difference",
    "Financing_difference_pct",
}.issubset(DF.columns):

    source_known = to_num(
        DF["Known_financing_total"]
    )

    source_difference = to_num(
        DF["Financing_difference"]
    )

    source_difference_pct = to_num(
        DF["Financing_difference_pct"]
    )

    source_financing_audit = pd.DataFrame({
        "metric": [
            "source_known_financing_total",
            "source_financing_difference",
            "source_financing_difference_pct",
            "recomputed_v0_known_financing_total",
            "recomputed_v0_difference",
            "recomputed_v0_difference_pct",
        ],
        "mean": [
            source_known.mean(),
            source_difference.mean(),
            source_difference_pct.mean(),
            known_v0.mean(),
            gap_v0.mean(),
            gap_pct_v0.mean(),
        ],
        "median": [
            source_known.median(),
            source_difference.median(),
            source_difference_pct.median(),
            known_v0.median(),
            gap_v0.median(),
            gap_pct_v0.median(),
        ],
    })

    comparison_mask = (
        source_difference_pct.notna()
        & gap_pct_v0.notna()
    )

    if comparison_mask.any():

        source_financing_audit = pd.concat([
            source_financing_audit,
            pd.DataFrame({
                "metric": [
                    "n_source_vs_recomputed",
                    "mean_abs_difference_pct_points",
                    "median_abs_difference_pct_points",
                    "max_abs_difference_pct_points",
                ],
                "mean": [
                    comparison_mask.sum(),
                    (
                        source_difference_pct[comparison_mask]
                        - gap_pct_v0[comparison_mask]
                    ).abs().mean(),
                    (
                        source_difference_pct[comparison_mask]
                        - gap_pct_v0[comparison_mask]
                    ).abs().median(),
                    (
                        source_difference_pct[comparison_mask]
                        - gap_pct_v0[comparison_mask]
                    ).abs().max(),
                ],
                "median": [
                    np.nan,
                    np.nan,
                    np.nan,
                    np.nan,
                ],
            }),
        ], ignore_index=True)


# =============================================================================
# 45. SAMPLE FLOW
# =============================================================================

sample_flow_rows = []

all_mask = pd.Series(
    True,
    index=DF.index,
)

step_masks = [
    (
        "Step 0: all episodes in frozen episode file",
        all_mask,
    ),
    (
        "Step 1: Total Cost > 0",
        valid_cost,
    ),
    (
        "Step 2: Patient share available",
        valid_cost
        & patient_paid.notna(),
    ),
    (
        "Step 3: OOP within [0,1]",
        valid_cost
        & patient_paid.notna()
        & oop_valid,
    ),
    (
        "Step 4: Valid admission date",
        valid_cost
        & patient_paid.notna()
        & oop_valid
        & admit.notna(),
    ),
    (
        "Step 5: No known discharge-before-admission error",
        primary_bool,
    ),
]

previous_n = len(DF)

for step_name, mask in step_masks:

    n_remaining = int(mask.sum())

    sample_flow_rows.append({
        "step": step_name,
        "n_remaining": n_remaining,
        "n_excluded_at_step": previous_n - n_remaining,
    })

    previous_n = n_remaining

sample_flow = pd.DataFrame(
    sample_flow_rows
)


# =============================================================================
# 46. SAMPLE SIZE TABLE
# =============================================================================

sample_size_specs = [
    (
        "analytic_primary",
        "Primary analytic sample.",
    ),
    (
        "complete_case_covariates",
        "Primary sample with complete core covariates.",
    ),
    (
        "sens_excl_anonymous",
        "Primary sample excluding anonymous episodes.",
    ),
    (
        "sens_excl_zero_los",
        "Primary sample excluding zero-LOS episodes.",
    ),
    (
        "sens_excl_financing_gap",
        "Primary sample excluding >5% V0 financing gap.",
    ),
    (
        "sens_excl_financing_gap_v3_requested",
        "Primary sample excluding >5% V3 requested-basis gap.",
    ),
    (
        "sens_excl_duplicates",
        "Primary sample excluding duplicate-claim episodes.",
    ),
    (
        "sens_excl_missing_discharge",
        "Primary sample with observed discharge date.",
    ),
]

sample_sizes = pd.DataFrame([
    {
        "sample": name,
        "description": description,
        "n": int(
            OUT[name].eq(1).sum()
        ),
    }
    for name, description in sample_size_specs
])


# =============================================================================
# 47. OUTCOME SUMMARY BY GROUP
# =============================================================================

def outcome_by_group(
    frame,
    group_variable,
):
    rows = []

    for category, g in frame.groupby(
        group_variable,
        dropna=False,
    ):

        y = g["oop_share"].dropna()

        rows.append({
            "grouping_variable": group_variable,
            "category": (
                category
                if pd.notna(category)
                else MISSING_LABEL
            ),
            "n": len(g),
            "mean_oop_share": y.mean(),
            "median_oop_share": y.median(),
            "p25": y.quantile(0.25),
            "p75": y.quantile(0.75),
            "pct_zero_oop": (
                100 * g["zero_oop"].eq(1).mean()
            ),
            "pct_high_oop_primary": (
                100 * g["high_oop_ge10pct"].eq(1).mean()
            ),
        })

    return pd.DataFrame(rows)


primary = OUT.loc[
    OUT["analytic_primary"].eq(1)
].copy()

outcome_tables = []

for group_variable in [
    "insurance_fund_grp",
    "ownership_grp",
    "department_grp",
    "diagnosis_type_grp",
    "sex",
    "age_group",
]:

    outcome_tables.append(
        outcome_by_group(
            primary,
            group_variable,
        )
    )

outcome_by_group_qc = pd.concat(
    outcome_tables,
    ignore_index=True,
)


# =============================================================================
# 48. OOP BAND TABLE
# =============================================================================

oop_band_rows = []

for group_variable in [
    "insurance_fund_grp",
    "ownership_grp",
    "diagnosis_type_grp",
]:

    tab = pd.crosstab(
        primary[group_variable],
        primary["oop_band"],
        dropna=False,
    )

    for category, row in tab.iterrows():

        oop_band_rows.append({
            "grouping_variable": group_variable,
            "category": category,
            "n": int(row.sum()),
            **{
                label: int(row.get(label, 0))
                for label in OOP_BAND_LABELS
            },
        })

oop_band_qc = pd.DataFrame(
    oop_band_rows
)


# =============================================================================
# 49. OUTCOME COMPONENT CORRELATIONS
# =============================================================================

correlation_columns = [
    "oop_share",
    "insurer_share",
    "unreconciled_share",
    "org_deduction_share",
    "log_total_cost",
    "los_days",
    "age",
    "financing_gap_v0_pct",
    "financing_gap_v3_pct",
]

correlation_columns = [
    c for c in correlation_columns
    if c in primary.columns
]

if len(correlation_columns) >= 2:

    outcome_component_correlations = (
        primary[correlation_columns]
        .corr(
            method="spearman"
        )
    )

else:

    outcome_component_correlations = pd.DataFrame()


# =============================================================================
# 50. ZERO-LOS AUDIT
# =============================================================================

zero_los_by_setting = []

for group_variable in [
    "department_grp",
    "diagnosis_type_grp",
    "ownership_grp",
]:

    for category, g in primary.groupby(
        group_variable,
        dropna=False,
    ):

        zero_los_by_setting.append({
            "grouping_variable": group_variable,
            "category": category,
            "n": len(g),
            "n_zero_los": int(
                g["los_zero"].eq(1).sum()
            ),
            "pct_zero_los": (
                100
                * g["los_zero"].eq(1).mean()
            ),
        })

zero_los_by_setting = pd.DataFrame(
    zero_los_by_setting
)


# =============================================================================
# 51. FINANCING GAP BY GROUP
# =============================================================================

financing_gap_by_group_rows = []

for group_variable in [
    "insurance_fund_grp",
    "ownership_grp",
    "diagnosis_type_grp",
]:

    for category, g in primary.groupby(
        group_variable,
        dropna=False,
    ):

        gap = g[
            "financing_gap_v0_pct"
        ].dropna()

        financing_gap_by_group_rows.append({
            "grouping_variable": group_variable,
            "category": category,
            "n": len(g),
            "n_gap_gt_5pct": int(
                gap.abs().gt(
                    FINANCING_GAP_TOLERANCE_PCT
                ).sum()
            ),
            "pct_gap_gt_5pct": (
                100
                * gap.abs().gt(
                    FINANCING_GAP_TOLERANCE_PCT
                ).mean()
                if len(gap)
                else np.nan
            ),
            "mean_gap_pct": gap.mean(),
            "median_gap_pct": gap.median(),
        })

financing_gap_by_group = pd.DataFrame(
    financing_gap_by_group_rows
)


# =============================================================================
# 52. DISCHARGE STATUS QC
# =============================================================================

discharge_status_qc = (
    primary.groupby(
        "discharge_status",
        dropna=False,
    )
    .agg(
        n=("discharge_status", "size"),
        flagged_as_death=(
            "died_in_hospital",
            lambda x: int(x.eq(1).sum()),
        ),
    )
    .reset_index()
)


# =============================================================================
# 53. SETTING AUDIT SUMMARY
# =============================================================================
#
# IMPORTANT:
# This table does NOT call observations "implausible for cholera".
# It reports only the prevalence of individual audit signals.
#
# =============================================================================

setting_audit_summary_rows = []

for feature_name in setting_audit_columns:

    x = primary[feature_name]

    setting_audit_summary_rows.append({
        "audit_variable": feature_name,
        "n_flagged": int(
            x.eq(1).sum()
        ),
        "pct_flagged": (
            100 * x.eq(1).mean()
        ),
    })

setting_audit_summary = pd.DataFrame(
    setting_audit_summary_rows
)


# =============================================================================
# 54. DIAGNOSIS CODE AUDIT
# =============================================================================

diagnosis_audit = pd.DataFrame()

if diagnosis_code_candidates:

    diagnosis_rows = []

    for col in diagnosis_code_candidates:

        values = clean_text(
            DF[col]
        )

        diagnosis_rows.append({
            "source_column": col,
            "n_nonmissing": int(
                values.notna().sum()
            ),
            "n_unique": int(
                values.nunique(
                    dropna=True
                )
            ),
            "n_A00_like": int(
                values.str.upper()
                .str.startswith("A00", na=False)
                .sum()
            ),
            "pct_A00_like_among_nonmissing": (
                100
                * values.str.upper()
                .str.startswith("A00", na=False)
                .sum()
                / values.notna().sum()
                if values.notna().sum() > 0
                else np.nan
            ),
        })

    diagnosis_audit = pd.DataFrame(
        diagnosis_rows
    )


# =============================================================================
# 55. OPTIONAL: CLAIM-LINE DIAGNOSIS WARNING
# =============================================================================

log("")
log("=" * 90)
log("FINAL QC")
log("=" * 90)

log(
    f"Total episodes: {len(DF):,}"
)

log(
    f"Primary analytic sample: "
    f"{int(OUT['analytic_primary'].eq(1).sum()):,}"
)

log(
    f"Anonymous episodes: "
    f"{int(OUT['flag_anonymous_episode'].eq(1).sum()):,}"
)

identified_patients = (
    OUT["Patient_cluster"]
    .notna()
    .sum()
)

log(
    f"Identified patient episodes: "
    f"{int(identified_patients):,}"
)

if "n_episodes_patient" in OUT.columns:

    repeated_patients = (
        (
            OUT["n_episodes_patient"]
            .gt(1)
        )
        .sum()
    )

    log(
        f"Episodes belonging to patients with >1 episode: "
        f"{int(repeated_patients):,}"
    )

log(
    f"Zero-LOS episodes: "
    f"{int(OUT['los_zero'].eq(1).sum()):,} "
    f"({100 * OUT['los_zero'].eq(1).mean():.1f}%)"
)

log(
    f"LOS/date mismatches: "
    f"{int(OUT['flag_los_mismatch'].eq(1).sum()):,}"
)

log(
    f"Invalid total cost: "
    f"{int(OUT['flag_invalid_total_cost'].eq(1).sum()):,}"
)

log(
    f"OOP outside [0,1]: "
    f"{int(OUT['flag_oop_out_of_range'].eq(1).sum()):,}"
)

log("")
log("FINANCING RECONCILIATION")

for _, row in financing_reconciliation_summary.iterrows():

    log(
        f"{row['variant']}: "
        f"complete={int(row['n_complete'])}, "
        f"gap>5%={int(row['n_gap_gt_5pct'])} "
        f"({row['pct_gap_gt_5pct']:.1f}%) "
        f"median gap={row['median_gap_pct']:.2f}%"
    )


# =============================================================================
# 56. PRIMARY OOP SUMMARY
# =============================================================================

y = primary[
    "oop_share"
].dropna()

log("")
log("PRIMARY OOP OUTCOME")

if len(y):

    log(
        y.describe().to_string()
    )


# =============================================================================
# 57. CRITICAL DATA-VALIDITY WARNINGS
# =============================================================================

log("")
log("=" * 90)
log("DATA-VALIDITY WARNINGS")
log("=" * 90)

log(
    "1. The episode file does not independently establish the A00.x cholera case definition "
    "unless a diagnosis-code source is available."
)

log(
    "2. A00.x primary-vs-any-diagnosis status must be verified from the raw claims."
)

log(
    "3. The specialty/department audit variables are NOT clinical case definitions."
)

log(
    "4. No episode is automatically excluded because of specialty/department keywords."
)

log(
    "5. Financing V0-V4 are reported separately; primary OOP remains patient-paid / Total Cost."
)

log(
    "6. Insurance fund is treated as payer category, not socioeconomic rank, "
    "unless an independent SES ordering is supplied."
)

log(
    "7. Zero LOS is retained and must be interpreted as same-day/recorded zero LOS, "
    "not automatically treated as data error."
)

log(
    "8. Financing gaps are not silently imputed, redistributed, or assigned to OOP."
)


# =============================================================================
# 58. IMPORTANT NUMERICAL WARNINGS
# =============================================================================

n_v0_gap = int(
    OUT["flag_financing_gap"].eq(1).sum()
)

n_v3_gap = int(
    OUT["flag_financing_gap_v3_requested"].eq(1).sum()
)

n_setting_signal = int(
    OUT["audit_any_setting_signal"].eq(1).sum()
)

log("")
log("IMPORTANT NUMERICAL FLAGS")

log(
    f"V0 >5% financing gap: "
    f"{n_v0_gap:,} / {len(DF):,} "
    f"({100*n_v0_gap/len(DF):.1f}%)"
)

v3_valid_n = int(
    OUT["financing_complete_v3"].eq(1).sum()
)

log(
    f"V3 requested-basis reconciliation complete: "
    f"{v3_valid_n:,} / {len(DF):,}"
)

if v3_valid_n:

    log(
        f"V3 >5% financing gap among complete V3 episodes: "
        f"{n_v3_gap:,} / {v3_valid_n:,}"
    )

log(
    f"Specialty/department audit signal: "
    f"{n_setting_signal:,} / {len(DF):,} "
    f"({100*n_setting_signal/len(DF):.1f}%)"
)

log(
    "This percentage is an AUDIT SIGNAL, not a percentage of episodes "
    "proven not to be cholera."
)


# =============================================================================
# 59. FILE OUTPUTS
# =============================================================================

# -----------------------------------------------------------------------------
# Analysis dataset
# -----------------------------------------------------------------------------
#
# Direct Episode_ID and raw Patient_ID are deliberately excluded.
#
# -----------------------------------------------------------------------------

analysis_dataset = OUT.copy()


# =============================================================================
# 60. CONFIDENTIAL LINKAGE FILE
# =============================================================================

linkage_columns = [
    "Episode_key",
    "Episode_ID",
]

if "Patient_ID" in DF.columns:
    linkage_columns.append(
        "Patient_ID"
    )

if "Episode_Identification_Type" in DF.columns:
    linkage_columns.append(
        "Episode_Identification_Type"
    )

confidential_linkage = pd.DataFrame({
    "Episode_key": OUT["Episode_key"],
    "Episode_ID": DF["Episode_ID"],
})

if "Patient_ID" in DF.columns:

    confidential_linkage["Patient_ID"] = DF[
        "Patient_ID"
    ]

if "Episode_Identification_Type" in DF.columns:

    confidential_linkage[
        "Episode_Identification_Type"
    ] = DF[
        "Episode_Identification_Type"
    ]


# =============================================================================
# 61. WRITE OUTPUTS
# =============================================================================

analysis_dataset.to_csv(
    OUTPUT_DIR / "01_analysis_dataset.csv",
    index=False,
    encoding="utf-8-sig",
)

dictionary_df.to_csv(
    OUTPUT_DIR / "02_feature_dictionary.csv",
    index=False,
    encoding="utf-8-sig",
)

sample_flow.to_csv(
    OUTPUT_DIR / "03_analytic_sample_flow.csv",
    index=False,
    encoding="utf-8-sig",
)

sample_sizes.to_csv(
    OUTPUT_DIR / "03b_sample_sizes.csv",
    index=False,
    encoding="utf-8-sig",
)

category_collapse_map.to_csv(
    OUTPUT_DIR / "04_category_collapse_map.csv",
    index=False,
    encoding="utf-8-sig",
)

optional_availability.to_csv(
    OUTPUT_DIR / "05_optional_column_availability.csv",
    index=False,
    encoding="utf-8-sig",
)

outcome_by_group_qc.to_csv(
    OUTPUT_DIR / "06_outcome_by_group_qc.csv",
    index=False,
    encoding="utf-8-sig",
)

OUT[
    [
        c for c in [
            "Episode_key",
            "flag_anonymous_episode",
            "flag_invalid_date_order",
            "flag_missing_discharge_date",
            "flag_los_mismatch",
            "flag_duplicate_claim",
            "flag_invalid_age",
            "flag_invalid_total_cost",
            "flag_oop_out_of_range",
            "flag_financing_gap",
            "flag_financing_gap_v3_requested",
            "audit_any_setting_signal",
        ]
        if c in OUT.columns
    ]
].to_csv(
    OUTPUT_DIR / "07_missingness_and_flag_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

if not outcome_component_correlations.empty:

    outcome_component_correlations.to_csv(
        OUTPUT_DIR / "08_outcome_component_correlations.csv",
        encoding="utf-8-sig",
    )

zero_los_by_setting.to_csv(
    OUTPUT_DIR / "09_zero_los_by_setting.csv",
    index=False,
    encoding="utf-8-sig",
)

financing_gap_by_group.to_csv(
    OUTPUT_DIR / "10_financing_gap_by_group.csv",
    index=False,
    encoding="utf-8-sig",
)

oop_band_qc.to_csv(
    OUTPUT_DIR / "11_oop_bands_by_group.csv",
    index=False,
    encoding="utf-8-sig",
)

discharge_status_qc.to_csv(
    OUTPUT_DIR / "12_discharge_status_categories.csv",
    index=False,
    encoding="utf-8-sig",
)

setting_audit_summary.to_csv(
    OUTPUT_DIR / "13_setting_audit_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

financing_reconciliation_summary.to_csv(
    OUTPUT_DIR / "14_financing_reconciliation_V0_V4.csv",
    index=False,
    encoding="utf-8-sig",
)

if not source_financing_audit.empty:

    source_financing_audit.to_csv(
        OUTPUT_DIR / "15_source_vs_recomputed_financing.csv",
        index=False,
        encoding="utf-8-sig",
    )

if not diagnosis_audit.empty:

    diagnosis_audit.to_csv(
        OUTPUT_DIR / "16_diagnosis_code_audit.csv",
        index=False,
        encoding="utf-8-sig",
    )

# Confidential file.
confidential_linkage.to_csv(
    OUTPUT_DIR / "CONFIDENTIAL_id_linkage.csv",
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# 62. FINAL REPORT
# =============================================================================

report_lines = []

report_lines.append(
    "=" * 90
)

report_lines.append(
    "FINAL FEATURE ENGINEERING v4"
)

report_lines.append(
    "FINANCIAL PROTECTION / INSURANCE COVERAGE / CHOLERA EPISODES"
)

report_lines.append(
    "=" * 90
)

report_lines.extend(LOG)

report_lines.append("")
report_lines.append(
    "PRIMARY ANALYTICAL DEFINITION"
)

report_lines.append(
    "Primary OOP outcome = Insured Person's Share / Total Cost."
)

report_lines.append(
    "Financing reconciliation V0-V4 is retained as an audit/sensitivity layer."
)

report_lines.append(
    "No keyword-based specialty/department exclusion is used."
)

report_lines.append(
    "No automatic exclusion of anonymous episodes."
)

report_lines.append(
    "No automatic exclusion of zero-LOS episodes."
)

report_lines.append(
    "No imputation, winsorization, scaling, or ML feature construction."
)

report_lines.append("")
report_lines.append(
    "CASE-DEFINITION LIMITATION"
)

if diagnosis_code_candidates:

    report_lines.append(
        "Diagnosis-code candidates were found in the episode file: "
        + "; ".join(diagnosis_code_candidates)
    )

else:

    report_lines.append(
        "No diagnosis-code column was found in the episode file."
    )

    report_lines.append(
        "A00.x cholera status must therefore be verified from raw claims."
    )

    report_lines.append(
        "Primary-vs-any diagnosis status must be verified."
    )

    report_lines.append(
        "Claim-line linkage to the episode must be verified."
    )

report_lines.append("")
report_lines.append(
    "FINANCING LIMITATION"
)

report_lines.append(
    "The presence of a financing gap does not automatically invalidate "
    "the patient-payment / total-cost OOP ratio."
)

report_lines.append(
    "However, financing gaps must be reported because they limit "
    "interpretation of payer-share and coverage variables."
)

report_lines.append("")
report_lines.append(
    "CLINICAL SETTING AUDIT LIMITATION"
)

report_lines.append(
    "Specialty/department keyword signals are not validated clinical "
    "case definitions and are not used to define the primary cohort."
)

report_lines.append("")
report_lines.append(
    "OUTPUT FILES"
)

report_lines.append(
    "01_analysis_dataset.csv"
)

report_lines.append(
    "02_feature_dictionary.csv"
)

report_lines.append(
    "03_analytic_sample_flow.csv"
)

report_lines.append(
    "03b_sample_sizes.csv"
)

report_lines.append(
    "04_category_collapse_map.csv"
)

report_lines.append(
    "05_optional_column_availability.csv"
)

report_lines.append(
    "06_outcome_by_group_qc.csv"
)

report_lines.append(
    "07_missingness_and_flag_summary.csv"
)

report_lines.append(
    "08_outcome_component_correlations.csv"
)

report_lines.append(
    "09_zero_los_by_setting.csv"
)

report_lines.append(
    "10_financing_gap_by_group.csv"
)

report_lines.append(
    "11_oop_bands_by_group.csv"
)

report_lines.append(
    "12_discharge_status_categories.csv"
)

report_lines.append(
    "13_setting_audit_summary.csv"
)

report_lines.append(
    "14_financing_reconciliation_V0_V4.csv"
)

report_lines.append(
    "15_source_vs_recomputed_financing.csv"
)

report_lines.append(
    "16_diagnosis_code_audit.csv"
)

report_lines.append(
    "CONFIDENTIAL_id_linkage.csv"
)

report_lines.append("")
report_lines.append(
    "IMPORTANT: CONFIDENTIAL_id_linkage.csv must not be shared with "
    "analysis results or submitted as supplementary material."
)


report_text = "\n".join(
    report_lines
)

with open(
    OUTPUT_DIR / "17_feature_engineering_report.txt",
    "w",
    encoding="utf-8",
) as f:

    f.write(report_text)


# =============================================================================
# 63. FINAL CONSOLE SUMMARY
# =============================================================================

print("")
print("=" * 90)
print("FEATURE ENGINEERING v4 COMPLETED")
print("=" * 90)

print(
    f"Output directory:\n{OUTPUT_DIR}"
)

print(
    f"Analysis dataset: {len(analysis_dataset):,} episodes"
)

print(
    f"Engineered variables: {len(analysis_dataset.columns):,}"
)

print(
    f"Dictionary entries: {len(dictionary_df):,}"
)

print(
    f"Primary analytic sample: "
    f"{int(OUT['analytic_primary'].eq(1).sum()):,}"
)

print(
    f"V0 financing gap >5%: "
    f"{int(OUT['flag_financing_gap'].eq(1).sum()):,}"
)

print(
    f"V3 requested-basis complete: "
    f"{int(OUT['financing_complete_v3'].eq(1).sum()):,}"
)

print(
    f"V3 requested-basis gap >5%: "
    f"{int(OUT['flag_financing_gap_v3_requested'].eq(1).sum()):,}"
)

print(
    f"Zero LOS: "
    f"{int(OUT['los_zero'].eq(1).sum()):,}"
)

print(
    f"Any setting audit signal: "
    f"{int(OUT['audit_any_setting_signal'].eq(1).sum()):,}"
)

print("")
print(
    "PRIMARY OOP DEFINITION:"
)

print(
    "    OOP_share = Insured Person's Share / Total Cost"
)

print("")
print(
    "No keyword-based cohort exclusion was applied."
)

print(
    "No financing-gap observations were automatically deleted."
)

print(
    "Case-definition verification against raw A00.x claims remains required."
)

print("=" * 90)

FINAL FEATURE ENGINEERING v4
FINANCIAL PROTECTION / INSURANCE COVERAGE / CHOLERA EPISODES
Run time : 2026-09-29 20:17
Input    : C:\Users\Shekoofeh\Documents\Health Insurance Data\Cholera\04_episode_level_data.csv
Output   : C:\Users\Shekoofeh\Documents\Health Insurance Data\Cholera\feature engineering v4_FINAL
Episodes loaded: 1,171   Columns: 31
Columns present: Episode_ID; Total Cost; Insured Person's Share; Organization Requested Share; Organization Paid Share; Terminally Patients Paid Share; Terminally Patients Requested Share; Allocated subsidized foreign currency share; Requested subsidized foreign currency share; Patient_ID; Age; Gender; Ownership; Insurance Fund; Treating Physician Specialty; Admision date; Discharge date; Discharge Status; Diagnosis type; Department Type; Episode_Identification_Type; Episode_Linkage_Method; Number_of_claim_lines; Number_of_service_groups; Episode_LOS_days; invalid_date_order; contains_exact_duplicate_claim; contains_invalid_age; Known_financi

In [7]:
# =============================================================================
# CHOLERA FINANCIAL PROTECTION STUDY
# NEXT-LEVEL ANALYTICAL PIPELINE
#
# Purpose:
#   1. Load and validate the engineered analytical dataset
#   2. Define the primary analytical cohort
#   3. Audit SES availability and independence from outcomes
#   4. Validate financial-protection outcomes
#   5. Validate candidate composite index components
#   6. Quantify hospital clustering
#   7. Prepare data for ridit concentration indices
#   8. Prepare multilevel regression analyses
#   9. Run core sensitivity checks
#
# IMPORTANT:
#   This script DOES NOT assume that Cluster_for_SE is SES.
#   It audits whether an actual independent SES variable exists.
#
# Primary conceptual framework:
#   Inequality in financial protection and insurance coverage
#
# NOT:
#   Value-based care
#   Clinical quality
#   Causal effects
# =============================================================================


# =============================================================================
# 0. IMPORTS
# =============================================================================

import os
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from scipy import stats
from scipy.stats import spearmanr, pearsonr

import statsmodels.api as sm
import statsmodels.formula.api as smf

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Optional:
# pip install pingouin
try:
    import pingouin as pg
    PINGOUIN_AVAILABLE = True
except Exception:
    PINGOUIN_AVAILABLE = False


# =============================================================================
# 1. INPUT
# =============================================================================

INPUT_FILE = (
    r"C:\Users\Shekoofeh\Documents\Health Insurance Data"
    r"\Cholera\feature engineering v4_FINAL\01_analysis_dataset.csv"
)

OUTPUT_DIR = (
    r"C:\Users\Shekoofeh\Documents\Health Insurance Data"
    r"\Cholera\feature engineering v4_FINAL\next_level_analysis"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 80)
print("NEXT-LEVEL ANALYSIS")
print("=" * 80)

print("\nInput:")
print(INPUT_FILE)

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(
        f"\nInput file not found:\n{INPUT_FILE}\n"
        "Check the path before continuing."
    )

df = pd.read_csv(INPUT_FILE, low_memory=False)

print("\nDataset loaded.")
print("Rows:", len(df))
print("Columns:", len(df.columns))


# =============================================================================
# 2. BASIC STRUCTURE AUDIT
# =============================================================================

print("\n" + "=" * 80)
print("2. BASIC STRUCTURE AUDIT")
print("=" * 80)

print("\nDuplicate Episode_key:")
print(df["Episode_key"].duplicated().sum())

print("\nEpisode_key missing:")
print(df["Episode_key"].isna().sum())

print("\nUnique episodes:")
print(df["Episode_key"].nunique())

if "Patient_cluster" in df.columns:
    print("\nUnique Patient_cluster:")
    print(df["Patient_cluster"].nunique())

if "Cluster_for_SE" in df.columns:
    print("\nUnique Cluster_for_SE:")
    print(df["Cluster_for_SE"].nunique())


# =============================================================================
# 3. VARIABLE INVENTORY
# =============================================================================

print("\n" + "=" * 80)
print("3. VARIABLE INVENTORY")
print("=" * 80)

for col in df.columns:
    print(col)


# =============================================================================
# 4. PRIMARY ANALYTICAL COHORT
# =============================================================================
#
# The primary cohort should be defined transparently.
#
# We do NOT automatically remove:
#   - anonymous episodes
#   - zero-LOS episodes
#   - financing-gap episodes
#
# Those are sensitivity analyses unless they make the primary outcome
# mathematically undefined.
#
# We DO require:
#   - valid episode identifier
#   - valid OOP share
#   - valid total cost
#
# =============================================================================

required_primary = [
    "Episode_key",
    "oop_share",
    "log_total_cost"
]

missing_required = [
    x for x in required_primary
    if x not in df.columns
]

if missing_required:
    raise ValueError(
        "Missing required primary-analysis variables: "
        + str(missing_required)
    )

analysis = df.copy()

# Valid primary outcome
analysis["valid_primary_outcome"] = (
    analysis["oop_share"].notna()
    &
    np.isfinite(analysis["oop_share"])
    &
    (analysis["oop_share"] >= 0)
    &
    (analysis["oop_share"] <= 1)
)

# Valid cost
analysis["valid_cost"] = (
    analysis["log_total_cost"].notna()
    &
    np.isfinite(analysis["log_total_cost"])
)

primary = analysis[
    analysis["valid_primary_outcome"]
    &
    analysis["valid_cost"]
].copy()

print("\nPrimary analytical cohort:")
print("N =", len(primary))

print("\nExcluded because outcome invalid:")
print((~analysis["valid_primary_outcome"]).sum())

print("\nExcluded because cost invalid:")
print(
    (
        analysis["valid_primary_outcome"]
        &
        ~analysis["valid_cost"]
    ).sum()
)


# =============================================================================
# 5. COHORT FLOW TABLE
# =============================================================================

flow = []

flow.append(("Engineered dataset", len(df)))

flow.append((
    "Valid OOP share",
    analysis["valid_primary_outcome"].sum()
))

flow.append((
    "Valid OOP share + valid total cost",
    len(primary)
))

if "sens_excl_anonymous" in primary.columns:
    flow.append((
        "After excluding anonymous episodes",
        primary["sens_excl_anonymous"].eq(1).sum()
    ))

if "sens_excl_zero_los" in primary.columns:
    flow.append((
        "After excluding zero-LOS episodes",
        primary["sens_excl_zero_los"].eq(1).sum()
    ))

flow_df = pd.DataFrame(
    flow,
    columns=["Cohort_step", "N"]
)

flow_df.to_csv(
    os.path.join(OUTPUT_DIR, "01_cohort_flow.csv"),
    index=False
)

print("\nCohort flow:")
print(flow_df.to_string(index=False))


# =============================================================================
# 6. PRIMARY OUTCOME VALIDATION
# =============================================================================

print("\n" + "=" * 80)
print("6. PRIMARY OUTCOME VALIDATION")
print("=" * 80)

outcomes = [
    "oop_share",
    "high_oop_ge05pct",
    "high_oop_ge10pct",
    "high_oop_ge20pct",
    "high_oop_ge30pct",
    "zero_oop"
]

outcome_rows = []

for var in outcomes:

    if var not in primary.columns:
        continue

    x = primary[var]

    outcome_rows.append({
        "variable": var,
        "n": x.notna().sum(),
        "missing": x.isna().sum(),
        "mean": x.mean(),
        "median": x.median(),
        "sd": x.std(),
        "min": x.min(),
        "max": x.max(),
        "n_unique": x.nunique()
    })

outcome_summary = pd.DataFrame(outcome_rows)

print(outcome_summary.to_string(index=False))

outcome_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "02_primary_outcome_summary.csv"
    ),
    index=False
)


# =============================================================================
# 7. ZERO-LOS INVESTIGATION
# =============================================================================

print("\n" + "=" * 80)
print("7. ZERO-LOS INVESTIGATION")
print("=" * 80)

if "los_zero" in primary.columns:

    zero_los_tab = (
        primary["los_zero"]
        .value_counts(dropna=False)
        .rename_axis("los_zero")
        .reset_index(name="N")
    )

    zero_los_tab["percent"] = (
        100 *
        zero_los_tab["N"] /
        zero_los_tab["N"].sum()
    )

    print(zero_los_tab)

    # Compare primary outcomes
    zero_los_comparison = (
        primary
        .groupby("los_zero", dropna=False)
        .agg(
            n=("Episode_key", "size"),
            oop_mean=("oop_share", "mean"),
            oop_median=("oop_share", "median"),
            high_oop_10=("high_oop_ge10pct", "mean"),
            high_oop_20=("high_oop_ge20pct", "mean"),
            high_oop_30=("high_oop_ge30pct", "mean")
        )
        .reset_index()
    )

    print("\nZero-LOS outcome comparison:")
    print(zero_los_comparison.to_string(index=False))

    zero_los_comparison.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "03_zero_los_comparison.csv"
        ),
        index=False
    )


# =============================================================================
# 8. SES AUDIT
# =============================================================================
#
# THIS IS CRITICAL.
#
# We must distinguish:
#
#   A. Genuine SES variable
#   B. Proxy for SES
#   C. Administrative grouping
#   D. Identifier / clustering variable
#
# Cluster_for_SE must NOT automatically be interpreted as SES.
# =============================================================================

print("\n" + "=" * 80)
print("8. SES AUDIT")
print("=" * 80)

candidate_ses = [
    "ses",
    "ses_index",
    "socioeconomic_status",
    "income",
    "income_decile",
    "income_quintile",
    "deprivation",
    "deprivation_index",
    "education",
    "occupation",
    "residence_income",
    "area_income",
    "province_income",
    "county_income",
    "insurance_fund_grp",
    "insurance_fund",
    "Cluster_for_SE"
]

ses_candidates_found = [
    x for x in candidate_ses
    if x in primary.columns
]

print("\nCandidate SES-related variables found:")
for x in ses_candidates_found:
    print("  -", x)

print("\nIMPORTANT:")
print(
    "`Cluster_for_SE` is currently treated as a grouping identifier, "
    "NOT as a socioeconomic ranking."
)


# =============================================================================
# 9. CHECK CLUSTER_FOR_SE
# =============================================================================

if "Cluster_for_SE" in primary.columns:

    print("\n" + "-" * 70)
    print("Cluster_for_SE diagnostic")
    print("-" * 70)

    print(
        primary["Cluster_for_SE"]
        .describe()
    )

    print("\nNumber of unique clusters:")
    print(primary["Cluster_for_SE"].nunique())

    cluster_sizes = (
        primary
        .groupby("Cluster_for_SE")
        .size()
        .describe()
    )

    print("\nCluster size distribution:")
    print(cluster_sizes)


# =============================================================================
# 10. DETECT POTENTIAL HOSPITAL IDENTIFIER
# =============================================================================

print("\n" + "=" * 80)
print("10. HOSPITAL-CLUSTERING VARIABLE AUDIT")
print("=" * 80)

possible_hospital_vars = [
    "hospital_id",
    "Hospital_ID",
    "hospital",
    "Hospital",
    "provider_id",
    "provider",
    "hospital_code",
    "Hospital_code",
    "facility_id",
    "facility_code"
]

hospital_candidates = [
    x for x in possible_hospital_vars
    if x in primary.columns
]

print("Hospital candidates:")
print(hospital_candidates)

if not hospital_candidates:
    print(
        "\nWARNING: No explicit hospital identifier was detected."
    )
    print(
        "Hospital-clustered inference cannot be performed correctly "
        "until a hospital/provider identifier is available."
    )


# =============================================================================
# 11. HOSPITAL OWNERSHIP / INSURANCE DISTRIBUTIONS
# =============================================================================

print("\n" + "=" * 80)
print("11. INSURANCE AND HOSPITAL CHARACTERISTICS")
print("=" * 80)

group_vars = [
    "insurance_fund_grp",
    "ownership_grp",
    "department_grp",
    "diagnosis_type_grp",
    "specialty_grp",
    "age_group",
    "sex",
    "admission_year",
    "admission_quarter"
]

for var in group_vars:

    if var not in primary.columns:
        continue

    tab = (
        primary[var]
        .value_counts(dropna=False)
        .rename_axis(var)
        .reset_index(name="N")
    )

    tab["percent"] = (
        100 * tab["N"] / len(primary)
    )

    print(f"\n--- {var} ---")
    print(tab.to_string(index=False))

    tab.to_csv(
        os.path.join(
            OUTPUT_DIR,
            f"04_distribution_{var}.csv"
        ),
        index=False
    )


# =============================================================================
# 12. FINANCIAL-PROTECTION OUTCOME BY INSURANCE / OWNERSHIP
# =============================================================================

def grouped_financial_summary(data, group_var):

    if group_var not in data.columns:
        return None

    result = (
        data
        .groupby(group_var, dropna=False)
        .agg(
            n=("Episode_key", "size"),
            oop_mean=("oop_share", "mean"),
            oop_median=("oop_share", "median"),
            oop_sd=("oop_share", "std"),
            high_oop_5=("high_oop_ge05pct", "mean"),
            high_oop_10=("high_oop_ge10pct", "mean"),
            high_oop_20=("high_oop_ge20pct", "mean"),
            high_oop_30=("high_oop_ge30pct", "mean"),
            zero_oop=("zero_oop", "mean"),
            cost_mean=("log_total_cost", "mean")
        )
        .reset_index()
    )

    for col in [
        "high_oop_5",
        "high_oop_10",
        "high_oop_20",
        "high_oop_30",
        "zero_oop"
    ]:
        result[col] *= 100

    return result


for var in [
    "insurance_fund_grp",
    "ownership_grp",
    "department_grp",
    "diagnosis_type_grp",
    "age_group",
    "sex"
]:

    result = grouped_financial_summary(primary, var)

    if result is None:
        continue

    print("\n" + "-" * 70)
    print(var)
    print("-" * 70)
    print(result.to_string(index=False))

    result.to_csv(
        os.path.join(
            OUTPUT_DIR,
            f"05_financial_protection_by_{var}.csv"
        ),
        index=False
    )


# =============================================================================
# 13. CORRELATION / CIRCULARITY AUDIT
# =============================================================================
#
# This is one of the most important analyses.
#
# We test whether OOP is mechanically embedded in other candidate
# financial-protection variables.
#
# A very high correlation would warn us not to treat the variables
# as independent components of a composite index.
# =============================================================================

print("\n" + "=" * 80)
print("13. CIRCULARITY / COMPONENT CORRELATION AUDIT")
print("=" * 80)

component_candidates = [
    "oop_share",
    "oop_share_of_known",
    "insurer_share",
    "insurer_deduction_share",
    "org_deduction_share",
    "financing_gap_pct",
    "financing_gap_v0_pct",
    "financing_gap_v1_pct",
    "financing_gap_v2_pct",
    "financing_gap_v3_pct",
    "financing_gap_v4_pct",
    "unreconciled_share",
    "log_total_cost",
    "cost_per_day"
]

component_candidates = [
    x for x in component_candidates
    if x in primary.columns
]

corr = primary[component_candidates].corr(
    method="spearman"
)

print(corr.round(4).to_string())

corr.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "06_spearman_component_correlation.csv"
    )
)


# =============================================================================
# 14. VIF AUDIT
# =============================================================================
#
# VIF is used only for continuous candidate components.
# It is NOT used as proof of validity.
# =============================================================================

from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_vars = [
    x for x in component_candidates
    if x != "oop_share"
]

vif_data = primary[vif_vars].replace(
    [np.inf, -np.inf],
    np.nan
).dropna()

# Avoid impossible calculation if too few observations
if len(vif_data) > 20 and len(vif_vars) >= 2:

    X = sm.add_constant(vif_data)

    vif_rows = []

    for i, var in enumerate(X.columns):

        if var == "const":
            continue

        vif_rows.append({
            "variable": var,
            "VIF": variance_inflation_factor(
                X.values,
                i
            )
        })

    vif_df = pd.DataFrame(vif_rows)

    print("\nVIF:")
    print(vif_df.to_string(index=False))

    vif_df.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "07_VIF_components.csv"
        ),
        index=False
    )


# =============================================================================
# 15. MISSINGNESS PROFILE
# =============================================================================

print("\n" + "=" * 80)
print("15. MISSING DATA PROFILE")
print("=" * 80)

missing = (
    primary
    .isna()
    .mean()
    .sort_values(ascending=False)
    .mul(100)
    .rename("missing_percent")
    .reset_index()
    .rename(columns={"index": "variable"})
)

missing = missing[missing["missing_percent"] > 0]

print(
    missing.head(50).to_string(index=False)
)

missing.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "08_missingness_profile.csv"
    ),
    index=False
)


# =============================================================================
# 16. MISSINGNESS BY PRIMARY OUTCOME
# =============================================================================

key_covariates = [
    "age",
    "sex",
    "insurance_fund_grp",
    "ownership_grp",
    "department_grp",
    "diagnosis_type_grp",
    "specialty_grp",
    "los_days",
    "admission_quarter"
]

key_covariates = [
    x for x in key_covariates
    if x in primary.columns
]

missing_outcome_rows = []

for var in key_covariates:

    miss = primary[var].isna()

    if miss.sum() == 0:
        continue

    missing_outcome_rows.append({
        "variable": var,
        "missing_n": miss.sum(),
        "missing_percent": 100 * miss.mean(),
        "oop_missing_mean": primary.loc[miss, "oop_share"].mean(),
        "oop_observed_mean": primary.loc[~miss, "oop_share"].mean()
    })

missing_outcome_df = pd.DataFrame(missing_outcome_rows)

if not missing_outcome_df.empty:
    print(
        missing_outcome_df.to_string(index=False)
    )

missing_outcome_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "09_missingness_vs_outcome.csv"
    ),
    index=False
)


# =============================================================================
# 17. AGE / SEX / INSURANCE / OWNERSHIP ASSOCIATIONS
# =============================================================================

print("\n" + "=" * 80)
print("17. BASIC ASSOCIATIONS WITH OOP SHARE")
print("=" * 80)

continuous_vars = [
    "age",
    "los_days",
    "log_total_cost",
    "cost_per_day",
    "n_claim_lines",
    "n_service_groups"
]

association_rows = []

for var in continuous_vars:

    if var not in primary.columns:
        continue

    tmp = primary[[var, "oop_share"]].replace(
        [np.inf, -np.inf],
        np.nan
    ).dropna()

    if len(tmp) < 10:
        continue

    rho, p = spearmanr(
        tmp[var],
        tmp["oop_share"]
    )

    association_rows.append({
        "variable": var,
        "n": len(tmp),
        "spearman_rho": rho,
        "p_value": p
    })

association_df = pd.DataFrame(association_rows)

print(
    association_df.to_string(index=False)
)

association_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "10_continuous_associations.csv"
    ),
    index=False
)


# =============================================================================
# 18. DEFINE CANDIDATE PRIMARY OUTCOME
# =============================================================================
#
# Recommendation:
#
# Primary continuous outcome:
#       oop_share
#
# Secondary binary outcomes:
#       high_oop_ge10pct
#       high_oop_ge20pct
#
# DO NOT call these "catastrophic expenditure" unless household
# capacity-to-pay information exists.
# =============================================================================

primary["oop_share_pct"] = (
    primary["oop_share"] * 100
)

print("\nPrimary outcome:")
print("OOP share of episode cost")

print("\nMean:", primary["oop_share"].mean())
print("Median:", primary["oop_share"].median())
print("SD:", primary["oop_share"].std())


# =============================================================================
# 19. FINANCIAL-PROTECTION INDEX CANDIDATES
# =============================================================================
#
# IMPORTANT:
#
# We do NOT automatically construct a composite index here.
#
# First, we identify candidate components and determine whether they
# represent distinct constructs.
#
# A composite should not contain mathematically overlapping quantities
# simply because they are available.
# =============================================================================

index_candidates = [
    "oop_share",
    "insurer_share",
    "financing_gap_pct",
    "unreconciled_share"
]

index_candidates = [
    x for x in index_candidates
    if x in primary.columns
]

print("\nCandidate index components:")
for x in index_candidates:
    print(" -", x)


# =============================================================================
# 20. COMPONENT DISTRIBUTIONS
# =============================================================================

component_summary = []

for var in index_candidates:

    x = primary[var].replace(
        [np.inf, -np.inf],
        np.nan
    ).dropna()

    component_summary.append({
        "variable": var,
        "n": len(x),
        "mean": x.mean(),
        "sd": x.std(),
        "median": x.median(),
        "q01": x.quantile(.01),
        "q25": x.quantile(.25),
        "q75": x.quantile(.75),
        "q99": x.quantile(.99),
        "min": x.min(),
        "max": x.max()
    })

component_summary = pd.DataFrame(component_summary)

print(
    component_summary.to_string(index=False)
)

component_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "11_index_component_summary.csv"
    ),
    index=False
)


# =============================================================================
# 21. HOSPITAL-LEVEL CLUSTERING
# =============================================================================
#
# If an explicit hospital ID exists, calculate:
#   - number of hospitals
#   - hospital size
#   - hospital-level OOP distribution
#   - between-hospital variation
#
# =============================================================================

if hospital_candidates:

    hospital_id = hospital_candidates[0]

    print("\nUsing hospital clustering variable:")
    print(hospital_id)

    hospital_summary = (
        primary
        .groupby(hospital_id)
        .agg(
            n=("Episode_key", "size"),
            oop_mean=("oop_share", "mean"),
            oop_median=("oop_share", "median"),
            high_oop_10=("high_oop_ge10pct", "mean"),
            high_oop_20=("high_oop_ge20pct", "mean"),
            zero_los=("los_zero", "mean") if "los_zero" in primary.columns
                      else ("Episode_key", lambda x: np.nan)
        )
        .reset_index()
    )

    print("\nHospital-level summary:")
    print(
        hospital_summary.describe(
            include="all"
        ).to_string()
    )

    hospital_summary.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "12_hospital_level_summary.csv"
        ),
        index=False
    )


# =============================================================================
# 22. INTRACLASS CORRELATION — RAW OOP SHARE
# =============================================================================
#
# Approximate one-way random-effects ICC.
#
# This is a diagnostic, not yet the final multilevel model.
# =============================================================================

if hospital_candidates:

    hospital_id = hospital_candidates[0]

    temp = primary[
        [hospital_id, "oop_share"]
    ].dropna()

    group_sizes = (
        temp
        .groupby(hospital_id)
        .size()
    )

    valid_groups = group_sizes[
        group_sizes >= 2
    ].index

    temp = temp[
        temp[hospital_id].isin(valid_groups)
    ]

    if temp[hospital_id].nunique() >= 2:

        grand_mean = temp["oop_share"].mean()

        group_stats = (
            temp
            .groupby(hospital_id)["oop_share"]
            .agg(["mean", "count"])
        )

        between_ss = (
            group_stats["count"]
            *
            (group_stats["mean"] - grand_mean) ** 2
        ).sum()

        within_ss = (
            temp
            .merge(
                group_stats["mean"],
                left_on=hospital_id,
                right_index=True
            )
            .assign(
                residual_sq=lambda d:
                    (
                        d["oop_share"]
                        -
                        d["mean"]
                    ) ** 2
            )["residual_sq"]
            .sum()
        )

        n_groups = len(group_stats)
        n_total = len(temp)

        ms_between = (
            between_ss /
            (n_groups - 1)
        )

        ms_within = (
            within_ss /
            (n_total - n_groups)
        )

        mean_n = (
            group_stats["count"].mean()
        )

        icc = (
            (ms_between - ms_within)
            /
            (
                ms_between
                +
                (mean_n - 1) * ms_within
            )
        )

        print("\nApproximate hospital ICC:")
        print(icc)

        pd.DataFrame({
            "metric": ["hospital_ICC"],
            "value": [icc]
        }).to_csv(
            os.path.join(
                OUTPUT_DIR,
                "13_hospital_ICC.csv"
            ),
            index=False
        )


# =============================================================================
# 23. PREPARE RIDIT RANK
# =============================================================================
#
# This is the correct direction for concentration-index analysis.
#
# We DO NOT use min-max scaling as a socioeconomic rank.
#
# Once a valid SES variable is identified, this function should be applied
# to that SES variable.
# =============================================================================

def fractional_rank(series):
    """
    Fractional rank / ridit:
    rank percentile centered within each ordered group.
    """

    x = pd.to_numeric(
        series,
        errors="coerce"
    )

    valid = x.notna()

    ranks = pd.Series(
        np.nan,
        index=series.index,
        dtype=float
    )

    ranks.loc[valid] = (
        x.loc[valid]
        .rank(method="average")
        - 0.5
    ) / valid.sum()

    return ranks


# =============================================================================
# 24. DO NOT CREATE SES RANK FROM OOP
# =============================================================================
#
# This explicit safeguard is intentional.
#
# If OOP is used as SES, the concentration index becomes circular.
# =============================================================================

if "oop_share" in primary.columns:

    primary["SES_RIDIT_FORBIDDEN"] = np.nan

    print(
        "\nWARNING:"
    )
    print(
        "No OOP-derived SES rank has been created."
    )
    print(
        "This is intentional because OOP is one of the financial "
        "protection outcomes."
    )


# =============================================================================
# 25. TEST INSURANCE FUND AS POSSIBLE STRATIFICATION VARIABLE
# =============================================================================
#
# Insurance fund may be substantively important, but it should NOT
# automatically be renamed SES.
#
# We therefore analyze it as an insurance/financing characteristic.
# =============================================================================

if "insurance_fund_grp" in primary.columns:

    insurance_analysis = (
        primary
        .groupby("insurance_fund_grp")
        .agg(
            n=("Episode_key", "size"),
            oop_mean=("oop_share", "mean"),
            oop_median=("oop_share", "median"),
            high_oop_10=("high_oop_ge10pct", "mean"),
            high_oop_20=("high_oop_ge20pct", "mean"),
            high_oop_30=("high_oop_ge30pct", "mean"),
            zero_oop=("zero_oop", "mean")
        )
        .reset_index()
    )

    for c in [
        "high_oop_10",
        "high_oop_20",
        "high_oop_30",
        "zero_oop"
    ]:
        insurance_analysis[c] *= 100

    insurance_analysis.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "14_insurance_fund_financial_protection.csv"
        ),
        index=False
    )


# =============================================================================
# 26. SENSITIVITY COHORTS
# =============================================================================

print("\n" + "=" * 80)
print("26. SENSITIVITY COHORTS")
print("=" * 80)

sensitivity_cohorts = {
    "primary": primary
}

if "sens_excl_anonymous" in primary.columns:
    sensitivity_cohorts["exclude_anonymous"] = primary[
        primary["sens_excl_anonymous"] == 1
    ].copy()

if "sens_excl_zero_los" in primary.columns:
    sensitivity_cohorts["exclude_zero_los"] = primary[
        primary["sens_excl_zero_los"] == 1
    ].copy()

if "sens_excl_financing_gap" in primary.columns:
    sensitivity_cohorts["exclude_financing_gap"] = primary[
        primary["sens_excl_financing_gap"] == 1
    ].copy()

if "sens_excl_duplicates" in primary.columns:
    sensitivity_cohorts["exclude_duplicates"] = primary[
        primary["sens_excl_duplicates"] == 1
    ].copy()

if "sens_excl_missing_discharge" in primary.columns:
    sensitivity_cohorts["exclude_missing_discharge"] = primary[
        primary["sens_excl_missing_discharge"] == 1
    ].copy()


sensitivity_rows = []

for name, d in sensitivity_cohorts.items():

    sensitivity_rows.append({
        "cohort": name,
        "n": len(d),
        "oop_mean": d["oop_share"].mean(),
        "oop_median": d["oop_share"].median(),
        "high_oop_10_pct":
            100 * d["high_oop_ge10pct"].mean(),
        "high_oop_20_pct":
            100 * d["high_oop_ge20pct"].mean(),
        "high_oop_30_pct":
            100 * d["high_oop_ge30pct"].mean()
    })

sensitivity_summary = pd.DataFrame(
    sensitivity_rows
)

print(
    sensitivity_summary.to_string(index=False)
)

sensitivity_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "15_sensitivity_cohort_summary.csv"
    ),
    index=False
)


# =============================================================================
# 27. MAIN ANALYTICAL DATASET
# =============================================================================

# Keep only the variables needed for the next analytical stage.

core_variables = [
    "Episode_key",
    "Patient_cluster",
    "Cluster_for_SE",
    "oop_share",
    "high_oop_ge05pct",
    "high_oop_ge10pct",
    "high_oop_ge20pct",
    "high_oop_ge30pct",
    "zero_oop",
    "financing_gap_pct",
    "unreconciled_share",
    "insurer_deduction_share",
    "insurer_share",
    "age",
    "age_group",
    "sex",
    "insurance_fund_grp",
    "ownership_grp",
    "department_grp",
    "diagnosis_type_grp",
    "specialty_grp",
    "los_days",
    "los_zero",
    "cost_per_day",
    "log_total_cost",
    "admission_quarter",
    "discharge_status",
    "died_in_hospital"
]

core_variables = [
    x for x in core_variables
    if x in primary.columns
]

analysis_core = primary[core_variables].copy()

analysis_core.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "16_next_level_core_analysis_dataset.csv"
    ),
    index=False
)


# =============================================================================
# 28. FINAL STATUS REPORT
# =============================================================================

print("\n" + "=" * 80)
print("NEXT-LEVEL ANALYSIS STATUS")
print("=" * 80)

print("\nPrimary cohort N:", len(primary))

print(
    "\nCandidate SES variables found:",
    ses_candidates_found
)

print(
    "\nHospital identifier candidates:",
    hospital_candidates
)

print(
    "\nCandidate financial-protection components:",
    index_candidates
)

print("\nFiles written to:")
print(OUTPUT_DIR)

print("\n" + "=" * 80)
print("IMPORTANT DECISION POINT")
print("=" * 80)

if not any(
    x in primary.columns
    for x in [
        "ses",
        "ses_index",
        "socioeconomic_status",
        "income",
        "income_decile",
        "income_quintile",
        "deprivation",
        "deprivation_index",
        "education",
        "occupation",
        "area_income",
        "residence_income"
    ]
):

    print(
        """
NO INDEPENDENT INDIVIDUAL/AREA SES VARIABLE WAS DETECTED.

Therefore:
1. DO NOT calculate a SES-based concentration index yet.
2. DO NOT use oop_share as SES.
3. DO NOT describe Cluster_for_SE as SES without establishing what it represents.
4. The defensible interim analysis is inequality by insurance fund,
   hospital ownership/type, and other observed characteristics.
5. If an external area-level income/deprivation variable exists,
   merge it BEFORE calculating the ridit concentration index.
"""
    )

else:

    print(
        """
An SES candidate variable was detected.

Before using it:
1. Establish that it is measured independently of OOP.
2. Establish its temporal precedence.
3. Define its ordering.
4. Construct the fractional/ridit rank.
5. Then proceed to the Erreygers/Wagstaff concentration analysis.
"""
    )

print("\nDONE.")

NEXT-LEVEL ANALYSIS

Input:
C:\Users\Shekoofeh\Documents\Health Insurance Data\Cholera\feature engineering v4_FINAL\01_analysis_dataset.csv

Dataset loaded.
Rows: 1171
Columns: 103

2. BASIC STRUCTURE AUDIT

Duplicate Episode_key:
0

Episode_key missing:
0

Unique episodes:
1171

Unique Patient_cluster:
1077

Unique Cluster_for_SE:
1144

3. VARIABLE INVENTORY
Episode_key
Patient_cluster
n_episodes_patient
Cluster_for_SE
oop_share_raw
oop_share
oop_band
zero_oop
high_oop_ge05pct
high_oop_ge10pct
high_oop_ge20pct
high_oop_ge30pct
high_oop_share
financing_gap_v0_rial
financing_gap_v0_pct
financing_gap_v1_rial
financing_gap_v1_pct
financing_gap_v2_rial
financing_gap_v2_pct
financing_gap_v3_rial
financing_gap_v3_pct
financing_gap_v4_rial
financing_gap_v4_pct
financing_complete_v0
financing_complete_v1
financing_complete_v2
financing_complete_v3
financing_complete_v4
financing_gap_pct
unreconciled_share
financing_gap_positive
financing_gap_negative
flag_financing_gap
flag_gap_unexplained
fla

In [8]:
# =============================================================================
# CHOLERA FINANCIAL PROTECTION STUDY
# ANALYTICAL VALIDATION v2
#
# PURPOSE
# -------
# This stage is deliberately BEFORE the final inequality analysis.
#
# It validates:
#   1. Primary cohort
#   2. Primary financial-protection outcome
#   3. Sensitivity flags
#   4. Zero-LOS structure
#   5. Financing-variable redundancy
#   6. Candidate SES variables
#   7. Cluster_for_SE
#   8. Hospital identifier availability
#   9. Death/discharge variables
#  10. Potential coding/referral anomalies
#
# IMPORTANT:
#   DO NOT calculate SES concentration indices in this script.
#   DO NOT treat Cluster_for_SE as SES until its construction is verified.
# =============================================================================


import os
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings("ignore")


# =============================================================================
# 1. PATHS
# =============================================================================

INPUT_FILE = (
    r"C:\Users\Shekoofeh\Documents\Health Insurance Data"
    r"\Cholera\feature engineering v4_FINAL\01_analysis_dataset.csv"
)

OUTPUT_DIR = (
    r"C:\Users\Shekoofeh\Documents\Health Insurance Data"
    r"\Cholera\feature engineering v4_FINAL\analytical_validation_v2"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)


# =============================================================================
# 2. LOAD DATA
# =============================================================================

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(INPUT_FILE)

df = pd.read_csv(
    INPUT_FILE,
    low_memory=False
)

print("=" * 90)
print("CHOLERA ANALYTICAL VALIDATION v2")
print("=" * 90)

print("\nRows:", len(df))
print("Columns:", len(df.columns))


# =============================================================================
# 3. EPISODE UNIQUENESS
# =============================================================================

print("\n" + "=" * 90)
print("3. EPISODE UNIQUENESS")
print("=" * 90)

episode_report = pd.DataFrame({
    "metric": [
        "Rows",
        "Unique Episode_key",
        "Duplicated Episode_key rows",
        "Missing Episode_key"
    ],
    "value": [
        len(df),
        df["Episode_key"].nunique(dropna=True),
        df["Episode_key"].duplicated().sum(),
        df["Episode_key"].isna().sum()
    ]
})

print(episode_report.to_string(index=False))

episode_report.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "01_episode_uniqueness.csv"
    ),
    index=False
)


# =============================================================================
# 4. PRIMARY OUTCOME VALIDATION
# =============================================================================

print("\n" + "=" * 90)
print("4. PRIMARY OUTCOME VALIDATION")
print("=" * 90)

required = [
    "Episode_key",
    "oop_share",
    "log_total_cost"
]

missing_required = [
    x for x in required
    if x not in df.columns
]

if missing_required:
    raise ValueError(
        "Missing required variables: "
        + str(missing_required)
    )


valid_oop = (
    df["oop_share"].notna()
    &
    np.isfinite(df["oop_share"])
    &
    (df["oop_share"] >= 0)
    &
    (df["oop_share"] <= 1)
)

valid_cost = (
    df["log_total_cost"].notna()
    &
    np.isfinite(df["log_total_cost"])
)

primary = df[
    valid_oop & valid_cost
].copy()

print("\nPrimary N:", len(primary))

print(
    "\nOOP share:"
)

print(
    primary["oop_share"]
    .describe()
    .to_string()
)


# =============================================================================
# 5. OUTCOME MONOTONICITY CHECK
# =============================================================================

print("\n" + "=" * 90)
print("5. HIGH-OOP OUTCOME MONOTONICITY")
print("=" * 90)

thresholds = [
    ("high_oop_ge05pct", 0.05),
    ("high_oop_ge10pct", 0.10),
    ("high_oop_ge20pct", 0.20),
    ("high_oop_ge30pct", 0.30)
]

monotonic_rows = []

for var, threshold in thresholds:

    if var not in primary.columns:
        continue

    expected = (
        primary["oop_share"] >= threshold
    ).astype(int)

    observed = (
        pd.to_numeric(
            primary[var],
            errors="coerce"
        )
        .fillna(-999)
        .astype(int)
    )

    mismatch = (
        expected != observed
    )

    monotonic_rows.append({
        "variable": var,
        "threshold": threshold,
        "observed_positive": observed.eq(1).sum(),
        "expected_positive": expected.eq(1).sum(),
        "mismatch_n": mismatch.sum()
    })

monotonic_df = pd.DataFrame(
    monotonic_rows
)

print(
    monotonic_df.to_string(index=False)
)

monotonic_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "02_outcome_threshold_validation.csv"
    ),
    index=False
)


# =============================================================================
# 6. ZERO-LOS VALIDATION
# =============================================================================

print("\n" + "=" * 90)
print("6. ZERO-LOS VALIDATION")
print("=" * 90)

if "los_zero" in primary.columns:

    zero_los = (
        primary["los_zero"]
        .astype("Int64")
    )

    expected_zero = (
        pd.to_numeric(
            primary["los_days"],
            errors="coerce"
        ) == 0
    )

    zero_los_mismatch = (
        zero_los.notna()
        &
        (zero_los.astype(int) != expected_zero.astype(int))
    )

    print(
        "Zero-LOS flag mismatches:",
        zero_los_mismatch.sum()
    )

    zero_los_table = (
        primary
        .groupby("los_zero", dropna=False)
        .agg(
            n=("Episode_key", "size"),
            oop_mean=("oop_share", "mean"),
            oop_median=("oop_share", "median"),
            oop_sd=("oop_share", "std"),
            high_oop_10=("high_oop_ge10pct", "mean"),
            high_oop_20=("high_oop_ge20pct", "mean"),
            high_oop_30=("high_oop_ge30pct", "mean")
        )
        .reset_index()
    )

    for c in [
        "high_oop_10",
        "high_oop_20",
        "high_oop_30"
    ]:
        zero_los_table[c] *= 100

    print(
        zero_los_table.to_string(index=False)
    )

    zero_los_table.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "03_zero_los_validation.csv"
        ),
        index=False
    )


# =============================================================================
# 7. SENSITIVITY FLAG AUDIT
# =============================================================================
#
# This section does NOT assume that a flag's name tells us its meaning.
#
# It compares the flag with the underlying variable whenever possible.
# =============================================================================

print("\n" + "=" * 90)
print("7. SENSITIVITY FLAG AUDIT")
print("=" * 90)


def audit_binary_flag(data, flag, expected_keep):

    if flag not in data.columns:
        return {
            "flag": flag,
            "status": "NOT_FOUND"
        }

    x = pd.to_numeric(
        data[flag],
        errors="coerce"
    )

    return {
        "flag": flag,
        "status": "FOUND",
        "n": len(x),
        "n_0": (x == 0).sum(),
        "n_1": (x == 1).sum(),
        "missing": x.isna().sum(),
        "unique_values": sorted(
            x.dropna().unique().tolist()
        )
    }


flags = [
    "sens_excl_anonymous",
    "sens_excl_zero_los",
    "sens_excl_financing_gap",
    "sens_excl_financing_gap_v3_requested",
    "sens_excl_duplicates",
    "sens_excl_missing_discharge"
]

flag_rows = [
    audit_binary_flag(
        primary,
        flag,
        None
    )
    for flag in flags
]

flag_df = pd.DataFrame(flag_rows)

print(
    flag_df.to_string(index=False)
)

flag_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "04_sensitivity_flag_audit.csv"
    ),
    index=False
)


# =============================================================================
# 8. DERIVE SENSITIVITY COHORTS FROM SOURCE VARIABLES
# =============================================================================
#
# IMPORTANT:
# We do NOT rely on pre-existing sensitivity flags for the main report.
#
# We explicitly construct the cohorts from underlying variables.
# =============================================================================

sensitivity = {}

# Primary
sensitivity["primary_all"] = primary.copy()

# Anonymous
if "flag_anonymous_episode" in primary.columns:

    sensitivity["exclude_anonymous"] = primary[
        primary["flag_anonymous_episode"].fillna(0).eq(0)
    ].copy()

# Zero LOS
if "los_days" in primary.columns:

    sensitivity["exclude_zero_los"] = primary[
        pd.to_numeric(
            primary["los_days"],
            errors="coerce"
        ) > 0
    ].copy()

# Missing discharge
if "flag_missing_discharge_date" in primary.columns:

    sensitivity["exclude_missing_discharge"] = primary[
        primary["flag_missing_discharge_date"]
        .fillna(0)
        .eq(0)
    ].copy()

# Duplicate claims
if "flag_duplicate_claim" in primary.columns:

    # Only apply this if the variable contains actual information.
    duplicate_flag = pd.to_numeric(
        primary["flag_duplicate_claim"],
        errors="coerce"
    )

    if duplicate_flag.notna().any():

        sensitivity["exclude_duplicates"] = primary[
            duplicate_flag.eq(0)
        ].copy()


# =============================================================================
# 9. SENSITIVITY SUMMARY
# =============================================================================

sensitivity_rows = []

for name, d in sensitivity.items():

    sensitivity_rows.append({
        "cohort": name,
        "N": len(d),
        "mean_oop_share":
            d["oop_share"].mean(),
        "median_oop_share":
            d["oop_share"].median(),
        "sd_oop_share":
            d["oop_share"].std(),
        "high_oop_10_pct":
            100 * d["high_oop_ge10pct"].mean(),
        "high_oop_20_pct":
            100 * d["high_oop_ge20pct"].mean(),
        "high_oop_30_pct":
            100 * d["high_oop_ge30pct"].mean()
    })

sensitivity_summary = pd.DataFrame(
    sensitivity_rows
)

print(
    "\nSensitivity analysis:"
)

print(
    sensitivity_summary.to_string(index=False)
)

sensitivity_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "05_sensitivity_summary_reconstructed.csv"
    ),
    index=False
)


# =============================================================================
# 10. FINANCING VARIABLE REDUNDANCY
# =============================================================================

print("\n" + "=" * 90)
print("10. FINANCING VARIABLE REDUNDANCY")
print("=" * 90)

financing_vars = [
    "oop_share",
    "oop_share_of_known",
    "insurer_share",
    "insurer_deduction_share",
    "org_deduction_share",
    "financing_gap_pct",
    "financing_gap_v0_pct",
    "financing_gap_v1_pct",
    "financing_gap_v2_pct",
    "financing_gap_v3_pct",
    "financing_gap_v4_pct",
    "unreconciled_share"
]

financing_vars = [
    x for x in financing_vars
    if x in primary.columns
]

corr = (
    primary[financing_vars]
    .apply(pd.to_numeric, errors="coerce")
    .corr(method="spearman")
)

print(
    corr.round(3).to_string()
)

corr.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "06_financing_spearman_matrix.csv"
    )
)


# =============================================================================
# 11. AUTOMATIC REDUNDANCY DETECTION
# =============================================================================

redundant_pairs = []

for i in range(len(financing_vars)):

    for j in range(i + 1, len(financing_vars)):

        a = financing_vars[i]
        b = financing_vars[j]

        r = corr.loc[a, b]

        if pd.notna(r) and abs(r) >= 0.95:

            redundant_pairs.append({
                "variable_1": a,
                "variable_2": b,
                "spearman_rho": r
            })

redundant_df = pd.DataFrame(
    redundant_pairs
)

print("\nHighly redundant pairs |rho| >= 0.95:")

if len(redundant_df):
    print(
        redundant_df
        .sort_values(
            "spearman_rho",
            key=lambda x: x.abs(),
            ascending=False
        )
        .to_string(index=False)
    )
else:
    print("None.")

redundant_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "07_high_redundancy_pairs.csv"
    ),
    index=False
)


# =============================================================================
# 12. DO NOT USE THESE AS SIMULTANEOUS INDEX COMPONENTS
# =============================================================================
#
# We explicitly create a "do not combine" list.
# =============================================================================

do_not_combine = [
    "financing_gap_pct",
    "financing_gap_v0_pct",
    "financing_gap_v1_pct",
    "financing_gap_v2_pct",
    "financing_gap_v3_pct",
    "financing_gap_v4_pct",
    "unreconciled_share",
    "insurer_deduction_share",
    "org_deduction_share"
]

do_not_combine = [
    x for x in do_not_combine
    if x in primary.columns
]

pd.DataFrame({
    "variable": do_not_combine,
    "reason": [
        "Highly redundant financing-derived quantity"
        for _ in do_not_combine
    ]
}).to_csv(
    os.path.join(
        OUTPUT_DIR,
        "08_do_not_combine_components.csv"
    ),
    index=False
)


# =============================================================================
# 13. CANDIDATE SES VARIABLE SEARCH
# =============================================================================

print("\n" + "=" * 90)
print("13. SES VARIABLE SEARCH")
print("=" * 90)

ses_keywords = [
    "ses",
    "socio",
    "economic",
    "income",
    "wealth",
    "depriv",
    "poverty",
    "education",
    "educ",
    "occupation",
    "job",
    "employment",
    "residence",
    "residential",
    "area",
    "district",
    "county",
    "province",
    "income_decile",
    "income_quintile"
]

ses_candidates = []

for col in primary.columns:

    col_lower = str(col).lower()

    if any(
        keyword in col_lower
        for keyword in ses_keywords
    ):
        ses_candidates.append(col)

print("\nPotential SES-related variables:")

for x in ses_candidates:
    print(" -", x)

pd.DataFrame({
    "candidate_variable": ses_candidates
}).to_csv(
    os.path.join(
        OUTPUT_DIR,
        "09_ses_candidate_variables.csv"
    ),
    index=False
)


# =============================================================================
# 14. CLUSTER_FOR_SE AUDIT
# =============================================================================

print("\n" + "=" * 90)
print("14. Cluster_for_SE AUDIT")
print("=" * 90)

if "Cluster_for_SE" not in primary.columns:

    print(
        "Cluster_for_SE is NOT present."
    )

else:

    x = primary["Cluster_for_SE"]

    print(
        "\nData type:",
        x.dtype
    )

    print(
        "Missing:",
        x.isna().sum()
    )

    print(
        "Unique:",
        x.nunique(dropna=True)
    )

    print(
        "\nTop values:"
    )

    print(
        x.value_counts(dropna=False)
        .head(30)
        .to_string()
    )

    print(
        "\nDescriptive statistics:"
    )

    print(
        x.describe()
        .to_string()
    )

    # Relationship with financial outcomes
    cluster_summary = (
        primary
        .groupby("Cluster_for_SE", dropna=False)
        .agg(
            n=("Episode_key", "size"),
            oop_mean=("oop_share", "mean"),
            oop_median=("oop_share", "median"),
            high_oop_10=("high_oop_ge10pct", "mean"),
            high_oop_20=("high_oop_ge20pct", "mean")
        )
        .reset_index()
    )

    print(
        "\nCluster summary:"
    )

    print(
        cluster_summary.head(30)
        .to_string(index=False)
    )

    cluster_summary.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "10_cluster_for_SE_summary.csv"
        ),
        index=False
    )


# =============================================================================
# 15. HOSPITAL IDENTIFIER SEARCH
# =============================================================================

print("\n" + "=" * 90)
print("15. HOSPITAL IDENTIFIER SEARCH")
print("=" * 90)

hospital_keywords = [
    "hospital",
    "facility",
    "provider",
    "institution",
    "center",
    "centre",
    "hospital_id",
    "facility_id",
    "provider_id"
]

hospital_candidates = []

for col in primary.columns:

    col_lower = str(col).lower()

    if any(
        keyword in col_lower
        for keyword in hospital_keywords
    ):
        hospital_candidates.append(col)

print(
    "\nPotential hospital/provider variables:"
)

for x in hospital_candidates:
    print(" -", x)

pd.DataFrame({
    "candidate_hospital_variable": hospital_candidates
}).to_csv(
    os.path.join(
        OUTPUT_DIR,
        "11_hospital_candidate_variables.csv"
    ),
    index=False
)


# =============================================================================
# 16. OWNERSHIP AS HOSPITAL-LEVEL INFORMATION
# =============================================================================

if "ownership_grp" in primary.columns:

    ownership_summary = (
        primary
        .groupby("ownership_grp")
        .agg(
            n=("Episode_key", "size"),
            oop_mean=("oop_share", "mean"),
            oop_median=("oop_share", "median"),
            oop_sd=("oop_share", "std"),
            high_oop_10=("high_oop_ge10pct", "mean"),
            high_oop_20=("high_oop_ge20pct", "mean"),
            high_oop_30=("high_oop_ge30pct", "mean")
        )
        .reset_index()
    )

    for c in [
        "high_oop_10",
        "high_oop_20",
        "high_oop_30"
    ]:
        ownership_summary[c] *= 100

    print(
        "\nOwnership summary:"
    )

    print(
        ownership_summary.to_string(index=False)
    )

    ownership_summary.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "12_ownership_financial_protection.csv"
        ),
        index=False
    )


# =============================================================================
# 17. DEATH / DISCHARGE CONSISTENCY
# =============================================================================

print("\n" + "=" * 90)
print("17. DEATH / DISCHARGE CONSISTENCY")
print("=" * 90)

death_vars = [
    "died_in_hospital",
    "discharge_status",
    "flag_missing_discharge_date",
    "flag_invalid_date_order"
]

death_vars = [
    x for x in death_vars
    if x in primary.columns
]

for var in death_vars:

    print(
        f"\n--- {var} ---"
    )

    print(
        primary[var]
        .value_counts(dropna=False)
        .head(30)
        .to_string()
    )


if (
    "died_in_hospital" in primary.columns
    and
    "discharge_status" in primary.columns
):

    death_cross = pd.crosstab(
        primary["died_in_hospital"],
        primary["discharge_status"],
        dropna=False
    )

    print(
        "\nDeath x discharge-status table:"
    )

    print(
        death_cross.to_string()
    )

    death_cross.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "13_death_discharge_crosscheck.csv"
        )
    )


# =============================================================================
# 18. DEPARTMENT / SPECIALTY AUDIT
# =============================================================================
#
# The presence of cardiovascular emergency / cardiovascular specialty
# in a cholera cohort is not automatically an error.
#
# We need to determine whether this reflects:
#   - genuine comorbidity/referral,
#   - service assignment,
#   - coding structure,
#   - or case-definition contamination.
#
# =============================================================================

print("\n" + "=" * 90)
print("18. DEPARTMENT / SPECIALTY AUDIT")
print("=" * 90)

audit_vars = [
    "department_grp",
    "specialty_grp",
    "diagnosis_type_grp",
    "audit_cardiac_specialty_or_dept",
    "audit_any_setting_signal"
]

for var in audit_vars:

    if var not in primary.columns:
        continue

    print(
        f"\n--- {var} ---"
    )

    print(
        primary[var]
        .value_counts(dropna=False)
        .head(30)
        .to_string()
    )


# =============================================================================
# 19. CARDIOVASCULAR SIGNAL VS OOP
# =============================================================================

if "audit_cardiac_specialty_or_dept" in primary.columns:

    cardiac_summary = (
        primary
        .groupby(
            "audit_cardiac_specialty_or_dept",
            dropna=False
        )
        .agg(
            n=("Episode_key", "size"),
            oop_mean=("oop_share", "mean"),
            oop_median=("oop_share", "median"),
            high_oop_10=("high_oop_ge10pct", "mean"),
            high_oop_20=("high_oop_ge20pct", "mean")
        )
        .reset_index()
    )

    cardiac_summary["high_oop_10"] *= 100
    cardiac_summary["high_oop_20"] *= 100

    print(
        "\nCardiovascular audit:"
    )

    print(
        cardiac_summary.to_string(index=False)
    )

    cardiac_summary.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "14_cardiovascular_signal_audit.csv"
        ),
        index=False
    )


# =============================================================================
# 20. TIME STRUCTURE
# =============================================================================

print("\n" + "=" * 90)
print("20. TIME STRUCTURE")
print("=" * 90)

time_vars = [
    "admission_year",
    "admission_month",
    "admission_quarter",
    "admission_ym"
]

for var in time_vars:

    if var not in primary.columns:
        continue

    print(
        f"\n--- {var} ---"
    )

    print(
        primary[var]
        .value_counts(dropna=False)
        .sort_index()
        .to_string()
    )


# =============================================================================
# 21. OUTCOME BY YEAR
# =============================================================================

if "admission_year" in primary.columns:

    year_summary = (
        primary
        .groupby("admission_year")
        .agg(
            n=("Episode_key", "size"),
            oop_mean=("oop_share", "mean"),
            oop_median=("oop_share", "median"),
            high_oop_10=("high_oop_ge10pct", "mean"),
            high_oop_20=("high_oop_ge20pct", "mean"),
            high_oop_30=("high_oop_ge30pct", "mean")
        )
        .reset_index()
    )

    for c in [
        "high_oop_10",
        "high_oop_20",
        "high_oop_30"
    ]:
        year_summary[c] *= 100

    print(
        "\nFinancial protection by year:"
    )

    print(
        year_summary.to_string(index=False)
    )

    year_summary.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "15_financial_protection_by_year.csv"
        ),
        index=False
    )


# =============================================================================
# 22. CORE ANALYTICAL DATASET
# =============================================================================
#
# We retain primary outcome + prespecified covariates.
#
# We DO NOT create a composite index here.
# =============================================================================

core_vars = [
    "Episode_key",
    "Patient_cluster",
    "Cluster_for_SE",

    # primary outcome
    "oop_share",

    # secondary outcomes
    "high_oop_ge05pct",
    "high_oop_ge10pct",
    "high_oop_ge20pct",
    "high_oop_ge30pct",
    "zero_oop",

    # financial descriptors
    "insurer_share",
    "insurer_deduction_share",
    "org_deduction_share",
    "financing_gap_pct",

    # demographics
    "age",
    "age_group",
    "sex",

    # insurance/hospital
    "insurance_fund_grp",
    "ownership_grp",
    "department_grp",
    "diagnosis_type_grp",
    "specialty_grp",

    # utilization
    "los_days",
    "los_zero",
    "n_claim_lines",
    "n_service_groups",

    # cost
    "log_total_cost",
    "cost_per_day",

    # time
    "admission_year",
    "admission_month",
    "admission_quarter",

    # outcome
    "discharge_status",
    "died_in_hospital"
]

core_vars = [
    x for x in core_vars
    if x in primary.columns
]

core = primary[
    core_vars
].copy()

core.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "16_core_analytical_dataset_v2.csv"
    ),
    index=False
)


# =============================================================================
# 23. FINAL DECISION REPORT
# =============================================================================

print("\n" + "=" * 90)
print("FINAL ANALYTICAL DECISION REPORT")
print("=" * 90)

print(
    """
CURRENT STATUS
--------------

1. Primary outcome:
   oop_share is suitable as the primary financial-protection outcome.

2. Secondary outcomes:
   high_oop_ge10pct, high_oop_ge20pct and high_oop_ge30pct
   are suitable as secondary high-cost-share indicators.

3. Zero LOS:
   60.6% of episodes have zero LOS.
   This must remain in the primary cohort and be handled through
   a prespecified sensitivity analysis.

4. Financing variables:
   Several financing variables are mathematically/redundantly related.
   They must NOT all be combined into one composite index.

5. SES:
   No concentration index should be calculated until Cluster_for_SE
   is verified or an independent SES variable is identified.

6. Hospital clustering:
   A true hospital identifier is required for hospital-clustered
   inference. Ownership_grp alone should NOT automatically be treated
   as a hospital cluster identifier.

7. Clinical outcome:
   died_in_hospital should be audited before being used for validation.
   It should NOT be inserted into the financial-protection index.

8. Case definition:
   The cardiovascular/specialty signals require a cohort/coding audit
   before being interpreted substantively.

9. Composite index:
   NOT constructed at this stage.

10. Final inequality analysis:
    NOT performed at this stage.
"""
)

print(
    "\nOutput directory:"
)

print(
    OUTPUT_DIR
)

print("\nDONE.")

CHOLERA ANALYTICAL VALIDATION v2

Rows: 1171
Columns: 103

3. EPISODE UNIQUENESS
                     metric  value
                       Rows   1171
         Unique Episode_key   1171
Duplicated Episode_key rows      0
        Missing Episode_key      0

4. PRIMARY OUTCOME VALIDATION

Primary N: 1171

OOP share:
count    1171.000000
mean        0.198746
std         0.250839
min         0.000000
25%         0.032033
50%         0.062028
75%         0.258666
max         0.831186

5. HIGH-OOP OUTCOME MONOTONICITY
        variable  threshold  observed_positive  expected_positive  mismatch_n
high_oop_ge05pct       0.05                706                706           0
high_oop_ge10pct       0.10                446                446           0
high_oop_ge20pct       0.20                328                328           0
high_oop_ge30pct       0.30                276                276           0

6. ZERO-LOS VALIDATION
Zero-LOS flag mismatches: 0
 los_zero   n  oop_mean  oop_median   oo

In [16]:
# =============================================================================
# OBJECTIVE 1 - DIAGNOSTICS (run BEFORE trusting / revising the models)
# Usage:  python objective1_diagnostics.py                (uses default paths)
#         python objective1_diagnostics.py <input.csv> <out_dir>
# Output: everything is printed AND saved to <out_dir>/diagnostics_log.txt
#         -> paste the content of diagnostics_log.txt back to me.
# =============================================================================
import itertools
import re
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from patsy import dmatrix
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 45)

BASE = Path(r"C:\Users\Shekoofeh\Documents\Health Insurance Data\Cholera")
INPUT_FILE = BASE / "feature engineering v4_FINAL" / "01_analysis_dataset.csv"
OUT_DIR = INPUT_FILE.parent / "objective1_diagnostics"
OUT_DIR.mkdir(parents=True, exist_ok=True)
LOG = open(OUT_DIR / "diagnostics_log.txt", "w", encoding="utf-8")


def say(x=""):
    s = x.to_string() if isinstance(x, (pd.DataFrame, pd.Series)) else str(x)
    print(s)
    LOG.write(s + "\n")
    LOG.flush()


def head(t):
    say("\n" + "=" * 90)
    say(t)
    say("=" * 90)


OUT = ["los_days", "n_claim_lines", "n_service_groups"]
COV = ["age_group", "sex", "insurance_fund_grp", "diagnosis_type_grp",
       "specialty_grp", "department_grp", "ownership_grp", "admission_year"]
MIN_LEVEL_N = 30

# -----------------------------------------------------------------------------
# D0. Load + replicate the exact analytic sample used before (with attrition)
# -----------------------------------------------------------------------------
head("D0. SAMPLE DEFINITION AND ATTRITION")
df = pd.read_csv(INPUT_FILE, low_memory=False)
say(f"Rows x columns in file: {df.shape}")


def flag(col):
    if col not in df.columns:
        return pd.Series(False, index=df.index)
    return pd.to_numeric(df[col], errors="coerce").fillna(0).eq(1)


for c in OUT:
    df[c] = pd.to_numeric(df[c], errors="coerce")

m1 = flag("analytic_primary")
m2 = m1 & flag("complete_case_covariates") if "complete_case_covariates" in df.columns else m1
m3 = m2 & df[OUT + COV].notna().all(axis=1)
say(pd.DataFrame({"step": ["all rows", "analytic_primary==1", "+ complete_case_covariates",
                           "+ no NaN in outcomes/covariates"],
                  "n": [len(df), m1.sum(), m2.sum(), m3.sum()]}))

d = df.loc[m3].copy()
d["admission_year"] = d["admission_year"].astype(float).astype(int).astype(str)
for v in COV:
    d[v] = d[v].astype(str).str.strip()
d["cl"] = d["Cluster_for_SE"].fillna(d["Episode_key"]).astype(str)
d["los_pos"] = (d["los_days"] > 0).astype(int)
say(f"Analytic sample: {len(d):,} episodes")

# -----------------------------------------------------------------------------
# D1. Clustering structure
# -----------------------------------------------------------------------------
head("D1. CLUSTER STRUCTURE (validity of cluster-robust SEs)")
sz = d["cl"].value_counts()
say(f"Episodes: {len(d):,} | clusters: {len(sz):,} | singleton clusters: {(sz == 1).sum():,} "
    f"| largest cluster: {sz.max():,} ({sz.max() / len(d):.1%} of episodes)")
say("Cluster size distribution (top 10):")
say(sz.head(10))
say("Cluster-size summary:")
say(sz.describe())
say(f"Blank Cluster_for_SE in sample: {d['Cluster_for_SE'].isna().sum()}")
if "flag_anonymous_episode" in d.columns:
    an = pd.to_numeric(d["flag_anonymous_episode"], errors="coerce").fillna(0).eq(1)
    say(f"Anonymous episodes in sample: {an.sum()} | distinct clusters among them: "
        f"{d.loc[an, 'cl'].nunique()}")
if "n_episodes_patient" in d.columns:
    say("n_episodes_patient (in sample):")
    say(pd.to_numeric(d["n_episodes_patient"], errors="coerce").describe())

# -----------------------------------------------------------------------------
# D2. Outcome integrity
# -----------------------------------------------------------------------------
head("D2a. LOS: tail, concentration, data-quality flags")
say(d["los_days"].value_counts().sort_index().head(12).rename("n (LOS value, first 12)"))
say("Quantiles: " + str(d["los_days"].quantile([.5, .75, .9, .95, .99, .995, 1]).round(1).to_dict()))
srt = d["los_days"].sort_values(ascending=False)
tot = srt.sum()
say("Share of TOTAL LOS-days contributed by the top k episodes: " +
    str({k: f"{srt.head(k).sum() / tot:.1%}" for k in (1, 5, 10, 25)}))
say(f"Mean LOS all={d['los_days'].mean():.2f} | without top 5={srt.iloc[5:].mean():.2f} "
    f"| without top 10={srt.iloc[10:].mean():.2f}")
cols = ["los_days", "n_claim_lines", "department_grp", "specialty_grp", "diagnosis_type_grp",
        "insurance_fund_grp", "ownership_grp", "admission_year"]
say("15 longest stays:")
say(d.sort_values("los_days", ascending=False)[cols].head(15))
fl = [c for c in ["flag_los_mismatch", "flag_invalid_date_order", "flag_missing_discharge_date",
                  "flag_duplicate_claim", "flag_anonymous_episode", "died_in_hospital"] if c in d.columns]
say("Flag counts inside the analytic sample:")
say(pd.Series({c: int(pd.to_numeric(d[c], errors="coerce").fillna(0).eq(1).sum()) for c in fl}))

head("D2b. CLAIM LINES vs SERVICE GROUPS (are they really two dimensions?)")
say(f"n_claim_lines: min={d.n_claim_lines.min()} max={d.n_claim_lines.max()} | "
    f"episodes AT the max value: {(d.n_claim_lines == d.n_claim_lines.max()).sum()}")
say(d["n_claim_lines"].value_counts().sort_index().rename("n episodes"))
diff = d["n_claim_lines"] - d["n_service_groups"]
say(f"Identical (lines == groups): {(diff == 0).mean():.1%} | groups > lines (logical error): "
    f"{(diff < 0).sum()} | lines - groups: max={diff.max()}")
say(f"Spearman(lines, groups) = {stats.spearmanr(d.n_claim_lines, d.n_service_groups)[0]:.4f} | "
    f"Pearson = {np.corrcoef(d.n_claim_lines, d.n_service_groups)[0, 1]:.4f}")
g = d.groupby("department_grp")["n_claim_lines"].agg(
    n="size", min="min", median="median", max="max", sd="std",
    pct_equal_4=lambda s: (s == 4).mean() * 100)
say("Claim lines by department (fixed billing bundles would show SD ~ 0):")
say(g.round(2))
g2 = d.groupby("diagnosis_type_grp")["n_claim_lines"].agg(n="size", min="min", median="median", max="max", sd="std")
say(g2.round(2))

# -----------------------------------------------------------------------------
# D3. Categorical structure: levels, sparsity, separation, cohort content
# -----------------------------------------------------------------------------
head("D3a. RAW LEVELS (before pooling): size, LOS>0 rate, separation risk")
for v in COV:
    t = d.groupby(v).agg(n=("los_pos", "size"), n_los_pos=("los_pos", "sum"),
                         mean_los=("los_days", "mean"), max_los=("los_days", "max"),
                         mean_claims=("n_claim_lines", "mean"),
                         sd_claims=("n_claim_lines", "std")).reset_index()
    t["pct_los_pos"] = t["n_los_pos"] / t["n"] * 100
    t["SMALL_N(<30)"] = np.where(t["n"] < MIN_LEVEL_N, "<<<", "")
    t["SEPARATION?"] = np.where((t["n_los_pos"] <= 1) | (t["n"] - t["n_los_pos"] <= 1), "<<<", "")
    say(f"\n--- {v} ---")
    say(t.round(2))


def cramers_v(x, y):
    ct = pd.crosstab(x, y)
    chi2 = stats.chi2_contingency(ct, correction=False)[0]
    n = ct.values.sum()
    phi2 = chi2 / n
    r, k = ct.shape
    phi2c = max(0, phi2 - (k - 1) * (r - 1) / (n - 1))
    rc, kc = r - (r - 1) ** 2 / (n - 1), k - (k - 1) ** 2 / (n - 1)
    den = min(kc - 1, rc - 1)
    return np.sqrt(phi2c / den) if den > 0 else np.nan


head("D3b. PAIRWISE ASSOCIATION AND EMPTY CELLS BETWEEN DETERMINANTS")
rows = []
for a, b in itertools.combinations(COV, 2):
    ct = pd.crosstab(d[a], d[b])
    rows.append({"var_a": a, "var_b": b, "cells": ct.size, "empty_cells": int((ct == 0).sum().sum()),
                 "cells_n<5": int(((ct > 0) & (ct < 5)).sum().sum()),
                 "cramers_v": round(cramers_v(d[a], d[b]), 3)})
pw = pd.DataFrame(rows).sort_values("cramers_v", ascending=False)
pw["pct_empty"] = (pw["empty_cells"] / pw["cells"] * 100).round(1)
say(pw)
pw.to_csv(OUT_DIR / "D3b_pairwise.csv", index=False)
say("\nDepartment x Ownership:")
say(pd.crosstab(d["department_grp"], d["ownership_grp"]))
say("\nDepartment x Diagnosis type:")
say(pd.crosstab(d["department_grp"], d["diagnosis_type_grp"]))

head("D3c. COHORT CONTENT CHECK (case-definition plausibility)")
aud = [c for c in df.columns if c.startswith("audit_")]
if aud:
    a = df.loc[d.index, aud].apply(pd.to_numeric, errors="coerce").fillna(0)
    say("Audit flags inside analytic sample (n flagged, % of sample):")
    say(pd.DataFrame({"n": a.sum().astype(int), "pct": (a.mean() * 100).round(1)}))
    if "audit_any_setting_signal" in a.columns:
        d["audit_any"] = a["audit_any_setting_signal"].values
        say("Outcomes by audit_any_setting_signal:")
        say(d.groupby("audit_any")[OUT].agg(["size", "mean", "median"]).round(2))
        say("Specialty x audit_any_setting_signal:")
        say(pd.crosstab(d["specialty_grp"], d["audit_any"]))
else:
    say("No audit_* columns found.")

# -----------------------------------------------------------------------------
# Modelling frame: same pooling as before
# -----------------------------------------------------------------------------
dm = d.copy()
for v in COV:
    vc = dm[v].value_counts()
    rare = vc[vc < MIN_LEVEL_N].index
    dm.loc[dm[v].isin(rare), v] = "Other (pooled)"
REF = {v: dm[v].value_counts().idxmax() for v in COV}


def rhs(vars_):
    return " + ".join(f"C({v}, Treatment(reference='{REF[v]}'))" for v in vars_)


# -----------------------------------------------------------------------------
# D4. Design-matrix collinearity
# -----------------------------------------------------------------------------
head("D4. DESIGN MATRIX: rank, conditioning, GENERALISED VIF")
Xfull = dmatrix(rhs(COV), dm, return_type="dataframe")
npar = Xfull.shape[1]
say(f"Parameters (incl. intercept): {npar} | rank: {np.linalg.matrix_rank(Xfull.values)} "
    f"| n = {len(dm):,} | n LOS>0 = {int(dm.los_pos.sum())}")
say(f"Episodes per parameter: all={len(dm) / npar:.1f} | LOS>0 only={dm.los_pos.sum() / npar:.1f}")
say(f"Clusters per parameter: {dm['cl'].nunique() / npar:.1f} (need >1; ideally >>1)")
Xs = Xfull.drop(columns="Intercept")
Xs = (Xs - Xs.mean()) / Xs.std().replace(0, 1)
say(f"Condition number (standardised X): {np.linalg.cond(Xs.values):.1f}  (>30 = concerning)")

blocks = {v: dmatrix(f"C({v}, Treatment(reference='{REF[v]}'))", dm, return_type="dataframe").drop(columns="Intercept")
          for v in COV}
Xb = pd.concat(blocks.values(), axis=1)
R = np.corrcoef(Xb.values, rowvar=False)
ld_all = np.linalg.slogdet(R)[1]
gv = []
for v, M in blocks.items():
    idx = [Xb.columns.get_loc(c) for c in M.columns]
    rest = [i for i in range(Xb.shape[1]) if i not in idx]
    val = np.exp(np.linalg.slogdet(R[np.ix_(idx, idx)])[1] + np.linalg.slogdet(R[np.ix_(rest, rest)])[1] - ld_all)
    gv.append({"variable": v, "df": len(idx), "GVIF": val, "GVIF^(1/2df)": val ** (1 / (2 * len(idx)))})
gv = pd.DataFrame(gv)
gv["flag"] = np.where(gv["GVIF^(1/2df)"] > 3.16, "SEVERE (>3.16)",
                      np.where(gv["GVIF^(1/2df)"] > 2.24, "concerning (>2.24)", ""))
say(gv.round(3))
gv.to_csv(OUT_DIR / "D4_gvif.csv", index=False)

# -----------------------------------------------------------------------------
# D5. Model stability
# -----------------------------------------------------------------------------
def qp(data, y, covs):
    m = smf.glm(f"{y} ~ {rhs(covs)}", data=data, family=sm.families.Poisson())
    return m.fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(data["cl"])[0]})


def wald_by_var(res):
    out = {}
    try:
        tab = res.wald_test_terms(scalar=True).table
        for term, row in tab.iterrows():
            mm = re.search(r"C\((\w+),", str(term))
            if mm:
                out[mm.group(1)] = float(row.iloc[1])
    except Exception:
        pass
    return out


def short(t):
    return re.sub(r"C\((\w+), Treatment\(reference='.*?'\)\)\[T\.(.*)\]", r"\1: \2", t)


def sign_flips(a, b):
    common = [c for c in a.params.index.intersection(b.params.index) if c not in ("Intercept", "alpha")]
    bad = [c for c in common if np.sign(a.params[c]) != np.sign(b.params[c])
           and abs(a.params[c]) > 0.10 and abs(b.params[c]) > 0.10]
    return len(common), bad


head("D5a. SPECIFICATION STABILITY (joint Wald p per determinant, cluster-robust)")
for y in OUT:
    say(f"\n########## OUTCOME: {y} ##########")
    fits = {}
    fits["A_full"] = qp(dm, y, COV)
    fits["B_no_dept"] = qp(dm, y, [v for v in COV if v != "department_grp"])
    fits["C_dept_only_context"] = qp(dm, y, ["age_group", "sex", "insurance_fund_grp",
                                             "department_grp", "admission_year"])
    top5 = dm[y].sort_values(ascending=False).index[:5]
    fits["D_drop_top5_y"] = qp(dm.drop(index=top5), y, COV)
    dm["_yw"] = dm[y].clip(upper=dm[y].quantile(0.99))
    fits["E_winsor_p99"] = smf.glm(f"_yw ~ {rhs(COV)}", data=dm, family=sm.families.Poisson()).fit(
        cov_type="cluster", cov_kwds={"groups": pd.factorize(dm["cl"])[0]})
    try:
        nb = smf.negativebinomial(f"{y} ~ {rhs(COV)}", data=dm).fit(
            method="bfgs", maxiter=500, disp=0, cov_type="cluster",
            cov_kwds={"groups": pd.factorize(dm["cl"])[0]})
        fits["F_NB2_MLE"] = nb
        say(f"NB2 MLE alpha = {float(nb.params['alpha']):.4f} | converged = {nb.mle_retvals.get('converged')}")
    except Exception as e:
        say(f"NB2 MLE failed: {e}")
    tab = pd.DataFrame({k: wald_by_var(r) for k, r in fits.items()}).reindex(COV)
    say("Wald p-values:")
    say(tab.map(lambda x: f"{x:.3g}" if pd.notna(x) else "-"))
    tab.to_csv(OUT_DIR / f"D5a_wald_{y}.csv")
    for k in ("D_drop_top5_y", "E_winsor_p99", "F_NB2_MLE"):
        if k in fits:
            n_c, bad = sign_flips(fits["A_full"], fits[k])
            say(f"Sign flips A_full vs {k}: {len(bad)}/{n_c} terms"
                + (" -> " + "; ".join(short(b) for b in bad[:12]) if bad else ""))
    A = fits["A_full"]
    naive = smf.glm(f"{y} ~ {rhs(COV)}", data=dm, family=sm.families.Poisson()).fit(scale="X2")
    ratio = (A.bse / naive.bse).drop("Intercept")
    say(f"Pearson dispersion={naive.scale:.3f} | robust/naive SE ratio: median={ratio.median():.2f}, "
        f"min={ratio.min():.2f}, max={ratio.max():.2f}")
    big = A.params.drop("Intercept")
    big = big[(np.exp(big) > 20) | (np.exp(big) < 0.05)]
    say(f"Terms with implausible rate ratios (>20 or <0.05): {len(big)}")
    if len(big):
        say(pd.DataFrame({"term": [short(i) for i in big.index], "RR": np.exp(big.values).round(3),
                          "SE(log)": A.bse[big.index].values.round(2)}))

head("D5b. LOGISTIC PART-1 (LOS>0): separation check")
try:
    lg = smf.logit(f"los_pos ~ {rhs(COV)}", data=dm).fit(disp=0, maxiter=300)
    cf = pd.DataFrame({"term": [short(i) for i in lg.params.index], "coef": lg.params.values,
                       "se": lg.bse.values})
    cf["SUSPECT"] = np.where((cf.coef.abs() > 8) | (cf.se > 5), "<<<", "")
    say(f"converged={lg.mle_retvals.get('converged')} | max|coef|={cf.coef.abs().max():.1f}")
    say(cf[cf["SUSPECT"] != ""].round(2))
    say(f"Number of suspect coefficients: {(cf['SUSPECT'] != '').sum()}")
except Exception as e:
    say(f"Logit failed (itself a sign of separation): {e}")
say("\nDepartment-level LOS>0 (the usual source of separation):")
say(dm.groupby("department_grp")["los_pos"].agg(n="size", n_pos="sum", pct=lambda s: s.mean() * 100).round(1))
say("\nOwnership-level LOS>0:")
say(dm.groupby("ownership_grp")["los_pos"].agg(n="size", n_pos="sum", pct=lambda s: s.mean() * 100).round(1))

head("D5c. UNADJUSTED (one determinant at a time) joint Wald p, quasi-Poisson")
uni = {}
for y in OUT:
    uni[y] = {v: wald_by_var(qp(dm, y, [v])).get(v, np.nan) for v in COV}
say(pd.DataFrame(uni).map(lambda x: f"{x:.3g}"))

# -----------------------------------------------------------------------------
# D6. Sensitivity flags and time coverage
# -----------------------------------------------------------------------------
head("D6. SENSITIVITY FLAGS INSIDE THE ANALYTIC SAMPLE and TIME COVERAGE")
sf = [c for c in df.columns if c.startswith("sens_excl_")]
say(pd.DataFrame({"flag": sf,
                  "kept": [int(flag(c).loc[d.index].sum()) for c in sf],
                  "dropped_vs_primary": [len(d) - int(flag(c).loc[d.index].sum()) for c in sf]}))
if "admission_ym" in d.columns:
    say("Episodes per admission month:")
    say(d["admission_ym"].value_counts().sort_index())
say(d.groupby("admission_year")["los_days"].agg(n="size", mean="mean", pct_gt10=lambda s: (s > 10).mean() * 100).round(2))

head("END OF DIAGNOSTICS - please paste diagnostics_log.txt back")
say(f"Saved to: {OUT_DIR / 'diagnostics_log.txt'}")
LOG.close()


D0. SAMPLE DEFINITION AND ATTRITION
Rows x columns in file: (1171, 103)
                              step     n
0                         all rows  1171
1              analytic_primary==1  1171
2       + complete_case_covariates  1171
3  + no NaN in outcomes/covariates  1171
Analytic sample: 1,171 episodes

D1. CLUSTER STRUCTURE (validity of cluster-robust SEs)
Episodes: 1,171 | clusters: 1,144 | singleton clusters: 1,130 | largest cluster: 14 (1.2% of episodes)
Cluster size distribution (top 10):
cl
194     14
939      3
720      2
186      2
458      2
481      2
671      2
1053     2
989      2
990      2
Cluster-size summary:
count    1144.000000
mean        1.023601
std         0.401618
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max        14.000000
Blank Cluster_for_SE in sample: 0
Anonymous episodes in sample: 67 | distinct clusters among them: 67
n_episodes_patient (in sample):
count    1104.000000
mean        1.192029
std         1.46

In [18]:
# =============================================================================
# OBJECTIVE 1 (v2) - Claims-observable healthcare resource utilization
# Revised after diagnostics. No SES variables, no cost/OOP outcomes, no composite index.
#
# Outcomes (separate, multidimensional RU):
#   (a) Service intensity  = n_claim_lines  (n_service_groups is 99.9% identical -> not modelled)
#   (b) LOS, as a two-part construct:
#         Part 1: any overnight stay (LOS >= 1)         -> Firth penalised logistic
#         Part 2: LOS among LOS >= 1, winsorised at P99 -> quasi-Poisson (log link)
# Determinants enter in Andersen blocks (sequential models M1..M4).
# =============================================================================
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from patsy import dmatrix
from scipy import stats
from scipy.special import expit

warnings.filterwarnings("ignore")

# ----------------------------------------------------------------------------
# 0. SETTINGS AND ANALYTIC DECISIONS (all driven by the diagnostics)
# ----------------------------------------------------------------------------
BASE = Path(r"C:\Users\Shekoofeh\Documents\Health Insurance Data\Cholera")
INPUT_FILE = BASE / "feature engineering v4_FINAL" / "01_analysis_dataset.csv"
OUT_DIR = INPUT_FILE.parent / "objective1_outputs_v2"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MIN_LEVEL_N = 30            # levels smaller than this are pooled
LOS_WINSOR_Q = 0.99         # LOS cap for modelling (raw LOS kept as sensitivity)
DROP_TOP = 5                # influence check: drop the k largest outcome values
BUNDLE_DEPT = "Limited surgery - D Clinic"   # ~98% of episodes have exactly 4 claim lines

# Level merges (CONFIRM these are the same kind of payer in your data)
INSURANCE_MERGE = {"Other layers": "Other funds",
                   "Other layers of the referral system": "Other funds",
                   "Other (rare)": "Other funds"}
# Reference levels with enough events (the old reference had 1 LOS>0 out of 353)
REF_OVERRIDE = {"department_grp": "Cardiovascular emergency",
                "specialty_grp": "Specialist in cardiovascular diseases"}

ANDERSEN = {
    "Predisposing": ["age_group", "sex"],
    "Enabling": ["insurance_fund_grp"],
    "Need / episode type": ["diagnosis_type_grp", "specialty_grp"],
    "Service setting": ["department_grp"],
    "Time / control": ["admission_year"],
}
BLOCKS = [
    ("M1_predisposing+time", ["age_group", "sex", "admission_year"]),
    ("M2_+enabling", ["age_group", "sex", "admission_year", "insurance_fund_grp"]),
    ("M3_+need", ["age_group", "sex", "admission_year", "insurance_fund_grp",
                  "diagnosis_type_grp", "specialty_grp"]),
    ("M4_+service_setting_PRIMARY", ["age_group", "sex", "admission_year", "insurance_fund_grp",
                                     "diagnosis_type_grp", "specialty_grp", "department_grp"]),
]
PRIMARY = BLOCKS[-1][1]
DESC_VARS = PRIMARY + ["ownership_grp"]        # ownership: descriptive + sensitivity only
ADJ_VARS = ["age_group", "sex", "insurance_fund_grp", "admission_year"]  # full overlap only

# ----------------------------------------------------------------------------
# 1. LOAD DATA, SAMPLE, CLEAN CATEGORIES
# ----------------------------------------------------------------------------
df = pd.read_csv(INPUT_FILE, low_memory=False)
print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} cols")


def flag(col):
    if col not in df.columns:
        return pd.Series(False, index=df.index)
    return pd.to_numeric(df[col], errors="coerce").fillna(0).eq(1)


need = ["los_days", "n_claim_lines", "n_service_groups", "analytic_primary", "Cluster_for_SE",
        "Episode_key"] + DESC_VARS
miss = [c for c in need if c not in df.columns]
assert not miss, f"Missing columns: {miss}"
for c in ["los_days", "n_claim_lines", "n_service_groups"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

mask = flag("analytic_primary")
if "complete_case_covariates" in df.columns:
    mask &= flag("complete_case_covariates")
d = df.loc[mask].dropna(subset=["los_days", "n_claim_lines"] + DESC_VARS).copy()
d = d[(d["los_days"] >= 0) & (d["n_claim_lines"] >= 1)]
d["admission_year"] = d["admission_year"].astype(float).astype(int).astype(str)
for v in DESC_VARS:
    d[v] = d[v].astype(str).str.strip()
d["insurance_fund_grp"] = d["insurance_fund_grp"].replace(INSURANCE_MERGE)
for v in DESC_VARS:                                   # pool remaining sparse levels
    vc = d[v].value_counts()
    rare = vc[vc < MIN_LEVEL_N].index
    if len(rare):
        print(f"  {v}: pooling {list(rare)} -> 'Other (pooled)'")
        d.loc[d[v].isin(rare), v] = "Other (pooled)"

d["cl"] = d["Cluster_for_SE"].fillna(d["Episode_key"]).astype(str)
d["los_pos"] = (d["los_days"] >= 1).astype(int)
CAP = int(round(d["los_days"].quantile(LOS_WINSOR_Q)))
d["los_w"] = d["los_days"].clip(upper=CAP)
anon = flag("flag_anonymous_episode").loc[d.index]
cardiac = flag("audit_cardiac_specialty_or_dept").loc[d.index]
print(f"Analytic sample: {len(d):,} episodes | {d['cl'].nunique():,} clusters | "
      f"LOS cap (P{int(LOS_WINSOR_Q * 100)}) = {CAP} days | LOS>=1: {d['los_pos'].sum()}")


# Reference levels are fixed ONCE on the full analytic sample (same in every model/subset)
REF = {}
for _v in DESC_VARS:
    _o = REF_OVERRIDE.get(_v)
    REF[_v] = _o if (_o is not None and (d[_v] == _o).any()) else d[_v].value_counts().idxmax()
print("Reference levels:", REF)


def ref_for(data, v):
    if (data[v] == REF[v]).any():
        return REF[v]
    return data[v].value_counts().idxmax()      # only if the level is absent (e.g. sensitivity S4)


def rhs(data, covs):
    covs = [v for v in covs if data[v].nunique() > 1]
    return " + ".join(f"C({v}, Treatment(reference='{ref_for(data, v)}'))" for v in covs) or "1"


def var_of(term):
    m = re.search(r"C\((\w+),", str(term))
    return m.group(1) if m else str(term)


def short(t):
    return re.sub(r"C\((\w+), Treatment\(reference='.*?'\)\)\[T\.(.*)\]", r"\1: \2", str(t))


# ----------------------------------------------------------------------------
# 2. DESCRIPTIVES
# ----------------------------------------------------------------------------
def desc(s):
    return pd.Series({"N": s.notna().sum(), "Mean": s.mean(), "SD": s.std(), "Median": s.median(),
                      "Q1": s.quantile(.25), "Q3": s.quantile(.75), "P90": s.quantile(.90),
                      "P99": s.quantile(.99), "Min": s.min(), "Max": s.max(),
                      "% zero": (s == 0).mean() * 100})


T1 = pd.DataFrame({"LOS days (raw)": desc(d["los_days"]),
                   f"LOS days (winsorised at {CAP})": desc(d["los_w"]),
                   "Claim lines": desc(d["n_claim_lines"]),
                   "Service groups (near-duplicate of claim lines)": desc(d["n_service_groups"])}).T
print("\n=== T1 overall resource utilization ===")
print(T1.round(2))
T1.to_csv(OUT_DIR / "T1_overall.csv")

srt = d["los_days"].sort_values(ascending=False)
conc = pd.Series({f"top {k} stays": srt.head(k).sum() / srt.sum() * 100 for k in (1, 5, 10, 25)},
                 name="% of total LOS-days")
print("\nLOS concentration:\n", conc.round(1))
conc.to_csv(OUT_DIR / "T1b_los_concentration.csv")

show = [c for c in ["Episode_key", "los_days", "n_claim_lines", "department_grp", "specialty_grp",
                    "diagnosis_type_grp", "ownership_grp", "admission_ym"] if c in d.columns]
d.sort_values("los_days", ascending=False)[show].head(15).to_csv(
    OUT_DIR / "T1c_extreme_LOS_episodes_VERIFY_IN_RAW_DATA.csv", index=False)

sp = d[["los_days", "n_claim_lines", "n_service_groups"]].corr(method="spearman")
print("\nSpearman:\n", sp.round(3))
sp.to_csv(OUT_DIR / "T1d_spearman.csv")


def by_group(var):
    rows = []
    for lvl, g in d.groupby(var):
        rows.append({"Variable": var, "Level": lvl, "N": len(g), "% sample": len(g) / len(d) * 100,
                     "% LOS>=1": g["los_pos"].mean() * 100,
                     "LOS median [IQR] (raw)": f"{g.los_days.median():.0f} [{g.los_days.quantile(.25):.0f}-{g.los_days.quantile(.75):.0f}]",
                     "LOS mean (winsorised)": g["los_w"].mean(),
                     "Claim lines median [IQR]": f"{g.n_claim_lines.median():.0f} [{g.n_claim_lines.quantile(.25):.0f}-{g.n_claim_lines.quantile(.75):.0f}]",
                     "Claim lines mean (SD)": f"{g.n_claim_lines.mean():.2f} ({g.n_claim_lines.std():.2f})"})
    out = pd.DataFrame(rows)
    for y, lab in [("los_days", "LOS"), ("n_claim_lines", "Claims")]:
        try:
            out[f"{lab} KW p (exploratory)"] = stats.kruskal(*[g[y].values for _, g in d.groupby(var)]).pvalue
        except ValueError:
            out[f"{lab} KW p (exploratory)"] = np.nan
    return out


T2 = pd.concat([by_group(v) for v in DESC_VARS], ignore_index=True)
T2.to_csv(OUT_DIR / "T2_descriptives_by_determinant.csv", index=False)
print("\nSaved T2 (descriptives by determinant, ownership included descriptively).")


# ----------------------------------------------------------------------------
# 3. MODEL ENGINE
# ----------------------------------------------------------------------------
def firth(X, y, maxit=100, tol=1e-7):
    """Firth (Jeffreys-penalised) logistic regression. Returns beta, cov, penalised loglik."""
    k = X.shape[1]
    b = np.zeros(k)

    def pll(beta):
        eta = X @ beta
        p = expit(eta)
        W = p * (1 - p)
        ld = np.linalg.slogdet(X.T @ (X * W[:, None]))[1]
        return np.sum(y * eta - np.logaddexp(0, eta)) + 0.5 * ld

    for _ in range(maxit):
        eta = X @ b
        p = expit(eta)
        W = p * (1 - p)
        Iinv = np.linalg.pinv(X.T @ (X * W[:, None]))
        h = np.einsum("ij,jk,ik->i", X, Iinv, X) * W
        step = Iinv @ (X.T @ (y - p + h * (0.5 - p)))
        m = np.max(np.abs(step))
        if m > 5:
            step *= 5 / m
        f0, t = pll(b), 1.0
        while pll(b + t * step) < f0 - 1e-10 and t > 1e-4:
            t /= 2
        b_new = b + t * step
        done = np.max(np.abs(b_new - b)) < tol
        b = b_new
        if done:
            break
    p = expit(X @ b)
    W = p * (1 - p)
    cov = np.linalg.pinv(X.T @ (X * W[:, None]))
    return b, cov, pll(b)


YCOL = {"claims": "n_claim_lines", "los_part1": "los_pos", "los_part2": "los_w",
        "los_part2_gamma": "los_w", "los_overall": "los_w", "los_overall_raw": "los_days"}
LABEL = {"claims": "Claim lines (rate ratio)", "los_part1": "LOS part 1: any overnight stay (odds ratio, Firth)",
         "los_part2": "LOS part 2: LOS | >=1 day, winsorised (ratio of means)"}


def run(kind, data, covs, drop_top=0):
    if kind in ("los_part2", "los_part2_gamma"):
        data = data[data["los_days"] >= 1]
    ycol = YCOL[kind]
    if drop_top and kind != "los_part1":
        data = data.drop(index=data[ycol].nlargest(drop_top).index)
    data = data.copy()
    cv = [v for v in covs if data[v].nunique() > 1]

    if kind == "los_part1":
        X = dmatrix(rhs(data, covs), data, return_type="dataframe")
        y = data["los_pos"].values.astype(float)
        b, cov, pl = firth(X.values, y)
        se = np.sqrt(np.diag(cov))
        pv = {}
        for v in cv:
            Xr = dmatrix(rhs(data, [u for u in cv if u != v]), data, return_type="dataframe")
            plr = firth(Xr.values, y)[2]
            dfv = sum(var_of(c) == v for c in X.columns)
            pv[v] = stats.chi2.sf(max(2 * (pl - plr), 0), dfv)
        params = pd.Series(b, index=X.columns)
        tab = pd.DataFrame({"term": X.columns, "ratio": np.exp(b), "ci_low": np.exp(b - 1.96 * se),
                            "ci_high": np.exp(b + 1.96 * se),
                            "p": 2 * stats.norm.sf(np.abs(b / se))})
        return dict(kind=kind, n=len(data), params=params, pvals=pv, tab=tab[tab.term != "Intercept"],
                    data=data, design_info=X.design_info, npar=X.shape[1], dev_expl=np.nan, disp=np.nan,
                    refs={v: ref_for(data, v) for v in cv})

    fam = sm.families.Poisson()
    if kind == "los_part2_gamma":
        link = getattr(sm.families.links, "Log", None) or sm.families.links.log
        fam = sm.families.Gamma(link())
    res = smf.glm(f"{ycol} ~ {rhs(data, covs)}", data=data, family=fam).fit(
        cov_type="cluster", cov_kwds={"groups": pd.factorize(data["cl"])[0]})
    pv = {}
    try:
        for term, row in res.wald_test_terms(scalar=True).table.iterrows():
            if var_of(term) in cv:
                pv[var_of(term)] = float(row.iloc[1])
    except Exception:
        pass
    ci = res.conf_int()
    tab = pd.DataFrame({"term": res.params.index, "ratio": np.exp(res.params.values),
                        "ci_low": np.exp(ci[0].values), "ci_high": np.exp(ci[1].values),
                        "p": res.pvalues.values})
    return dict(kind=kind, n=int(res.nobs), params=res.params, pvals=pv, tab=tab[tab.term != "Intercept"],
                data=data, res=res, npar=len(res.params),
                dev_expl=1 - res.deviance / res.null_deviance, disp=res.pearson_chi2 / res.df_resid,
                refs={v: ref_for(data, v) for v in cv})


def clean(tab, label, refs):
    t = tab.copy()
    t["reference"] = t["term"].map(lambda x: refs.get(var_of(x), ""))
    t["term"] = t["term"].map(short)
    t.insert(0, "model", label)
    return t


# ----------------------------------------------------------------------------
# 4. ANDERSEN SEQUENTIAL BLOCKS (+ primary model tables)
# ----------------------------------------------------------------------------
print("\n=== ANDERSEN SEQUENTIAL MODELS ===")
block_rows, prim_tabs, prim = [], [], {}
for kind in LABEL:
    for bname, covs in BLOCKS:
        r = run(kind, d, covs)
        block_rows.append({"outcome": kind, "model": bname, "n": r["n"], "n_params": r["npar"],
                           "deviance_explained": r["dev_expl"],
                           **{f"p_{v}": r["pvals"].get(v, np.nan) for v in PRIMARY}})
        if bname == BLOCKS[-1][0]:
            prim[kind] = r
            prim_tabs.append(clean(r["tab"], LABEL[kind], r["refs"]))
            epp = r["n"] / r["npar"]
            print(f"  {kind}: n={r['n']:,}, params={r['npar']}, n/param={epp:.1f}"
                  + (f", Pearson dispersion={r['disp']:.2f}" if kind != "los_part1" else ""))
T4 = pd.DataFrame(block_rows)
T4.to_csv(OUT_DIR / "T4_andersen_blocks_joint_tests.csv", index=False)
pd.concat(prim_tabs).to_csv(OUT_DIR / "T5_primary_model_ratios.csv", index=False)
print(T4.round(4).to_string())

# ----------------------------------------------------------------------------
# 5. SENSITIVITY (primary model re-fitted under alternative samples / specifications)
# ----------------------------------------------------------------------------
print("\n=== SENSITIVITY ===")
SPECS = {
    "S0_primary": dict(data=d),
    "S1_excl_anonymous": dict(data=d[~anon]),
    "S2_drop_top5_outcome": dict(data=d, drop_top=DROP_TOP),
    "S3_excl_bundle_dept": dict(data=d[d["department_grp"] != BUNDLE_DEPT]),
    "S4_excl_cardiac_audit": dict(data=d[~cardiac]),
    "S5_plus_ownership": dict(data=d, covs=PRIMARY + ["ownership_grp"]),
}
VARS_S = PRIMARY + ["ownership_grp"]
flip_rows = []
for kind in LABEL:
    cols, base_params = {}, None
    for sname, sp_ in SPECS.items():
        dd = sp_["data"]
        if len(dd) < 100 or dd["los_pos"].sum() < 30:
            print(f"  {kind} | {sname}: skipped (too few observations)")
            continue
        try:
            r = run(kind, dd, sp_.get("covs", PRIMARY), sp_.get("drop_top", 0))
        except Exception as e:
            print(f"  {kind} | {sname}: failed ({e})")
            continue
        cols[sname] = pd.Series({v: r["pvals"].get(v, np.nan) for v in VARS_S})
        cols[sname]["n"] = r["n"]
        if sname == "S0_primary":
            base_params, base_refs = r["params"], r["refs"]
        elif base_params is not None:
            same = {v for v in base_refs if r["refs"].get(v) == base_refs[v]}
            common = [c for c in base_params.index.intersection(r["params"].index)
                      if c != "Intercept" and var_of(c) in same]
            bad = [c for c in common if np.sign(base_params[c]) != np.sign(r["params"][c])
                   and abs(base_params[c]) > .10 and abs(r["params"][c]) > .10]
            flip_rows.append({"outcome": kind, "spec": sname, "terms_compared": len(common),
                              "sign_flips": len(bad), "flipped_terms": "; ".join(short(b) for b in bad[:10])})
    W = pd.DataFrame(cols)
    W.to_csv(OUT_DIR / f"T6_sensitivity_wald_{kind}.csv")
    print(f"\n--- {LABEL[kind]}: joint Wald/LR p-values ---")
    print(W.map(lambda x: f"{x:.3g}" if pd.notna(x) else "-").to_string())
pd.DataFrame(flip_rows).to_csv(OUT_DIR / "T6b_sign_flips.csv", index=False)
print("\nSign flips vs primary:\n", pd.DataFrame(flip_rows).to_string(index=False))

print("\n--- LOS outlier / estimator checks (primary covariates) ---")
chk = {}
for lab, kind, dt in [("overall LOS raw", "los_overall_raw", 0), ("overall LOS winsorised", "los_overall", 0),
                      ("overall LOS raw, drop top5", "los_overall_raw", DROP_TOP),
                      ("part2 quasi-Poisson", "los_part2", 0), ("part2 Gamma-log", "los_part2_gamma", 0)]:
    r = run(kind, d, PRIMARY, dt)
    chk[lab] = pd.Series({v: r["pvals"].get(v, np.nan) for v in PRIMARY})
T7 = pd.DataFrame(chk)
T7.to_csv(OUT_DIR / "T7_los_outlier_estimator_checks.csv")
print(T7.map(lambda x: f"{x:.3g}" if pd.notna(x) else "-").to_string())

# ----------------------------------------------------------------------------
# 6. ADJUSTED (MODEL-STANDARDISED) MEANS - only variables with full overlap
#    (setting variables have 40-50% empty cells between them -> not standardised)
# ----------------------------------------------------------------------------
rows = []
for kind in LABEL:
    f = prim[kind]
    data = f["data"]
    for v in ADJ_VARS:
        for lvl in sorted(data[v].unique()):
            tmp = data.copy()
            tmp[v] = lvl
            if kind == "los_part1":
                Xn = np.asarray(dmatrix(f["design_info"], tmp, return_type="dataframe"))
                pred = expit(Xn @ f["params"].values)
            else:
                pred = np.asarray(f["res"].predict(tmp))
            rows.append({"outcome": kind, "variable": v, "level": lvl,
                         "n_in_level": int((data[v] == lvl).sum()),
                         "adjusted_mean_or_prob": float(np.mean(pred))})
pd.DataFrame(rows).to_csv(OUT_DIR / "T8_adjusted_means_overlap_vars.csv", index=False)

d.groupby("admission_year")[["los_days", "los_w", "n_claim_lines"]].agg(["count", "mean", "median"]).to_csv(
    OUT_DIR / "T9_by_admission_year.csv")

print(f"\nDone. All outputs in:\n{OUT_DIR}")

Loaded 1,171 rows x 103 cols
  ownership_grp: pooling ['Other (rare)'] -> 'Other (pooled)'
Analytic sample: 1,171 episodes | 1,144 clusters | LOS cap (P99) = 25 days | LOS>=1: 461
Reference levels: {'age_group': '15_44', 'sex': 'Male', 'admission_year': '2024', 'insurance_fund_grp': 'rural', 'diagnosis_type_grp': 'Hospitalization / Inpatient', 'specialty_grp': 'Specialist in cardiovascular diseases', 'department_grp': 'Cardiovascular emergency', 'ownership_grp': 'private'}

=== T1 overall resource utilization ===
                                                    N  Mean     SD  Median   Q1   Q3   P90   P99  Min    Max  % zero
LOS days (raw)                                 1171.0  1.94  12.89     0.0  0.0  1.0   3.0  25.5  0.0  335.0   60.63
LOS days (winsorised at 25)                    1171.0  1.22   3.32     0.0  0.0  1.0   3.0  24.3  0.0   25.0   60.63
Claim lines                                    1171.0  6.62   3.50     5.0  4.0  9.0  12.0  18.0  1.0   20.0    0.00
Service group

In [2]:
# =============================================================================
# OBJECTIVE 2 (v3) - Healthcare expenditure, claims-observed episode cost
# Framework: cost-of-illness / resource-use costing + health production (C = f(RU, X)); Andersen blocks
#
# WHAT CHANGED vs v2 (each item comes from the v2 output):
#  A. Reference-cell collinearity fixed. In v2, non-inpatient episodes carried the label of the D-clinic
#     reference level. When a sub-sample had no D-clinic INPATIENT episodes (S6, S6b, S10), that label
#     became an exact linear combination of the care-type dummies -> rank-deficient design, Wald p = 1,
#     junk specialty ratios (0.14-0.46) and spurious "sign flips". Now:
#       - harmonise_dept(): non-inpatient rows always carry the reference level chosen among INPATIENT rows
#       - ref_for(): the dept_in reference is picked among inpatient rows
#       - run(): design-matrix rank is checked; rank-deficient specifications are reported and skipped
#  B. S8 (month fixed effects) crashed because REF had no entry for 'ym' -> ref_for() now has a fallback.
#  C. Out-of-fold comparison: 2 of 5 folds failed for the "+ ownership" model (n = 673), so it was not
#     comparable. Gamma fit now retries from quasi-Poisson start values, and all candidates are compared
#     on the SAME rows (common rows), with the number of failed rows printed.
#  D. Ratio-stability table only compares terms whose reference level is identical to the primary model.
#  E. Robustness scorecard (how many specifications each block is significant in) + 'Other (rare)' check.
# Primary outcome : Total Cost per episode (rial; analysed in millions of rial)
# Primary model   : GLM, Gamma family, log link -> ratios of mean cost
# =============================================================================
import re
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

warnings.filterwarnings("ignore")

# ----------------------------------------------------------------------------
# 0. SETTINGS
# ----------------------------------------------------------------------------
BASE = Path(r"C:\Users\Shekoofeh\Documents\Health Insurance Data\Cholera")
DEFAULT_INPUT = BASE / "feature engineering v4_FINAL" / "01_analysis_dataset.csv"


def resolve_input():
    """Use a command-line path only if it is a real file; ignore Jupyter/IPython kernel arguments."""
    for a in sys.argv[1:]:
        if a.startswith("-"):
            continue
        p = Path(a)
        if p.suffix.lower() == ".csv" and p.is_file():
            return p
    return DEFAULT_INPUT


INPUT_FILE = resolve_input()
assert INPUT_FILE.is_file(), f"Input file not found: {INPUT_FILE}"
OUT_DIR = INPUT_FILE.parent / "objective2_outputs_v3"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Input : {INPUT_FILE}\nOutput: {OUT_DIR}")


class Tee:
    def __init__(self, path, original):
        self.f = open(path, "w", encoding="utf-8")
        self.o = original

    def write(self, s):
        self.o.write(s)
        self.f.write(s)

    def flush(self):
        self.o.flush()
        self.f.flush()

    def __getattr__(self, name):
        return getattr(self.o, name)


if isinstance(sys.stdout, Tee):
    sys.stdout.f.close()
    sys.stdout = sys.stdout.o
sys.stdout = Tee(OUT_DIR / "objective2_log.txt", sys.stdout)

MIN_LEVEL_N = 30
MIN_STRATUM_N = 20
SPEC_MIN_N = 50
DROP_TOP = 5
WINSOR_Q = 0.99
N_BOOT = 2000
SEED = 42
UNIT = 1e6
N_FOLDS = 5
COOK_THRESHOLD = 1.0
MASS_MIN_N = 12
BUNDLE_DEPT = "Limited surgery - D Clinic"
INP = "Hospitalization / Inpatient"
REF_DEPT = "Limited surgery - D Clinic"
INSURANCE_MERGE = {"Other layers": "Other funds", "Other layers of the referral system": "Other funds",
                   "Other (rare)": "Other funds"}
DEPT_MAP = {"General emergency": "Emergency / triage", "General emergency and triage": "Emergency / triage",
            "Internal General": "Internal (general + emergency)", "Internal emergency": "Internal (general + emergency)",
            "Other (rare)": "Other departments"}
ATTRIBUTION_COL = None   # name of a 0/1 column for cholera as PRINCIPAL diagnosis, if available

DESC_CATS = ["age_group", "sex", "insurance_fund_grp", "diagnosis_type_grp", "specialty_grp", "department_grp",
             "ownership_grp"]
NUMERIC = {"time_m", "n_claim_lines", "gapflag"}
REF_OVERRIDE = {"diagnosis_type_grp": INP, "specialty_c": "General surgery specialist", "dept_in": REF_DEPT,
                "los_cat": "LOS0", "specialty_grp": "Specialist in cardiovascular diseases",
                "department_grp": "Cardiovascular emergency"}

BLOCKS = [
    ("M1_predisposing+time", ["age_group", "sex", "time_m"]),
    ("M2_+enabling", ["age_group", "sex", "time_m", "insurance_fund_grp"]),
    ("M3_+need(care type, specialty)", ["age_group", "sex", "time_m", "insurance_fund_grp",
                                        "diagnosis_type_grp", "specialty_c"]),
    ("M4_+service_setting_PRIMARY", ["age_group", "sex", "time_m", "insurance_fund_grp",
                                     "diagnosis_type_grp", "specialty_c", "dept_in"]),
    ("M5_+resource_use (explanatory)", ["age_group", "sex", "time_m", "insurance_fund_grp",
                                        "diagnosis_type_grp", "specialty_c", "dept_in",
                                        "n_claim_lines", "los_cat"]),
]
PRIMARY = BLOCKS[3][1]
OLD_PRIMARY = ["age_group", "sex", "time_m", "insurance_fund_grp", "diagnosis_type_grp", "specialty_grp",
               "department_grp"]
INP_COVS = ["age_group", "sex", "time_m", "insurance_fund_grp", "specialty_c", "dept_in"]
PERDAY_COVS = ["age_group", "sex", "time_m", "insurance_fund_grp", "dept_in"]
ADJ_VARS = ["age_group", "sex", "insurance_fund_grp"]


def banner(t):
    print("\n" + "=" * 90 + f"\n{t}\n" + "=" * 90)


# ----------------------------------------------------------------------------
# 1. LOAD, RECOVER TOTAL COST, DEFINE SAMPLE
# ----------------------------------------------------------------------------
df = pd.read_csv(INPUT_FILE, low_memory=False)
print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns")


def flag(col):
    if col not in df.columns:
        return pd.Series(False, index=df.index)
    return pd.to_numeric(df[col], errors="coerce").fillna(0).eq(1)


need = ["log_total_cost", "los_days", "n_claim_lines", "analytic_primary", "Cluster_for_SE", "Episode_key"] + DESC_CATS
miss = [c for c in need if c not in df.columns]
assert not miss, f"Missing columns: {miss}"
for c in ["log_total_cost", "los_days", "n_claim_lines", "cost_per_day", "admission_days_since_start"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

raw_col = next((c for c in ["Total Cost", "total_cost", "Total_Cost"] if c in df.columns), None)
if raw_col:
    df["total_cost"] = pd.to_numeric(df[raw_col], errors="coerce")
    print(f"Total cost taken from column '{raw_col}'")
else:
    cands = {"exp": np.exp(df["log_total_cost"]), "expm1": np.expm1(df["log_total_cost"])}
    best = "exp"
    if "cost_per_day" in df.columns:
        z = df["los_days"].eq(0) & df["cost_per_day"].gt(0)
        errs = {k: float(((v[z] - df.loc[z, "cost_per_day"]).abs() / df.loc[z, "cost_per_day"]).median())
                for k, v in cands.items()}
        best = min(errs, key=errs.get)
        print(f"Total cost recovered as {best}(log_total_cost); median rel. error vs cost_per_day at LOS=0: {errs[best]:.2e}")
        if errs[best] > 1e-3:
            print("  WARNING: recovered cost does not match cost_per_day at LOS=0 - verify cost definition!")
    df["total_cost"] = cands[best]

mask = flag("analytic_primary")
if "complete_case_covariates" in df.columns:
    mask &= flag("complete_case_covariates")
n_flag_bad = int(flag("flag_invalid_total_cost").loc[mask].sum())
mask &= ~flag("flag_invalid_total_cost")
d = df.loc[mask].copy()
d = d[np.isfinite(d["total_cost"]) & (d["total_cost"] > 0)]
d = d.dropna(subset=["los_days", "n_claim_lines"] + DESC_CATS)
print(f"Sample: {len(d):,} episodes (invalid-cost flag removed: {n_flag_bad})")

HAVE_YM = "admission_ym" in d.columns
if "admission_days_since_start" in d.columns and d["admission_days_since_start"].notna().all():
    d["time_m"] = d["admission_days_since_start"] / 30.4375
elif HAVE_YM:
    _ym = pd.PeriodIndex(d["admission_ym"].astype(str), freq="M")
    d["time_m"] = (_ym - _ym.min()).map(lambda x: x.n).astype(float)
else:
    raise SystemExit("Need admission_days_since_start or admission_ym to build the time control")
if HAVE_YM:
    d["ym_raw"] = d["admission_ym"].astype(str).str[:7]
    _vcm = d["ym_raw"].value_counts()
    d["ym"] = np.where(d["ym_raw"].map(_vcm) < 30, "small_months_pooled", d["ym_raw"])

for v in DESC_CATS:
    d[v] = d[v].astype(str).str.strip()
d["insurance_fund_grp"] = d["insurance_fund_grp"].replace(INSURANCE_MERGE)
for v in DESC_CATS:
    vc = d[v].value_counts()
    rare = vc[vc < MIN_LEVEL_N].index
    if len(rare):
        print(f"  {v}: pooling {list(rare)} -> 'Other (pooled)'")
        d.loc[d[v].isin(rare), v] = "Other (pooled)"
assert (d["diagnosis_type_grp"] == INP).any(), f"'{INP}' not found in diagnosis_type_grp"

# ---- derived service-setting variables ----
_vc = d["specialty_grp"].value_counts()
d["specialty_c"] = d["specialty_grp"].where(~d["specialty_grp"].isin(_vc[_vc < SPEC_MIN_N].index),
                                            "Other small specialties")
is_inp = d["diagnosis_type_grp"].eq(INP)
d["dept_in"] = REF_DEPT
_dep = d.loc[is_inp, "department_grp"].replace(DEPT_MAP)
_cnt = _dep.value_counts()
d.loc[is_inp, "dept_in"] = _dep.where(~_dep.isin(_cnt[_cnt < MIN_STRATUM_N].index), "Other departments")
d["setting_label"] = np.where(is_inp, d["dept_in"], "n/a (non-inpatient)")
d["los_cat"] = pd.cut(d["los_days"], [-1, 0, 1, 3, 7, 1e9],
                      labels=["LOS0", "LOS1", "LOS2_3", "LOS4_7", "LOS8plus"]).astype(str)
_vc = d["los_cat"].value_counts()
d.loc[d["los_cat"].isin(_vc[_vc < MIN_LEVEL_N].index), "los_cat"] = "LOS8plus"

d["cost_m"] = d["total_cost"] / UNIT
d["cost_w"] = d["cost_m"].clip(upper=d["cost_m"].quantile(WINSOR_Q))
d["log_cost"] = np.log(d["cost_m"])
d["cost_pd"] = np.where(d["los_days"] >= 1, d["cost_m"] / d["los_days"].clip(lower=1), np.nan)
d["cl"] = d["Cluster_for_SE"].fillna(d["Episode_key"]).astype(str)
anon = flag("flag_anonymous_episode").loc[d.index]
cardiac = flag("audit_cardiac_specialty_or_dept").loc[d.index]
d["gapflag"] = flag("flag_financing_gap").loc[d.index].astype(int)
d["gapflag_v3"] = flag("flag_financing_gap_v3_requested").loc[d.index].astype(int)
FIN_OK_COL = next((c for c in ["sens_excl_financing_gap_v3_requested", "sens_excl_financing_gap"] if c in df.columns), None)
fin_ok = flag(FIN_OK_COL).loc[d.index] if FIN_OK_COL else pd.Series(False, index=d.index)
fin_ok_orig = flag("sens_excl_financing_gap").loc[d.index]

_lb = np.floor(np.log(d["cost_m"]) / np.log(1.01)).astype(int)
d["near_mass"] = (_lb.map(_lb.value_counts()) >= MASS_MIN_N).astype(int)

print(f"Clusters: {d['cl'].nunique():,} (share of singleton clusters: "
      f"{(d.groupby('cl').size() == 1).mean():.1%}) | anonymous: {int(anon.sum())}")
print(f"Financing-reconciled flag used for sensitivity: {FIN_OK_COL} (n={int(fin_ok.sum())})")

codes = pd.factorize(d["cl"])[0]

REF = {}
for _v in DESC_CATS + ["specialty_c", "dept_in", "los_cat"]:
    _o = REF_OVERRIDE.get(_v)
    REF[_v] = _o if (_o is not None and (d[_v] == _o).any()) else d[_v].value_counts().idxmax()
print("Reference levels:", REF)
_rc = ((d["diagnosis_type_grp"] == REF["diagnosis_type_grp"]) & (d["specialty_c"] == REF["specialty_c"])
       & (d["dept_in"] == REF["dept_in"]))
print(f"Joint reference cell (inpatient x D-clinic x general surgery): n={int(_rc.sum())}")

# how much of the data sits in the uninformative 'Other (rare)' specialty (limitation, see notes)
_or = d["specialty_grp"].eq("Other (rare)")
print(f"'Other (rare)' specialty: {_or.mean():.1%} of episodes, "
      f"{d.loc[_or, 'cost_m'].sum() / d['cost_m'].sum():.1%} of total expenditure; "
      f"{int(_or.loc[d['cost_m'].nlargest(15).index].sum())} of the 15 most expensive episodes")


def harmonise_dept(data):
    """Non-inpatient rows carry the dept_in reference chosen among INPATIENT rows, so the reference label
    can never become a linear combination of the care-type dummies."""
    data = data.copy()
    inp = data["diagnosis_type_grp"].eq(INP)
    vc = data.loc[inp, "dept_in"].value_counts()
    if len(vc) == 0:
        return data
    ref = REF["dept_in"] if REF["dept_in"] in vc.index else vc.index[0]
    data.loc[~inp, "dept_in"] = ref
    return data


def ref_for(data, v):
    if v == "dept_in" and "diagnosis_type_grp" in data.columns:
        inp = data["diagnosis_type_grp"].eq(INP)
        vc = data.loc[inp, v].value_counts()
        if len(vc):
            return REF[v] if REF[v] in vc.index else vc.index[0]
    r = REF.get(v)
    if r is not None and (data[v] == r).any():
        return r
    return data[v].value_counts().idxmax()


def rhs(data, covs):
    terms = []
    for v in covs:
        if v in NUMERIC:
            terms.append(v)
            continue
        if data[v].nunique() < 2:
            continue
        terms.append(f"C({v}, Treatment(reference='{ref_for(data, v)}'))")
    return " + ".join(terms) or "1"


def var_of(term):
    m = re.search(r"C\((\w+),", str(term))
    return m.group(1) if m else str(term)


def short(t):
    return re.sub(r"C\((\w+), Treatment\(reference='.*?'\)\)\[T\.(.*)\]", r"\1: \2", str(t))


def same_ref(r, base, t):
    v = var_of(t)
    # care-type coefficients = non-inpatient vs inpatient-in-the-reference-department, so they are only
    # comparable when the dept_in reference is unchanged too
    if v == "diagnosis_type_grp" and r["refs"].get("dept_in") != base["refs"].get("dept_in"):
        return False
    return v not in base["refs"] or r["refs"].get(v) == base["refs"][v]


# ----------------------------------------------------------------------------
# 2. DESCRIPTION OF EXPENDITURE AND DATA-QUALITY CHECKS
# ----------------------------------------------------------------------------
banner("[2] DESCRIPTION OF EXPENDITURE")


def desc(s):
    return pd.Series({"N": s.notna().sum(), "Mean": s.mean(), "SD": s.std(), "Median": s.median(),
                      "Q1": s.quantile(.25), "Q3": s.quantile(.75), "P90": s.quantile(.90),
                      "P99": s.quantile(.99), "Min": s.min(), "Max": s.max(),
                      "Skewness": stats.skew(s.dropna()), "Sum": s.sum()})


inp_lt = d[is_inp & (d["los_days"] >= 1)]
T1 = pd.DataFrame({"Total cost (million rial)": desc(d["cost_m"]),
                   "Cost per inpatient day, inpatient & LOS>=1 (million rial)": desc(inp_lt["cost_pd"])}).T
print(T1.round(2).to_string())
T1.to_csv(OUT_DIR / "T1_expenditure_overall.csv")

rng = np.random.default_rng(SEED)
groups = [np.where(codes == k)[0] for k in range(codes.max() + 1)]
cost_arr = d["cost_m"].values
bs = []
for _ in range(N_BOOT):
    idx = np.concatenate([groups[k] for k in rng.integers(0, len(groups), len(groups))])
    s = cost_arr[idx]
    bs.append((s.mean(), np.exp(np.log(s).mean()), np.median(s)))
bs = np.array(bs)
T1b = pd.DataFrame({"estimate": [cost_arr.mean(), np.exp(np.log(cost_arr).mean()), np.median(cost_arr)],
                    "ci_low": np.percentile(bs, 2.5, axis=0), "ci_high": np.percentile(bs, 97.5, axis=0)},
                   index=["Arithmetic mean", "Geometric mean", "Median"])
print("\nBootstrap 95% CI (million rial; clusters ~ episodes):\n", T1b.round(2).to_string())
T1b.to_csv(OUT_DIR / "T1b_bootstrap_ci.csv")

srt = d["cost_m"].sort_values(ascending=False)
conc = pd.Series({f"top {k} episodes": srt.head(k).sum() / srt.sum() * 100 for k in (1, 5, 10, 25, 50)},
                 name="% of total expenditure")
print("\nExpenditure concentration:\n", conc.round(1).to_string())
conc.to_csv(OUT_DIR / "T1c_concentration.csv")

sp = d[["cost_m", "los_days", "n_claim_lines"]].corr(method="spearman")
sp.to_csv(OUT_DIR / "T1e_spearman.csv")
print("\nSpearman(cost, LOS, claim lines):\n", sp.round(3).to_string())

if HAVE_YM:
    d.groupby("admission_ym")["cost_m"].agg(n="size", median="median", mean="mean").to_csv(
        OUT_DIR / "T1g_cost_by_month_nominal.csv")


def by_group(var):
    tot = d["cost_m"].sum()
    rows = []
    for lvl, g in d.groupby(var):
        c = g["cost_m"]
        rows.append({"Variable": var, "Level": lvl, "N": len(g), "% episodes": len(g) / len(d) * 100,
                     "% total expenditure": c.sum() / tot * 100,
                     "Median [IQR]": f"{c.median():.1f} [{c.quantile(.25):.1f}-{c.quantile(.75):.1f}]",
                     "Mean (SD)": f"{c.mean():.1f} ({c.std():.1f})",
                     "Geometric mean": np.exp(np.log(c).mean()), "P90": c.quantile(.9),
                     "CV": c.std() / c.mean()})
    out = pd.DataFrame(rows)
    try:
        out["KW p (exploratory)"] = stats.kruskal(*[g["cost_m"].values for _, g in d.groupby(var)]).pvalue
    except ValueError:
        out["KW p (exploratory)"] = np.nan
    return out


T2 = pd.concat([by_group(v) for v in DESC_CATS + ["specialty_c", "setting_label", "los_cat"]], ignore_index=True)
T2.to_csv(OUT_DIR / "T2_cost_by_determinant.csv", index=False)
T2b = (d.groupby(["diagnosis_type_grp", "setting_label"])["cost_m"]
       .agg(n="size", total="sum", mean="mean", median="median").reset_index())
T2b["pct_expenditure"] = T2b["total"] / d["cost_m"].sum() * 100
T2b.to_csv(OUT_DIR / "T2b_service_setting_cells.csv", index=False)
print("\nExpenditure by care type x setting (share of total expenditure):")
print(T2b.round(1).to_string(index=False))
T2c = d.pivot_table(index="los_cat", columns="diagnosis_type_grp", values="cost_m", aggfunc=["size", "mean"]).round(1)
T2c.to_csv(OUT_DIR / "T2c_cost_by_los_and_type.csv")


# ----------------------------------------------------------------------------
# 3. MODEL ENGINE
# ----------------------------------------------------------------------------
def gamma_family():
    link = getattr(sm.families.links, "Log", None) or sm.families.links.log
    return sm.families.Gamma(link())


def pool_rare(data, covs, m):
    data = data.copy()
    for v in covs:
        if v in NUMERIC:
            continue
        vc = data[v].value_counts()
        rare = vc[vc < m].index
        if len(rare) and (len(vc) - len(rare)) >= 1:
            data.loc[data[v].isin(rare), v] = "Other (pooled)"
    return data


def run(kind, data, covs, drop_top=0, ycol="cost_m", pool_min=0, cov_type="cluster"):
    """kind: gamma | qpois | logols | gamma_perday"""
    if kind == "gamma_perday":
        data, ycol = data[data["los_days"] >= 1], "cost_pd"
    if drop_top:
        data = data.drop(index=data[ycol].nlargest(drop_top).index)
    if "dept_in" in covs:
        data = harmonise_dept(data)
    if pool_min:
        data = pool_rare(data, covs, pool_min)
    data = data.copy()
    cv = [v for v in covs if data[v].nunique() > 1]
    f = rhs(data, covs)
    cl = {"groups": pd.factorize(data["cl"])[0]}
    fit_kw = dict(cov_type="cluster", cov_kwds=cl) if cov_type == "cluster" else dict(cov_type=cov_type)
    if kind == "logols":
        data["_ly"] = np.log(data[ycol])
        model = smf.ols(f"_ly ~ {f}", data=data)
        res = model.fit(**fit_kw)
        dev, disp = res.rsquared, np.nan
    else:
        fam = sm.families.Poisson() if kind == "qpois" else gamma_family()
        model = smf.glm(f"{ycol} ~ {f}", data=data, family=fam)
        try:
            res = model.fit(**fit_kw)
        except Exception:
            # retry from quasi-Poisson start values (IRLS can diverge in small folds / sub-samples)
            p0 = smf.glm(f"{ycol} ~ {f}", data=data, family=sm.families.Poisson()).fit().params
            res = model.fit(start_params=p0, **fit_kw)
        dev, disp = 1 - res.deviance / res.null_deviance, res.pearson_chi2 / res.df_resid
    X = np.asarray(model.exog)
    rank_def = int(X.shape[1] - np.linalg.matrix_rank(X))
    pv = {}
    try:
        for term, row in res.wald_test_terms(scalar=True).table.iterrows():
            if var_of(term) in cv:
                pv[var_of(term)] = float(row.iloc[1])
    except Exception:
        pass
    ci = res.conf_int()
    tab = pd.DataFrame({"term": res.params.index, "ratio": np.exp(res.params.values),
                        "ci_low": np.exp(ci[0].values), "ci_high": np.exp(ci[1].values), "p": res.pvalues.values})
    return dict(kind=kind, n=int(res.nobs), params=res.params, pvals=pv, tab=tab[tab.term != "Intercept"],
                data=data, res=res, npar=len(res.params), dev_expl=dev, disp=disp, ycol=ycol,
                rank_def=rank_def,
                refs={v: (None if v in NUMERIC else ref_for(data, v)) for v in cv})


def clean(tab, label, refs):
    t = tab.copy()
    t["reference"] = t["term"].map(lambda x: refs.get(var_of(x)) or "")
    t["term"] = t["term"].map(short)
    t.insert(0, "model", label)
    return t


fold = np.random.default_rng(SEED).integers(0, N_FOLDS, codes.max() + 1)[codes]


def oof(covs, ycol="cost_m"):
    pred = pd.Series(np.nan, index=d.index)
    for k in range(N_FOLDS):
        tr, te = d[fold != k], d[fold == k]
        try:
            r = run("gamma", tr, covs, ycol=ycol)
            pred.loc[te.index] = np.asarray(r["res"].predict(harmonise_dept(te) if "dept_in" in covs else te))
        except Exception as e:
            print(f"   fold {k} failed: {str(e)[:70]}")
    return pred


def oof_metrics(pred, rows=None):
    m = pred.notna() & (pred > 0)
    if rows is not None:
        m &= rows
    y, p = d.loc[m, "cost_m"], pred[m]
    return {"n": int(m.sum()), "RMSE": np.sqrt(((y - p) ** 2).mean()), "MAE": (y - p).abs().mean(),
            "mean_obs/mean_pred": y.mean() / p.mean(),
            "mean_gamma_deviance": np.mean(2 * ((y - p) / p - np.log(y / p)))}


# ----------------------------------------------------------------------------
# 4. FAMILY / LINK / CALIBRATION CHECKS ON THE PRIMARY MODEL
# ----------------------------------------------------------------------------
banner("[4] MODEL-FIT CHECKS (primary, Gamma-log GLM)")
g0 = run("gamma", d, PRIMARY)
assert g0["rank_def"] == 0, "Primary design matrix is rank-deficient - check reference levels / pooling"
fit_ = g0["res"].fittedvalues
r2 = (d.loc[g0["data"].index, "cost_m"] - fit_) ** 2
park = sm.OLS(np.log(np.maximum(r2, 1e-12)), sm.add_constant(np.log(fit_))).fit(cov_type="HC1")
lam, lam_se = float(park.params.iloc[1]), float(park.bse.iloc[1])
suggest = ("Gaussian (~0)" if abs(lam) < .5 else "Poisson (~1)" if abs(lam - 1) < .5 else
           "Gamma (~2)" if abs(lam - 2) < .5 else "Inverse Gaussian (~3)" if abs(lam - 3) < .5 else "unclear")
d2 = g0["data"].copy()
d2["_xb2"] = np.log(fit_) ** 2
link_p = float(smf.glm(f"cost_m ~ {rhs(d2, PRIMARY)} + _xb2", data=d2, family=gamma_family()).fit(
    cov_type="cluster", cov_kwds={"groups": pd.factorize(d2["cl"])[0]}).pvalues["_xb2"])
print(f"Modified Park test: lambda = {lam:.2f} (SE {lam_se:.2f}) -> closest to {suggest}")
print(f"Pregibon-type link test p = {link_p:.3g} | Pearson dispersion = {g0['disp']:.2f} | "
      f"deviance explained = {g0['dev_expl']:.3f}")
print(f"Primary model: n={g0['n']:,}, params={g0['npar']}, n/param={g0['n'] / g0['npar']:.1f}, "
      f"rank deficiency={g0['rank_def']}")
pd.DataFrame({"park_lambda": [lam], "park_se": [lam_se], "closest_family": [suggest],
              "link_test_p": [link_p], "pearson_dispersion": [g0["disp"]]}).to_csv(
    OUT_DIR / "T3b_family_link_tests.csv", index=False)

print("\nOut-of-fold comparison of mean-function specifications (lower deviance = better):")
CANDS = {"primary (nested service setting)": PRIMARY,
         "primary without specialty_c": [c for c in PRIMARY if c != "specialty_c"],
         "primary + ownership": PRIMARY + ["ownership_grp"],
         "OLD primary (diagnosis + specialty_grp + department_grp)": OLD_PRIMARY}
oofp = {}
for nm, covs in CANDS.items():
    oofp[nm] = oof(covs)
    print(f"   {nm}: predictions for {int(oofp[nm].notna().sum())} of {len(d)} episodes")
OWN = "primary + ownership"
main_c = {k: v for k, v in oofp.items() if k != OWN}
common = pd.concat(main_c, axis=1).notna().all(axis=1)
print(f"Main comparison on the {int(common.sum())} episodes predicted by all non-ownership candidates.")
T3c = pd.DataFrame({nm: oof_metrics(p, common) for nm, p in main_c.items()}).T
own_rows = oofp[OWN].notna()
T3c.loc["-- same-rows check (folds where +ownership converged) --"] = np.nan
T3c.loc["primary, same rows as +ownership"] = pd.Series(oof_metrics(oofp["primary (nested service setting)"], own_rows))
T3c.loc["primary + ownership"] = pd.Series(oof_metrics(oofp[OWN], own_rows))
print(T3c.round(3).to_string())
print("Ownership is judged on the same-rows check only (its folds fail because of the 16-episode pooled level).")
T3c.to_csv(OUT_DIR / "T3c_oof_specification_comparison.csv")

pg = oofp["primary (nested service setting)"]
cal = pd.DataFrame({"pred": pg, "obs": d["cost_m"]}).dropna()
cal["decile"] = pd.qcut(cal["pred"], 10, duplicates="drop")
cal = cal.groupby("decile").agg(n=("obs", "size"), observed_mean=("obs", "mean"), predicted_mean=("pred", "mean"))
cal["obs/pred"] = cal["observed_mean"] / cal["predicted_mean"]
print("\nOut-of-fold calibration by decile of predicted cost:\n", cal.round(2).to_string())
cal.to_csv(OUT_DIR / "T3_oof_calibration_deciles.csv")
rows = []
for v in ["diagnosis_type_grp", "dept_in", "specialty_c", "insurance_fund_grp", "age_group", "ownership_grp", "los_cat"]:
    tmp = pd.DataFrame({"g": d[v], "obs": d["cost_m"], "pred": pg}).dropna()
    gg = tmp.groupby("g").agg(n=("obs", "size"), obs=("obs", "mean"), pred=("pred", "mean"))
    gg["obs/pred"] = gg["obs"] / gg["pred"]
    gg.insert(0, "variable", v)
    rows.append(gg.reset_index())
CG = pd.concat(rows)
CG.to_csv(OUT_DIR / "T3d_oof_calibration_by_group.csv", index=False)
bad = CG[(CG.n >= 30) & ((CG["obs/pred"] < .8) | (CG["obs/pred"] > 1.25))]
print("\nGroups (n>=30) mis-predicted by >20% out-of-fold (LOS is NOT in M4, so LOS bins are expected to miss):")
print(bad.round(2).to_string(index=False) if len(bad) else "  none")

# ----------------------------------------------------------------------------
# 5. INFLUENTIAL EPISODES (verify in raw data) AND FINANCING-RECONCILIATION VALIDITY
# ----------------------------------------------------------------------------
banner("[5] INFLUENTIAL EPISODES AND COST VALIDITY")
try:
    infl = g0["res"].get_influence()
    cd = infl.cooks_distance
    cook = pd.Series(np.asarray(cd[0] if isinstance(cd, tuple) else cd), index=g0["data"].index)
except Exception as e:
    print("  GLM influence unavailable:", e)
    cook = None
show = [c for c in ["Episode_key", "cost_m", "los_days", "n_claim_lines", "diagnosis_type_grp", "department_grp",
                    "specialty_grp", "ownership_grp", "admission_ym"] if c in d.columns]
ext = d.sort_values("cost_m", ascending=False)[show].head(15).copy()
if cook is not None:
    ext["cook_d"] = cook.reindex(ext.index)
    top_c = d.assign(cook_d=cook).nlargest(10, "cook_d")[show + ["cook_d"]]
    infl_idx = cook[cook > COOK_THRESHOLD].index
    print(f"Episodes with Cook's D > {COOK_THRESHOLD}: {list(d.loc[infl_idx, 'Episode_key'])}")
    print("Top 10 by Cook's D (VERIFY IN RAW DATA - e.g. ED-observation episodes costing >>10x their group):")
    print(top_c.round(2).to_string(index=False))
    top_c.to_csv(OUT_DIR / "T1d2_influential_by_cook_VERIFY_IN_RAW_DATA.csv", index=False)
else:
    infl_idx = d.index[:0]
ext.to_csv(OUT_DIR / "T1d_extreme_cost_episodes_VERIFY_IN_RAW_DATA.csv", index=False)

print("\nCost by financing flags:")
rows = []
for c in ["flag_financing_gap", "flag_financing_gap_v3_requested"]:
    if c in df.columns:
        f = flag(c).loc[d.index]
        for val in (False, True):
            g = d[f == val]
            rows.append({"flag": c, "value": int(val), "n": len(g), "median_cost_m": g["cost_m"].median(),
                         "mean_cost_m": g["cost_m"].mean()})
if rows:
    T1f = pd.DataFrame(rows)
    print(T1f.round(2).to_string(index=False))
    T1f.to_csv(OUT_DIR / "T1f_cost_by_financing_flags.csv", index=False)
for c in ["financing_gap_pct", "financing_gap_v3_pct", "unreconciled_share", "insurer_deduction_share",
          "org_deduction_share"]:
    if c in df.columns:
        print(f"\n{c}:\n", pd.to_numeric(df.loc[d.index, c], errors="coerce").describe().round(3).to_string())
for lab, col in [("flag_financing_gap", "gapflag"), ("flag_financing_gap_v3_requested", "gapflag_v3")]:
    if d[col].nunique() > 1:
        NUMERIC.add(col)
        r = run("gamma", d, PRIMARY + [col])
        row = r["tab"][r["tab"].term == col].iloc[0]
        print(f"Adjusted ratio of mean cost, {lab}=1 vs 0: {row.ratio:.2f} (95% CI {row.ci_low:.2f}-{row.ci_high:.2f}), "
              f"p={row.p:.3g}  (financing flags are post-service data-quality markers, not causes of cost)")

# ----------------------------------------------------------------------------
# 6. ANDERSEN SEQUENTIAL BLOCKS + PRIMARY TABLES
# ----------------------------------------------------------------------------
banner("[6] ANDERSEN SEQUENTIAL MODELS (Gamma-log, ratio of mean cost)")
allvars = PRIMARY + ["n_claim_lines", "los_cat"]
block_rows, prim = [], {}
for bname, covs in BLOCKS:
    r = run("gamma", d, covs)
    block_rows.append({"model": bname, "n": r["n"], "n_params": r["npar"], "deviance_explained": r["dev_expl"],
                       "rank_deficiency": r["rank_def"],
                       **{f"p_{v}": r["pvals"].get(v, np.nan) for v in allvars}})
    prim[bname] = r
T4 = pd.DataFrame(block_rows)
T4.to_csv(OUT_DIR / "T4_andersen_blocks_joint_tests.csv", index=False)
print(T4.round(4).to_string())
for bname in (BLOCKS[3][0], BLOCKS[4][0]):
    r = prim[bname]
    clean(r["tab"], f"Gamma-log: {bname}", r["refs"]).to_csv(OUT_DIR / f"T5_ratios_{bname.split('_')[0]}.csv", index=False)
print("\nM5 is an accounting/explanatory decomposition (cost is literally the sum of claim lines): "
      "do NOT interpret its department/setting coefficients as determinants.")

# ----------------------------------------------------------------------------
# 7. SENSITIVITY (primary model, alternative samples / specifications)
# ----------------------------------------------------------------------------
banner("[7] SENSITIVITY")
SPECS = {
    "S0_primary": dict(data=d),
    "S1_excl_anonymous": dict(data=d[~anon]),
    "S2_drop_top5_cost": dict(data=d, drop_top=DROP_TOP),
    "S3_drop_influential_cook": dict(data=d.drop(index=infl_idx)),
    "S4_excl_repeated_cost_values": dict(data=d[d["near_mass"] == 0]),
    "S5_plus_ownership": dict(data=d, covs=PRIMARY + ["ownership_grp"]),
    "S6_financing_reconciled": dict(data=d[fin_ok], pool_min=MIN_STRATUM_N),
    "S6b_original_gap_flag_complete": dict(data=d[fin_ok_orig], pool_min=MIN_STRATUM_N),
    "S7_cost_winsorised_P99": dict(data=d, ycol="cost_w"),
    "S9_excl_cardiac_audit": dict(data=d[~cardiac]),
    "S10_excl_Dclinic": dict(data=d[d["department_grp"] != BUNDLE_DEPT], pool_min=MIN_STRATUM_N),
    "S11_plus_gap_flag": dict(data=d, covs=PRIMARY + ["gapflag"]),
    "S12_inpatient_only": dict(data=d[is_inp], covs=INP_COVS, pool_min=MIN_STRATUM_N),
}
if HAVE_YM:
    SPECS["S8_month_FE"] = dict(data=d, covs=[c for c in PRIMARY if c != "time_m"] + ["ym"])
if ATTRIBUTION_COL and ATTRIBUTION_COL in df.columns:
    SPECS["S13_cholera_principal_dx_only"] = dict(data=d[flag(ATTRIBUTION_COL).loc[d.index]])
VARS_S = PRIMARY + ["ownership_grp", "ym", "gapflag"]
cols_p, cols_r, flips, base, skipped = {}, {}, [], None, []
for sname, sp_ in SPECS.items():
    dd = sp_["data"]
    if len(dd) < 100:
        print(f"  {sname}: skipped (n={len(dd)})")
        continue
    try:
        r = run("gamma", dd, sp_.get("covs", PRIMARY), sp_.get("drop_top", 0), sp_.get("ycol", "cost_m"),
                sp_.get("pool_min", 0))
    except Exception as e:
        print(f"  {sname}: failed ({e})")
        skipped.append(sname)
        continue
    if r["rank_def"] > 0:
        print(f"  {sname}: SKIPPED - design matrix rank-deficient by {r['rank_def']} column(s) (n={r['n']}); "
              f"cells too sparse for this parametrisation")
        skipped.append(sname)
        continue
    cols_p[sname] = pd.Series({**{v: r["pvals"].get(v, np.nan) for v in VARS_S}, "n": r["n"]})
    pr = np.exp(r["params"].drop("Intercept"))
    if sname == "S0_primary":
        base = r
    elif base is not None:
        pr = pr[[same_ref(r, base, t) for t in pr.index]]      # compare only terms with the SAME reference level
        same = {v for v in base["refs"] if r["refs"].get(v, "__x") == base["refs"][v]}
        if r["refs"].get("dept_in") != base["refs"].get("dept_in"):
            same.discard("diagnosis_type_grp")
        print(f"  {sname}: dept_in reference = {r['refs'].get('dept_in')}")
        common_t = [c for c in base["params"].index.intersection(r["params"].index)
                    if c != "Intercept" and var_of(c) in same]
        badt = [c for c in common_t if np.sign(base["params"][c]) != np.sign(r["params"][c])
                and abs(base["params"][c]) > .10 and abs(r["params"][c]) > .10]
        flips.append({"spec": sname, "terms_compared": len(common_t), "sign_flips": len(badt),
                      "flipped_terms": "; ".join(short(b) for b in badt[:10])})
    cols_r[sname] = pr
T6 = pd.DataFrame(cols_p)
T6.to_csv(OUT_DIR / "T6_sensitivity_wald.csv")
print(T6.map(lambda x: f"{x:.3g}" if pd.notna(x) else "-").to_string())
if skipped:
    print(f"\nSpecifications not estimable/skipped: {skipped}")
pd.DataFrame(flips).to_csv(OUT_DIR / "T6b_sign_flips.csv", index=False)
print("\nSign flips vs primary (same-reference terms only):\n", pd.DataFrame(flips).to_string(index=False))
RR = pd.DataFrame(cols_r)
RR.index = [short(i) for i in RR.index]
RR.to_csv(OUT_DIR / "T6c_ratio_stability.csv")
ld = np.abs(np.log(RR.drop(columns="S0_primary")).sub(np.log(RR["S0_primary"]), axis=0)).max(axis=1)
stab = pd.DataFrame({"max_abs_logdiff": ld, "primary_ratio": RR["S0_primary"], "min_ratio": RR.min(axis=1),
                     "max_ratio": RR.max(axis=1)}).sort_values("max_abs_logdiff", ascending=False)
print("\nTerms whose ratio moves most across specifications (0.25 ~ 28% change; same-reference comparisons only):")
print(stab.head(12).round(2).to_string())
print(f"Terms moving >0.25: {int((ld > .25).sum())} of {int(ld.notna().sum())}  (v2 value, inflated by degenerate specs: 15 of 29)")
stab.to_csv(OUT_DIR / "T6d_ratio_stability_ranked.csv")

sc = T6.drop(index=[i for i in ["n", "ownership_grp", "ym", "gapflag"] if i in T6.index])
SCORE = pd.DataFrame({"specs_estimated": sc.notna().sum(axis=1), "specs_p<0.05": (sc < 0.05).sum(axis=1),
                      "median_p": sc.median(axis=1), "max_p": sc.max(axis=1)})
print("\nRobustness scorecard (joint Wald tests across sensitivity specifications):\n", SCORE.round(4).to_string())
SCORE.to_csv(OUT_DIR / "T6e_robustness_scorecard.csv")

# ----------------------------------------------------------------------------
# 8. ESTIMATOR CHECKS + INPATIENT-ONLY MODEL + SECONDARY OUTCOME
# ----------------------------------------------------------------------------
banner("[8] ESTIMATOR CHECKS, INPATIENT-ONLY MODEL, COST PER INPATIENT DAY")
chk, coefs = {}, {}
for lab, kind in [("Gamma-log (primary)", "gamma"), ("quasi-Poisson-log", "qpois"), ("log-OLS", "logols")]:
    r = run(kind, d, PRIMARY)
    chk[lab] = pd.Series({v: r["pvals"].get(v, np.nan) for v in PRIMARY})
    coefs[lab] = r["params"]
T7 = pd.DataFrame(chk)
T7.to_csv(OUT_DIR / "T7_estimator_checks_wald.csv")
print(T7.map(lambda x: f"{x:.3g}" if pd.notna(x) else "-").to_string())
ref_c = coefs["Gamma-log (primary)"].drop("Intercept")
for lab in ("quasi-Poisson-log", "log-OLS"):
    c = coefs[lab].drop("Intercept").reindex(ref_c.index)
    big = [short(i) for i in ref_c.index if np.sign(ref_c[i]) != np.sign(c[i]) and abs(ref_c[i]) > .1 and abs(c[i]) > .1]
    print(f"  Sign flips Gamma vs {lab}: {len(big)}/{len(ref_c)}" + (f" -> {big[:8]}" if big else ""))

ri = run("gamma", d[is_inp], INP_COVS, pool_min=MIN_STRATUM_N)
print(f"\nINPATIENT-ONLY (co-primary robustness): n={ri['n']}, params={ri['npar']}, "
      f"n/param={ri['n'] / ri['npar']:.1f}, deviance explained={ri['dev_expl']:.3f}, rank deficiency={ri['rank_def']}")
print("Joint Wald p:", {k: round(v, 4) for k, v in ri["pvals"].items()})
clean(ri["tab"], "Gamma-log: inpatient episodes only", ri["refs"]).to_csv(OUT_DIR / "T10_inpatient_only_ratios.csv", index=False)
print(clean(ri["tab"], "inpatient-only", ri["refs"]).drop(columns="model").round(3).to_string(index=False))
try:
    fi = ri["res"].fittedvalues
    pi = sm.OLS(np.log(np.maximum((ri["data"]["cost_m"] - fi) ** 2, 1e-12)), sm.add_constant(np.log(fi))).fit(cov_type="HC1")
    print(f"Park lambda (inpatient-only): {pi.params.iloc[1]:.2f} (SE {pi.bse.iloc[1]:.2f}); "
          f"2 = Gamma, 3 = inverse Gaussian (point estimates stay consistent under the log link; "
          f"SEs are cluster-robust and quasi-Poisson agrees)")
except Exception as e:
    print("  Park (inpatient-only) failed:", e)

rpd = run("gamma_perday", d[is_inp], PERDAY_COVS, pool_min=MIN_STRATUM_N)
clean(rpd["tab"], "SECONDARY: cost per inpatient day (inpatient, LOS>=1), Gamma-log", rpd["refs"]).to_csv(
    OUT_DIR / "T8_secondary_cost_per_day.csv", index=False)
print(f"\nSecondary (cost per inpatient day): n={rpd['n']}, params={rpd['npar']}, n/param={rpd['n'] / rpd['npar']:.1f}")
print("Department levels in this sub-sample (a label 'Other (pooled)' merges every level with <"
      f"{MIN_STRATUM_N} episodes):")
print(rpd["data"]["dept_in"].value_counts().to_string())
print(pd.Series({v: rpd["pvals"].get(v, np.nan) for v in PERDAY_COVS}).map(lambda x: f"{x:.3g}").to_string())
print("Cost per day is mechanically tied to LOS and to fixed per-admission costs: report as secondary/descriptive.")

# ----------------------------------------------------------------------------
# 9. ADJUSTED MEAN COST (g-computation) - only variables with full overlap
# ----------------------------------------------------------------------------
rows = []
for lab, r in [("primary_all_episodes", prim[BLOCKS[3][0]]), ("inpatient_only", ri)]:
    res, dat = r["res"], r["data"]
    for v in ADJ_VARS:
        for lvl in sorted(dat[v].unique()):
            tmp = dat.copy()
            tmp[v] = lvl
            rows.append({"model": lab, "variable": v, "level": lvl, "n_in_level": int((dat[v] == lvl).sum()),
                         "adjusted_mean_cost_m": float(np.mean(np.asarray(res.predict(tmp)))),
                         "crude_mean_cost_m": float(dat.loc[dat[v] == lvl, "cost_m"].mean())})
T9 = pd.DataFrame(rows)
T9.to_csv(OUT_DIR / "T9_adjusted_mean_cost_overlap_vars.csv", index=False)
print("\nAdjusted vs crude mean cost (million rial):")
print(T9.round(1).to_string(index=False))

banner("DONE")
print(f"All outputs (tables + objective2_log.txt) in:\n{OUT_DIR}")

Input : C:\Users\Shekoofeh\Documents\Health Insurance Data\Cholera\feature engineering v4_FINAL\01_analysis_dataset.csv
Output: C:\Users\Shekoofeh\Documents\Health Insurance Data\Cholera\feature engineering v4_FINAL\objective2_outputs_v3
Loaded 1,171 rows x 103 columns
Total cost recovered as exp(log_total_cost); median rel. error vs cost_per_day at LOS=0: 7.31e-16
Sample: 1,171 episodes (invalid-cost flag removed: 0)
  ownership_grp: pooling ['Other (rare)'] -> 'Other (pooled)'
Clusters: 1,144 (share of singleton clusters: 98.8%) | anonymous: 67
Financing-reconciled flag used for sensitivity: sens_excl_financing_gap_v3_requested (n=650)
Reference levels: {'age_group': '15_44', 'sex': 'Male', 'insurance_fund_grp': 'rural', 'diagnosis_type_grp': 'Hospitalization / Inpatient', 'specialty_grp': 'Specialist in cardiovascular diseases', 'department_grp': 'Cardiovascular emergency', 'ownership_grp': 'private', 'specialty_c': 'General surgery specialist', 'dept_in': 'Limited surgery - D Clini

In [28]:
# =============================================================================
# OBJECTIVE 3 - DIAGNOSTICS (run BEFORE fixing objective3_v1.py)
# Tests: (1) which reconciliation definition reproduces the earlier diagnostics (521 flagged),
#        (2) what the v0 gap really contains, (3) how discordant episodes differ,
#        (4) cell sizes / pooling heterogeneity, (5) out-of-fold competition of cell structures,
#        (6) specialty nesting, (7) fund robustness across samples, (8) tariff/mass points per cell,
#        (9) binary-threshold separation grid, (10) worst out-of-fold residuals, (11) link choice,
#        (12) clusters, age labels, time coverage.
# Output: objective3_diagnostics/ (csv) + objective3_diagnostics_log.txt  -> send me the log.
# =============================================================================
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm

warnings.filterwarnings("ignore")
BASE = Path(r"C:\Users\Shekoofeh\Documents\Health Insurance Data\Cholera")
DEFAULT_INPUT = BASE / "feature engineering v4_FINAL" / "01_analysis_dataset.csv"


def resolve_input():
    for a in sys.argv[1:]:
        if not a.startswith("-") and Path(a).suffix.lower() == ".csv" and Path(a).is_file():
            return Path(a)
    return DEFAULT_INPUT


INPUT_FILE = resolve_input()
OUT = INPUT_FILE.parent / "objective3_diagnostics"
OUT.mkdir(parents=True, exist_ok=True)


class Tee:
    def __init__(self, path, original):
        self.f, self.o = open(path, "w", encoding="utf-8"), original

    def write(self, s):
        self.o.write(s)
        self.f.write(s)

    def flush(self):
        self.o.flush()
        self.f.flush()

    def __getattr__(self, n):
        return getattr(self.o, n)


if isinstance(sys.stdout, Tee):
    sys.stdout.f.close()
    sys.stdout = sys.stdout.o
sys.stdout = Tee(OUT / "objective3_diagnostics_log.txt", sys.stdout)
pd.set_option("display.width", 250, "display.max_columns", 40, "display.max_colwidth", 70)

# flagged-episode counts per setting from the EARLIER diagnostics (F3, gapflag=1)
DIAG_F3 = {"Emergency Department Observation | n/a (non-inpatient)": 4, "Global | n/a (non-inpatient)": 5,
           "Hospitalization / Inpatient | Adult general surgery": 9,
           "Hospitalization / Inpatient | Emergency / triage": 49,
           "Hospitalization / Inpatient | General surgery for adults and children": 1,
           "Hospitalization / Inpatient | Internal (general + emergency)": 48,
           "Hospitalization / Inpatient | Limited surgery - D Clinic": 353,
           "Hospitalization / Inpatient | Other departments": 50,
           "Temporary Hospitalization | n/a (non-inpatient)": 2}
INP, SEED, N_FOLDS = "Hospitalization / Inpatient", 42, 5
INSURANCE_MERGE = {"Other layers": "Other funds", "Other layers of the referral system": "Other funds",
                   "Other (rare)": "Other funds"}
DEPT_MAP = {"General emergency": "Emergency / triage", "General emergency and triage": "Emergency / triage",
            "Internal General": "Internal (general + emergency)", "Internal emergency": "Internal (general + emergency)",
            "Other (rare)": "Other departments"}
CATS = ["age_group", "sex", "insurance_fund_grp", "diagnosis_type_grp", "specialty_grp", "department_grp", "ownership_grp"]


def banner(t):
    print("\n" + "=" * 100 + f"\n{t}\n" + "=" * 100)


df = pd.read_csv(INPUT_FILE, low_memory=False)
num = lambda c: pd.to_numeric(df[c], errors="coerce") if c in df.columns else pd.Series(np.nan, index=df.index)
flag = lambda c: num(c).fillna(0).eq(1) if c in df.columns else pd.Series(False, index=df.index)
for c in ["oop_share", "log_total_cost", "los_days", "unreconciled_share", "insurer_share", "insurer_deduction_share",
          "admission_days_since_start"]:
    if c in df.columns:
        df[c] = num(c)
mask = flag("analytic_primary") & ~flag("flag_invalid_total_cost") & ~flag("flag_oop_out_of_range") & df["oop_share"].between(0, 1)
if "complete_case_covariates" in df.columns:
    mask &= flag("complete_case_covariates")
d = df.loc[mask].dropna(subset=["oop_share"] + CATS).copy()
for v in CATS:
    d[v] = d[v].astype(str).str.strip()
d["insurance_fund_grp"] = d["insurance_fund_grp"].replace(INSURANCE_MERGE)
for v in CATS:
    vc = d[v].value_counts()
    d.loc[d[v].isin(vc[vc < 30].index), v] = "Other (pooled)"
_vc = d["specialty_grp"].value_counts()
d["specialty_c"] = d["specialty_grp"].where(~d["specialty_grp"].isin(_vc[_vc < 50].index), "Other small specialties")
is_inp = d["diagnosis_type_grp"].eq(INP)
dep = d.loc[is_inp, "department_grp"].replace(DEPT_MAP)
cnt = dep.value_counts()
d["setting_label"] = "n/a (non-inpatient)"
d.loc[is_inp, "setting_label"] = dep.where(~dep.isin(cnt[cnt < 20].index), "Other departments")
d["setting_cell"] = d["diagnosis_type_grp"] + " | " + d["setting_label"]
d["time_m"] = d["admission_days_since_start"] / 30.4375 if "admission_days_since_start" in d.columns else 0.0
d["cl"] = d["Cluster_for_SE"].fillna(d["Episode_key"]).astype(str)
d["cost_m"] = np.exp(d["log_total_cost"]) / 1e6
print(f"Loaded {len(d)} episodes from {INPUT_FILE.name}")
cl_codes = pd.factorize(d["cl"])[0]
fold = pd.Series(np.random.default_rng(SEED).integers(0, N_FOLDS, cl_codes.max() + 1)[cl_codes], index=d.index)

# ----------------------------------------------------------------------------
banner("[1] WHICH RECONCILIATION DEFINITION REPRODUCES THE EARLIER DIAGNOSTICS (F3: 521 flagged)?")
defs = {"flag_financing_gap (v0, used as primary in v1)": flag("flag_financing_gap").loc[d.index],
        "flag_financing_gap_v3_requested": flag("flag_financing_gap_v3_requested").loc[d.index]}
for k in range(5):
    c = f"financing_gap_v{k}_pct"
    if c in df.columns:
        for th in (2, 5, 10):
            defs[f"|v{k} gap| > {th}%"] = (num(c).abs() > th).loc[d.index]
rows = []
for nm, fl in defs.items():
    per = fl.groupby(d["setting_cell"]).sum()
    diff = sum(abs(int(per.get(s, 0)) - n) for s, n in DIAG_F3.items())
    rows.append({"definition": nm, "n_flagged": int(fl.sum()), "abs diff vs F3 by setting (0 = identical)": diff})
T1 = pd.DataFrame(rows).sort_values("abs diff vs F3 by setting (0 = identical)")
print("F3 total flagged = 521. Best-matching definitions first:\n", T1.to_string(index=False))
T1.to_csv(OUT / "D1_flag_definitions.csv", index=False)
v0f, v3f = defs["flag_financing_gap (v0, used as primary in v1)"], defs["flag_financing_gap_v3_requested"]
print("\nFlag cross-tab (rows v0 flag, cols v3 flag):\n", pd.crosstab(v0f, v3f).to_string())

banner("[2] WHAT DOES EACH GAP VERSION CONTAIN?  residual_k = 1 - oop - insurer_share - ded_share - gap_k (pct/100)")
rows = []
for k in range(5):
    c = f"financing_gap_v{k}_pct"
    if c in df.columns and "insurer_share" in df.columns:
        g = (num(c) / 100).loc[d.index]
        r = 1 - d["oop_share"] - d["insurer_share"] - d["insurer_deduction_share"].fillna(0) - g
        r0 = 1 - d["oop_share"] - d["insurer_share"] - g
        rows.append({"gap version": f"v{k}", "median gap %": g.median() * 100, "mean gap %": g.mean() * 100,
                     "resid (net of deduction) median %": r.median() * 100, "p5 %": r.quantile(.05) * 100,
                     "p95 %": r.quantile(.95) * 100, "resid (NOT net) median %": r0.median() * 100})
T2 = pd.DataFrame(rows)
print("A version whose residual is ~0 everywhere is a pure remainder of the identity; if 'NOT net' is ~0 the gap\n"
      "already contains the deductions (then flagging on it flags deductions, not missing payments):\n", T2.round(2).to_string(index=False))
T2.to_csv(OUT / "D2_gap_versions.csv", index=False)

banner("[3] DISCORDANT EPISODES (v0-flagged but v3-reconciled) vs concordant, mean oop_share % by setting")
d["grp"] = np.select([~v0f & ~v3f, v0f & ~v3f, ~v0f & v3f, v0f & v3f],
                     ["reconciled both", "v0-flag only", "v3-flag only", "flagged both"], "")
T3 = d.pivot_table(index="setting_cell", columns="grp", values="oop_share", aggfunc=["size", "mean"]).round(3)
print(T3.to_string())
T3.to_csv(OUT / "D3_discordant.csv")

# ----------------------------------------------------------------------------
SAMPLES = {"v3-reconciled (should match F3, n=650)": d[~v3f], "v0-reconciled (v1 primary, n=480)": d[~v0f]}
main_name = list(SAMPLES)[0]

banner("[4] CELL SIZES AND WITHIN-CELL HETEROGENEITY (setting x ownership)")
for nm, s in SAMPLES.items():
    t = s.groupby(["setting_cell", "ownership_grp"]).oop_share.agg(n="size", mean="mean", sd="std").reset_index()
    t["mean"] *= 100
    t["sd"] *= 100
    print(f"\n{nm}: {len(t)} observed cells; cells n<20: {int((t.n < 20).sum())} covering "
          f"{t.loc[t.n < 20, 'n'].sum() / len(s):.1%} of episodes; cells n<10: {int((t.n < 10).sum())}")
    print(t.round(1).to_string(index=False))
    t.to_csv(OUT / f"D4_cells_{nm.split()[0]}.csv", index=False)


# ---------------- model helpers ----------------
def link(name):
    L = sm.families.links
    return {"logit": getattr(L, "Logit", None) or L.logit, "probit": getattr(L, "Probit", None) or L.probit,
            "cloglog": getattr(L, "CLogLog", None) or getattr(L, "cloglog", None)}[name]()


def build_cell(data, m, by="so"):
    if by == "setting":
        lab = data["setting_cell"]
    else:
        lab = data["setting_cell"] + " @ " + data["ownership_grp"]
        lab = lab.where(lab.map(lab.value_counts()) >= m, data["setting_cell"] + " @ sparse")
    return lab.where(lab.map(lab.value_counts()) >= m, "Other (pooled)")


def pool(s, m):
    return s.where(s.map(s.value_counts()) >= m, "Other (pooled)")


def design(data, spec, ref_spec=None):
    """spec: list of (name, series_or_None numeric) pairs given as dict name->Series, cat or num."""
    X = pd.DataFrame({"const": 1.0}, index=data.index)
    cols_of = {}
    for name, s in spec.items():
        if not pd.api.types.is_numeric_dtype(s):
            vc = s.value_counts()
            ref = vc.index[0]
            cs = []
            for l in sorted(vc.index):
                if l != ref:
                    X[f"{name}: {l}"] = (s == l).astype(float)
                    cs.append(f"{name}: {l}")
            cols_of[name] = cs
        else:
            X[name] = s.astype(float)
            cols_of[name] = [name]
    keep, r = [], 0          # drop exactly collinear columns (e.g. setting and ownership nested in each other)
    for c in X.columns:
        r2 = np.linalg.matrix_rank(X[keep + [c]].values)
        if r2 > r:
            keep.append(c)
            r = r2
    dropped = [c for c in X.columns if c not in keep]
    if dropped:
        X = X[keep]
        cols_of = {k: [c for c in v if c in keep] for k, v in cols_of.items()}
    return X, cols_of


def fit(kind, y, X, groups=None):
    kw = dict(cov_type="cluster", cov_kwds={"groups": groups}) if groups is not None else {}
    if kind == "ols":
        return sm.OLS(y, X).fit(**kw)
    fam = sm.families.Binomial(link=link(kind))
    m = sm.GLM(y, X, family=fam)
    try:
        return m.fit(**kw)
    except Exception:
        return m.fit(method="bfgs", maxiter=3000, **kw)


def metrics(y, p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    xl = lambda a, b: np.where(a > 0, a * np.log(a / b), 0.0)
    dev = 2 * (xl(y, p) + xl(1 - y, 1 - p))
    return {"n": len(y), "OOF_R2": 1 - ((y - p) ** 2).sum() / ((y - y.mean()) ** 2).sum(),
            "RMSE_pp": np.sqrt(((y - p) ** 2).mean()) * 100, "binom_dev": dev.mean()}


def oof(data, spec, kind="logit"):
    """spec: dict name -> Series (already on data.index). Categories can differ between folds -> rebuilt per fold,
    unseen test levels get the reference prediction (counted)."""
    pred, unseen = pd.Series(np.nan, index=data.index), 0
    f = fold.loc[data.index]
    for k in range(N_FOLDS):
        tr, te = data.index[f != k], data.index[f == k]
        Xtr, cols = design(data.loc[tr], {n: s.loc[tr] for n, s in spec.items()})
        try:
            res = fit(kind, data.loc[tr, "oop_share"], Xtr)
        except Exception as e:
            print("   fold failed", str(e)[:60])
            continue
        Xte = pd.DataFrame({"const": 1.0}, index=te)
        for c in Xtr.columns[1:]:
            n, _, l = c.partition(": ")
            Xte[c] = (spec[n].loc[te] == l).astype(float) if l else spec[n].loc[te].astype(float)
        for n, s in spec.items():
            if not pd.api.types.is_numeric_dtype(s):
                known = set(s.loc[tr].unique())
                unseen += int((~s.loc[te].isin(known)).sum())
        pred.loc[te] = np.asarray(res.predict(Xte[Xtr.columns]))
    return pred, unseen


base_cov = lambda s: {"age_group": s["age_group"], "sex": s["sex"], "time_m": s["time_m"]}

banner("[5] OUT-OF-FOLD COMPETITION OF CELL STRUCTURES (fractional logit, cluster folds, same rows)")
res5 = {}
for nm, s in SAMPLES.items():
    cands = {}
    c20, c10, c5 = (build_cell(s, m) for m in (20, 10, 5))
    cands["null"] = {}
    cands["setting only (pool<5)"] = {"setting": build_cell(s, 5, "setting")}
    cands["setting + ownership (additive, pool<5)"] = {"setting": build_cell(s, 5, "setting"), "own": pool(s["ownership_grp"], 5)}
    cands["cell_so pool<20 (v1 primary structure)"] = {"cell": c20}
    cands["cell_so pool<10"] = {"cell": c10}
    cands["cell_so pool<5"] = {"cell": c5}
    S5, O5 = {"setting": build_cell(s, 5, "setting"), "own": pool(s["ownership_grp"], 5)}, None
    cands["add + fund"] = {**S5, "fund": s["insurance_fund_grp"]}
    cands["add + fund + age + sex + time"] = {**S5, "fund": s["insurance_fund_grp"], **base_cov(s)}
    cands["add + fund + age + sex + time + specialty"] = {**S5, "fund": s["insurance_fund_grp"], **base_cov(s),
                                                            "spec": pool(s["specialty_c"], 20)}
    cands["add + specialty only"] = {**S5, "spec": pool(s["specialty_c"], 20)}
    cands["cell_so pool<20 + fund + age + sex + time + specialty (v1 PRIMARY)"] = {
        "cell": c20, "fund": s["insurance_fund_grp"], **base_cov(s), "spec": pool(s["specialty_c"], 20)}
    preds = {k: oof(s, v) for k, v in cands.items()}
    ok = pd.concat({k: p[0] for k, p in preds.items()}, axis=1).notna().all(axis=1)
    T5 = pd.DataFrame({k: {**metrics(s.loc[ok, "oop_share"].values, p[0][ok].values), "unseen_levels": p[1]}
                       for k, p in preds.items()}).T
    print(f"\n{nm}: compared on {int(ok.sum())} rows")
    print(T5.round(3).to_string())
    T5.to_csv(OUT / f"D5_oof_structures_{nm.split()[0]}.csv")
    res5[nm] = (s, preds, ok)

banner("[6] IS SPECIALTY JUST A PROXY FOR SETTING?  (v3 sample)")
s = SAMPLES[main_name]
ct = pd.crosstab(s["setting_cell"], pool(s["specialty_c"], 20))
print("Row % of episodes by specialty within setting:\n", (ct.div(ct.sum(axis=1), axis=0) * 100).round(0).to_string())
acc = (ct.max(axis=1).sum()) / ct.values.sum()
print(f"\nShare of episodes whose specialty equals the modal specialty of their setting: {acc:.1%} "
      f"(high -> specialty is nearly nested in setting; its odds ratios cannot be read as independent 'need' effects)")
ct.to_csv(OUT / "D6_specialty_by_setting.csv")

banner("[7] FUND ROBUSTNESS ACROSS SAMPLES (additive setting+ownership structure, fractional logit, cluster SE)")
rows = []
samples7 = {**SAMPLES, "v3 & |v3 gap|<=2%": d[~v3f & (num("financing_gap_v3_pct").loc[d.index].abs() <= 2)],
            "ALL episodes": d}
for nm, s in samples7.items():
    if len(s) < 100:
        continue
    for lab, extra in [("age+sex+time+fund", {}), ("+ setting + ownership", {"setting": build_cell(s, 5, "setting"),
                                                                           "own": pool(s["ownership_grp"], 5)})]:
        spec = {**base_cov(s), "fund": s["insurance_fund_grp"], **extra}
        X, cols = design(s, spec)
        try:
            r = fit("logit", s["oop_share"], X, pd.factorize(s["cl"])[0])
            R = np.zeros((len(cols["fund"]), X.shape[1]))
            for i, c in enumerate(cols["fund"]):
                R[i, list(X.columns).index(c)] = 1
            p = float(r.wald_test(R, scalar=True).pvalue)
            ors = {c.split(": ")[1]: np.exp(r.params[c]) for c in cols["fund"]}
            rows.append({"sample": nm, "model": lab, "n": len(s), "fund joint Wald p": p,
                         **{f"OR {k}": v for k, v in ors.items()}})
        except Exception as e:
            rows.append({"sample": nm, "model": lab, "n": len(s), "fund joint Wald p": f"failed {str(e)[:40]}"})
T7 = pd.DataFrame(rows)
print(T7.round(3).to_string(index=False))
T7.to_csv(OUT / "D7_fund_robustness.csv", index=False)
s = SAMPLES[main_name]
cc = s.groupby("setting_cell").size()
big = cc[cc >= 30].index
t = s[s.setting_cell.isin(big)].pivot_table(index="setting_cell", columns="insurance_fund_grp", values="oop_share",
                                            aggfunc=["size", "mean"]).round(3)
print("\nWithin-setting mean share (v3 sample) by fund - n and mean:\n", t.to_string())

banner("[8] TARIFF / MASS POINTS PER CELL  (v3 sample): share of episodes within +-1pp of the cell mode")
rows = []
s = SAMPLES[main_name]
for (sc, ow), g in s.groupby(["setting_cell", "ownership_grp"]):
    if len(g) < 20:
        continue
    r = (g.oop_share * 200).round() / 200
    vc = r.value_counts()
    mode = vc.index[0]
    rows.append({"cell": f"{sc} @ {ow}", "n": len(g), "mode %": mode * 100, "share at mode": vc.iloc[0] / len(g),
                 "share within +-1pp of mode": ((g.oop_share - mode).abs() <= .01).mean(), "sd pp": g.oop_share.std() * 100,
                 "2nd mode %": (vc.index[1] * 100 if len(vc) > 1 else np.nan)})
T8 = pd.DataFrame(rows)
print(T8.round(2).to_string(index=False))
print("Cells where >50% of episodes sit within 1pp of one value have rule-set (tariff) cost-sharing: covariates cannot "
      "explain within-cell variation there, and standard errors from a smooth model are not meaningful for them.")
T8.to_csv(OUT / "D8_mass_points.csv", index=False)

banner("[9] THRESHOLD / SEPARATION GRID  (v3 sample, cells = setting x ownership with n>=10)")
s = SAMPLES[main_name]
cell = (s["setting_cell"] + " @ " + s["ownership_grp"])
cell = cell[cell.map(cell.value_counts()) >= 10]
rows = []
for th in (5, 10, 20, 30, 40, 50, 60, 70):
    y = (s["oop_share"] >= th / 100).astype(int)
    prev = y.groupby(cell).mean().dropna()
    rows.append({"threshold %": th, "overall prevalence": y.mean(), "cells": len(prev),
                 "cells with prevalence 0": int((prev == 0).sum()), "cells with prevalence 1": int((prev == 1).sum()),
                 "cells informative (5-95%)": int(((prev > .05) & (prev < .95)).sum()),
                 "episodes in informative cells": int(cell.map(prev).between(.05, .95).sum())})
T9 = pd.DataFrame(rows)
print(T9.round(3).to_string(index=False))
T9.to_csv(OUT / "D9_threshold_grid.csv", index=False)

banner("[10] WORST OUT-OF-FOLD RESIDUALS (best additive structure, v3 sample)")
s, preds, ok = res5[main_name]
bestk = "add + fund + age + sex + time + specialty"
p = preds[bestk][0]
w = s.assign(pred=p, err_pp=(s.oop_share - p) * 100).loc[ok]
cols = [c for c in ["Episode_key", "setting_cell", "ownership_grp", "specialty_c", "insurance_fund_grp", "age_group", "oop_share",
                    "pred", "err_pp", "cost_m"] if c in w.columns]
top = w.reindex(w.err_pp.abs().sort_values(ascending=False).index)[cols].head(15)
print(top.round(3).to_string(index=False))
top.to_csv(OUT / "D10_worst_residuals.csv", index=False)
w["dec"] = pd.qcut(w.pred, 10, duplicates="drop")
print("\nCalibration by decile (obs vs pred, pp):\n", (w.groupby("dec").agg(n=("oop_share", "size"), obs=("oop_share", "mean"),
                                                                       pred=("pred", "mean")) * [1, 100, 100]).round(1).to_string())

banner("[11] LINK / ESTIMATOR CHOICE under the additive structure (OOF, v3 sample)")
s = SAMPLES[main_name]
spec = {"setting": build_cell(s, 5, "setting"), "own": pool(s["ownership_grp"], 5), "fund": s["insurance_fund_grp"],
        **base_cov(s), "spec": pool(s["specialty_c"], 20)}
rows = {}
for kind in ("logit", "probit", "cloglog", "ols"):
    try:
        pp, un = oof(s, spec, kind)
        okk = pp.notna()
        rows[kind] = metrics(s.loc[okk, "oop_share"].values, pp[okk].values)
    except Exception as e:
        rows[kind] = {"n": f"failed {str(e)[:40]}"}
T11 = pd.DataFrame(rows).T
print(T11.round(3).to_string())
T11.to_csv(OUT / "D11_link_choice.csv")

banner("[12] CLUSTERS, AGE LABELS, TIME COVERAGE")
for nm, s in SAMPLES.items():
    sz = s.groupby("cl").size()
    print(f"{nm}: {len(sz)} clusters; episodes in clusters >1: {sz[sz > 1].sum() / len(s):.1%}; max cluster size {sz.max()}; "
          f"anonymous-flagged episodes {int(flag('flag_anonymous_episode').loc[s.index].sum())}")
    print("   age_group counts:", s["age_group"].value_counts().to_dict())
print("\nAdmission month (time_m) range by setting, v3 sample:")
print(SAMPLES[main_name].groupby("setting_cell").time_m.agg(["min", "median", "max"]).round(1).to_string())

banner("AUTO-SUMMARY")
b1 = T1.iloc[0]
print(f"1. Best-matching reconciliation definition: {b1['definition']} (abs diff {b1['abs diff vs F3 by setting (0 = identical)']}).")
for nm, (s, preds, ok) in res5.items():
    m = {k: metrics(s.loc[ok, 'oop_share'].values, p[0][ok].values)["OOF_R2"] for k, p in preds.items()}
    print(f"2. {nm}: OOF R2 v1-primary structure = {m['cell_so pool<20 + fund + age + sex + time + specialty (v1 PRIMARY)']:.3f} | "
          f"additive setting+ownership(+covariates) = {m['add + fund + age + sex + time + specialty']:.3f}")
print("3. See [6] specialty nesting, [7] fund p-values by sample, [8] tariff cells, [9] separation grid.")
banner("DONE - send me objective3_diagnostics_log.txt")
print(f"Outputs in {OUT}")

In [31]:
# =============================================================================
# OBJECTIVE 3 (v3) - DIAGNOSTIC SCRIPT  (run BEFORE finalising the v4 code)
# Purpose: test the three things the v3 results cannot settle on their own
#   A. WHAT the 521 unreconciled episodes are (76.6% of private episodes!) -> decides whether the
#      population-level cost-sharing is ~20% (as recorded) or ~45% (scenario/upper bound).
#   B. WHETHER the State-University binary-model findings (sex, fund) are real or threshold/cost artefacts,
#      and why the >=20% model shows separation / EPV 8.4.
#   C. WHETHER the State-University cost gradient / worst residuals point to a missing covariate or a
#      mis-typed setting.
# Output: console + objective3_diagnostics_v3/ (E1..E10 csv files + diagnostics_log.txt)
# Paste the whole log back.
# =============================================================================
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats

warnings.filterwarnings("ignore")

BASE = Path(r"C:\Users\Shekoofeh\Documents\Health Insurance Data\Cholera")
DEFAULT_INPUT = BASE / "feature engineering v4_FINAL" / "01_analysis_dataset.csv"


def resolve_input():
    for a in sys.argv[1:]:
        if a.startswith("-"):
            continue
        p = Path(a)
        if p.suffix.lower() == ".csv" and p.is_file():
            return p
    return DEFAULT_INPUT


INPUT_FILE = resolve_input()
assert INPUT_FILE.is_file(), f"Input file not found: {INPUT_FILE}"
OUT_DIR = INPUT_FILE.parent / "objective3_diagnostics_v3"
OUT_DIR.mkdir(parents=True, exist_ok=True)


class Tee:
    def __init__(self, path, original):
        self.f = open(path, "w", encoding="utf-8")
        self.o = original

    def write(self, s):
        self.o.write(s)
        self.f.write(s)

    def flush(self):
        self.o.flush()
        self.f.flush()

    def __getattr__(self, name):
        return getattr(self.o, name)


if isinstance(sys.stdout, Tee):
    sys.stdout.f.close()
    sys.stdout = sys.stdout.o
sys.stdout = Tee(OUT_DIR / "diagnostics_log.txt", sys.stdout)
pd.set_option("display.width", 250, "display.max_columns", 50, "display.max_colwidth", 70)
print(f"Input : {INPUT_FILE}\nOutput: {OUT_DIR}")

UNIT = 1e6
SU = "State University"
INP = "Hospitalization / Inpatient"
BUNDLE_DEPT = "Limited surgery - D Clinic"
INSURANCE_MERGE = {"Other layers": "Other funds", "Other layers of the referral system": "Other funds",
                   "Other (rare)": "Other funds"}
DEPT_MAP = {"General emergency": "Emergency / triage", "General emergency and triage": "Emergency / triage",
            "Internal General": "Internal (general + emergency)", "Internal emergency": "Internal (general + emergency)",
            "Other (rare)": "Other departments"}
DESC_CATS = ["age_group", "sex", "insurance_fund_grp", "diagnosis_type_grp", "specialty_grp", "department_grp",
             "ownership_grp"]


def banner(t):
    print("\n" + "=" * 100 + f"\n{t}\n" + "=" * 100)


def show(tab, name, nd=2, index=True):
    tab.to_csv(OUT_DIR / f"{name}.csv", index=index)
    print(tab.round(nd).to_string(index=index))


# ----------------------------------------------------------------------------
# 0. SAMPLE (identical to the v3 script)
# ----------------------------------------------------------------------------
df = pd.read_csv(INPUT_FILE, low_memory=False)


def flag(col):
    if col not in df.columns:
        return pd.Series(False, index=df.index)
    return pd.to_numeric(df[col], errors="coerce").fillna(0).eq(1)


for c in ["oop_share", "log_total_cost", "los_days", "n_claim_lines", "admission_days_since_start",
          "financing_gap_v3_pct"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
raw_col = next((c for c in ["Total Cost", "total_cost", "Total_Cost"] if c in df.columns), None)
df["total_cost"] = pd.to_numeric(df[raw_col], errors="coerce") if raw_col else np.exp(df["log_total_cost"])
df["gap_signed"] = df["financing_gap_v3_pct"] / 100
df["gap_share"] = df["gap_signed"].clip(lower=0)

mask = flag("analytic_primary")
if "complete_case_covariates" in df.columns:
    mask &= flag("complete_case_covariates")
mask &= ~flag("flag_invalid_total_cost") & ~flag("flag_oop_out_of_range") & df["oop_share"].between(0, 1)
d = df.loc[mask].copy()
d = d[np.isfinite(d["total_cost"]) & (d["total_cost"] > 0)]
d = d.dropna(subset=["los_days", "n_claim_lines", "oop_share"] + DESC_CATS)
if "admission_days_since_start" in d.columns and d["admission_days_since_start"].notna().all():
    d["time_m"] = d["admission_days_since_start"] / 30.4375
else:
    _ym = pd.PeriodIndex(d["admission_ym"].astype(str).str[:7], freq="M")
    d["time_m"] = (_ym - _ym.min()).map(lambda x: x.n).astype(float)
d["ym"] = d["admission_ym"].astype(str).str[:7] if "admission_ym" in d.columns else "NA"
for v in DESC_CATS:
    d[v] = d[v].astype(str).str.strip()
d["insurance_fund_grp"] = d["insurance_fund_grp"].replace(INSURANCE_MERGE)
for v in DESC_CATS:
    vc = d[v].value_counts()
    d.loc[d[v].isin(vc[vc < 30].index), v] = "Other (pooled)"
_vc = d["specialty_grp"].value_counts()
d["specialty_c"] = d["specialty_grp"].where(~d["specialty_grp"].isin(_vc[_vc < 50].index), "Other small specialties")
is_inp = d["diagnosis_type_grp"].eq(INP)
d["dept_in"] = BUNDLE_DEPT
_dep = d.loc[is_inp, "department_grp"].replace(DEPT_MAP)
_cnt = _dep.value_counts()
d.loc[is_inp, "dept_in"] = _dep.where(~_dep.isin(_cnt[_cnt < 20].index), "Other departments")
d["setting_cell"] = d["diagnosis_type_grp"] + " | " + np.where(is_inp, d["dept_in"], "n/a (non-inpatient)")
d["cost_m"] = d["total_cost"] / UNIT
d["lc_c"] = np.log(d["cost_m"]) - np.log(d["cost_m"]).median()
d["y_upper"] = np.clip(d["oop_share"] + d["gap_share"].fillna(0), 0, 1)
d["cl"] = d["Cluster_for_SE"].fillna(d["Episode_key"]).astype(str)
d["gapflag"] = flag("flag_financing_gap_v3_requested").loc[d.index].astype(int)
d["dclinic"] = d["setting_cell"].str.contains(BUNDLE_DEPT, regex=False)
d["is_priv"] = d["ownership_grp"].eq("private")
dr = d[d["gapflag"] == 0].copy()
print(f"Episodes: {len(d)}; reconciled: {len(dr)}; flagged: {int(d.gapflag.sum())}; D-clinic: {int(d.dclinic.sum())}")

# ----------------------------------------------------------------------------
# E0. COLUMN INVENTORY (so that the financing components can be located)
# ----------------------------------------------------------------------------
banner("E0. COLUMNS THAT DESCRIBE FINANCING COMPONENTS, SITES, DATES")
KW = ["share", "gap", "deduct", "fx", "subsid", "paid", "request", "insurer", "org", "hospital", "provider",
      "center", "centre", "facility", "branch", "ward", "unit", "date", "month", "ym", "claim", "price", "tariff"]
cand_cols = [c for c in df.columns if any(k in c.lower() for k in KW)]
print(cand_cols)
COMP = [c for c in ["oop_share", "insurer_share", "insurer_deduction_share", "org_deduction_share",
                    "unreconciled_share", "gap_signed", "y_upper"] if c in d.columns]
print("Components used below:", COMP)
SITE_COLS = [c for c in df.columns if any(k in c.lower() for k in ["hospital", "provider", "center", "centre",
                                                                    "facility", "branch", "ward"])
             and c in d.columns and d[c].nunique() <= 80]
print("Site-like columns used below:", SITE_COLS)

# ----------------------------------------------------------------------------
# E1. ANATOMY OF THE FLAGGED EPISODES
# ----------------------------------------------------------------------------
banner("E1. SIZE OF THE v3 GAP BY OWNERSHIP (is flagging borderline or gross?)")
d["gap_bin"] = pd.cut(d["gap_signed"], [-np.inf, -0.20, -0.05, 0.05, 0.10, 0.20, 0.50, np.inf],
                      labels=["<-20%", "-20..-5%", "within +-5% (reconciled)", "5-10%", "10-20%", "20-50%", ">50%"])
show(pd.crosstab(d["ownership_grp"], d["gap_bin"], margins=True), "E1a_gap_bins_by_ownership", 0)
print("\nPrivate, non-D-clinic vs D-clinic:")
show(pd.crosstab(np.where(d["dclinic"], "D-clinic", d["ownership_grp"] + " (other)"), d["gap_bin"]), "E1b_gap_bins_dclinic", 0)
print("\nIf nearly all flagged episodes have gaps >20%, the 5% threshold is irrelevant; if many sit at 5-10% "
      "they are borderline and could be re-admitted.")

# ----------------------------------------------------------------------------
# E2. IS THE UNRECONCILED MONEY PLAUSIBLY PATIENT-PAID?  (private episodes)
# ----------------------------------------------------------------------------
banner("E2. PRIVATE EPISODES: recorded components by group (means, % of cost)")
pv = d[d["is_priv"]].copy()
pv["grp"] = np.where(pv["dclinic"], "D-clinic (all flagged)", np.where(pv["gapflag"] == 1, "flagged, other settings",
                                                                       "reconciled"))
agg = pv.groupby("grp")[COMP].mean() * 100
agg.insert(0, "n", pv.groupby("grp").size())
show(agg, "E2a_private_components_by_group")
print("\nImplied patient share = recorded oop + whole gap. Quantiles (%), compared with RECONCILED private oop_share:")
qs = [.05, .25, .50, .75, .95]
qt = {"reconciled: oop_share": pv.loc[pv.grp == "reconciled", "oop_share"].quantile(qs) * 100}
for g in ["flagged, other settings", "D-clinic (all flagged)"]:
    qt[f"{g}: oop+gap"] = pv.loc[pv.grp == g, "y_upper"].quantile(qs) * 100
    qt[f"{g}: oop only"] = pv.loc[pv.grp == g, "oop_share"].quantile(qs) * 100
show(pd.DataFrame(qt, index=[f"P{int(q * 100)}" for q in qs]), "E2b_implied_share_quantiles")
rec_priv = pv.loc[pv.grp == "reconciled", "oop_share"]
lo, hi = rec_priv.quantile(.05), rec_priv.quantile(.95)
rows = []
for g in ["flagged, other settings", "D-clinic (all flagged)"]:
    x = pv.loc[pv.grp == g, "y_upper"]
    ks = stats.ks_2samp(x, rec_priv) if len(x) > 5 and len(rec_priv) > 5 else None
    rows.append({"group": g, "n": len(x), "share of oop+gap inside reconciled-private P5-P95 (%)":
                 x.between(lo, hi).mean() * 100, "KS stat vs reconciled oop": ks.statistic if ks else np.nan,
                 "KS p": ks.pvalue if ks else np.nan})
show(pd.DataFrame(rows), "E2c_balance_billing_plausibility", index=False)
if "insurer_share" in pv.columns:
    print("\nInsurer share (the part the insurer actually paid): if flagged == reconciled, the insurer behaved the same "
          "and the unexplained money is what the patient paid.")
    rows = []
    for g, x in pv.groupby("grp"):
        rows.append({"group": g, "n": len(x), "median insurer share %": x["insurer_share"].median() * 100,
                     "IQR low": x["insurer_share"].quantile(.25) * 100, "IQR high": x["insurer_share"].quantile(.75) * 100})
    show(pd.DataFrame(rows), "E2d_insurer_share_by_group", index=False)
    nd_ = pv[~pv["dclinic"]]
    a, b = nd_.loc[nd_.gapflag == 0, "insurer_share"], nd_.loc[nd_.gapflag == 1, "insurer_share"]
    if len(a) > 5 and len(b) > 5:
        print(f"Mann-Whitney insurer share, reconciled vs flagged (private, non-D-clinic): p = "
              f"{stats.mannwhitneyu(a, b).pvalue:.3g}")
else:
    print("\n(no 'insurer_share' column - look at E0 for the column holding the insurer/organisation payment and add it to COMP)")
print("\nSame comparison inside each private setting (flagged vs reconciled), means in %:")
t = pv[~pv["dclinic"]].groupby(["setting_cell", "gapflag"]).agg(n=("oop_share", "size"), oop=("oop_share", "mean"),
                                                               implied=("y_upper", "mean"), cost_m=("cost_m", "median"))
t[["oop", "implied"]] *= 100
show(t, "E2e_private_by_setting_flag")

# ----------------------------------------------------------------------------
# E3. WHO / WHAT / WHEN GETS FLAGGED  (data-entry or batch effect?)
# ----------------------------------------------------------------------------
banner("E3. DETERMINANTS OF FLAGGING (private episodes; then all episodes)")
for data_, lab in [(d[d["is_priv"] & ~d["dclinic"]].copy(), "private non-D-clinic"), (d.copy(), "all")]:
    data_["cost_q"] = pd.qcut(data_["cost_m"], 4, labels=["Q1", "Q2", "Q3", "Q4"], duplicates="drop").astype(str)
    data_["los_g"] = pd.cut(data_["los_days"], [-1, 0, 1, 3, 7, 1e9], labels=["0", "1", "2-3", "4-7", "8+"]).astype(str)
    data_["lines_q"] = pd.qcut(data_["n_claim_lines"].rank(method="first"), 4, labels=["Q1", "Q2", "Q3", "Q4"]).astype(str)
    out = []
    for var in ["ym", "setting_cell", "specialty_c", "insurance_fund_grp", "age_group", "sex", "cost_q", "los_g",
                "lines_q"] + SITE_COLS:
        ct = pd.crosstab(data_[var].astype(str), data_["gapflag"])
        if ct.shape[0] < 2 or ct.shape[1] < 2:
            continue
        p = stats.chi2_contingency(ct)[1]
        for lv, r in ct.iterrows():
            out.append({"sample": lab, "variable": var, "level": lv, "n": int(r.sum()),
                        "pct_flagged": r.get(1, 0) / r.sum() * 100, "chi2_p": p})
    o = pd.DataFrame(out)
    o.to_csv(OUT_DIR / f"E3_flag_determinants_{lab.replace(' ', '_')}.csv", index=False)
    print(f"\n[{lab}] chi-square p by variable (small p = flagging is structured by that variable):")
    print(o.groupby("variable")["chi2_p"].first().sort_values().round(4).to_string())
    print(o[o.variable.isin(["ym", "cost_q", "los_g"] + SITE_COLS)].round(1).to_string(index=False))
cs = d.groupby("cl").agg(n=("gapflag", "size"), f=("gapflag", "sum"))
multi = cs[cs.n > 1]
p_f = d["gapflag"].mean()
if len(multi):
    obs = ((multi.f == 0) | (multi.f == multi.n)).mean()
    exp = np.mean([(1 - p_f) ** n + p_f ** n for n in multi.n])
    print(f"\nClusters with >1 episode: {len(multi)}; all-or-none flagged: observed {obs:.1%} vs expected under independence "
          f"{exp:.1%} (much higher observed = flagging is a cluster-level/patient-level or site-level phenomenon).")

# ----------------------------------------------------------------------------
# E4. D-CLINIC PROFILE vs OTHER PRIVATE EPISODES
# ----------------------------------------------------------------------------
banner("E4. D-CLINIC (353 episodes, 100% flagged) vs other private episodes")
pv["cost_per_line"] = pv["cost_m"] / pv["n_claim_lines"].clip(lower=1)
prof = pv.groupby("grp").agg(n=("cost_m", "size"), median_cost_m=("cost_m", "median"), p90_cost_m=("cost_m", lambda s: s.quantile(.9)),
                             median_LOS=("los_days", "median"), median_lines=("n_claim_lines", "median"),
                             median_cost_per_line=("cost_per_line", "median"))
show(prof, "E4a_dclinic_profile")
print("\nD-clinic by month:")
print(pv[pv.dclinic]["ym"].value_counts().sort_index().to_string())
print("\nD-clinic by specialty and fund:")
print(pv[pv.dclinic]["specialty_c"].value_counts().to_string())
print(pv[pv.dclinic]["insurance_fund_grp"].value_counts().to_string())

# ----------------------------------------------------------------------------
# E5. TARIFF-RULE TEST AND SCENARIO SENSITIVITY
# ----------------------------------------------------------------------------
banner("E5. IS THE PRIVATE SHARE COST-INDEPENDENT? AND HOW SENSITIVE IS THE POPULATION MEAN?")
rows = []
for g, x in pv.groupby("grp"):
    for col in ["oop_share", "y_upper"]:
        if len(x) > 10:
            rho, p = stats.spearmanr(x["lc_c"], x[col])
            rows.append({"group": g, "variable": col, "n": len(x), "spearman_with_log_cost": rho, "p": p})
show(pd.DataFrame(rows), "E5a_cost_dependence", 3, index=False)
own_mean = dr.groupby("ownership_grp")["oop_share"].mean()
base_other = d["ownership_grp"].map(own_mean).fillna(dr["oop_share"].mean())
is_pf = d["is_priv"] & (d["gapflag"] == 1)
rows = []
for x in [None, 0.30, 0.40, 0.50, 0.60, 0.65, 0.70, float(own_mean.get("private", 0.727))]:
    y = np.where(d["gapflag"] == 1, base_other, d["oop_share"])
    y = np.where(is_pf, d["oop_share"] if x is None else x, y)
    rows.append({"assumed share for flagged PRIVATE episodes": "as recorded" if x is None else f"{x:.1%}",
                 "population mean episode share %": np.mean(y) * 100,
                 "cost-weighted aggregate share %": (y * d["cost_m"]).sum() / d["cost_m"].sum() * 100})
print("Flagged non-private episodes are held at their ownership group's reconciled mean in every row.")
show(pd.DataFrame(rows), "E5b_scenario_sensitivity", index=False)

# ----------------------------------------------------------------------------
# E6. STATE-UNIVERSITY BINARY MODELS: separation, sex/fund artefacts
# ----------------------------------------------------------------------------
banner("E6. STATE-UNIVERSITY HIGH COST-SHARING MODELS")
su = dr[dr["ownership_grp"] == SU].copy()
for v, m in [("age_group", 10), ("insurance_fund_grp", 20)]:
    vc = su[v].value_counts()
    su.loc[su[v].isin(vc[vc < m].index), v] = "Other (pooled)"
vcs = su["setting_cell"].value_counts()
su["setting_b"] = su["setting_cell"].where(su["setting_cell"].map(vcs) >= 30, "Other settings (pooled)")
su["hi10"] = (su["oop_share"] >= .10).astype(int)
su["hi20"] = (su["oop_share"] >= .20).astype(int)
print(f"State-University reconciled episodes: {len(su)}")
for h in ["hi10", "hi20"]:
    print(f"\nEvents by setting_b for {h}:")
    show(pd.crosstab(su["setting_b"], su[h], margins=True), f"E6a_cross_{h}", 0)


def design(data, cats, nums):
    X = pd.DataFrame({"const": 1.0}, index=data.index)
    for v in cats:
        vc = data[v].astype(str).value_counts()
        for lv in vc.index[1:]:
            X[f"{v}: {lv}"] = (data[v].astype(str) == lv).astype(float)
    for v in nums:
        X[v] = data[v].astype(float)
    return X


def fit_glm(data, y, cats, nums, family=None):
    X = design(data, cats, nums)
    g = pd.factorize(data["cl"])[0]
    res = sm.GLM(data[y].astype(float), X, family=family or sm.families.Binomial()).fit(
        cov_type="cluster", cov_kwds={"groups": g})
    p = np.asarray(res.fittedvalues)
    ci = res.conf_int()
    info = dict(n=len(data), events=int(data[y].sum()) if family is None else np.nan, npar=X.shape[1],
                max_abs_coef=float(res.params.abs().max()), nonfinite_se=int((~np.isfinite(res.bse)).sum()),
                fitted_lt_0005_or_gt_0995=int(((p < .005) | (p > .995)).sum()))
    tab = pd.DataFrame({"OR": np.exp(res.params), "lo": np.exp(ci[0]), "hi": np.exp(ci[1]), "p": res.pvalues})
    return info, tab


CATS = ["age_group", "sex", "insurance_fund_grp", "setting_b"]
ed = su[su["setting_cell"].str.startswith("Emergency Department Observation")].copy()
res_rows = []
for h in ["hi10", "hi20"]:
    for lab, data_, cats_, nums_ in [("A v3 model", su, CATS, ["time_m"]),
                                      ("B + log cost", su, CATS, ["time_m", "lc_c"]),
                                      ("C ED-observation only, + log cost", ed, ["age_group", "sex", "insurance_fund_grp"],
                                       ["time_m", "lc_c"])]:
        try:
            info, tab = fit_glm(data_, h, cats_, nums_)
        except Exception as e:
            print(f"  {h} {lab} failed: {str(e)[:80]}")
            continue
        print(f"\n{h} | {lab}: {info}")
        keep = tab[tab.index.str.startswith(("sex", "insurance_fund_grp", "lc_c"))]
        print(keep.round(3).to_string())
        for t_, r_ in keep.iterrows():
            res_rows.append({"outcome": h, "model": lab, "term": t_, **r_.to_dict(), **{f"info_{k}": v for k, v in info.items()}})
pd.DataFrame(res_rows).to_csv(OUT_DIR / "E6b_binary_models_sex_fund.csv", index=False)
print("\nFractional-logit (share, not threshold) inside ED-observation State University, with log cost:")
try:
    info, tab = fit_glm(ed, "oop_share", ["age_group", "sex", "insurance_fund_grp"], ["time_m", "lc_c"])
    print(tab[tab.index.str.startswith(("sex", "insurance_fund_grp", "lc_c"))].round(3).to_string())
except Exception as e:
    print("  failed:", str(e)[:80])
print("\nRead: if sex/fund effects vanish once log cost is added (B, C) they are cost-composition artefacts near the 10% "
      "threshold, consistent with the primary model (sex p=0.25). If they persist, they are candidates for a real finding "
      "but still need the multiplicity caveat (>=20 terms tested).")
print("Crude sex difference in ED-observation (% >=10): ", ed.groupby("sex")["hi10"].mean().mul(100).round(1).to_dict())

# ----------------------------------------------------------------------------
# E7. STATE-UNIVERSITY COST GRADIENT AND HIGH-SHARE EPISODES (worst residuals)
# ----------------------------------------------------------------------------
banner("E7. STATE UNIVERSITY: SHARE BY COST BIN, AND PROFILE OF HIGH-SHARE EPISODES")
su["cost_bin"] = pd.qcut(su["cost_m"], 10, duplicates="drop")
tb = su.groupby("cost_bin").agg(n=("oop_share", "size"), median_cost_m=("cost_m", "median"), mean_share=("oop_share", "mean"),
                                share_ge_50=("oop_share", lambda s: (s >= .5).mean()))
tb[["mean_share", "share_ge_50"]] *= 100
show(tb, "E7a_su_share_by_cost_decile")
print("\nBy setting x cost tercile (State University):")
rows = []
for sb, g in su.groupby("setting_b"):
    if len(g) < 30:
        continue
    g = g.assign(tercile=pd.qcut(g["cost_m"], 3, labels=["low", "mid", "high"], duplicates="drop"))
    for tr, x in g.groupby("tercile"):
        rows.append({"setting_b": sb, "cost_tercile": tr, "n": len(x), "median_cost_m": x["cost_m"].median(),
                     "mean_share_%": x["oop_share"].mean() * 100, "pct_share_ge_50": (x["oop_share"] >= .5).mean() * 100})
show(pd.DataFrame(rows), "E7b_su_setting_by_cost_tercile", index=False)
su["hs"] = su["oop_share"] >= .5
print(f"\nHigh-share (>=50%) State-University episodes: {int(su.hs.sum())} of {len(su)}")
prof = su.groupby("hs").agg(n=("oop_share", "size"), median_cost_m=("cost_m", "median"), median_LOS=("los_days", "median"),
                            median_lines=("n_claim_lines", "median"))
show(prof, "E7c_su_highshare_profile")
show(pd.crosstab(su["setting_b"], su["hs"]), "E7d_su_highshare_by_setting", 0)
show(pd.crosstab(su["specialty_c"], su["hs"]), "E7e_su_highshare_by_specialty", 0)
keys = [c for c in ["Episode_key", "setting_cell", "specialty_c", "insurance_fund_grp", "oop_share", "cost_m", "los_days",
                    "n_claim_lines"] if c in su.columns]
print("\nTop 15 most expensive State-University episodes:")
show(su.sort_values("cost_m", ascending=False)[keys].head(15), "E7f_su_top_cost", 3, index=False)

# ----------------------------------------------------------------------------
# E8. COMPOSITION: why age and fund are fragile
# ----------------------------------------------------------------------------
banner("E8. COMPOSITION OF AGE AND FUND ACROSS OWNERSHIP / SPECIALTY / SETTING (reconciled sample, row %)")
for a, b in [("age_group", "specialty_c"), ("age_group", "ownership_grp"), ("insurance_fund_grp", "ownership_grp"),
             ("insurance_fund_grp", "setting_cell")]:
    print(f"\n{a} x {b}")
    show(pd.crosstab(dr[a], dr[b], normalize="index") * 100, f"E8_{a}_x_{b}", 1)

# ----------------------------------------------------------------------------
# E9. DATA INTEGRITY
# ----------------------------------------------------------------------------
banner("E9. DATA INTEGRITY")
print("Duplicate Episode_key in analysis sample:", int(d["Episode_key"].duplicated().sum()))
print("Episodes per cluster (reconciled):", dr.groupby("cl").size().describe().round(2).to_dict())
z = dr[dr["oop_share"] == 0]
print(f"\nZero cost-sharing episodes ({len(z)}) - verify in raw claims:")
show(z[[c for c in keys if c in z.columns]], "E9a_zero_share", 3, index=False)
top = dr.assign(abs_dev=(dr["oop_share"] - dr.groupby("setting_cell")["oop_share"].transform("median")).abs())
print("\nEpisodes furthest from their setting's median share (verify in raw data):")
show(top.sort_values("abs_dev", ascending=False)[[c for c in keys + ["ownership_grp"] if c in top.columns]].head(15),
     "E9b_far_from_setting_median", 3, index=False)

banner("HOW TO READ THE RESULTS")
print("""A. E1: if flagged private episodes have gaps >20%, the 5% rule is not the issue (the gap is real money, not rounding).
   E2: if oop+gap of flagged private episodes sits inside the reconciled-private P5-P95 (>~80%) and the insurer share
       matches the reconciled group, the 'balance billing / patient-paid remainder' reading is supported -> population
       cost-sharing is near the scenario/upper figure (~45%), not the recorded 20%. If not, only bounds can be reported.
   E3/E4: flagging concentrated by month, site, cluster or department = a data-capture artefact, not a patient trait;
       then the reconciled private episodes are a non-random 23% and IPW cannot repair it.
   E5: cost-independence of the private share supports applying the reconciled tariff to flagged episodes; the sensitivity table
       gives the population mean under any assumed share.
B. E6: separation criteria triggered at >=20%; sex/fund results that vanish with log cost are artefacts.
C. E7: a cost threshold above which State-University shares jump to private-like levels would need a cost-band covariate;
       E7c-E7f show whether high-share episodes look like mis-typed (inpatient-like) episodes in an ED/other setting.
D. E8: explains why the age effect appears only after adjusting for specialty and why crude fund differences disappear.""")
print(f"\nAll outputs in: {OUT_DIR}")

In [32]:
# =============================================================================
# OBJECTIVE 3 (v4, FINAL) - Patient financial burden: episode-level patient cost-sharing
# Outcome : oop_share = Insured Person's Share / Total Cost  (fraction in [0,1])
#           -> "episode-level patient cost-sharing", NOT catastrophic health expenditure.
# Primary : fractional logit (Papke-Wooldridge), cluster-robust SE, v3-reconciled episodes,
#           additive setting + ownership + specialty structure.
#
# CHANGES vs v3 - every item is driven by the v3 diagnostics (E1-E9); no further diagnostics are needed:
#  1. POPULATION ESTIMATE. E1/E2: all 412 unreconciled private episodes have gaps >20% (gross, not rounding);
#     their implied patient share (oop + gap) lies inside the reconciled-private P5-P95 in 87-95% of cases and
#     the insurer paid the same ~23-27% in flagged and reconciled private episodes. So the unreconciled money
#     is plausibly patient-paid. v4 reports LOWER / SCENARIO A / SCENARIO B / UPPER with cluster-bootstrap CIs
#     and a sensitivity grid (T3e, T3f). Scenario B (flagged private = oop + gap, others as recorded) uses each
#     episode's own record; A (ownership rule) is kept for comparison. The assumption itself is not testable.
#  2. FUND. E3: inside private hospitals reconciliation is determined by fund (Iranians 0% flagged, rural 4%,
#     Government employees 76%, Other funds 67%). Fund contrasts are therefore estimable only through
#     State-University episodes; IPW violates positivity. Table T3g; fund is reported as 'not robust'.
#  3. BINARY MODELS. E6: the v3 separation flag was a false alarm (1 extreme fitted value of 427) while the
#     >=10% model had a near-separated cell (Temporary Hospitalization 96% high, OR 82). v4 (a) drops settings
#     where the threshold is rule-determined (prevalence <5% or >95%), (b) counts shares within 0.5 pp below the
#     threshold as high (tariff mass point at 10%), (c) flags separation only if max|coef|>8, non-finite SE or
#     >1% extreme fitted values. Sex/fund effects survived log-cost control (E6B) -> reported as exploratory,
#     threshold-specific findings, not as robust determinants.
#  4. NEW SECONDARY MODEL: fractional logit inside State University hospitals only (T13), where shares vary.
#  5. NEW SENSITIVITY: S12 reconciliation tolerance 20% (E1: 83 of 104 flagged State-University episodes have
#     gaps of 5-20%), S13 excluding zero-share episodes (E9: two single-claim-line high-cost zero-share episodes).
#  6. Reporting notes updated with the E7/E8/E9 findings (4 State-University surgery episodes with private-like
#     shares to verify; age is confounded with ownership and specialty).
# =============================================================================
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats

warnings.filterwarnings("ignore")

# ----------------------------------------------------------------------------
# 0. SETTINGS
# ----------------------------------------------------------------------------
BASE = Path(r"C:\Users\Shekoofeh\Documents\Health Insurance Data\Cholera")
DEFAULT_INPUT = BASE / "feature engineering v4_FINAL" / "01_analysis_dataset.csv"


def resolve_input():
    for a in sys.argv[1:]:
        if a.startswith("-"):
            continue
        p = Path(a)
        if p.suffix.lower() == ".csv" and p.is_file():
            return p
    return DEFAULT_INPUT


INPUT_FILE = resolve_input()
assert INPUT_FILE.is_file(), f"Input file not found: {INPUT_FILE}"
OUT_DIR = INPUT_FILE.parent / "objective3_outputs_v4"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Input : {INPUT_FILE}\nOutput: {OUT_DIR}")


class Tee:
    def __init__(self, path, original):
        self.f = open(path, "w", encoding="utf-8")
        self.o = original

    def write(self, s):
        self.o.write(s)
        self.f.write(s)

    def flush(self):
        self.o.flush()
        self.f.flush()

    def __getattr__(self, name):
        return getattr(self.o, name)


if isinstance(sys.stdout, Tee):
    sys.stdout.f.close()
    sys.stdout = sys.stdout.o
sys.stdout = Tee(OUT_DIR / "objective3_log.txt", sys.stdout)
pd.set_option("display.width", 250, "display.max_columns", 40, "display.max_colwidth", 80)

MIN_LEVEL_N = 30
MIN_STRATUM_N = 20
SPEC_MIN_N = 50
POOL = {"age_group": 10, "sex": 10, "insurance_fund_grp": 20, "specialty_c": 20, "setting_cell": 5,
        "ownership_grp": 5, "ym": 15}
CELL_N = 5
N_BOOT = 2000
N_BOOT_ADJ = 1000
SEED = 42
UNIT = 1e6
N_FOLDS = 5
THRESHOLDS = (0.10, 0.20)          # binary models (State University only); >=30% descriptive only
MIN_CLUSTERS = 50
TOL = 0.005                  # shares within 0.5 pp below a threshold count as 'high' (tariff mass point)
N_BOOT_POP = 1000
MIN_EPV = 10                 # preferred; 5-10 = report as exploratory (Vittinghoff & McCulloch 2007)
SEP_FRAC = 0.01              # separation flagged if >1% of fitted probabilities are <0.005 or >0.995
SU = "State University"
TIGHT_GAP = 0.02
EXPECTED_FLAGGED, EXPECTED_RECONCILED = 521, 650
BUNDLE_DEPT = "Limited surgery - D Clinic"
INP = "Hospitalization / Inpatient"
REF_DEPT = "Limited surgery - D Clinic"
INSURANCE_MERGE = {"Other layers": "Other funds", "Other layers of the referral system": "Other funds",
                   "Other (rare)": "Other funds"}
DEPT_MAP = {"General emergency": "Emergency / triage", "General emergency and triage": "Emergency / triage",
            "Internal General": "Internal (general + emergency)", "Internal emergency": "Internal (general + emergency)",
            "Other (rare)": "Other departments"}
DESC_CATS = ["age_group", "sex", "insurance_fund_grp", "diagnosis_type_grp", "specialty_grp", "department_grp",
             "ownership_grp"]
NUMERIC = {"time_m", "log_cost", "lc_c", "lc_su", "lc_ns"}
LEAK = {"s_gap", "gap_share", "gap_signed", "gapflag", "gapflag_v0", "insurer_share", "insurer_deduction_share",
        "org_deduction_share", "unreconciled_share", "oop_share_of_known", "financing_gap_pct", "y_upper",
        "y_known", "oop_amt_m", "y_scen", "y_scenB"}

BLOCKS = [
    ("M1_predisposing+time", ["age_group", "sex", "time_m"]),
    ("M2_enabling(fund)", ["age_group", "sex", "time_m", "insurance_fund_grp"]),
    ("M3_service setting+ownership", ["age_group", "sex", "time_m", "insurance_fund_grp", "setting_cell",
                                      "ownership_grp"]),
    ("M4_need proxy(specialty)_PRIMARY", ["age_group", "sex", "time_m", "insurance_fund_grp", "setting_cell",
                                         "ownership_grp", "specialty_c"]),
    ("M5_cost gradient (exploratory, mechanical)", ["age_group", "sex", "time_m", "insurance_fund_grp",
                                                    "setting_cell", "ownership_grp", "specialty_c", "lc_su", "lc_ns"]),
]
PRIM = list(BLOCKS[3][1])
ADJ_VARS = ["age_group", "sex", "insurance_fund_grp"]
SEL_COVS = ["age_group", "sex", "time_m", "insurance_fund_grp", "setting_cell", "ownership_grp"]
BIN_SU = ["age_group", "sex", "time_m", "insurance_fund_grp", "setting_b"]
CHECKS = []


def check(name, ok, detail):
    CHECKS.append((name, "PASS" if ok else "WARN", detail))


def banner(t):
    print("\n" + "=" * 90 + f"\n{t}\n" + "=" * 90)


def fmt(x):
    return f"{x:.3g}" if pd.notna(x) else "-"


# ----------------------------------------------------------------------------
# 1. LOAD, DEFINE SAMPLES
# ----------------------------------------------------------------------------
df = pd.read_csv(INPUT_FILE, low_memory=False)
print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns")


def flag(col):
    if col not in df.columns:
        return pd.Series(False, index=df.index)
    return pd.to_numeric(df[col], errors="coerce").fillna(0).eq(1)


need = ["oop_share", "log_total_cost", "los_days", "n_claim_lines", "analytic_primary", "Cluster_for_SE",
        "Episode_key", "flag_financing_gap_v3_requested"] + DESC_CATS
miss = [c for c in need if c not in df.columns]
assert not miss, f"Missing columns: {miss}"
for c in ["oop_share", "log_total_cost", "los_days", "n_claim_lines", "cost_per_day", "admission_days_since_start",
          "financing_gap_v3_pct", "financing_gap_v0_pct", "oop_share_of_known"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

raw_col = next((c for c in ["Total Cost", "total_cost", "Total_Cost"] if c in df.columns), None)
if raw_col:
    df["total_cost"] = pd.to_numeric(df[raw_col], errors="coerce")
else:
    cands = {"exp": np.exp(df["log_total_cost"]), "expm1": np.expm1(df["log_total_cost"])}
    best = "exp"
    if "cost_per_day" in df.columns:
        z = df["los_days"].eq(0) & df["cost_per_day"].gt(0)
        errs = {k: float(((v[z] - df.loc[z, "cost_per_day"]).abs() / df.loc[z, "cost_per_day"]).median())
                for k, v in cands.items()}
        best = min(errs, key=errs.get)
        print(f"Total cost recovered as {best}(log_total_cost); median rel. error vs cost_per_day at LOS=0: {errs[best]:.2e}")
    df["total_cost"] = cands[best]

df["gap_signed"] = df["financing_gap_v3_pct"] / 100 if "financing_gap_v3_pct" in df.columns else np.nan
df["gap_share"] = df["gap_signed"].clip(lower=0)

mask = flag("analytic_primary")
if "complete_case_covariates" in df.columns:
    mask &= flag("complete_case_covariates")
n_bad_cost = int((mask & flag("flag_invalid_total_cost")).sum())
n_bad_oop = int((mask & flag("flag_oop_out_of_range")).sum())
mask &= ~flag("flag_invalid_total_cost") & ~flag("flag_oop_out_of_range")
mask &= df["oop_share"].between(0, 1)
d = df.loc[mask].copy()
d = d[np.isfinite(d["total_cost"]) & (d["total_cost"] > 0)]
d = d.dropna(subset=["los_days", "n_claim_lines", "oop_share"] + DESC_CATS)
print(f"Analysis sample: {len(d):,} episodes (invalid-cost removed: {n_bad_cost}; out-of-range OOP removed: {n_bad_oop})")

if "admission_days_since_start" in d.columns and d["admission_days_since_start"].notna().all():
    d["time_m"] = d["admission_days_since_start"] / 30.4375
elif "admission_ym" in d.columns:
    _ym = pd.PeriodIndex(d["admission_ym"].astype(str).str[:7], freq="M")
    d["time_m"] = (_ym - _ym.min()).map(lambda x: x.n).astype(float)
else:
    raise SystemExit("Need admission_days_since_start or admission_ym for the time control")
HAVE_YM = "admission_ym" in d.columns
if HAVE_YM:
    d["ym"] = d["admission_ym"].astype(str).str[:7]
    _vcm = d["ym"].value_counts()
    d["ym"] = np.where(d["ym"].map(_vcm) < 30, "small_months_pooled", d["ym"])

for v in DESC_CATS:
    d[v] = d[v].astype(str).str.strip()
d["insurance_fund_grp"] = d["insurance_fund_grp"].replace(INSURANCE_MERGE)
for v in DESC_CATS:
    vc = d[v].value_counts()
    rare = vc[vc < MIN_LEVEL_N].index
    if len(rare):
        print(f"  {v}: pooling {list(rare)} -> 'Other (pooled)'")
        d.loc[d[v].isin(rare), v] = "Other (pooled)"
assert (d["diagnosis_type_grp"] == INP).any(), f"'{INP}' not found in diagnosis_type_grp"

_vc = d["specialty_grp"].value_counts()
d["specialty_c"] = d["specialty_grp"].where(~d["specialty_grp"].isin(_vc[_vc < SPEC_MIN_N].index),
                                            "Other small specialties")
is_inp = d["diagnosis_type_grp"].eq(INP)
d["dept_in"] = REF_DEPT
_dep = d.loc[is_inp, "department_grp"].replace(DEPT_MAP)
_cnt = _dep.value_counts()
d.loc[is_inp, "dept_in"] = _dep.where(~_dep.isin(_cnt[_cnt < MIN_STRATUM_N].index), "Other departments")
d["setting_label"] = np.where(is_inp, d["dept_in"], "n/a (non-inpatient)")
d["setting_cell"] = d["diagnosis_type_grp"] + " | " + d["setting_label"]

d["cost_m"] = d["total_cost"] / UNIT
d["log_cost"] = np.log(d["cost_m"])
d["lc_c"] = d["log_cost"] - d["log_cost"].median()
_su = d["ownership_grp"].eq(SU)
d["lc_su"] = np.where(_su, d["lc_c"], 0.0)
d["lc_ns"] = np.where(~_su, d["lc_c"], 0.0)
d["oop_amt_m"] = d["oop_share"] * d["cost_m"]
d["y_known"] = (d["oop_share_of_known"].where(flag("valid_oop_known_denominator").loc[d.index])
                if "oop_share_of_known" in d.columns else np.nan)
d["y_upper"] = np.clip(d["oop_share"] + d["gap_share"].fillna(0), 0, 1)
d["cl"] = d["Cluster_for_SE"].fillna(d["Episode_key"]).astype(str)
anon = flag("flag_anonymous_episode").loc[d.index]
cardiac = flag("audit_cardiac_specialty_or_dept").loc[d.index]
d["gapflag"] = flag("flag_financing_gap_v3_requested").loc[d.index].astype(int)
d["gapflag_v0"] = flag("flag_financing_gap").loc[d.index].astype(int)
d["regime"] = pd.cut(d["oop_share"], [-1, 0.10 - 1e-12, 0.50 - 1e-12, 2],
                     labels=["low<10%", "mid10-50%", "high>=50%"]).astype(str)
dclinic = d["setting_cell"].str.contains(BUNDLE_DEPT, regex=False)

dr = d[d["gapflag"] == 0].copy()
n_fl = int(d["gapflag"].sum())
print(f"v3-flagged: {n_fl} (expected {EXPECTED_FLAGGED}); PRIMARY reconciled sample: {len(dr):,} "
      f"(expected {EXPECTED_RECONCILED}); D-clinic: {int(dclinic.sum())}, of which reconciled: "
      f"{int((dclinic & (d['gapflag'] == 0)).sum())}")
check("primary sample reproduces earlier diagnostics (650 reconciled / 521 flagged)",
      len(d) != 1171 or (len(dr) == EXPECTED_RECONCILED and n_fl == EXPECTED_FLAGGED),
      f"n_reconciled={len(dr)}, n_flagged={n_fl}")
N_CL = dr["cl"].nunique()
print(f"Clusters in primary sample: {N_CL:,}")
check(f"cluster-robust inference: >= {MIN_CLUSTERS} clusters", N_CL >= MIN_CLUSTERS,
      f"{N_CL} clusters (with fewer, cluster-robust Wald p-values are anti-conservative)")
codes_all = pd.factorize(d["cl"])[0]
fold_all = pd.Series(np.random.default_rng(SEED).integers(0, N_FOLDS, codes_all.max() + 1)[codes_all], index=d.index)


def build_cell(data, m):
    raw = data["setting_cell"] + " @ " + data["ownership_grp"]
    lab = raw.where(raw.map(raw.value_counts()) >= m, data["setting_cell"] + " @ sparse")
    return lab.where(lab.map(lab.value_counts()) >= m, "Other (pooled) @ sparse")


def prep(data):
    data = data.copy()
    for v, m in POOL.items():
        if v in data.columns:
            vc = data[v].value_counts()
            rare = vc[vc < m].index
            if len(rare) and len(vc) - len(rare) >= 1:
                data.loc[data[v].isin(rare), v] = "Other (pooled)"
    data["cell_so"] = build_cell(data, CELL_N)
    data["cell_so20"] = build_cell(data, 20)
    return data


dp, drp = prep(d), prep(dr)

# ----------------------------------------------------------------------------
# 2. DESCRIPTION  (primary sample = reconciled episodes ONLY; see [3b] for population bounds)
# ----------------------------------------------------------------------------
banner("[2] DESCRIPTION OF EPISODE-LEVEL PATIENT COST-SHARING (oop_share, %)")


def desc(s, thr=None):
    s = s.dropna()
    out = {"N": len(s), "Mean": s.mean(), "SD": s.std(), "Median": s.median(), "Q1": s.quantile(.25),
           "Q3": s.quantile(.75), "P90": s.quantile(.90), "P99": s.quantile(.99), "Min": s.min(), "Max": s.max(),
           "% zero": (s == 0).mean() * 100}
    out["% >=10"] = (s >= thr[0]).mean() * 100 if thr else np.nan
    out["% >=30"] = (s >= thr[1]).mean() * 100 if thr else np.nan
    out["Skewness"] = stats.skew(s)
    return pd.Series(out)


T1 = pd.DataFrame({
    "ALL episodes (incl. unreconciled)": desc(d["oop_share"] * 100, (10, 30)),
    "PRIMARY: v3-reconciled": desc(dr["oop_share"] * 100, (10, 30)),
    "Unreconciled (v3-flagged)": desc(d.loc[d["gapflag"] == 1, "oop_share"] * 100, (10, 30)),
    "PRIMARY, patient payment (million rial)": desc(dr["oop_amt_m"])}).T
print(T1.round(2).to_string())
T1.to_csv(OUT_DIR / "T1_oop_overall.csv")


def mass_summary(s):
    r = s.round(3)
    vc = r.value_counts()
    n = len(r)
    return pd.Series({"n": n, "distinct_3dp": len(vc), "top1_share": vc.iloc[0] / n, "top5_share": vc.iloc[:5].sum() / n,
                      "top10_share": vc.iloc[:10].sum() / n,
                      "near_multiple_of_5pp (chance~10%)": ((s - (s / .05).round() * .05).abs() <= .005).mean()})


T1m = pd.DataFrame({"primary": mass_summary(dr["oop_share"]), "all": mass_summary(d["oop_share"])}).T
print("\nMass-point summary (shares cluster on tariff values -> cost-sharing is largely rule-based):\n", T1m.round(3).to_string())
T1m.to_csv(OUT_DIR / "T1m_mass_points.csv")


def boot_desc(data, B=N_BOOT):
    cc = pd.factorize(data["cl"])[0]
    grp = [np.where(cc == k)[0] for k in range(cc.max() + 1)]
    s, a, c = data["oop_share"].values, data["oop_amt_m"].values, data["cost_m"].values
    rng = np.random.default_rng(SEED)
    out = []
    for _ in range(B):
        idx = np.concatenate([grp[k] for k in rng.integers(0, len(grp), len(grp))])
        out.append((s[idx].mean() * 100, np.median(s[idx]) * 100, a[idx].sum() / c[idx].sum() * 100,
                    (s[idx] >= .10).mean() * 100))
    out = np.array(out)
    est = [s.mean() * 100, np.median(s) * 100, a.sum() / c.sum() * 100, (s >= .10).mean() * 100]
    return pd.DataFrame({"estimate": est, "ci_low": np.percentile(out, 2.5, axis=0),
                         "ci_high": np.percentile(out, 97.5, axis=0)},
                        index=["Mean episode share %", "Median episode share %",
                               "Aggregate share % (sum OOP / sum cost)", "% episodes >=10%"])


T1b = boot_desc(dr)
print("\nCluster-bootstrap 95% CI, primary (reconciled) sample ONLY:\n", T1b.round(2).to_string())
T1b.to_csv(OUT_DIR / "T1b_bootstrap_ci.csv")

srt = dr["oop_amt_m"].sort_values(ascending=False)
conc = pd.Series({f"top {p}% of episodes": srt.head(max(1, int(len(srt) * p / 100))).sum() / srt.sum() * 100
                  for p in (1, 5, 10, 25)}, name="% of all patient payments (primary)")
print("\nConcentration of patient payments:\n", conc.round(1).to_string())
conc.to_csv(OUT_DIR / "T1c_concentration.csv")


def by_group(data, var, label):
    tot_amt = data["oop_amt_m"].sum()
    rows = []
    for lvl, g in data.groupby(var):
        s = g["oop_share"]
        rows.append({"Sample": label, "Variable": var, "Level": lvl, "N": len(g), "% episodes": len(g) / len(data) * 100,
                     "Mean % (SD)": f"{s.mean() * 100:.1f} ({s.std() * 100:.1f})",
                     "Median [IQR] %": f"{s.median() * 100:.1f} [{s.quantile(.25) * 100:.1f}-{s.quantile(.75) * 100:.1f}]",
                     "% >=10": (s >= .10).mean() * 100, "% >=30": (s >= .30).mean() * 100,
                     "Aggregate share %": g["oop_amt_m"].sum() / g["cost_m"].sum() * 100,
                     "% of all patient payments": g["oop_amt_m"].sum() / tot_amt * 100})
    out = pd.DataFrame(rows)
    try:
        out["KW p (exploratory)"] = stats.kruskal(*[g["oop_share"].values for _, g in data.groupby(var)]).pvalue
    except ValueError:
        out["KW p (exploratory)"] = np.nan
    return out


GV = DESC_CATS + ["specialty_c", "setting_cell", "cell_so"]
T2 = pd.concat([by_group(drp, v, "primary") for v in GV] + [by_group(dp, v, "all") for v in GV], ignore_index=True)
T2.to_csv(OUT_DIR / "T2_oop_by_determinant.csv", index=False)
print("\nPrimary sample by setting x ownership cell (cells with N<10 are not interpretable on their own):")
print(T2[(T2.Sample == "primary") & (T2.Variable == "cell_so")].drop(columns=["Sample", "Variable"]).round(1).to_string(index=False))
T2r = pd.crosstab(dp["setting_cell"], dp["regime"])
T2r.to_csv(OUT_DIR / "T2r_regime_by_setting_all.csv")
print("\nCost-sharing regime by setting (ALL episodes):\n", T2r.to_string())

# ----------------------------------------------------------------------------
# 3. IDENTIFICATION AND POPULATION-LEVEL BOUNDS
# ----------------------------------------------------------------------------
banner("[3] IDENTIFICATION - v3-RECONCILED vs UNRECONCILED EPISODES AND BOUNDS")
rows = []
for sc, g in d.groupby("setting_cell"):
    ok, bad = g[g["gapflag"] == 0], g[g["gapflag"] == 1]
    rows.append({"setting_cell": sc, "n": len(g), "n_flagged": len(bad), "% flagged": len(bad) / len(g) * 100,
                 "mean_oop_reconciled %": ok["oop_share"].mean() * 100 if len(ok) else np.nan,
                 "mean_oop_flagged %": bad["oop_share"].mean() * 100 if len(bad) else np.nan,
                 "LOWER bound (oop_share) %": g["oop_share"].mean() * 100,
                 "mean oop_share_of_known %": g["y_known"].mean() * 100,
                 "UPPER bound (oop + v3 gap) %": g["y_upper"].mean() * 100,
                 "mean unreconciled (v3, >=0) %": g["gap_share"].mean() * 100})
T3 = pd.DataFrame(rows)
T3.to_csv(OUT_DIR / "T3_identification_bounds_by_setting.csv", index=False)
print(T3.round(1).to_string(index=False))
print("\nWhere % flagged is ~100 (D-clinic) the share is identified only up to [lower, upper]; never a point estimate.")
d["flag_combo"] = np.select([(d.gapflag_v0 == 0) & (d.gapflag == 0), (d.gapflag_v0 == 1) & (d.gapflag == 0),
                             (d.gapflag_v0 == 0) & (d.gapflag == 1)], ["reconciled both", "v0-flag only", "v3-flag only"],
                            "flagged both")
print("\nv0 vs v3 flag:")
print(d.groupby("flag_combo").oop_share.agg(n="size", mean_share_pct=lambda s: s.mean() * 100).round(2).to_string())

banner("[3b] POPULATION-LEVEL BOUNDS AND OWNERSHIP-RULE SCENARIO (all episodes)")
rec = d[d["gapflag"] == 0]
own_mean = rec.groupby("ownership_grp")["oop_share"].mean()
d["y_scen"] = np.where(d["gapflag"] == 1, d["ownership_grp"].map(own_mean).fillna(rec["oop_share"].mean()), d["oop_share"])


def pop_row(data, col):
    return {"n": len(data), "mean episode share %": data[col].mean() * 100,
            "aggregate share % (cost-weighted)": (data[col] * data["cost_m"]).sum() / data["cost_m"].sum() * 100}


dr_ = d[d["gapflag"] == 0]
d["y_scenB"] = np.where((d["ownership_grp"] == "private") & (d["gapflag"] == 1), d["y_upper"], d["oop_share"])
T3c = pd.DataFrame({
    "LOWER: oop_share as recorded (all episodes)": pop_row(d, "oop_share"),
    "SCENARIO A (ownership rule): flagged episodes take reconciled mean of their ownership group": pop_row(d, "y_scen"),
    "SCENARIO B (episode remainder): flagged PRIVATE = oop + gap; other flagged as recorded": pop_row(d, "y_scenB"),
    "UPPER: oop_share + whole v3 gap treated as patient-paid": pop_row(d, "y_upper"),
    "PRIMARY reconciled sample only (NOT the population)": pop_row(dr_, "oop_share")}).T
print(T3c.round(1).to_string())
T3c.to_csv(OUT_DIR / "T3c_population_bounds_scenario.csv")
g_own = []
for o, g in d.groupby("ownership_grp"):
    f = g[g["gapflag"] == 1]
    r_ = g[g["gapflag"] == 0]
    g_own.append({"ownership_grp": o, "n": len(g), "n_flagged": len(f), "% flagged": len(f) / len(g) * 100,
                  "mean_oop_reconciled %": r_["oop_share"].mean() * 100 if len(r_) else np.nan,
                  "mean_oop_flagged %": f["oop_share"].mean() * 100 if len(f) else np.nan,
                  "mean_gap_flagged % of cost": f["gap_share"].mean() * 100 if len(f) else np.nan})
T3d = pd.DataFrame(g_own)
T3d.to_csv(OUT_DIR / "T3d_flagging_by_ownership.csv", index=False)
print("\nFlagging by ownership:\n", T3d.round(1).to_string(index=False))
print("NOTE: if flagged private episodes show ~3% recorded OOP but a ~65% unreconciled gap, check in the raw claims whether the\n"
      "gap is balance billing paid by the patient. If so the recorded share understates their burden and the SCENARIO/UPPER\n"
      "rows are closer to the truth than the LOWER row. The scenario is an assumption, not an estimate.")

print("\nDiagnostics E1/E2 support the patient-paid-remainder reading for PRIVATE episodes (gaps >20% in all 412 flagged; "
      "oop+gap inside the reconciled-private P5-P95 in 87-95%; insurer share ~23-27% in every private group). "
      "Scenarios remain assumptions: the bootstrap CI below covers sampling error only.")
try:
    cc_p = pd.factorize(d["cl"])[0]
    grp_p = [np.where(cc_p == k)[0] for k in range(cc_p.max() + 1)]
    oc = pd.factorize(d["ownership_grp"])[0]
    n_o = oc.max() + 1
    fl_, o_, up_, c_, sB_ = (d["gapflag"].values, d["oop_share"].values, d["y_upper"].values, d["cost_m"].values,
                            d["y_scenB"].values)

    def _stats(y, c):
        return [y.mean() * 100, (y * c).sum() / c.sum() * 100]

    est_p = _stats(o_, c_) + _stats(d["y_scen"].values, c_) + _stats(sB_, c_) + _stats(up_, c_)
    rng_p = np.random.default_rng(SEED)
    bs = []
    for _ in range(N_BOOT_POP):
        idx = np.concatenate([grp_p[k] for k in rng_p.integers(0, len(grp_p), len(grp_p))])
        f_, ocb, ob, cb = fl_[idx], oc[idx], o_[idx], c_[idx]
        r_ = f_ == 0
        s_o = np.bincount(ocb[r_], weights=ob[r_], minlength=n_o)
        n_r = np.bincount(ocb[r_], minlength=n_o)
        m_o = np.where(n_r > 0, s_o / np.maximum(n_r, 1), ob[r_].mean())
        yA = np.where(f_ == 1, m_o[ocb], ob)
        bs.append(_stats(ob, cb) + _stats(yA, cb) + _stats(sB_[idx], cb) + _stats(up_[idx], cb))
    bs = np.array(bs)
    idx_p = pd.MultiIndex.from_product([["LOWER (recorded)", "SCENARIO A (ownership rule)", "SCENARIO B (episode remainder)",
                                         "UPPER (oop + gap)"], ["mean episode share %", "aggregate share % (cost-weighted)"]])
    T3e = pd.DataFrame({"estimate": est_p, "ci_low": np.percentile(bs, 2.5, axis=0),
                        "ci_high": np.percentile(bs, 97.5, axis=0)}, index=idx_p)
    print("\nPopulation-level patient cost-sharing, all episodes, cluster-bootstrap 95% CI:")
    print(T3e.round(1).to_string())
    T3e.to_csv(OUT_DIR / "T3e_population_bootstrap_ci.csv")
except Exception as e:
    print("  population bootstrap failed:", str(e)[:100])

try:
    own_priv = float(own_mean.get("private", np.nan))
    is_pf = (d["ownership_grp"] == "private") & (d["gapflag"] == 1)
    base_o = np.where(d["gapflag"] == 1, d["ownership_grp"].map(own_mean).fillna(rec["oop_share"].mean()), d["oop_share"])
    rows = []
    for x in [None, 0.30, 0.40, 0.50, 0.60, 0.65, 0.70, own_priv]:
        y = np.where(is_pf, d["oop_share"] if x is None else x, base_o)
        rows.append({"assumed share for flagged PRIVATE episodes": "as recorded" if x is None else f"{x:.1%}",
                     "population mean episode share %": np.mean(y) * 100,
                     "cost-weighted aggregate share %": (y * d["cost_m"]).sum() / d["cost_m"].sum() * 100})
    T3f = pd.DataFrame(rows)
    print("\nSensitivity of the population mean to the share assumed for flagged private episodes "
          "(other flagged episodes at their ownership group's reconciled mean):")
    print(T3f.round(1).to_string(index=False))
    T3f.to_csv(OUT_DIR / "T3f_scenario_sensitivity.csv", index=False)
    pvt = d[(d["ownership_grp"] == "private") & ~dclinic]
    T3g = pd.crosstab(pvt["insurance_fund_grp"], pvt["gapflag"]).reindex(columns=[0, 1], fill_value=0)
    T3g.columns = ["reconciled", "flagged"]
    T3g["% flagged"] = T3g["flagged"] / T3g.sum(axis=1) * 100
    print("\nPrivate non-D-clinic episodes: reconciliation by fund (fund determines whether a private claim reconciles):")
    print(T3g.round(1).to_string())
    T3g.to_csv(OUT_DIR / "T3g_private_flagging_by_fund.csv")
    T3h = pd.crosstab(d["ownership_grp"], d["insurance_fund_grp"], values=d["gapflag"], aggfunc="mean") * 100
    print("\n% flagged, ownership x fund (all episodes):")
    print(T3h.round(1).to_string())
    T3h.to_csv(OUT_DIR / "T3h_flagging_ownership_by_fund.csv")
except Exception as e:
    print("  scenario sensitivity failed:", str(e)[:100])

# ----------------------------------------------------------------------------
# 4. MODEL ENGINE
# ----------------------------------------------------------------------------
PROBIT = getattr(sm.families.links, "Probit", None) or sm.families.links.probit


def make_X(data, covs, spec=None):
    new = spec is None
    if new:
        spec = {"refs": {}, "levels": {}, "cols_of": {}, "cols": ["const"], "cv": [], "dropped": []}
    X = pd.DataFrame({"const": 1.0}, index=data.index)
    for v in covs:
        if v in NUMERIC:
            X[v] = data[v].astype(float)
            if new:
                spec["cols"].append(v)
                spec["cols_of"][v] = [v]
                spec["cv"].append(v)
            continue
        if new:
            vc = data[v].value_counts()
            if len(vc) < 2:
                continue
            vc = vc.sort_index().sort_values(ascending=False, kind="stable")
            ref = vc.index[0]
            spec["refs"][v] = ref
            spec["levels"][v] = [l for l in sorted(vc.index) if l != ref]
            cols = [f"{v}: {l}" for l in spec["levels"][v]]
            spec["cols"] += cols
            spec["cols_of"][v] = cols
            spec["cv"].append(v)
        if v in spec["levels"]:
            for l in spec["levels"][v]:
                X[f"{v}: {l}"] = (data[v] == l).astype(float)
    if new:
        keep, r = [], 0
        for c in spec["cols"]:
            r2 = np.linalg.matrix_rank(X[keep + [c]].values)
            if r2 > r:
                keep.append(c)
                r = r2
        spec["dropped"] = [c for c in spec["cols"] if c not in keep]
        spec["cols"] = keep
        spec["cols_of"] = {k: [c for c in v if c in keep] for k, v in spec["cols_of"].items()}
        spec["cv"] = [v for v in spec["cv"] if spec["cols_of"][v]]
    return X[spec["cols"]], spec


def fit_one(kind, y, X, groups, w=None, cov=True):
    kw = dict(cov_type="cluster", cov_kwds={"groups": groups}) if cov else {}
    if kind == "ols":
        return (sm.OLS(y, X) if w is None else sm.WLS(y, X, weights=w)).fit(**kw)
    fam = {"flogit": sm.families.Binomial(), "logit": sm.families.Binomial(),
           "fprobit": sm.families.Binomial(link=PROBIT()), "qpois": sm.families.Poisson()}[kind]
    m = sm.GLM(y, X, family=fam, var_weights=w) if w is not None else sm.GLM(y, X, family=fam)
    try:
        return m.fit(**kw)
    except Exception:
        return m.fit(method="bfgs", maxiter=3000, **kw)


def run(kind, data, covs, ycol="oop_share", weights=None, light=False):
    bad = set(covs) & LEAK
    assert not bad, f"Leakage: {bad} are components of the outcome's accounting identity"
    data = data.dropna(subset=[ycol]).copy()
    X, spec = make_X(data, covs)
    y = data[ycol].astype(float)
    groups = pd.factorize(data["cl"])[0]
    w = None if weights is None else weights.reindex(data.index).values.astype(float)
    res = fit_one(kind, y, X, groups, w, cov=not light)
    out = dict(kind=kind, n=int(len(data)), npar=X.shape[1], params=res.params, res=res, spec=spec, data=data,
               covs=covs, dropped=spec["dropped"], ycol=ycol, X=X, refs={v: spec["refs"].get(v) for v in spec["cv"]},
               dev_expl=(res.rsquared if kind == "ols" else 1 - res.deviance / res.null_deviance))
    if light:
        return out
    pv, cols = {}, list(X.columns)
    for v in spec["cv"]:
        idx = [cols.index(c) for c in spec["cols_of"][v]]
        R = np.zeros((len(idx), len(cols)))
        R[range(len(idx)), idx] = 1
        try:
            pv[v] = float(res.wald_test(R, scalar=True).pvalue)
        except Exception:
            pv[v] = np.nan
    ci = res.conf_int()
    ex = kind != "ols"
    tab = pd.DataFrame({"term": res.params.index, "estimate": np.exp(res.params.values) if ex else res.params.values,
                        "ci_low": np.exp(ci[0].values) if ex else ci[0].values,
                        "ci_high": np.exp(ci[1].values) if ex else ci[1].values, "p": res.pvalues.values})
    out.update(pvals=pv, tab=tab[tab.term != "const"])
    return out


def var_of(term):
    return str(term).split(": ")[0]


def predict(r, new):
    X, _ = make_X(new, r["covs"], r["spec"])
    un = sum(int((~new[v].isin(r["spec"]["levels"][v] + [r["spec"]["refs"][v]])).sum()) for v in r["spec"]["levels"])
    return np.asarray(r["res"].predict(X)), un


def clean(r, label):
    t = r["tab"].copy()
    t["reference"] = t["term"].map(lambda x: r["refs"].get(var_of(x)) or "")
    t["scale"] = "coefficient (share, fraction)" if r["kind"] == "ols" else (
        "ratio of mean amount" if r["kind"] == "qpois" else "odds ratio")
    t.insert(0, "model", label)
    return t


def oof(data, covs, kind="flogit", ycol="oop_share"):
    pred, unseen = pd.Series(np.nan, index=data.index), 0
    f = fold_all.loc[data.index]
    for k in range(N_FOLDS):
        tr, te = data[f != k], data[f == k]
        if len(te) == 0:
            continue
        try:
            r = run(kind, tr, covs, ycol, light=True)
            p, un = predict(r, te)
            pred.loc[te.index] = p
            unseen += un
        except Exception as e:
            print(f"   fold {k} failed: {str(e)[:70]}")
    return pred, unseen


def oof_metrics(data, pred, rows=None, ycol="oop_share"):
    m = pred.notna()
    if rows is not None:
        m &= rows
    y, p = data.loc[m, ycol].values, np.clip(pred[m].values, 1e-6, 1 - 1e-6)
    xl = lambda a, b: np.where(a > 0, a * np.log(a / b), 0.0)
    dev = 2 * (xl(y, p) + xl(1 - y, 1 - p))
    return {"n": int(m.sum()), "RMSE_pp": np.sqrt(((y - p) ** 2).mean()) * 100, "MAE_pp": np.abs(y - p).mean() * 100,
            "OOF_R2": 1 - ((y - p) ** 2).sum() / ((y - y.mean()) ** 2).sum(),
            "mean_obs/mean_pred": y.mean() / p.mean(), "mean_binomial_deviance": dev.mean()}


def ridge_logit(data, covs, ycol):
    """Ridge-penalised logit (intercept unpenalised): finite estimates under (quasi-)separation. Used for IPW only."""
    X, _ = make_X(data, covs)
    y = data[ycol].astype(float).values
    a = np.full(X.shape[1], 1.0 / len(data))
    a[0] = 0.0
    m = sm.GLM(y, X.values, family=sm.families.Binomial()).fit_regularized(alpha=a, L1_wt=0.0, maxiter=1000)
    b = np.asarray(m.params, dtype=float)
    p = 1 / (1 + np.exp(-(X.values @ b)))
    return pd.Series(p, index=data.index), pd.Series(b, index=X.columns)


# ----------------------------------------------------------------------------
# 5. PRIMARY MODEL + FIT CHECKS
# ----------------------------------------------------------------------------
banner("[5] PRIMARY MODEL (fractional logit; v3-reconciled; additive setting + ownership) AND FIT CHECKS")
g0 = run("flogit", drp, PRIM)
if g0["dropped"]:
    print("Collinear columns dropped from the primary design:", g0["dropped"])
check("primary design has no collinear columns dropped", not g0["dropped"], str(g0["dropped"]))
p0 = pd.Series(np.asarray(g0["res"].fittedvalues), index=g0["data"].index)
disp = g0["res"].pearson_chi2 / g0["res"].df_resid
print(f"Primary: n={g0['n']:,}, params={g0['npar']}, n/param={g0['n'] / g0['npar']:.1f}, "
      f"deviance explained={g0['dev_expl']:.3f}, Pearson dispersion={disp:.3f}")
print("(Dispersion << 1: shares are nearly deterministic given ownership/setting, so the binomial variance "
      "function over-states variance; inference therefore relies on the cluster-robust sandwich.)")
print(f"Fitted range: {p0.min():.4f} - {p0.max():.4f}")
cl_ = np.clip(p0, 1e-6, 1 - 1e-6)
X2 = g0["X"].copy()
X2["_xb2"] = np.log(cl_ / (1 - cl_)) ** 2
link_p = float(fit_one("flogit", g0["data"]["oop_share"], X2, pd.factorize(g0["data"]["cl"])[0]).pvalues["_xb2"])
print(f"Pregibon-type link test p = {link_p:.3g}")
check("logit link not rejected (link test p>0.05)", link_p > 0.05,
      f"p={link_p:.3g}; rejection reflects the mass-point/tariff structure; logit/probit/OLS give the same OOF fit (D11) -> report as limitation")
rs = (g0["data"]["oop_share"] - p0) ** 2 / (p0 * (1 - p0))
vf = pd.DataFrame({"q": pd.qcut(p0, 5, duplicates="drop"), "r": rs}).groupby("q").r.mean()
print("Variance-function check (mean squared Pearson residual by fitted quintile):\n", vf.round(3).to_string())
pd.DataFrame({"link_test_p": [link_p], "pearson_dispersion": [disp]}).to_csv(OUT_DIR / "T4b_link_variance_checks.csv", index=False)
clean(g0, "Fractional logit PRIMARY").to_csv(OUT_DIR / "T5_primary_odds_ratios.csv", index=False)
print("\nPrimary odds ratios (setting_cell and specialty_c ORs are NOT interpreted individually - see reporting notes):")
print(clean(g0, "").drop(columns=["model", "scale"]).round(3).to_string(index=False))

print("\nOut-of-fold comparison of structures (cluster folds; common rows):")
CANDS = {"null": [], "setting only": ["setting_cell"], "setting + ownership (additive)": ["setting_cell", "ownership_grp"],
         "cell_so pool<5": ["cell_so"], "cell_so pool<20 (v1 structure)": ["cell_so20"],
         "additive + fund": ["setting_cell", "ownership_grp", "insurance_fund_grp"],
         "additive + specialty": ["setting_cell", "ownership_grp", "specialty_c"],
         "PRIMARY w/o specialty": [c for c in PRIM if c != "specialty_c"], "PRIMARY": PRIM,
         "PRIMARY + cost gradient (M5)": BLOCKS[4][1]}
oofp = {}
for nm, cv_ in CANDS.items():
    oofp[nm], un = oof(drp, cv_)
    print(f"   {nm}: predicted {int(oofp[nm].notna().sum())}/{len(drp)}; unseen levels mapped to reference: {un}")
common = pd.concat(oofp, axis=1).notna().all(axis=1)
T4d = pd.DataFrame({nm: oof_metrics(drp, p, common) for nm, p in oofp.items()}).T
print(f"Compared on {int(common.sum())} common rows:\n", T4d.round(3).to_string())
T4d.to_csv(OUT_DIR / "T4d_oof_structure_comparison.csv")
best_r2 = T4d["OOF_R2"].max()
check("primary OOF R2 within 0.02 of the best candidate", T4d.loc["PRIMARY", "OOF_R2"] >= best_r2 - 0.02,
      f"primary={T4d.loc['PRIMARY', 'OOF_R2']:.3f}, best={best_r2:.3f} ({T4d['OOF_R2'].idxmax()})")

pg = oofp["PRIMARY"]
cal = pd.DataFrame({"pred": pg, "obs": drp["oop_share"]}).dropna()
cal["decile"] = pd.qcut(cal["pred"], 10, duplicates="drop")
cal = cal.groupby("decile").agg(n=("obs", "size"), observed=("obs", "mean"), predicted=("pred", "mean"))
cal["diff_pp"] = (cal.observed - cal.predicted) * 100
print("\nOut-of-fold calibration by decile of predicted share:\n", cal.round(3).to_string())
cal.to_csv(OUT_DIR / "T4e_oof_calibration_deciles.csv")
check("OOF calibration: every decile within 5pp", cal.diff_pp.abs().max() < 5, f"max |diff|={cal.diff_pp.abs().max():.1f}pp")
rows = []
for v in ["setting_cell", "ownership_grp", "cell_so", "specialty_c", "insurance_fund_grp", "age_group", "sex"]:
    tmp = pd.DataFrame({"g": drp[v], "obs": drp["oop_share"], "pred": pg}).dropna()
    gg = tmp.groupby("g").apply(lambda x: pd.Series({"n": len(x), "obs": x.obs.mean(), "pred": x.pred.mean(),
                                                     "rmse_pp": np.sqrt(((x.obs - x.pred) ** 2).mean()) * 100}))
    gg["diff_pp"] = (gg.obs - gg.pred) * 100
    gg.insert(0, "variable", v)
    rows.append(gg.reset_index())
CG = pd.concat(rows)
CG.to_csv(OUT_DIR / "T4f_oof_calibration_by_group.csv", index=False)
badg = CG[(CG.n >= 30) & (CG.diff_pp.abs() > 3)]
print("\nGroups (n>=30) mis-predicted by >3pp out-of-fold:")
print(badg.round(2).to_string(index=False) if len(badg) else "  none")
cc_ = CG[(CG.variable == "cell_so") & (CG.n >= 10)]
print("\nPer setting x ownership cell (n>=10): out-of-fold RMSE (pp)")
print(cc_.drop(columns="variable").round(2).to_string(index=False))
w_ = drp.assign(pred=pg, err_pp=(drp.oop_share - pg) * 100).dropna(subset=["pred"])
show = [c for c in ["Episode_key", "setting_cell", "ownership_grp", "specialty_c", "insurance_fund_grp", "age_group",
                    "oop_share", "pred", "err_pp", "cost_m"] if c in w_.columns]
top = w_.reindex(w_.err_pp.abs().sort_values(ascending=False).index)[show].head(15)
top.to_csv(OUT_DIR / "T4g_worst_oof_residuals_VERIFY_IN_RAW_DATA.csv", index=False)
print("\nWorst 15 out-of-fold residuals (VERIFY IN RAW DATA):")
print(top.round(3).to_string(index=False))

# ----------------------------------------------------------------------------
# 6. ANDERSEN SEQUENTIAL BLOCKS
# ----------------------------------------------------------------------------
banner("[6] ANDERSEN SEQUENTIAL MODELS (fractional logit, v3-reconciled episodes)")
allv = []
for _, c in BLOCKS:
    allv += [v for v in c if v not in allv]
rows, blk = [], {}
for bname, covs in BLOCKS:
    r = run("flogit", drp, covs)
    pr, _ = oof(drp, covs)
    mt = oof_metrics(drp, pr, common)
    rows.append({"model": bname, "n": r["n"], "n_params": r["npar"], "dropped_cols": len(r["dropped"]),
                 "deviance_explained": r["dev_expl"], "OOF_R2": mt["OOF_R2"], "OOF_RMSE_pp": mt["RMSE_pp"],
                 **{f"p_{v}": r["pvals"].get(v, np.nan) for v in allv}})
    blk[bname] = r
T6 = pd.DataFrame(rows)
T6.to_csv(OUT_DIR / "T6_andersen_blocks.csv", index=False)
print(T6.round(4).to_string())
print("M5 uses total cost, the denominator of the outcome: its coefficients describe the cost-sharing schedule "
      "(mechanical), not a behavioural determinant.")

# ----------------------------------------------------------------------------
# 7. SENSITIVITY
# ----------------------------------------------------------------------------
banner("[7] SENSITIVITY (core S*; contrasts C* are not expected to agree)")
nd = dp[~dp["setting_cell"].str.contains(BUNDLE_DEPT, regex=False)].copy()
nd["recon"] = 1 - nd["gapflag"]
ipw = None
try:
    ps, sel_b = ridge_logit(nd, SEL_COVS, "recon")
    print(f"Ridge selection model P(reconciled | X), non-D-clinic n={len(nd)}: reconciled share = {nd['recon'].mean():.1%}; "
          f"max |coef| = {sel_b.drop('const').abs().max():.2f} (finite by construction)")
    sel_b.rename("ridge_logit_coef").to_csv(OUT_DIR / "T3b_selection_model_ridge.csv")
    ok_idx = nd.index[nd["recon"] == 1]
    ipw = (nd["recon"].mean() / ps.loc[ok_idx].clip(lower=0.05)).clip(upper=10)
    ipw = ipw / ipw.mean()
    print(f"IPW (stabilised, trimmed at 10): min {ipw.min():.2f}, max {ipw.max():.2f}")
except Exception as e:
    print("  selection model failed:", str(e)[:100])

cook_idx = d.index[:0]
try:
    cd = g0["res"].get_influence().cooks_distance
    cook = pd.Series(np.asarray(cd[0] if isinstance(cd, tuple) else cd), index=g0["data"].index)
    cook_idx = cook.nlargest(5).index
    print(f"Top-5 influential episodes (Cook's D): {list(drp.loc[cook_idx, 'Episode_key'])} -> verify in raw data")
except Exception as e:
    print("  influence unavailable:", e)

alt_struct = [c for c in PRIM if c not in ("setting_cell", "ownership_grp")] + ["cell_so"]
SPECS = {
    "S0_primary": dict(data=dr),
    "S1_tight_reconciliation_v3<=2%": dict(data=d[d["gap_signed"].abs() <= TIGHT_GAP]),
    "S2_excl_anonymous": dict(data=dr[~anon.loc[dr.index]]),
    "S3_excl_cardiac_audit": dict(data=dr[~cardiac.loc[dr.index]]),
    "S4_inpatient_only": dict(data=dr[dr["diagnosis_type_grp"] == INP]),
    "S5_denominator_known_financing": dict(data=dr[dr["y_known"].notna()], ycol="y_known"),
    "S6_drop_top5_influential": dict(data=dr.drop(index=cook_idx)),
    "S7_alt_structure_cell_so": dict(data=dr, covs=alt_struct),
    "S8_state_university_only": dict(data=dr[dr["ownership_grp"] == SU]),
    "S9_without_specialty": dict(data=dr, covs=[c for c in PRIM if c != "specialty_c"]),
    "S12_tolerance_20pct_reconciliation": dict(data=d[d["gap_signed"].abs() <= 0.20]),
    "S13_excl_zero_share": dict(data=dr[dr["oop_share"] > 0]),
    "C1_ALL_episodes_incl_unreconciled": dict(data=d),
    "C2_ALL_episodes_upper_bound": dict(data=d, ycol="y_upper"),
    "C3_v0_flag_sample (v1 primary)": dict(data=d[d["gapflag_v0"] == 0]),
}
if ipw is not None:
    SPECS["S10_IPW_for_reconciliation"] = dict(data=dr.loc[ipw.index], weights=ipw)
if HAVE_YM:
    SPECS["S11_month_FE"] = dict(data=dr, covs=[c for c in PRIM if c != "time_m"] + ["ym"])
VARS_S = PRIM + ["cell_so", "ym"]
SETTING_TERMS = ("setting_cell", "cell_so")
cols_p, cols_r, flips, base, skipped = {}, {}, [], None, []
for sname, sp_ in SPECS.items():
    dd = sp_["data"]
    if len(dd) < 100:
        print(f"  {sname}: skipped (n={len(dd)})")
        skipped.append(sname)
        continue
    try:
        r = run("flogit", prep(dd), sp_.get("covs", PRIM), sp_.get("ycol", "oop_share"), sp_.get("weights"))
    except Exception as e:
        print(f"  {sname}: failed ({str(e)[:80]})")
        skipped.append(sname)
        continue
    if r["dropped"]:
        print(f"  {sname}: {len(r['dropped'])} collinear column(s) dropped: {r['dropped'][:4]}")
    cols_p[sname] = pd.Series({**{v: r["pvals"].get(v, np.nan) for v in VARS_S}, "n": r["n"]})
    if sname == "S0_primary":
        base = r
    same = (lambda t: var_of(t) not in base["refs"] or r["refs"].get(var_of(t)) == base["refs"][var_of(t)]) if base else None
    if base is not None and sname != "S0_primary":
        common_t = [c for c in base["params"].index.intersection(r["params"].index) if c != "const" and same(c)]
        badt = [c for c in common_t if np.sign(base["params"][c]) != np.sign(r["params"][c])
                and abs(base["params"][c]) > .10 and abs(r["params"][c]) > .10]
        badt_ns = [c for c in badt if var_of(c) not in SETTING_TERMS]
        flips.append({"spec": sname, "core": sname.startswith("S"), "terms_compared": len(common_t),
                      "sign_flips": len(badt), "sign_flips_excl_setting": len(badt_ns),
                      "flipped_terms": "; ".join(badt[:10])})
    cols_r[sname] = pd.Series({t: np.exp(v) for t, v in r["params"].items() if t != "const"
                               and (sname == "S0_primary" or same(t))})
T7 = pd.DataFrame(cols_p)
T7.to_csv(OUT_DIR / "T7_sensitivity_wald.csv")
print("Joint Wald p-values by specification:")
print(T7.apply(lambda c: c.map(fmt)).to_string())
if skipped:
    print("Not estimable/skipped:", skipped)
FL = pd.DataFrame(flips)
FL.to_csv(OUT_DIR / "T7b_sign_flips.csv", index=False)
print("\nSign flips vs primary (same-reference terms, |coef|>0.10 in both):\n", FL.to_string(index=False))
RR = pd.DataFrame(cols_r)
RR.to_csv(OUT_DIR / "T7c_odds_ratio_stability.csv")
core_cols = [c for c in RR.columns if c.startswith("S") and c != "S0_primary"]
if core_cols:
    allc = ["S0_primary"] + core_cols
    ld = np.abs(np.log(RR[core_cols]).sub(np.log(RR["S0_primary"]), axis=0)).max(axis=1)
    stab = pd.DataFrame({"max_abs_logdiff": ld, "primary_OR": RR["S0_primary"], "min_OR": RR[allc].min(axis=1),
                         "max_OR": RR[allc].max(axis=1)}).sort_values("max_abs_logdiff", ascending=False)
    print("\nTerms whose odds ratio moves most across CORE specs:")
    print(stab.head(12).round(2).to_string())
    stab.to_csv(OUT_DIR / "T7d_ratio_stability_ranked.csv")
core_p = T7[[c for c in T7.columns if c.startswith("S")]].drop(index=["n", "ym", "cell_so"], errors="ignore")
SCORE = pd.DataFrame({"core_specs_estimated": core_p.notna().sum(axis=1), "p<0.05": (core_p < .05).sum(axis=1),
                      "median_p": core_p.median(axis=1), "max_p": core_p.max(axis=1)})
SCORE["share_sig"] = SCORE["p<0.05"] / SCORE["core_specs_estimated"]
SCORE["verdict"] = np.where(SCORE.core_specs_estimated < 4, "too few specs",
                            np.where(SCORE.share_sig >= .8, "ROBUST",
                                     np.where(SCORE.share_sig <= .2, "NOT SUPPORTED", "INCONSISTENT")))
print("\nRobustness scorecard (core specifications only):\n", SCORE.round(4).to_string())
SCORE.to_csv(OUT_DIR / "T7e_robustness_scorecard.csv")
for v, row in SCORE.iterrows():
    check(f"block '{v}': verdict", row.verdict in ("ROBUST", "NOT SUPPORTED"),
          f"{row.verdict}: significant in {int(row['p<0.05'])}/{int(row.core_specs_estimated)} core specs")
if len(FL):
    nfl = int(FL[FL.core].sign_flips_excl_setting.sum())
    nfl_set = int(FL[FL.core].sign_flips.sum()) - nfl
    check("no sign flips (|coef|>0.10) for age/sex/fund/ownership/specialty/time across core specs", nfl == 0,
          f"{nfl} flipped terms outside setting (see T7b)")
    print(f"INFO: {nfl_set} sign flip(s) in setting_cell terms - expected because setting is nearly nested with specialty "
          f"(D6); setting is interpreted only through its joint Wald test.")

# ----------------------------------------------------------------------------
# 8. ESTIMATOR CHECKS, HIGH COST-SHARING (STATE UNIVERSITY ONLY), AMOUNT
# ----------------------------------------------------------------------------
banner("[8] ESTIMATOR CHECKS, HIGH COST-SHARING MODELS, PATIENT PAYMENT IN RIAL")
chk, coefs = {}, {}
for lab, kind in [("Fractional logit (primary)", "flogit"), ("Fractional probit", "fprobit"), ("OLS on share", "ols")]:
    r = run(kind, drp, PRIM)
    chk[lab] = pd.Series({v: r["pvals"].get(v, np.nan) for v in PRIM})
    coefs[lab] = r["params"]
T8 = pd.DataFrame(chk)
T8.to_csv(OUT_DIR / "T8_estimator_checks_wald.csv")
print(T8.apply(lambda c: c.map(fmt)).to_string())
ref_c = coefs["Fractional logit (primary)"].drop("const")
nbig = 0
for lab in ("Fractional probit", "OLS on share"):
    c = coefs[lab].drop("const").reindex(ref_c.index)
    sc_ = c * (ref_c.abs().median() / c.abs().median())
    big = [i for i in ref_c.index if np.sign(ref_c[i]) != np.sign(c[i]) and abs(sc_[i]) > .1 and abs(ref_c[i]) > .1]
    nbig += len(big)
    print(f"  Sign disagreements logit vs {lab}: {len(big)}/{len(ref_c)}" + (f" -> {big[:8]}" if big else ""))
check("probit/OLS agree with logit on direction", nbig == 0, f"{nbig} disagreements")

# --- descriptive prevalence of thresholds by cell (all thresholds, descriptive only)
tab9 = drp.groupby("cell_so").apply(lambda g: pd.Series({
    "n": len(g), "mean %": g.oop_share.mean() * 100, **{f"% >= {int(t * 100)}": (g.oop_share >= t).mean() * 100
                                                         for t in (0.10, 0.20, 0.30, 0.50)}}))
tab9.to_csv(OUT_DIR / "T9a_threshold_prevalence_by_cell.csv")
print("\nShare of episodes above each threshold by setting x ownership cell (0 or 100 = rule-determined; DESCRIPTIVE ONLY):")
print(tab9[tab9.n >= 10].round(1).to_string())

# --- binary models only where shares vary: State University
su = prep(dr[dr["ownership_grp"] == SU])
_vcs = su["setting_cell"].value_counts()
su["setting_b"] = su["setting_cell"].where(su["setting_cell"].map(_vcs) >= 30, "Other settings (pooled)")
print(f"\nBinary models restricted to {SU} reconciled episodes (n={len(su)}); settings with n<30 pooled:")
print(su["setting_b"].value_counts().to_string())
hi_rows = []
for t in THRESHOLDS:
    thr = t - TOL
    su["hi"] = (su["oop_share"] >= thr).astype(int)
    n_mass = int(su["oop_share"].between(t - TOL, t + TOL).sum())
    prev_b = su.groupby("setting_b")["hi"].mean()
    keep_b = list(prev_b[(prev_b >= 0.05) & (prev_b <= 0.95)].index)
    drop_b = [x for x in prev_b.index if x not in keep_b]
    sub = su[su["setting_b"].isin(keep_b)].copy()
    print(f"\nHigh cost-sharing >= {t:.0%} (shares >= {thr:.1%} count as high; {n_mass} episodes lie within +-{TOL:.1%} of "
          f"the threshold = tariff mass point)")
    print(f"  prevalence by setting_b: {prev_b.round(3).to_dict()}")
    print(f"  settings excluded because the threshold is rule-determined there (<5% or >95%): {drop_b}")
    if len(keep_b) < 2:
        print("  fewer than 2 informative settings - model skipped")
        check(f"binary model >= {t:.0%} (State University) estimable", False, "fewer than 2 informative settings")
        continue
    try:
        rb = run("logit", sub, BIN_SU, ycol="hi")
        ev = min(sub["hi"].sum(), (1 - sub["hi"]).sum())
        pb = np.asarray(rb["res"].fittedvalues)
        extreme = float(((pb < .005) | (pb > .995)).mean())
        sep = bool((np.abs(rb["params"]).max() > 8) or (not np.isfinite(rb["res"].bse).all()) or (extreme > SEP_FRAC))
        epv = ev / rb["npar"]
        print(f"  n={rb['n']}, events={int(sub['hi'].sum())}, EPV={epv:.1f}, max|coef|={np.abs(rb['params']).max():.2f}, "
              f"extreme fitted share={extreme:.1%}, separation: {sep}")
        print("  joint Wald p:", {k: fmt(v) for k, v in rb["pvals"].items()})
        tt = clean(rb, f"Logit share >= {t:.0%} ({SU} only, informative settings only)")
        tt["separation_warning"] = sep
        tt["EPV"] = epv
        hi_rows.append(tt)
        check(f"binary model >= {t:.0%} (State University): no separation and EPV>={MIN_EPV}",
              (not sep) and epv >= MIN_EPV,
              f"n={rb['n']}, EPV={epv:.1f}, separation={sep}; EPV 5-10 = report as exploratory")
    except Exception as e:
        print(f"  high>={t:.0%} failed: {str(e)[:80]}")
        check(f"binary model >= {t:.0%} estimable", False, str(e)[:60])
if hi_rows:
    pd.concat(hi_rows).to_csv(OUT_DIR / "T9_high_cost_sharing_SU_only_odds_ratios.csv", index=False)
print("Report these as 'high episode-level patient cost-sharing within State University hospitals', never as "
      "'catastrophic health expenditure'. If a WARN is raised, report only the descriptive prevalence table.")

print("\nSECONDARY: fractional logit inside State University hospitals only (where shares vary; no ownership term):")
try:
    rsu = run("flogit", su, ["age_group", "sex", "time_m", "insurance_fund_grp", "setting_cell", "specialty_c"])
    print("  n =", rsu["n"], "| joint Wald p:", {k: fmt(v) for k, v in rsu["pvals"].items()})
    clean(rsu, "Fractional logit, State University only").to_csv(OUT_DIR / "T13_su_only_fractional_logit.csv", index=False)
    print(clean(rsu, "").drop(columns=["model", "scale"]).round(3).to_string(index=False))
except Exception as e:
    print("  failed:", str(e)[:100])

ra = run("qpois", drp, PRIM, ycol="oop_amt_m")
rb2 = run("qpois", drp, PRIM + ["log_cost"], ycol="oop_amt_m")
pd.concat([clean(ra, "SECONDARY: patient payment (million rial), quasi-Poisson log"),
           clean(rb2, "SECONDARY: same with log total cost")]).to_csv(OUT_DIR / "T10_patient_payment_amount.csv", index=False)
el = rb2["tab"][rb2["tab"].term == "log_cost"].iloc[0]
print(f"\nPatient payment (million rial): joint Wald p (no cost control) { {k: fmt(v) for k, v in ra['pvals'].items()} }")
print(f"Elasticity of patient payment w.r.t. total cost (adjusted): {np.log(el.estimate):.2f} "
      f"(95% CI {np.log(el.ci_low):.2f} to {np.log(el.ci_high):.2f}); 1 = proportional, >1 = share rises with cost")
m5 = blk[BLOCKS[4][0]]
print("\nM5 cost gradient on the logit scale (per unit of log total cost; OR>1 = share rises with cost):")
print(m5["tab"][m5["tab"].term.isin(["lc_su", "lc_ns"])].round(3).to_string(index=False))

# ----------------------------------------------------------------------------
# 9. ADJUSTED MEAN COST-SHARING (g-computation; FIXED-design cluster bootstrap)
# ----------------------------------------------------------------------------
banner("[9] ADJUSTED MEAN PATIENT COST-SHARING (percentage points; cluster-bootstrap CI)")
Xcf = {}
for v in ADJ_VARS:
    if v not in g0["spec"]["refs"]:
        continue
    for lvl in [g0["spec"]["refs"][v]] + g0["spec"]["levels"][v]:
        tmp = g0["data"].copy()
        tmp[v] = lvl
        Xcf[(v, lvl)] = make_X(tmp, PRIM, g0["spec"])[0].values


def adj_from_params(b):
    return {k: float(np.mean(1 / (1 + np.exp(-(X @ b))))) * 100 for k, X in Xcf.items()}


est = adj_from_params(np.asarray(g0["params"], dtype=float))
X0, y0 = g0["X"].values, g0["data"]["oop_share"].values.astype(float)
cc = pd.factorize(g0["data"]["cl"])[0]
grp = [np.where(cc == k)[0] for k in range(cc.max() + 1)]
rng = np.random.default_rng(SEED)
draws, n_missing_level = [], 0
lvl_idx = {k: (g0["data"][k[0]] == k[1]).values for k in Xcf}
for _ in range(N_BOOT_ADJ):
    idx = np.concatenate([grp[k] for k in rng.integers(0, len(grp), len(grp))])
    if any(not m[idx].any() for m in lvl_idx.values()):
        n_missing_level += 1                  # a level absent from the resample: its coefficient is not identified
        continue
    try:
        b = np.asarray(sm.GLM(y0[idx], X0[idx], family=sm.families.Binomial()).fit(maxiter=100).params, dtype=float)
        if np.all(np.isfinite(b)) and np.abs(b).max() < 15:
            draws.append(adj_from_params(b))
    except Exception:
        continue
valid = len(draws)
print(f"Valid bootstrap replicates: {valid}/{N_BOOT_ADJ} (replicates lacking a level: {n_missing_level})")
check("bootstrap: >=80% valid replicates", valid >= 0.8 * N_BOOT_ADJ, f"{valid}/{N_BOOT_ADJ}")
rows = []
for (v, lvl), e in est.items():
    ref = g0["spec"]["refs"][v]
    row = {"variable": v, "level": lvl, "n": int((g0["data"][v] == lvl).sum()),
           "crude_mean_%": g0["data"].loc[g0["data"][v] == lvl, "oop_share"].mean() * 100, "adjusted_mean_%": e}
    if valid >= 50:
        a = np.array([x[(v, lvl)] for x in draws])
        row["adj_ci_low"], row["adj_ci_high"] = np.percentile(a, [2.5, 97.5])
        if lvl != ref:
            dif = a - np.array([x[(v, ref)] for x in draws])
            row["diff_vs_ref_pp"] = e - est[(v, ref)]
            row["diff_ci_low"], row["diff_ci_high"] = np.percentile(dif, [2.5, 97.5])
    rows.append(row)
T11 = pd.DataFrame(rows)
T11.to_csv(OUT_DIR / "T11_adjusted_mean_share.csv", index=False)
print(T11.round(2).to_string(index=False))
print("Setting and ownership are NOT standardised over (impossible-cell extrapolation). Differences for age and fund are "
      "descriptive: the scorecard classifies both as not robust.")

# ----------------------------------------------------------------------------
# 10. RELIABILITY CHECKLIST AND REPORTING NOTES
# ----------------------------------------------------------------------------
banner("RELIABILITY CHECKLIST")
CK = pd.DataFrame(CHECKS, columns=["check", "status", "detail"])
print(CK.to_string(index=False))
CK.to_csv(OUT_DIR / "T12_reliability_checklist.csv", index=False)
print(f"\n{(CK.status == 'PASS').sum()} PASS / {(CK.status == 'WARN').sum()} WARN. A WARN is a statement for the "
      "limitations section / the reason a block is reported as 'not robust', not necessarily an error.")
banner("REPORTING NOTES")
print("""1. Outcome = 'episode-level patient cost-sharing (insured person's share / total episode cost)'; never
   'catastrophic expenditure'.
2. POPULATION LEVEL (T3c, T3e, T3f): recorded cost-sharing across all episodes is a LOWER bound. For private episodes
   the unreconciled remainder is plausibly patient-paid (gaps >20% in all 412 flagged; implied share inside the
   reconciled-private range; insurer share equal across groups). Report Scenario B with its CI, the bounds and the
   sensitivity grid; state that the patient-paid assumption cannot be tested with claims data. D-clinic implied shares
   rise with cost (E5) and differ in distribution from reconciled private episodes (KS p<0.001): treat D-clinic as the
   least certain part of the scenario.
3. Primary model estimates apply to reconciled episodes (|v3 gap| <= 5%, ~55%). Reconciliation inside private hospitals is
   determined by fund (T3g): fund contrasts cannot be estimated inside private hospitals, IPW violates positivity (T7 S10
   is descriptive only). Report fund as 'not robustly associated' and say why.
4. Cost-sharing is largely rule-based by ownership (private ~73%, non-academic government ~60%; State University varies
   by setting/specialty). Describe it as the benefit/tariff schedule, not behaviour. Private share is cost-independent
   among reconciled episodes (Spearman 0.12, p=0.17).
5. Robust blocks (T7e): ownership, setting (jointly), specialty (jointly). Sex and time: not supported. Age and fund:
   inconsistent. Age is confounded with ownership and specialty (0-4 y: 79% private, 61% obstetrics; 5-14 y: 61% ENT),
   so it cannot be separated from service mix.
6. Do NOT interpret individual setting_cell or specialty_c odds ratios (nearly nested, flip sign across specs).
7. M5 (cost gradient) is mechanical; no cost threshold exists inside State University (E7).
8. High cost-sharing models exist only inside State University hospitals, informative settings only, with a 0.5 pp
   tolerance. Sex and fund effects there persist after log-cost control but do not appear in the fractional model: report
   them as exploratory, threshold-specific findings. The >=20% model has EPV <10 -> exploratory. T13 (fractional logit,
   State University only) is the better description of within-State-University determinants.
9. VERIFY IN RAW DATA before submission: E00058, E00494, E01036, E00042 (State University, general surgery, 67-75% share,
   480-685 M rial: possible ownership mis-coding or non-covered surgery); E00050 and E01152 (zero share, single claim
   line, 241-359 M rial); the five zero-share episodes; T4g residuals.
10. Limitations: link test rejects the logit mean function (OOF fit identical for logit/probit/OLS); Pearson dispersion
   << 1; associations only (cross-sectional claims); unreconciled money not independently verified.
11. Never add s_gap, insurer_share, deduction shares or oop_share_of_known as predictors (outcome identity).""")
banner("DONE")
print(f"All outputs (tables + objective3_log.txt) in:\n{OUT_DIR}")

In [35]:
# =============================================================================
# OBJECTIVE 4 - DIAGNOSTICS (run AFTER Part 1 and Part 2; reads only P4_cohort_with_profiles.csv + P4_meta.json)
# Questions answered:
#  D1  Are the profiles driven by tied values / one data block (D-clinic spike, tariff mass points)?
#  D2  Are mixture components degenerate (variance pinned at the regularisation floor)?
#  D3  Does the BIC-selected K survive regularisation and "de-quantisation" (jitter of discrete variables)?
#  D4  How stable is each K (cluster-bootstrap ARI) and is there ANY cluster structure (silhouette)?
#  D5  Do the profiles just reproduce ownership / setting / one variable (ARI, NMI)? Is there structure INSIDE strata?
#  D6  Why did the membership model fail (empty cells, separation, collinearity)?
#  D7  Automatic verdict flags
# Output: P4_diag_*.csv + objective4_diagnostics_log.txt. No model is changed by this script.
# =============================================================================
import json
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score as ARI
from sklearn.metrics import normalized_mutual_info_score as NMI
from sklearn.metrics import silhouette_score
from sklearn.mixture import GaussianMixture

warnings.filterwarnings("ignore")

BASE = Path(r"C:\Users\Shekoofeh\Documents\Health Insurance Data\Cholera")
OUT_DIR = BASE / "feature engineering v4_FINAL" / "objective4_outputs"
for a in sys.argv[1:]:
    if not a.startswith("-") and Path(a).is_dir():
        OUT_DIR = Path(a)
COHORT, META = OUT_DIR / "P4_cohort_with_profiles.csv", OUT_DIR / "P4_meta.json"
assert COHORT.is_file() and META.is_file(), f"Run Part 1 first; files not found in {OUT_DIR}"


class Tee:
    def __init__(self, path, original):
        self.f = open(path, "w", encoding="utf-8")
        self.o = original

    def write(self, s):
        self.o.write(s)
        self.f.write(s)

    def flush(self):
        self.o.flush()
        self.f.flush()

    def __getattr__(self, name):
        return getattr(self.o, name)


if isinstance(sys.stdout, Tee):
    sys.stdout.f.close()
    sys.stdout = sys.stdout.o
sys.stdout = Tee(OUT_DIR / "objective4_diagnostics_log.txt", sys.stdout)
pd.set_option("display.width", 250, "display.max_columns", 40, "display.max_colwidth", 60)

SEED = 42
N_INIT = 10
N_BOOT = 40
REGS = (1e-3, 1e-2, 5e-2)
KS_FULL = list(range(1, 8))
MIN_CLASS = 0.05
MIN_ENT = 0.70
FLAGS = []


def banner(t):
    print("\n" + "=" * 90 + f"\n{t}\n" + "=" * 90)


def flag(name, bad, detail):
    FLAGS.append((name, "WARN" if bad else "OK", detail))


meta = json.loads(META.read_text())
d = pd.read_csv(COHORT, low_memory=False)
V, K, COV = meta["profile_vars"], int(meta["K"]), meta["cov"]
n = len(d)
d["profile"] = d["profile"].astype(int)
RES_COST = [v for v in V if v != "oop"]
print(f"Loaded {n:,} episodes | profile variables {V} | primary K={K}, covariance={COV}")


def zmat(X):
    X = np.asarray(X, float)
    return (X - X.mean(0)) / X.std(0)


def gmm(k, cov, reg, n_init=N_INIT, seed=SEED):
    return GaussianMixture(k, covariance_type=cov, n_init=n_init, reg_covar=reg, max_iter=500, tol=1e-4, random_state=seed)


def rel_entropy(P, k):
    return 1 - (-(P * np.log(np.clip(P, 1e-300, 1))).sum()) / (len(P) * np.log(k)) if k > 1 else np.nan


def jittered(rng):
    X = pd.DataFrame(index=d.index)
    for v in V:
        if v == "los_t":
            X[v] = np.log1p(d["los_days"] + rng.uniform(0, 1, n))
        elif v == "claims_t":
            X[v] = np.log1p(d["n_claim_lines"] + rng.uniform(0, 1, n))
        elif v == "serv_t":
            X[v] = np.log1p(d["n_service_groups"] + rng.uniform(0, 1, n))
        elif v == "oop":
            X[v] = (d["oop"] + rng.normal(0, 0.005, n)).clip(0, 1)
        else:
            X[v] = d[v]
    return X


Zp = zmat(d[V])
dclin = d["setting_cell"].str.contains("D Clinic", regex=False)

# ----------------------------------------------------------------------------
# D1. TIED VALUES AND THE D-CLINIC BLOCK
# ----------------------------------------------------------------------------
banner("[D1] TIED VALUES / MASS POINTS / D-CLINIC BLOCK")
key = d["los_days"].astype(str) + "|" + d["n_claim_lines"].astype(str) + "|" + d["oop"].round(3).astype(str)
vc = key.value_counts()
print(f"Distinct (LOS, claim lines, share@3dp) combinations: {len(vc)} of {n}; the 5 most frequent hold "
      f"{vc.iloc[:5].sum() / n:.1%} of episodes (top: {vc.iloc[0]} episodes)")
spike = (d["los_days"] == 0) & (d["n_claim_lines"] == 4) & (d["oop"] < 0.05)
print(f"Spike (LOS=0, 4 claim lines, recorded share <5%): n={int(spike.sum())} ({spike.mean():.1%}); D-clinic n={int(dclin.sum())} "
      f"({dclin.mean():.1%}); overlap={int((spike & dclin).sum())}")
sd_ratio = pd.DataFrame({"SD within D-clinic": d.loc[dclin, V].std(), "SD whole cohort": d[V].std(),
                         "ratio": d.loc[dclin, V].std() / d[V].std()})
print("\nVariability INSIDE the D-clinic block relative to the cohort (ratio near 0 = a point mass, not a 'profile'):")
print(sd_ratio.round(3).to_string())
ct = pd.crosstab(d["profile"], [dclin.rename("D-clinic"), d["gapflag"].rename("unreconciled")])
print("\nProfile x D-clinic x unreconciled:\n", ct.to_string())
pd.DataFrame({"n_distinct_combos": [len(vc)], "top5_share": [vc.iloc[:5].sum() / n], "spike_n": [int(spike.sum())],
              "dclinic_n": [int(dclin.sum())]}).to_csv(OUT_DIR / "P4_diag_D1_mass_points.csv", index=False)
sd_ratio.to_csv(OUT_DIR / "P4_diag_D1_dclinic_variability.csv")
share_dc_profile = (ct.xs(True, axis=1, level=0).sum(axis=1) / ct.sum(axis=1)).max()
flag("D1 one profile is (almost) the D-clinic block", share_dc_profile > 0.8,
     f"largest D-clinic share inside a single profile = {share_dc_profile:.0%}")
flag("D1 large tied-value spike", spike.mean() > 0.20, f"spike share = {spike.mean():.1%}")

# ----------------------------------------------------------------------------
# D2. DEGENERATE COMPONENTS
# ----------------------------------------------------------------------------
banner("[D2] COMPONENT DEGENERACY AT THE PRIMARY SETTING (reg_covar = 1e-3)")
g = gmm(K, COV, 1e-3).fit(Zp)
lp = g.score_samples(Zp)
lab = g.predict(Zp)
rows = []
for k in range(K):
    cv = g.covariances_[k]
    mn = np.linalg.eigvalsh(cv).min() if COV == "full" else np.min(cv)
    m = lab == k
    rows.append({"component": k, "weight": g.weights_[k], "n": int(m.sum()), "min_variance_or_eigenvalue": mn,
                 "at_floor(<=3*reg)": bool(mn <= 3e-3), "mean_logdensity": lp[m].mean() if m.any() else np.nan,
                 "max_logdensity": lp[m].max() if m.any() else np.nan})
T2 = pd.DataFrame(rows)
print(T2.round(4).to_string(index=False))
print(f"Total log-likelihood of the K={K} model = {lp.sum():.1f}; per-episode maximum log-density = {lp.max():.1f}")
print("Positive/very large log-densities and variances at the regularisation floor mean the component is fitted to "
      "tied values (a spike), which inflates the likelihood and BIC.")
T2.to_csv(OUT_DIR / "P4_diag_D2_components.csv", index=False)
flag("D2 components pinned at the variance floor", bool(T2["at_floor(<=3*reg)"].any()),
     f"{int(T2['at_floor(<=3*reg)'].sum())} of {K} components")

# ----------------------------------------------------------------------------
# D3. DOES THE CHOSEN K SURVIVE REGULARISATION AND DE-QUANTISATION?
# ----------------------------------------------------------------------------
banner("[D3] K SELECTION UNDER DIFFERENT REGULARISATION AND JITTER (K = 1..7)")
rng = np.random.default_rng(SEED)
Zj = zmat(jittered(rng))
rows = []
for jit, Z in (("none", Zp), ("de-quantised", Zj)):
    for reg in REGS:
        for cov in ("full", "diag"):
            for k in KS_FULL:
                gm = gmm(k, cov, reg).fit(Z)
                P = gm.predict_proba(Z)
                lb = P.argmax(1)
                rows.append({"jitter": jit, "reg": reg, "cov": cov, "K": k, "loglik": gm.score(Z) * n, "BIC": gm.bic(Z),
                             "rel_entropy": rel_entropy(P, k), "min_class": np.bincount(lb, minlength=k).min() / n})
T3 = pd.DataFrame(rows)
T3["admissible"] = (T3.min_class >= MIN_CLASS) & ((T3.rel_entropy >= MIN_ENT) | (T3.K == 1))
T3.to_csv(OUT_DIR / "P4_diag_D3_K_grid.csv", index=False)
best = []
for (jit, reg, cov), g_ in T3.groupby(["jitter", "reg", "cov"]):
    b_all = g_.loc[g_.BIC.idxmin()]
    a = g_[g_.admissible]
    b_adm = a.loc[a.BIC.idxmin()] if len(a) else None
    best.append({"jitter": jit, "reg": reg, "cov": cov, "K_best_BIC_any": int(b_all.K),
                 "K_best_BIC_admissible": int(b_adm.K) if b_adm is not None else np.nan,
                 "loglik_at_K7": float(g_[g_.K == 7].loglik.iloc[0]), "loglik_at_K1": float(g_[g_.K == 1].loglik.iloc[0])})
T3b = pd.DataFrame(best)
print(T3b.round(1).to_string(index=False))
T3b.to_csv(OUT_DIR / "P4_diag_D3_best_K.csv", index=False)
dq = T3b[(T3b.jitter == "de-quantised") & (T3b.reg >= 1e-2)]
print("\nReading: if K_best changes a lot across rows, or is 1 once the data are de-quantised and regularised, the selected K "
      "was produced by tied values, not by real groups.")
flag("D3 selected K not robust across settings", T3b["K_best_BIC_admissible"].nunique() > 2,
     f"admissible-best K takes values {sorted(T3b['K_best_BIC_admissible'].dropna().astype(int).unique())}")
flag("D3 de-quantised data prefer K<=2", bool((dq["K_best_BIC_admissible"] <= 2).any()),
     f"de-quantised, reg>=1e-2: {dq['K_best_BIC_admissible'].tolist()}")

# ----------------------------------------------------------------------------
# D4. STABILITY PER K AND EXISTENCE OF CLUSTER STRUCTURE
# ----------------------------------------------------------------------------
banner("[D4] STABILITY (cluster bootstrap ARI) PER K, AND SILHOUETTE")
cc = pd.factorize(d["cl"])[0]
groups = [np.where(cc == k)[0] for k in range(cc.max() + 1)]
rows = []
for nm, Z, reg in (("primary data, reg 1e-3", Zp, 1e-3), ("de-quantised, reg 1e-2", Zj, 1e-2)):
    for k in range(2, 7):
        base_lab = gmm(k, COV, reg).fit(Z).predict(Z)
        r = np.random.default_rng(SEED + k)
        aris = []
        for b in range(N_BOOT):
            idx = np.concatenate([groups[j] for j in r.integers(0, len(groups), len(groups))])
            try:
                aris.append(ARI(base_lab, gmm(k, COV, reg, 3, SEED + b).fit(Z[idx]).predict(Z)))
            except Exception:
                pass
        rows.append({"setting": nm, "K": k, "ARI_mean": np.mean(aris), "ARI_p2.5": np.percentile(aris, 2.5)})
T4 = pd.DataFrame(rows)
print(T4.round(3).to_string(index=False))
T4.to_csv(OUT_DIR / "P4_diag_D4_stability_by_K.csv", index=False)
rows = []
for nm, cols in (("all profile variables", V), ("resource + cost only", RES_COST), ("cost-sharing only", ["oop"])):
    Z = zmat(d[cols])
    for k in range(2, 9):
        lb = KMeans(k, n_init=10, random_state=SEED).fit_predict(Z)
        rows.append({"variables": nm, "K": k, "silhouette": silhouette_score(Z, lb)})
T4s = pd.DataFrame(rows)
print("\nk-means silhouette (>0.50 clear structure; 0.26-0.50 weak; <0.25 none):")
print(T4s.pivot(index="K", columns="variables", values="silhouette").round(3).to_string())
T4s.to_csv(OUT_DIR / "P4_diag_D4_silhouette.csv", index=False)
prim_rows = T4[(T4.setting == "primary data, reg 1e-3") & (T4.K == K)]
flag("D4 primary-K bootstrap ARI 2.5th pct < 0.60", float(prim_rows["ARI_p2.5"].iloc[0]) < 0.60,
     f"{float(prim_rows['ARI_p2.5'].iloc[0]):.2f}")
flag("D4 weak cluster structure (max silhouette, all variables < 0.50)",
     T4s[T4s.variables == "all profile variables"].silhouette.max() < 0.50,
     f"{T4s[T4s.variables == 'all profile variables'].silhouette.max():.2f}")

# ----------------------------------------------------------------------------
# D5. DO THE PROFILES JUST REPRODUCE OWNERSHIP / SETTING / ONE VARIABLE?
# ----------------------------------------------------------------------------
banner("[D5] PROFILES VS TARIFF STRUCTURE AND SINGLE VARIABLES; STRUCTURE INSIDE STRATA")
d["own_setting"] = d["ownership_grp"] + " @ " + d["setting_cell"]
rows = []
for nm in ("ownership_grp", "setting_cell", "own_setting", "specialty_c"):
    rows.append({"partition": nm, "ARI": ARI(d["profile"], d[nm]), "NMI": NMI(d["profile"], d[nm])})
for v in V:
    lb = KMeans(K, n_init=10, random_state=SEED).fit_predict(zmat(d[[v]]))
    rows.append({"partition": f"1-D k-means on {v}", "ARI": ARI(d["profile"], lb), "NMI": NMI(d["profile"], lb)})
T5 = pd.DataFrame(rows)
print(T5.round(3).to_string(index=False))
T5.to_csv(OUT_DIR / "P4_diag_D5_partition_agreement.csv", index=False)
flag("D5 profiles mirror ownership x setting (NMI > 0.60)", float(T5.loc[T5.partition == "own_setting", "NMI"].iloc[0]) > 0.60,
     f"NMI={float(T5.loc[T5.partition == 'own_setting', 'NMI'].iloc[0]):.2f}")

print("\nStructure INSIDE homogeneous strata (resource + cost variables, full covariance, reg 1e-2; BIC and silhouette):")
rows = []
strata = {"State University, reconciled": (d["ownership_grp"] == "State University") & (d["gapflag"] == 0),
          "all non-D-clinic": ~dclin,
          "inpatient, reconciled": (d["diagnosis_type_grp"] == "Hospitalization / Inpatient") & (d["gapflag"] == 0)}
for nm, m in strata.items():
    sub = d[m]
    if len(sub) < 80:
        continue
    Z = zmat(sub[RES_COST])
    bic = {k: gmm(k, "full", 1e-2).fit(Z).bic(Z) for k in range(1, 6)}
    kb = min(bic, key=bic.get)
    sil = {k: silhouette_score(Z, KMeans(k, n_init=10, random_state=SEED).fit_predict(Z)) for k in range(2, 6)}
    rows.append({"stratum": nm, "n": len(sub), "K_best_BIC": kb, "max_silhouette": max(sil.values()),
                 "BIC_by_K": {k: round(v, 1) for k, v in bic.items()}})
T5b = pd.DataFrame(rows)
print(T5b.to_string(index=False))
T5b.to_csv(OUT_DIR / "P4_diag_D5_within_strata.csv", index=False)

# ----------------------------------------------------------------------------
# D6. WHY THE MEMBERSHIP MODEL FAILED
# ----------------------------------------------------------------------------
banner("[D6] MEMBERSHIP-MODEL FEASIBILITY: EMPTY CELLS, SEPARATION, COLLINEARITY")
PRED = ["age_group", "sex", "admission_year", "insurance_fund_grp", "ownership_grp", "setting_cell", "specialty_c"]
rows, zero_cells = [], []
for v in PRED:
    ctv = pd.crosstab(d[v], d["profile"])
    chi = stats.chi2_contingency(ctv)[0]
    cramer = np.sqrt(chi / (ctv.values.sum() * (min(ctv.shape) - 1)))
    zc = [(lvl, int(p)) for lvl in ctv.index for p in ctv.columns if ctv.loc[lvl, p] == 0]
    sc = int((ctv.values < 5).sum())
    rows.append({"predictor": v, "levels": ctv.shape[0], "Cramers_V": cramer, "empty_cells": len(zc),
                 "cells_<5": sc, "total_cells": ctv.size})
    zero_cells += [{"predictor": v, "level": lvl, "profile": p} for lvl, p in zc]
T6 = pd.DataFrame(rows)
print(T6.round(3).to_string(index=False))
T6.to_csv(OUT_DIR / "P4_diag_D6_predictor_cells.csv", index=False)
pd.DataFrame(zero_cells).to_csv(OUT_DIR / "P4_diag_D6_empty_cells.csv", index=False)
print("\nEmpty (level, profile) cells = (quasi-)complete separation: the unpenalised multinomial logit cannot converge. "
      "First 25 empty cells:")
print(pd.DataFrame(zero_cells).head(25).to_string(index=False))
X = pd.get_dummies(d[PRED], drop_first=True).astype(float)
X.insert(0, "const", 1.0)
cond = np.linalg.cond(X.values)
npar = X.shape[1] * (K - 1)
epp = d["profile"].value_counts().min() / X.shape[1]
print(f"\nDesign: {X.shape[1]} columns, {npar} free parameters; condition number = {cond:.0f} (>1000 = serious collinearity); "
      f"smallest-profile events per design column = {epp:.1f} (<10 = unstable)")
print("Profile x ownership:\n", pd.crosstab(d["ownership_grp"], d["profile"]).to_string())
flag("D6 empty cells in membership predictors", len(zero_cells) > 0, f"{len(zero_cells)} empty level x profile cells")
flag("D6 events per design column < 10", epp < 10, f"{epp:.1f}")
flag("D6 collinear design (condition number > 1000)", cond > 1000, f"{cond:.0f}")

# ----------------------------------------------------------------------------
# D7. VERDICT
# ----------------------------------------------------------------------------
banner("[D7] AUTOMATIC VERDICT FLAGS")
F = pd.DataFrame(FLAGS, columns=["check", "status", "detail"])
print(F.to_string(index=False))
F.to_csv(OUT_DIR / "P4_diag_D7_flags.csv", index=False)
print(f"\n{(F.status == 'WARN').sum()} WARN / {(F.status == 'OK').sum()} OK. Send me the P4_diag_*.csv files (or the log) and I "
      "will adjust Part 1 and Part 2 accordingly.")
banner("DIAGNOSTICS DONE")

In [1]:
# =============================================================================
# OBJECTIVE 4 (FINAL, v5) - PART 1 of 2: EMPIRICAL PROFILES OF RESOURCE USE, EXPENDITURE AND PATIENT COST-SHARING
#
# WHAT CHANGED FROM v4 (every change answers a specific weakness found when the v4 output was audited):
#  C1  TARIFF-LIKE EPISODES. A near-fixed-payment "global package" (Global diagnosis type, non-academic government, 4 claim
#      lines, cost 53-56 M rial, OOP ~60%) sits inside the reconciled cohort and makes up a large part of Profile 2. A second,
#      data-driven definition flags episodes that sit in a tied (ownership x diagnosis type x claim lines x cost bin x OOP bin)
#      combination seen >= TIE_MIN times. Both flags are tabulated per profile (P4_tariff_like_by_profile.csv) and saved.
#  C2  S8 = re-clustering WITHOUT the global-package group; S9 = re-clustering WITHOUT any tariff-like episode.
#  C3  OUT-OF-SAMPLE CHECKS (answers "are 0.9 stability values real or in-sample?"):
#        a) bootstrap assignment stability is now also computed on OUT-OF-BAG episodes only (assign_stability_oob);
#        b) grouped k-fold HELD-OUT stability: standardisation + k-means fitted on training folds only, test folds assigned
#           to the nearest training centroid, ARI vs the full-data partition computed per fold (never pooled across folds);
#        c) NULL-STRUCTURE test: observed silhouette vs (i) independently column-permuted data and (ii) a multivariate
#           Gaussian continuum with the same correlation matrix. High stability alone does not prove clusters exist.
#  C4  Per-profile verdict now flags a profile that is stable on average but unreliable in some resamples (2.5th Jaccard < 0.5).
#  C5  K is reported as a chosen resolution: the K table also shows held-out ARI; coarser/finer solutions stay visible.
# Everything else (cohort rule R1, k-means primary, rule-based K, Ward / mixture checks, S1-S7) is unchanged from v4.
#
# Profile variables (NO thresholds, NO weights, NO index): log(1+LOS), log(1+claim lines), [log(1+service groups) only if NOT a
# near-duplicate of claim lines], log(1+total cost), oop_share. z-scores only equalise scales.
# Requires: numpy, pandas, scipy, scikit-learn.   Run:  python objective4_part1_v5.py  [optional: path\to\01_analysis_dataset.csv]
# Environment variables for quick tests: P4_N_BOOT (200), P4_SENS_BOOT (60), P4_N_INIT (30), P4_HELD_REPS (10), P4_NULL_B (100)
# =============================================================================
import json
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from scipy.cluster.hierarchy import fcluster, linkage
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score as ARI
from sklearn.metrics import silhouette_samples, silhouette_score
from sklearn.mixture import GaussianMixture

warnings.filterwarnings("ignore")

BASE = Path(r"C:\Users\Shekoofeh\Documents\Health Insurance Data\Cholera")
DEFAULT_INPUT = BASE / "feature engineering v4_FINAL" / "01_analysis_dataset.csv"


def resolve_input():
    for a in sys.argv[1:]:
        if not a.startswith("-") and Path(a).suffix.lower() == ".csv" and Path(a).is_file():
            return Path(a)
    return DEFAULT_INPUT


INPUT_FILE = resolve_input()
assert INPUT_FILE.is_file(), f"Input file not found: {INPUT_FILE}"
OUT_DIR = INPUT_FILE.parent / "objective4_outputs_v5"
OUT_DIR.mkdir(parents=True, exist_ok=True)


class Tee:
    def __init__(self, path, original):
        self.f = open(path, "w", encoding="utf-8")
        self.o = original

    def write(self, s):
        self.o.write(s)
        self.f.write(s)

    def flush(self):
        self.o.flush()
        self.f.flush()

    def __getattr__(self, name):
        return getattr(self.o, name)


if isinstance(sys.stdout, Tee):
    sys.stdout.f.close()
    sys.stdout = sys.stdout.o
sys.stdout = Tee(OUT_DIR / "objective4_part1_log.txt", sys.stdout)
pd.set_option("display.width", 250, "display.max_columns", 40, "display.max_colwidth", 60)
print(f"Input : {INPUT_FILE}\nOutput: {OUT_DIR}")

KS = list(range(2, 7))
N_INIT = int(os.environ.get("P4_N_INIT", 30))
N_BOOT = int(os.environ.get("P4_N_BOOT", 200))
SENS_BOOT = int(os.environ.get("P4_SENS_BOOT", 60))
HELD_REPS = int(os.environ.get("P4_HELD_REPS", 10))
NULL_B = int(os.environ.get("P4_NULL_B", 100))
SEED = 42
FINAL_K = None                     # integer ONLY to override the rule-based K after reviewing P4_K_selection.csv
MIN_CLASS_PROP = 0.05
MIN_CLASS_N = 30
JAC_STABLE = 0.75
JAC_RELAXED = 0.60
JAC_TAIL = 0.50                    # 2.5th-percentile Jaccard below this = unreliable in some resamples
SERV_DUP_THR = 0.95
MIN_LEVEL_N = 30
SPEC_MIN_N = 50
MIN_STRATUM_N = 20
TIE_MIN = 5                        # a tied tariff-like combination must contain >= TIE_MIN episodes
GLOBAL_DX = "Global"
GLOBAL_OWN = "Non-academic government"
GLOBAL_LINES = 4
GLOBAL_COST_LO, GLOBAL_COST_HI = 53e6, 56e6
INP = "Hospitalization / Inpatient"
BUNDLE_DEPT = "Limited surgery - D Clinic"
INSURANCE_MERGE = {"Other layers": "Other funds", "Other layers of the referral system": "Other funds",
                   "Other (rare)": "Other funds"}
DEPT_MAP = {"General emergency": "Emergency / triage", "General emergency and triage": "Emergency / triage",
            "Internal General": "Internal (general + emergency)", "Internal emergency": "Internal (general + emergency)",
            "Other (rare)": "Other departments"}
DESC_CATS = ["age_group", "sex", "insurance_fund_grp", "diagnosis_type_grp", "specialty_grp", "department_grp",
             "ownership_grp"]


def banner(t):
    print("\n" + "=" * 90 + f"\n{t}\n" + "=" * 90)


# ----------------------------------------------------------------------------
# 1. LOAD, VALIDATE, DEFINE COHORTS
# ----------------------------------------------------------------------------
banner("[1] COHORT AND DATA-QUALITY VALIDATION")
df = pd.read_csv(INPUT_FILE, low_memory=False)
print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns")


def flag(col):
    if col not in df.columns:
        return pd.Series(False, index=df.index)
    return pd.to_numeric(df[col], errors="coerce").fillna(0).eq(1)


need = ["los_days", "n_claim_lines", "n_service_groups", "oop_share", "log_total_cost", "analytic_primary",
        "Cluster_for_SE", "Episode_key", "flag_financing_gap_v3_requested"] + DESC_CATS
miss = [c for c in need if c not in df.columns]
assert not miss, f"Missing columns: {miss}"
for c in ["los_days", "n_claim_lines", "n_service_groups", "oop_share", "log_total_cost", "cost_per_day",
          "financing_gap_v3_pct"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

raw_col = next((c for c in ["Total Cost", "total_cost", "Total_Cost"] if c in df.columns), None)
if raw_col:
    df["total_cost"] = pd.to_numeric(df[raw_col], errors="coerce")
    print(f"Total cost taken from raw column '{raw_col}'")
else:
    df["total_cost"] = np.exp(df["log_total_cost"])
    if "cost_per_day" in df.columns:
        z = df["los_days"].eq(0) & df["cost_per_day"].gt(0)
        err = float(((df.loc[z, "total_cost"] - df.loc[z, "cost_per_day"]).abs() / df.loc[z, "cost_per_day"]).median())
        print(f"Total cost recovered as exp(log_total_cost); median relative error vs cost_per_day at LOS=0: {err:.2e}")
        if err > 1e-3:
            print("  WARNING: recovered cost does not match cost_per_day - verify the cost definition!")

mask0 = flag("analytic_primary")
if "complete_case_covariates" in df.columns:
    mask0 &= flag("complete_case_covariates")
base = df.loc[mask0]
VAL = pd.DataFrame({
    v: {"n": len(base), "missing": int(base[v].isna().sum()), "zeros": int((base[v] == 0).sum()),
        "negatives": int((base[v] < 0).sum()), "min": base[v].min(), "P99": base[v].quantile(.99), "max": base[v].max()}
    for v in ["los_days", "n_claim_lines", "n_service_groups", "total_cost", "oop_share"]}).T
print(VAL.round(3).to_string())
VAL.to_csv(OUT_DIR / "P4_validation_profile_variables.csv")
mask = mask0 & ~flag("flag_invalid_total_cost") & ~flag("flag_oop_out_of_range")
mask &= df["oop_share"].between(0, 1) & np.isfinite(df["total_cost"]) & (df["total_cost"] > 0)
dall = df.loc[mask].copy()
dall = dall.dropna(subset=["los_days", "n_claim_lines", "n_service_groups", "oop_share"] + DESC_CATS)
dall = dall[(dall["los_days"] >= 0) & (dall["n_claim_lines"] >= 1)].copy()
print(f"Valid episodes (only invalid records removed; extreme and zero-LOS episodes KEPT): {len(dall):,}")

for v in DESC_CATS:
    dall[v] = dall[v].astype(str).str.strip()
dall["insurance_fund_grp"] = dall["insurance_fund_grp"].replace(INSURANCE_MERGE)
for v in DESC_CATS:
    vc = dall[v].value_counts()
    rare = vc[vc < MIN_LEVEL_N].index
    if len(rare):
        print(f"  {v}: pooling {list(rare)} -> 'Other (pooled)'")
        dall.loc[dall[v].isin(rare), v] = "Other (pooled)"
_vc = dall["specialty_grp"].value_counts()
dall["specialty_c"] = dall["specialty_grp"].where(~dall["specialty_grp"].isin(_vc[_vc < SPEC_MIN_N].index),
                                                   "Other small specialties")
is_inp = dall["diagnosis_type_grp"].eq(INP)
dall["dept_in"] = BUNDLE_DEPT
_dep = dall.loc[is_inp, "department_grp"].replace(DEPT_MAP)
_cnt = _dep.value_counts()
dall.loc[is_inp, "dept_in"] = _dep.where(~_dep.isin(_cnt[_cnt < MIN_STRATUM_N].index), "Other departments")
dall["setting_label"] = np.where(is_inp, dall["dept_in"], "n/a (non-inpatient)")
dall["setting_cell"] = dall["diagnosis_type_grp"] + " | " + dall["setting_label"]
dall["dclinic"] = dall["setting_cell"].str.contains(BUNDLE_DEPT, regex=False)
dall["admission_year"] = dall["admission_year"].astype(float).astype(int).astype(str)
dall["cl"] = dall["Cluster_for_SE"].fillna(dall["Episode_key"]).astype(str)
dall["cost_m"] = dall["total_cost"] / 1e6
dall["gapflag"] = flag("flag_financing_gap_v3_requested").loc[dall.index].astype(int)
gap = (dall["financing_gap_v3_pct"] / 100).clip(lower=0) if "financing_gap_v3_pct" in dall.columns \
    else pd.Series(0.0, index=dall.index)
dall["oop"] = dall["oop_share"]
dall["oop_B"] = np.where((dall["ownership_grp"] == "private") & (dall["gapflag"] == 1),
                         np.clip(dall["oop_share"] + gap.fillna(0), 0, 1), dall["oop_share"])
dall["los_t"] = np.log1p(dall["los_days"])
dall["claims_t"] = np.log1p(dall["n_claim_lines"])
dall["serv_t"] = np.log1p(dall["n_service_groups"])
dall["cost_t"] = np.log1p(dall["total_cost"])

# ---- C1: tariff-like episodes (flagged on ALL valid episodes so the tie counts do not depend on the cohort)
dall["global_pkg"] = ((dall["diagnosis_type_grp"] == GLOBAL_DX) & (dall["ownership_grp"] == GLOBAL_OWN)
                      & (dall["n_claim_lines"] == GLOBAL_LINES) & dall["total_cost"].between(GLOBAL_COST_LO, GLOBAL_COST_HI))
_tie_key = (dall["ownership_grp"] + "|" + dall["diagnosis_type_grp"] + "|" + dall["n_claim_lines"].astype(int).astype(str) + "|"
            + (np.log(dall["total_cost"]) * 20).round(0).astype(int).astype(str) + "|" + (dall["oop"] * 50).round(0).astype(int).astype(str))
dall["tied_tariff"] = _tie_key.map(_tie_key.value_counts()).ge(TIE_MIN)
dall["tariff_like"] = dall["global_pkg"] | dall["tied_tariff"]

# ---- evidence for the primary-cohort rule (R1)
EV = (dall.groupby(["ownership_grp", "gapflag"])
      .agg(n=("oop", "size"), median_oop=("oop", "median"), mean_oop=("oop", "mean"), pct_los0=("los_days", lambda s: (s == 0).mean() * 100),
           pct_dclinic=("dclinic", lambda s: s.mean() * 100)).reset_index())
print("\nRecorded OOP share by ownership x reconciliation flag (gapflag=1 means financing does not reconcile):")
print(EV.round(3).to_string(index=False))
EV.to_csv(OUT_DIR / "P4_oop_reliability_evidence.csv", index=False)
print("If private-sector OOP differs by an order of magnitude between gapflag 0 and 1 the unreconciled OOP is a recording "
      "artefact; profiling it would reproduce reconciliation status (R1).")

dcl = dall[dall["dclinic"]]
if len(dcl):
    DC = pd.DataFrame({"n": [len(dcl)], "pct_of_valid": [len(dcl) / len(dall) * 100], "pct_unreconciled": [dcl.gapflag.mean() * 100],
                       "pct_private": [(dcl.ownership_grp == "private").mean() * 100], "pct_los0": [(dcl.los_days == 0).mean() * 100],
                       "median_claim_lines": [dcl.n_claim_lines.median()], "median_cost_m_rial": [dcl.cost_m.median()],
                       "IQR_cost_m": [f"{dcl.cost_m.quantile(.25):.1f}-{dcl.cost_m.quantile(.75):.1f}"],
                       "median_oop_recorded": [dcl.oop.median()]})
    print("\nD-clinic bundle (point-mass tariff group; described separately, not profiled in the primary analysis):")
    print(DC.round(3).to_string(index=False))
    DC.to_csv(OUT_DIR / "P4_dclinic_bundle_description.csv", index=False)

d = dall[dall["gapflag"] == 0].copy()
print(f"\nPRIMARY COHORT (financing-reconciled episodes): {len(d):,} of {len(dall):,} valid episodes; "
      f"D-clinic episodes inside it: {int(d['dclinic'].sum())}; clusters: {d['cl'].nunique():,}")
assert len(d) >= 300, "Primary cohort too small"

# ---- C1: describe the global package and the tied-combination group inside the cohort
gp = d[d["global_pkg"]]
print(f"\nGLOBAL-PACKAGE GROUP (Global dx, {GLOBAL_OWN}, {GLOBAL_LINES} claim lines, cost {GLOBAL_COST_LO/1e6:.0f}-{GLOBAL_COST_HI/1e6:.0f} M rial): "
      f"{len(gp)} episodes = {len(gp)/len(d):.1%} of the cohort")
if len(gp):
    print(f"  cost M rial: median {gp.cost_m.median():.2f}, IQR {gp.cost_m.quantile(.25):.2f}-{gp.cost_m.quantile(.75):.2f}, "
          f"CV {gp.cost_m.std()/gp.cost_m.mean():.3f}; OOP share: median {gp.oop.median():.3f}, SD {gp.oop.std():.4f}; "
          f"LOS=0 {(gp.los_days == 0).mean():.0%}")
    other_g = d[(d["diagnosis_type_grp"] == GLOBAL_DX) & ~d["global_pkg"]]
    print(f"  other 'Global'-diagnosis episodes in the cohort that fall outside the rule: {len(other_g)}")
print(f"TIED TARIFF-LIKE COMBINATIONS (>= {TIE_MIN} episodes sharing ownership x diagnosis type x claim lines x ~5% cost bin x 0.02 OOP bin): "
      f"{int(d['tied_tariff'].sum())} episodes ({d['tied_tariff'].mean():.1%}); either definition: {int(d['tariff_like'].sum())} ({d['tariff_like'].mean():.1%})")

# ----------------------------------------------------------------------------
# 2. DISTRIBUTIONS AND TIED-VALUE DIAGNOSTICS
# ----------------------------------------------------------------------------
banner("[2] DISTRIBUTIONS, TRANSFORMATIONS AND TIED-VALUE DIAGNOSTICS (primary cohort)")
same_share = float((d["n_service_groups"] == d["n_claim_lines"]).mean())
rho = stats.spearmanr(d["n_claim_lines"], d["n_service_groups"])[0]
KEEP_SERV = same_share < SERV_DUP_THR
print(f"n_service_groups == n_claim_lines in {same_share:.1%} of episodes (Spearman {rho:.3f}) -> "
      f"{'kept' if KEEP_SERV else 'EXCLUDED (near-duplicate of claim lines)'}")
RES = ["los_t", "claims_t"] + (["serv_t"] if KEEP_SERV else [])
PRIM_VARS = RES + ["cost_t", "oop"]
DIST = pd.DataFrame({v: {"skew_raw": stats.skew(raw), "skew_transformed": stats.skew(d[t]), "% zero": (raw == 0).mean() * 100}
                     for v, raw, t in [("los", d["los_days"], "los_t"), ("claims", d["n_claim_lines"], "claims_t"),
                                       ("services", d["n_service_groups"], "serv_t"), ("cost", d["total_cost"], "cost_t"),
                                       ("oop_share", d["oop"], "oop")]}).T
print(DIST.round(2).to_string())
DIST.to_csv(OUT_DIR / "P4_distributions.csv")
key = d[PRIM_VARS].round(2).astype(str).agg("|".join, axis=1)
kc = key.map(key.value_counts())
print(f"Tied-value check (variables rounded to 2 decimals): distinct combinations {key.nunique()}, "
      f"share of episodes in a combination seen >=10 times = {(kc >= 10).mean():.1%}, top-5 combinations = "
      f"{key.value_counts().iloc[:5].sum() / len(d):.1%}")
sdv = pd.Series({v: d[v].std() for v in PRIM_VARS})
print("SD of each profile variable (a very small SD relative to the others forces a variance-floor component):")
print(sdv.round(3).to_string())
print("Zero LOS is kept (log1p preserves it). z-scores only equalise scales; they are NOT an economic weighting.")


def matrix(data, vars_, mode="raw", oop_col="oop"):
    """mode='dequant': counts + U(0,1) before log1p, cost x exp(N(0,0.01)), OOP + U(-0.005,0.005) clipped to [0,1]."""
    n = len(data)
    jit = mode == "dequant"
    rng = np.random.default_rng(SEED + 7)

    def u():
        return rng.uniform(0, 1, n) if jit else 0.0

    T = {"los_t": np.log1p(data["los_days"].values + u()),
         "claims_t": np.log1p(data["n_claim_lines"].values + u()),
         "serv_t": np.log1p(data["n_service_groups"].values + u()),
         "cost_t": np.log1p(data["total_cost"].values * (np.exp(rng.normal(0, 0.01, n)) if jit else 1.0)),
         "oop": np.clip(data[oop_col].values + (rng.uniform(-.005, .005, n) if jit else 0.0), 0, 1)}
    return pd.DataFrame({v: T[v] for v in vars_}, index=data.index)


def winsorise(X):
    return X.clip(lower=X.quantile(.01), upper=X.quantile(.99), axis=1)


def make_Z(X, wins=False):
    X = X.astype(float).copy()
    if wins:
        X = winsorise(X)
    sd = X.std(ddof=0)
    assert (sd > 0).all(), f"zero-variance variable(s): {list(sd[sd == 0].index)}"
    return ((X - X.mean()) / sd).values


# ----------------------------------------------------------------------------
# 3. CLUSTERING ENGINE: k-means + cluster-bootstrap stability (in-bag AND out-of-bag) + held-out stability + rule-based K
# ----------------------------------------------------------------------------
def km_fit(Z, K, n_init=N_INIT, seed=SEED):
    return KMeans(K, n_init=n_init, random_state=seed).fit(Z)


def order_by_cost(lab, cost_t):
    K = lab.max() + 1
    means = np.array([cost_t[lab == k].mean() for k in range(K)])
    order = np.argsort(means)
    inv = np.empty(K, int)
    inv[order] = np.arange(K)
    return inv[lab], order


def boot_stability(Z, lab, K, grp, B, seed=SEED):
    """Cluster bootstrap (resample patient clusters). Each replicate is re-clustered and ALL episodes are assigned to the
    nearest replicate centroid. Returns per-profile Jaccard (B x K), bootstrap ARI (all episodes), OUT-OF-BAG ARI, and per-episode
    assignment stability computed (a) over all replicates and (b) over replicates in which the episode was OUT-OF-BAG."""
    rng = np.random.default_rng(seed)
    G = [np.where(grp == k)[0] for k in range(grp.max() + 1)]
    J, A, A_oob = [], [], []
    ag_all, ag_oob, n_oob = np.zeros(len(Z)), np.zeros(len(Z)), np.zeros(len(Z))
    nb = 0
    for _ in range(B):
        idx = np.concatenate([G[k] for k in rng.integers(0, len(G), len(G))])
        inbag = np.zeros(len(Z), bool)
        inbag[idx] = True
        try:
            cent = KMeans(K, n_init=5, random_state=int(rng.integers(1e9))).fit(Z[idx]).cluster_centers_
        except Exception:
            continue
        pred = ((Z[:, None, :] - cent[None]) ** 2).sum(2).argmin(1)
        jj, match = [], np.zeros(K, int)
        for k in range(K):
            A_k = lab == k
            sc = [((A_k) & (pred == j)).sum() / max(((A_k) | (pred == j)).sum(), 1) for j in range(K)]
            match[k] = int(np.argmax(sc))
            jj.append(max(sc))
        J.append(jj)
        A.append(ARI(lab, pred))
        oob = ~inbag
        if oob.sum() > 10:
            A_oob.append(ARI(lab[oob], pred[oob]))
        hit = (pred == match[lab])
        ag_all += hit
        ag_oob += hit & oob
        n_oob += oob
        nb += 1
    stab_all = ag_all / max(nb, 1)
    stab_oob = np.where(n_oob > 0, ag_oob / np.maximum(n_oob, 1), np.nan)
    return np.array(J), np.array(A), np.array(A_oob), stab_all, stab_oob


def heldout_stability(X, lab, K, grp, reps=HELD_REPS, nfold=5, seed=SEED, wins=False):
    """GROUPED k-fold hold-out. Standardisation and k-means are fitted on TRAINING folds only; each test fold is assigned to the
    nearest training centroid. ARI against the full-data partition is computed PER FOLD (labels are arbitrary between fits, so
    predictions from different folds must never be pooled). Also returns the held-out silhouette on the training scale."""
    rng = np.random.default_rng(seed)
    X = X.astype(float)
    if wins:
        X = winsorise(X)
    grp = np.asarray(grp)
    a_list, s_list = [], []
    for r in range(reps):
        fold = rng.permutation(grp.max() + 1)[grp] % nfold
        for f in range(nfold):
            tr, te = fold != f, fold == f
            if te.sum() < 20 or len(np.unique(lab[te])) < 2:
                continue
            mu, sd = X[tr].mean(), X[tr].std(ddof=0)
            if (sd == 0).any():
                continue
            Ztr, Zte = ((X[tr] - mu) / sd).values, ((X[te] - mu) / sd).values
            km = KMeans(K, n_init=10, random_state=int(rng.integers(1e9))).fit(Ztr)
            pred = km.predict(Zte)
            a_list.append(ARI(lab[te], pred))
            if len(np.unique(pred)) > 1:
                s_list.append(silhouette_score(Zte, pred))
    a = np.array(a_list)
    return {"heldout_ARI_mean": float(a.mean()) if len(a) else np.nan,
            "heldout_ARI_p2.5": float(np.percentile(a, 2.5)) if len(a) else np.nan,
            "heldout_silhouette": float(np.mean(s_list)) if s_list else np.nan}


def select_K(Z, grp, B, X=None, reps=HELD_REPS, wins=False, seed=SEED):
    rows = []
    for K in KS:
        km = km_fit(Z, K, seed=seed)
        lab = km.labels_
        sz = np.bincount(lab, minlength=K)
        J, A, A_oob, _, _ = boot_stability(Z, lab, K, grp, B, seed=seed + K)
        r = {"K": K, "silhouette": silhouette_score(Z, lab), "min_profile_n": int(sz.min()),
             "min_profile_prop": sz.min() / len(Z), "mean_Jaccard": J.mean(), "min_profile_Jaccard": J.mean(0).min(),
             "boot_ARI_mean": A.mean(), "boot_ARI_p2.5": np.percentile(A, 2.5),
             "oob_ARI_mean": A_oob.mean() if len(A_oob) else np.nan, "inertia": km.inertia_}
        if X is not None:
            h = heldout_stability(X, lab, K, grp, reps=reps, wins=wins, seed=seed + 50 + K)
            r.update({"heldout_ARI_mean": h["heldout_ARI_mean"], "heldout_silhouette": h["heldout_silhouette"]})
        rows.append(r)
    T = pd.DataFrame(rows)
    size_ok = (T.min_profile_prop >= MIN_CLASS_PROP) & (T.min_profile_n >= MIN_CLASS_N)
    note = f"all profiles stable (min mean Jaccard >= {JAC_STABLE})"
    ok = T[size_ok & (T.min_profile_Jaccard >= JAC_STABLE)]
    if ok.empty:
        ok = T[size_ok & (T.min_profile_Jaccard >= JAC_RELAXED)]
        note = f"NO solution with all profiles >= {JAC_STABLE}; relaxed to {JAC_RELAXED} (report as limitation)"
    if ok.empty:
        ok, note = T[size_ok] if size_ok.any() else T, "NO solution meets the size/stability rules - interpret with great caution"
    best = ok.sort_values(["silhouette", "K"], ascending=[False, True]).iloc[0]
    T["eligible"] = T.K.isin(ok.K)
    return T, int(best["K"]), note


def null_structure(X, K, B=NULL_B, seed=SEED):
    """Is the observed silhouette larger than what k-means finds in data WITHOUT cluster structure?
    null 1: each column independently permuted (keeps every marginal incl. ties, destroys joint structure);
    null 2: multivariate Gaussian continuum with the same correlation matrix (no clusters by construction)."""
    rng = np.random.default_rng(seed)
    Z = make_Z(X)
    obs = silhouette_score(Z, km_fit(Z, K, n_init=10).labels_)
    C = np.corrcoef(Z.T)
    s1, s2 = [], []
    for _ in range(B):
        Zp_ = np.column_stack([rng.permutation(Z[:, j]) for j in range(Z.shape[1])])
        s1.append(silhouette_score(Zp_, km_fit(Zp_, K, n_init=5, seed=int(rng.integers(1e9))).labels_))
        Zg = rng.multivariate_normal(np.zeros(Z.shape[1]), C, len(Z))
        s2.append(silhouette_score(Zg, km_fit(Zg, K, n_init=5, seed=int(rng.integers(1e9))).labels_))
    return {"n": len(Z), "K": K, "obs_silhouette": obs,
            "perm_null_mean": float(np.mean(s1)), "perm_null_p97.5": float(np.percentile(s1, 97.5)),
            "perm_p": (1 + sum(x >= obs for x in s1)) / (B + 1),
            "gauss_null_mean": float(np.mean(s2)), "gauss_null_p97.5": float(np.percentile(s2, 97.5)),
            "gauss_p": (1 + sum(x >= obs for x in s2)) / (B + 1)}


# ----------------------------------------------------------------------------
# 4. K SELECTION AND PRIMARY MODEL
# ----------------------------------------------------------------------------
banner("[4] RULE-BASED K SELECTION AND PRIMARY MODEL (k-means on standardised profile variables)")
Xp = matrix(d, PRIM_VARS)
Zp = make_Z(Xp)
grp_p = pd.factorize(d["cl"])[0]
TK, K_rule, note_k = select_K(Zp, grp_p, N_BOOT, X=Xp)
print(TK.round(3).to_string(index=False))
print(f"\nRule-based choice: K = {K_rule} [{note_k}]")
print("K is a CHOSEN RESOLUTION, not a discovered number of types: report the K=2/3 (coarser) and K=5/6 (finer) rows alongside it.")
print("heldout_ARI_mean = agreement with the full-data partition when the model is fitted WITHOUT the test fold (out-of-sample).")
K0 = int(FINAL_K) if FINAL_K is not None else K_rule
if FINAL_K is not None:
    print(f"OVERRIDE in force: K = {K0}")
TK.to_csv(OUT_DIR / "P4_K_selection.csv", index=False)
print("Silhouette guide: <0.25 no substantial structure; 0.25-0.50 weak/overlapping; >0.50 reasonable.")

n = len(d)
km0 = km_fit(Zp, K0)
lab0, order_k = order_by_cost(km0.labels_, d["cost_t"].values)
d["profile"] = lab0 + 1
print("\nProfile sizes:", {int(k): int(v) for k, v in d["profile"].value_counts().sort_index().items()})
print(d.groupby("profile")[["los_days", "n_claim_lines", "cost_m", "oop"]].median().round(3).to_string())

J, A, A_oob, agree_all, agree_oob = boot_stability(Zp, lab0, K0, grp_p, N_BOOT, seed=SEED + 99)
sil_i = silhouette_samples(Zp, lab0)
sil0 = float(sil_i.mean())
d["assign_stability"] = agree_all
d["assign_stability_oob"] = agree_oob
d["silhouette_i"] = sil_i
JT = pd.DataFrame({"profile": np.arange(1, K0 + 1), "n": np.bincount(lab0, minlength=K0),
                   "silhouette": [sil_i[lab0 == k].mean() for k in range(K0)],
                   "mean_Jaccard": J.mean(0),
                   "p2.5_Jaccard": np.percentile(J, 2.5, axis=0),
                   "mean_assignment_stability": [agree_all[lab0 == k].mean() for k in range(K0)],
                   "oob_assignment_stability": [np.nanmean(agree_oob[lab0 == k]) for k in range(K0)]})
JT["verdict"] = np.where(JT.mean_Jaccard >= JAC_STABLE,
                         np.where(JT["p2.5_Jaccard"] >= JAC_TAIL, "stable",
                                  "stable on average, UNRELIABLE in some resamples"),
                         np.where(JT.mean_Jaccard >= JAC_RELAXED, "pattern but uncertain", "NOT stable"))
print(f"\nSilhouette (final partition) = {sil0:.3f}; cluster bootstrap ({len(A)} replicates): ARI mean {A.mean():.3f}, "
      f"2.5-97.5% [{np.percentile(A, 2.5):.3f}, {np.percentile(A, 97.5):.3f}]; OUT-OF-BAG ARI mean {A_oob.mean():.3f}, "
      f"2.5% {np.percentile(A_oob, 2.5):.3f}")
print("Per-profile stability (Hennig: Jaccard >= 0.75 stable, 0.60-0.75 uncertain, < 0.60 unreliable; "
      f"tail rule: 2.5th-percentile Jaccard < {JAC_TAIL} = unreliable in some resamples):")
print(JT.round(3).to_string(index=False))
JT.to_csv(OUT_DIR / "P4_profile_stability.csv", index=False)
print(f"Share of episodes whose profile is reproduced in >= 80% of ALL bootstrap replicates: {(agree_all >= 0.8).mean():.1%}; "
      f"in >= 80% of the replicates where they were OUT-OF-BAG: {np.nanmean(agree_oob >= 0.8):.1%}")

HO = heldout_stability(Xp, lab0, K0, grp_p, reps=HELD_REPS, seed=SEED + 3)
print(f"\nHELD-OUT (grouped 5-fold x {HELD_REPS}, train-only standardisation + k-means): ARI vs full-data partition "
      f"mean {HO['heldout_ARI_mean']:.3f} (2.5% {HO['heldout_ARI_p2.5']:.3f}); held-out silhouette {HO['heldout_silhouette']:.3f}")
print("Reading: these are OUT-OF-SAMPLE reproducibility figures. They show the partition is not a fragile in-sample fit; they do "
      "NOT show that the profiles are natural groups (see null test and tariff-like table).")

NS = null_structure(Xp, K0)
print(f"\nNULL-STRUCTURE TEST (K={K0}, {NULL_B} null datasets each): observed silhouette {NS['obs_silhouette']:.3f}")
print(f"  column-permuted null: mean {NS['perm_null_mean']:.3f}, 97.5th pct {NS['perm_null_p97.5']:.3f}, p = {NS['perm_p']:.3f}")
print(f"  Gaussian-continuum null (same correlations): mean {NS['gauss_null_mean']:.3f}, 97.5th pct {NS['gauss_null_p97.5']:.3f}, "
      f"p = {NS['gauss_p']:.3f}")
nt_rows = d.index[~d["tariff_like"]]
NS2 = null_structure(Xp.loc[nt_rows], K0, seed=SEED + 5) if len(nt_rows) > 150 else None
if NS2:
    print(f"  SAME TEST AFTER REMOVING TARIFF-LIKE EPISODES (n={NS2['n']}): observed silhouette {NS2['obs_silhouette']:.3f}; "
          f"permuted null mean {NS2['perm_null_mean']:.3f} (p = {NS2['perm_p']:.3f}); Gaussian null mean {NS2['gauss_null_mean']:.3f} "
          f"(p = {NS2['gauss_p']:.3f})")
    print("  If the silhouette collapses toward the Gaussian null here, the typology is carried by tariff point-masses.")
pd.DataFrame([dict(NS, sample="primary cohort")] + ([dict(NS2, sample="excluding tariff-like")] if NS2 else [])).to_csv(
    OUT_DIR / "P4_null_structure.csv", index=False)

best_in = km0.inertia_
inert = np.array([KMeans(K0, n_init=1, random_state=SEED + 1000 + i).fit(Zp).inertia_ for i in range(30)])
print(f"\nReproducibility: {(inert <= best_in * 1.001).mean():.0%} of 30 single-start k-means fits reach the best solution "
      f"(within 0.1% of inertia).")
if sil0 < 0.25:
    STR = "NO SUBSTANTIAL STRUCTURE"
elif (JT.mean_Jaccard < JAC_STABLE).any():
    STR = "WEAK / PARTLY UNSTABLE"
elif sil0 < 0.50:
    STR = "WEAK-TO-MODERATE (profiles reproducible but overlapping)"
else:
    STR = "ADEQUATE"
weak = JT.loc[JT["p2.5_Jaccard"] < JAC_TAIL, "profile"].tolist()
if weak:
    STR += f"; profile(s) {weak} unreliable in some resamples"
print(f"\nSTRUCTURE VERDICT: {STR}")
print("-> Whatever the verdict, call the profiles an exploratory typology of a continuum: they summarise the main combinations "
      "of resource use, expenditure and cost-sharing; they are not discrete patient types and carry no efficiency/value meaning.")
pd.DataFrame({"K": [K0], "silhouette": [sil0], "boot_ari_mean": [A.mean()], "boot_ari_p2.5": [np.percentile(A, 2.5)],
              "oob_ari_mean": [A_oob.mean()], "heldout_ari_mean": [HO["heldout_ARI_mean"]],
              "min_profile_Jaccard": [JT.mean_Jaccard.min()], "structure_verdict": [STR]}).to_csv(
    OUT_DIR / "P4_stability_primary.csv", index=False)

# ---- C1: share of each profile made up of tariff-like episodes
TL = d.groupby("profile").agg(n=("profile", "size"), n_global_pkg=("global_pkg", "sum"), n_tied_combo=("tied_tariff", "sum"),
                              n_tariff_like=("tariff_like", "sum")).reset_index()
TL.loc[len(TL)] = ["All", len(d), int(d.global_pkg.sum()), int(d.tied_tariff.sum()), int(d.tariff_like.sum())]
for c_, nm in [("n_global_pkg", "% global package"), ("n_tied_combo", "% tied tariff combination"),
               ("n_tariff_like", "% tariff-like (either)")]:
    TL[nm] = TL[c_] / TL["n"] * 100
print(f"\nTARIFF-LIKE EPISODES BY PROFILE (global package = rule above; tied combination = >= {TIE_MIN} episodes in the same "
      "ownership x diagnosis type x claim lines x cost bin x OOP bin):")
print(TL.round(1).to_string(index=False))
TL.to_csv(OUT_DIR / "P4_tariff_like_by_profile.csv", index=False)
print("A profile with a large tariff-like share is partly a billing-package artefact, not a free combination of use, cost and cost-sharing.")

# ----------------------------------------------------------------------------
# 5. CO-PRIMARY / SENSITIVITY METHODS AT THE SAME K
# ----------------------------------------------------------------------------
banner("[5] CROSS-METHOD CHECKS AT THE SAME K (Ward; Gaussian mixture = sensitivity only)")
d["lab_ward"] = fcluster(linkage(Zp, "ward"), K0, "maxclust") - 1
a_w = ARI(d["profile"], d["lab_ward"])
print(f"Ward vs k-means: ARI = {a_w:.3f} (>=0.6 broadly consistent); sizes {d['lab_ward'].value_counts().sort_index().tolist()}")
print(pd.crosstab(d["profile"], d["lab_ward"]).to_string())
gm = GaussianMixture(K0, covariance_type="full", n_init=N_INIT, reg_covar=0.05, max_iter=500, random_state=SEED).fit(Zp)
d["lab_gmm"] = gm.predict(Zp)
mvv = np.array([np.linalg.eigvalsh(m).min() for m in gm.covariances_])
a_g = ARI(d["profile"], d["lab_gmm"])
print(f"\nGaussian mixture (full covariance, floor 0.05) at K={K0}: ARI vs k-means = {a_g:.3f}; components at the floor: "
      f"{int((mvv <= 0.15).sum())}/{K0}; sizes {np.bincount(d['lab_gmm'], minlength=K0).tolist()}")
print("The mixture is reported ONLY as a sensitivity analysis: on discrete, mass-pointed log1p/OOP variables its likelihood "
      "collapses onto tied values, so it is not used for selection or inference.")
pd.DataFrame([{"method": "ward", "ARI_vs_primary": a_w}, {"method": "gaussian_mixture_sensitivity", "ARI_vs_primary": a_g}]
             ).to_csv(OUT_DIR / "P4_alternative_methods.csv", index=False)

# ----------------------------------------------------------------------------
# 6. SENSITIVITY SPECIFICATIONS (re-clustered; ARI on rows shared with the primary cohort)
# ----------------------------------------------------------------------------
banner("[6] SENSITIVITY: DOES THE PROFILE STRUCTURE CHANGE?")
all_idx = dall.index
SPECS = {
    "S1_resource_only": dict(vars=RES, rows=d.index),
    "S2_resource_plus_cost": dict(vars=RES + ["cost_t"], rows=d.index),
    "S3_all_valid_episodes_incl_unreconciled": dict(vars=PRIM_VARS, rows=all_idx),
    "S4_all_valid_scenarioB_cost_sharing": dict(vars=PRIM_VARS, rows=all_idx, oop_col="oop_B"),
    "S5_winsorised_P1_P99": dict(vars=PRIM_VARS, rows=d.index, wins=True),
    "S6_all_valid_excl_Dclinic": dict(vars=PRIM_VARS, rows=dall.index[~dall["dclinic"]]),
    "S7_dequantised": dict(vars=PRIM_VARS, rows=d.index, mode="dequant"),
    "S8_excl_global_package": dict(vars=PRIM_VARS, rows=d.index[~d["global_pkg"]], show_xtab=True),
    "S9_excl_all_tariff_like": dict(vars=PRIM_VARS, rows=d.index[~d["tariff_like"]], show_xtab=True),
}
SENS = []
for nm, sp in SPECS.items():
    dd = dall.loc[sp["rows"]]
    Xs = matrix(dd, sp["vars"], sp.get("mode", "raw"), sp.get("oop_col", "oop"))
    Z = make_Z(Xs, sp.get("wins", False))
    g_ = pd.factorize(dd["cl"])[0]
    T_s, K_s, note_s = select_K(Z, g_, SENS_BOOT, X=Xs, reps=max(3, HELD_REPS // 3), wins=sp.get("wins", False))
    lab_auto = pd.Series(km_fit(Z, K_s).labels_, index=dd.index)
    lab_fix = pd.Series(km_fit(Z, K0).labels_, index=dd.index)
    common = d.index.intersection(dd.index)
    row = {"spec": nm, "n": len(dd), "n_shared_with_primary": len(common), "auto_K": K_s,
           "silhouette_autoK": float(T_s.loc[T_s.K == K_s, "silhouette"].iloc[0]),
           "min_profile_Jaccard_autoK": float(T_s.loc[T_s.K == K_s, "min_profile_Jaccard"].iloc[0]),
           "heldout_ARI_autoK": float(T_s.loc[T_s.K == K_s, "heldout_ARI_mean"].iloc[0]),
           "silhouette_fixedK": float(silhouette_score(Z, lab_fix)),
           "ARI_fixedK_vs_primary": ARI(d.loc[common, "profile"], lab_fix.loc[common]),
           "ARI_autoK_vs_primary": ARI(d.loc[common, "profile"], lab_auto.loc[common]), "note": note_s}
    SENS.append(row)
    print(f"  {nm}: n={len(dd)}, rule-K={K_s}, silhouette {row['silhouette_autoK']:.2f}, min Jaccard "
          f"{row['min_profile_Jaccard_autoK']:.2f}, held-out ARI {row['heldout_ARI_autoK']:.2f}; fixed-K={K0}: silhouette "
          f"{row['silhouette_fixedK']:.2f}, ARI vs primary = {row['ARI_fixedK_vs_primary']:.3f}; auto-K ARI = {row['ARI_autoK_vs_primary']:.3f}")
    if sp.get("show_xtab"):
        print(f"    primary profile (rows) x {nm} fixed-K cluster (columns; labels arbitrary), episodes shared with the primary cohort:")
        print(pd.crosstab(d.loc[common, "profile"], lab_fix.loc[common]).to_string().replace("\n", "\n    "))
        rest = pd.DataFrame({"profile": d.loc[common, "profile"], "lab": lab_fix.loc[common]})
        print(f"    episodes retained from each primary profile: " +
              ", ".join(f"P{k}: {int((d.loc[common, 'profile'] == k).sum())}/{int((d.profile == k).sum())}" for k in range(1, K0 + 1)))
pd.DataFrame(SENS).to_csv(OUT_DIR / "P4_sensitivity_summary.csv", index=False)
print("\nReading guide: S1/S2 = effect of adding cost / cost-sharing; S3 = adding the UNRECONCILED episodes (expect structure "
      "driven by reconciliation/ownership - the reason the primary cohort excludes them); S4 = Scenario-B cost-sharing on all "
      "episodes; S5 = outliers; S6 = all episodes without the D-clinic bundle; S7 = tied-value artefact; S8 = without the "
      "government global package; S9 = without ANY tariff-like episode. ARI > 0.8 same partition; 0.5-0.8 broadly similar; "
      "< 0.5 different. K differs across specifications: report K as a resolution, not a finding.")

# ----------------------------------------------------------------------------
# 7. HAND-OFF TO PART 2
# ----------------------------------------------------------------------------
banner("[7] SAVE HAND-OFF FILE")
keep = (["Episode_key", "cl", "age_group", "sex", "insurance_fund_grp", "ownership_grp", "diagnosis_type_grp", "specialty_c",
         "department_grp", "dept_in", "setting_cell", "dclinic", "admission_year", "los_days", "n_claim_lines",
         "n_service_groups", "total_cost", "cost_m", "oop", "oop_B", "gapflag", "los_t", "claims_t", "serv_t", "cost_t",
         "global_pkg", "tied_tariff", "tariff_like",
         "profile", "assign_stability", "assign_stability_oob", "silhouette_i", "lab_ward", "lab_gmm"])
out = d[keep].copy()
out.insert(0, "orig_index", d.index)
out.to_csv(OUT_DIR / "P4_cohort_with_profiles.csv", index=False)
meta = {"K": int(K0), "method": "k-means (standardised profile variables), rule-based K, cluster-bootstrap Jaccard",
        "profile_vars": PRIM_VARS, "services_in_profile": bool(KEEP_SERV), "n": int(len(d)), "n_valid_all": int(len(dall)),
        "cohort": "financing-reconciled (flag_financing_gap_v3_requested == 0)", "input_file": str(INPUT_FILE),
        "silhouette": sil0, "min_profile_Jaccard": float(JT.mean_Jaccard.min()), "structure_verdict": STR,
        "heldout_ari_mean": HO["heldout_ARI_mean"], "global_pkg_n": int(d["global_pkg"].sum()),
        "tariff_like_n": int(d["tariff_like"].sum())}
(OUT_DIR / "P4_meta.json").write_text(json.dumps(meta, indent=2))
print(f"Saved {len(out):,} episodes, K={K0}, profile variables={PRIM_VARS}")
print("NEXT: review P4_K_selection.csv, P4_profile_stability.csv, P4_tariff_like_by_profile.csv, P4_null_structure.csv, the "
      "STRUCTURE VERDICT and the sensitivity ARIs; then run objective4_part2_v5.py.")
banner("PART 1 DONE")

Input : C:\Users\Shekoofeh\Documents\Health Insurance Data\Cholera\feature engineering v4_FINAL\01_analysis_dataset.csv
Output: C:\Users\Shekoofeh\Documents\Health Insurance Data\Cholera\feature engineering v4_FINAL\objective4_outputs_v5

[1] COHORT AND DATA-QUALITY VALIDATION
Loaded 1,171 rows x 103 columns
Total cost recovered as exp(log_total_cost); median relative error vs cost_per_day at LOS=0: 7.31e-16
                       n  missing  zeros  negatives       min           P99           max
los_days          1171.0      0.0  710.0        0.0       0.0  2.550000e+01  3.350000e+02
n_claim_lines     1171.0      0.0    0.0        0.0       1.0  1.800000e+01  2.000000e+01
n_service_groups  1171.0      0.0    0.0        0.0       1.0  1.800000e+01  2.000000e+01
total_cost        1171.0      0.0    0.0        0.0  190760.0  6.485628e+08  2.046654e+09
oop_share         1171.0      0.0    7.0        0.0       0.0  7.810000e-01  8.310000e-01
Valid episodes (only invalid records removed; ex

  File "c:\Users\Shekoofeh\AppData\Local\Programs\Python\Python312\Lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Shekoofeh\AppData\Local\Programs\Python\Python312\Lib\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\Shekoofeh\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Shekoofeh\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\Shekoofeh\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid

 K  silhouette  min_profile_n  min_profile_prop  mean_Jaccard  min_profile_Jaccard  boot_ARI_mean  boot_ARI_p2.5  oob_ARI_mean  inertia  heldout_ARI_mean  heldout_silhouette  eligible
 2       0.388            236             0.363         0.935                0.920          0.876          0.610         0.877 1632.013             0.963               0.387      True
 3       0.377            147             0.226         0.833                0.761          0.834          0.541         0.834 1231.435             0.900               0.371      True
 4       0.419             79             0.122         0.913                0.815          0.929          0.827         0.925  937.003             0.966               0.412      True
 5       0.422             14             0.022         0.835                0.643          0.854          0.650         0.852  774.543             0.888               0.402     False
 6       0.391             12             0.018         0.880                0.7

In [2]:
# =============================================================================
# OBJECTIVE 4 (FINAL, v5) - PART 2 of 2: CHARACTERISATION OF THE PROFILES AND DETERMINANTS OF PROFILE MEMBERSHIP
# Reads ONLY the hand-off written by Part 1 v5 (P4_cohort_with_profiles.csv + P4_meta.json).
# Requires: numpy, pandas, scipy, scikit-learn (no statsmodels).
#
# WHAT CHANGED FROM v4 (each change answers a weakness found when the v4 output was audited):
#  C1  Tariff-like share per profile is reported; a profile with >= 20% global-package episodes is labelled
#      "mixed: global-package and short-stay episodes" (rule-based, not hand-typed).
#  C2  CIRCULARITY AUDIT: how much of each profile-defining variable is already determined by each predictor (eta-squared).
#      Ownership/care type are billing categories that mechanically determine cost-sharing and stay. Not data leakage (no profile
#      variable is a predictor, asserted) but the membership model is then largely a restatement of the tariff schedule.
#  C3  Second mirror check: NMI(profile, ownership x DIAGNOSIS TYPE) - the billing category, not only inpatient/non-inpatient.
#  C4  Membership model robustness: the primary model is refitted (a) without the global package and (b) with specialty added;
#      sex/age odds ratios are compared across the three fits so that a tariff-driven "sex effect" is visible.
#  C5  Predictor tests: Holm-adjusted permutation p-values and an automatic SUPPORTED column (Holm p < 0.05 AND cross-validated
#      log-loss worsens by >= 0.005 when the predictor is dropped).
#  C6  Odds ratios from sparse cells (|log OR| > 4) are marked "direction only" in the table itself.
#  C7  Confident-subset check uses OUT-OF-BAG assignment stability (not in-bag).
#  C8  Extra single-block CV models (ownership only, care type only, tariff block) to show where the predictive power sits.
# Design (unchanged): cohort = financing-reconciled episodes; ridge-penalised multinomial logit (penalty by cluster-grouped CV);
#  CLUSTER-BOOTSTRAP CIs; additive Andersen blocks (age, sex, year | insurance fund | ownership + care type); specialty exploratory.
# Environment variables for quick tests: P4_N_BOOT_OR (default 300), P4_N_PERM (500)
# =============================================================================
import json
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, normalized_mutual_info_score
from sklearn.model_selection import GroupKFold

warnings.filterwarnings("ignore")

BASE = Path(r"C:\Users\Shekoofeh\Documents\Health Insurance Data\Cholera")
OUT_DIR = BASE / "feature engineering v4_FINAL" / "objective4_outputs_v5"
for a in sys.argv[1:]:
    if not a.startswith("-") and Path(a).is_dir():
        OUT_DIR = Path(a)
COHORT = OUT_DIR / "P4_cohort_with_profiles.csv"
META = OUT_DIR / "P4_meta.json"
assert COHORT.is_file() and META.is_file(), f"Run Part 1 v5 first; hand-off files not found in {OUT_DIR}"


class Tee:
    def __init__(self, path, original):
        self.f = open(path, "w", encoding="utf-8")
        self.o = original

    def write(self, s):
        self.o.write(s)
        self.f.write(s)

    def flush(self):
        self.o.flush()
        self.f.flush()

    def __getattr__(self, name):
        return getattr(self.o, name)


if isinstance(sys.stdout, Tee):
    sys.stdout.f.close()
    sys.stdout = sys.stdout.o
sys.stdout = Tee(OUT_DIR / "objective4_part2_log.txt", sys.stdout)
pd.set_option("display.width", 250, "display.max_columns", 40, "display.max_colwidth", 70)

MIN_POOL_N = 20
MIN_OWN_N = 30
N_BOOT_OR = int(os.environ.get("P4_N_BOOT_OR", 300))
N_PERM = int(os.environ.get("P4_N_PERM", 500))
C_GRID = [0.05, 0.1, 0.25, 0.5, 1, 2, 5, 10, 50]
SEED = 42
SPARSE_LOGOR = 4.0                 # |log OR| above this = sparse-cell / near-separation -> direction only
MIN_CV_GAIN = 0.005                # minimum cross-validated log-loss loss on dropping a predictor to call it supported
GLOBAL_MIX_THR = 0.20              # share of global-package episodes that makes a profile "mixed"
ETA_FLAG = 0.50
INP = "Hospitalization / Inpatient"
BUNDLE_DEPT = "Limited surgery - D Clinic"
CAT_VARS = ["age_group", "sex", "insurance_fund_grp", "ownership3", "care_type3", "diagnosis_type_grp", "specialty_c",
            "admission_year"]
BLOCKS = [
    ("M1_predisposing+year", ["age_group", "sex", "admission_year"]),
    ("M2_+enabling(fund)", ["age_group", "sex", "admission_year", "insurance_fund_grp"]),
    ("M3_+ownership+care_type_PRIMARY", ["age_group", "sex", "admission_year", "insurance_fund_grp", "ownership3", "care_type3"]),
    ("M4_+specialty_EXPLORATORY", ["age_group", "sex", "admission_year", "insurance_fund_grp", "ownership3", "care_type3",
                                    "specialty_c"]),
]
EXTRA_BLOCKS = [("X1_ownership_only", ["ownership3"]), ("X2_care_type_only", ["care_type3"]),
                ("X3_ownership+care_type_only", ["ownership3", "care_type3"])]
PRIM_COVS = BLOCKS[2][1]


def banner(t):
    print("\n" + "=" * 90 + f"\n{t}\n" + "=" * 90)


meta = json.loads(META.read_text())
d = pd.read_csv(COHORT, low_memory=False)
K = int(meta["K"])
d["profile"] = d["profile"].astype(int)
d["care_type3"] = np.where(d["setting_cell"].astype(str).str.contains(BUNDLE_DEPT, regex=False), "D-clinic bundle",
                           np.where(d["diagnosis_type_grp"].astype(str) == INP, "Inpatient", "Non-inpatient"))
_vo = d["ownership_grp"].astype(str).value_counts()
d["ownership3"] = d["ownership_grp"].astype(str).where(d["ownership_grp"].astype(str).map(_vo) >= MIN_OWN_N, "Other ownership (pooled)")
d["tariff_group"] = np.where(d["global_pkg"].astype(bool), "global package",
                             np.where(d["tariff_like"].astype(bool), "other tied tariff combination", "not tariff-like"))
for v in CAT_VARS:
    d[v] = d[v].astype(str)
STAB = "assign_stability_oob" if "assign_stability_oob" in d.columns else "assign_stability"
PROFILE_VARS = {"los_days", "n_claim_lines", "n_service_groups", "total_cost", "cost_m", "oop", "oop_B", "los_t", "claims_t",
                "serv_t", "cost_t", "profile", "gapflag", "assign_stability", "assign_stability_oob", "silhouette_i",
                "lab_ward", "lab_gmm", "global_pkg", "tied_tariff", "tariff_like", "tariff_group"}
assert not (set(sum([c for _, c in BLOCKS + EXTRA_BLOCKS], [])) & PROFILE_VARS), "Leakage: profile-defining variable among predictors"
print(f"Loaded {len(d):,} episodes; K={K}; method={meta['method']}; cohort: {meta.get('cohort')}; "
      f"profile variables={meta['profile_vars']}; Part-1 structure verdict: {meta.get('structure_verdict')}")
print("Leakage check passed: LOS, claim lines, service groups, cost, cost-sharing and the tariff-like flags are NOT predictors.")
n = len(d)

# ----------------------------------------------------------------------------
# A. ASSIGNMENT QUALITY
# ----------------------------------------------------------------------------
banner("[A] ASSIGNMENT QUALITY")
QA = d.groupby("profile").agg(n=("profile", "size"), mean_silhouette=("silhouette_i", "mean"),
                              pct_negative_silhouette=("silhouette_i", lambda s: (s < 0).mean() * 100),
                              mean_assignment_stability_oob=(STAB, "mean"),
                              pct_stable_ge80_oob=(STAB, lambda s: (s >= 0.8).mean() * 100))
print(QA.round(3).to_string())
print(f"Overall silhouette {d['silhouette_i'].mean():.3f}; episodes reproduced in >= 80% of the bootstrap replicates in which they "
      f"were OUT-OF-BAG: {(d[STAB] >= 0.8).mean():.1%}")
print("Silhouette 0.25-0.50 = overlapping profiles (a continuum); negative silhouette = episode closer to another profile. "
      "Assignment stability (out-of-bag) is the classification-uncertainty analogue of posterior probability. High stability = "
      "reproducible, not proof of natural groups (see Part 1 null test).")
QA.to_csv(OUT_DIR / "P4_assignment_quality.csv")

# ----------------------------------------------------------------------------
# B. PROFILE CHARACTERISATION
# ----------------------------------------------------------------------------
banner("[B] PROFILE CHARACTERISATION (profiles ordered from lowest to highest mean log-cost)")


def miq(s, p=1):
    return f"{s.median():.{p}f} [{s.quantile(.25):.{p}f}-{s.quantile(.75):.{p}f}]"


tot_cost = d["cost_m"].sum()
rows = []
for lab, g in list(d.groupby("profile")) + [("All", d)]:
    rows.append({"Profile": lab, "N": len(g), "% episodes": len(g) / n * 100,
                 "LOS median [IQR] (days)": miq(g["los_days"], 0), "% LOS=0": (g["los_days"] == 0).mean() * 100,
                 "Claim lines median [IQR]": miq(g["n_claim_lines"], 0),
                 "Cost median [IQR] (M rial)": miq(g["cost_m"], 1), "Cost mean (M rial)": g["cost_m"].mean(),
                 "% of total expenditure": g["cost_m"].sum() / tot_cost * 100,
                 "Cost-share median [IQR] %": miq(g["oop"] * 100, 1), "Cost-share mean %": g["oop"].mean() * 100,
                 "Aggregate cost-share % (sum OOP/sum cost)": (g["oop"] * g["cost_m"]).sum() / g["cost_m"].sum() * 100,
                 "% inpatient": (g["care_type3"] == "Inpatient").mean() * 100,
                 "% private": (g["ownership3"] == "private").mean() * 100,
                 "% State University": (g["ownership3"] == "State University").mean() * 100,
                 "% global package": g["global_pkg"].astype(bool).mean() * 100,
                 "% tariff-like (any)": g["tariff_like"].astype(bool).mean() * 100})
T_B = pd.DataFrame(rows)
print(T_B.round(1).T.to_string(header=False))
T_B.to_csv(OUT_DIR / "P4_profile_characteristics.csv", index=False)

zvars = [v for v in ["los_t", "claims_t", "serv_t", "cost_t", "oop"] if v in d.columns and v in meta["profile_vars"]]
Zm = (d.groupby("profile")[zvars].mean() - d[zvars].mean()) / d[zvars].std()
print("\nProfile means in SD units of the whole cohort:")
print(Zm.round(2).to_string())
Zm.to_csv(OUT_DIR / "P4_profile_z_means.csv")


def lvl(x):
    return "high" if x >= .25 else ("low" if x <= -.25 else "mid")


res_cols = [v for v in ["los_t", "claims_t", "serv_t"] if v in Zm.columns]
pkg_share = d.groupby("profile")["global_pkg"].apply(lambda s: s.astype(bool).mean())
labs = []
for k, r in Zm.iterrows():
    base_lab = f"{lvl(r[res_cols].mean())} resource use / {lvl(r['cost_t'])} expenditure / {lvl(r['oop'])} patient cost-sharing"
    mixed = pkg_share[k] >= GLOBAL_MIX_THR
    labs.append({"profile": k, "pct_global_package": pkg_share[k] * 100,
                 "descriptive_label": ("mixed: global-package and short-stay episodes (" + base_lab + ")") if mixed else base_lab})
SUG = pd.DataFrame(labs)
print(f"\nDESCRIPTIVE labels (rule: z-mean <= -0.25 low, >= 0.25 high; a profile with >= {GLOBAL_MIX_THR:.0%} global-package "
      "episodes is labelled 'mixed'; never 'efficient', 'wasteful' or 'high value'; edit by hand if needed):")
print(SUG.round(1).to_string(index=False))
SUG.to_csv(OUT_DIR / "P4_profile_suggested_labels.csv", index=False)

dcl_file = OUT_DIR / "P4_dclinic_bundle_description.csv"
if dcl_file.is_file():
    print("\nD-clinic bundle (excluded from the primary cohort; unreconciled point-mass tariff group), from Part 1:")
    print(pd.read_csv(dcl_file).round(2).to_string(index=False))

tl_file = OUT_DIR / "P4_tariff_like_by_profile.csv"
if tl_file.is_file():
    print("\nTariff-like episodes by profile, from Part 1:")
    print(pd.read_csv(tl_file).round(1).to_string(index=False))

# ---- C1: decomposition of the mixed profile(s)
for k in SUG.loc[SUG.pct_global_package >= GLOBAL_MIX_THR * 100, "profile"]:
    g = d[d.profile == k]
    print(f"\nPROFILE {k} DECOMPOSITION (global-package episodes vs the rest of the profile):")
    DEC = g.groupby(g["global_pkg"].astype(bool).map({True: "global package", False: "other short-stay / mixed"})).agg(
        n=("profile", "size"), cost_median_M=("cost_m", "median"), cost_IQR_low=("cost_m", lambda s: s.quantile(.25)),
        cost_IQR_high=("cost_m", lambda s: s.quantile(.75)), oop_median=("oop", "median"),
        claim_lines_median=("n_claim_lines", "median"), pct_los0=("los_days", lambda s: (s == 0).mean() * 100),
        pct_female=("sex", lambda s: (s == "Female").mean() * 100),
        pct_non_academic_gov=("ownership3", lambda s: (s == "Non-academic government").mean() * 100),
        pct_inpatient=("care_type3", lambda s: (s == "Inpatient").mean() * 100))
    print(DEC.round(2).to_string())
    print("Specialty mix of the non-package part:")
    print(g[~g["global_pkg"].astype(bool)]["specialty_c"].value_counts().head(6).to_string())
    DEC.to_csv(OUT_DIR / f"P4_profile{k}_decomposition.csv")

# ----------------------------------------------------------------------------
# C. COMPOSITION BY ANDERSEN VARIABLES + MIRROR CHECKS + CIRCULARITY AUDIT
# ----------------------------------------------------------------------------
banner("[C] COMPOSITION OF PROFILES (modal assignment)")
comp = []
for v in CAT_VARS + ["tariff_group"]:
    ct = pd.crosstab(d[v], d["profile"])
    try:
        chi2, pchi = stats.chi2_contingency(ct)[:2]
        cv_ = float(np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1))))
    except ValueError:
        pchi, cv_ = np.nan, np.nan
    for l_ in ct.index:
        m = (d[v] == l_).values
        r = {"variable": v, "level": l_, "N": int(m.sum()), "chi2 p (exploratory)": pchi, "Cramers_V": cv_}
        for k in range(1, K + 1):
            r[f"profile {k}: row % (modal)"] = ct.loc[l_, k] / ct.loc[l_].sum() * 100 if k in ct.columns else 0.0
        comp.append(r)
T_C = pd.DataFrame(comp)
T_C.to_csv(OUT_DIR / "P4_profile_composition.csv", index=False)
print(T_C[["variable", "level", "N"] + [f"profile {k}: row % (modal)" for k in range(1, K + 1)]
          + ["Cramers_V", "chi2 p (exploratory)"]].round(2).to_string(index=False))
print("Rows show, for each level, the % of its episodes falling in each profile. Ownership/setting rows reflect the "
      "benefit/tariff schedule (Objective 3), not behaviour.")
mirror = d["ownership3"] + " | " + d["care_type3"]
mirror2 = d["ownership3"] + " | " + d["diagnosis_type_grp"]
nmi_m = normalized_mutual_info_score(d["profile"], mirror)
nmi_s = normalized_mutual_info_score(d["profile"], d["setting_cell"])
nmi_d = normalized_mutual_info_score(d["profile"], mirror2)
print(f"\nMirror check: NMI(profile, ownership x care type) = {nmi_m:.3f}; NMI(profile, setting_cell) = {nmi_s:.3f}; "
      f"NMI(profile, ownership x diagnosis type) = {nmi_d:.3f}")
_mx = max(nmi_m, nmi_s, nmi_d)
print("  " + ("> 0.60: the profiles largely reproduce the tariff structure - say so in the paper." if _mx > .6 else
              ("0.45-0.60: the profiles overlap SUBSTANTIALLY with ownership/setting/billing category while still carrying some "
               "additional information - report the NMI values." if _mx > .45 else
               "<= 0.45: profiles carry information beyond ownership/setting/billing category.")))
print(pd.crosstab(d["profile"], d["care_type3"]).to_string())

print("\nCIRCULARITY AUDIT - share of each profile-defining variable's variance explained by each predictor (eta-squared):")


def eta2(y, g):
    y = np.asarray(y, float)
    gm = pd.Series(y).groupby(np.asarray(g)).transform("mean").values
    return float(((gm - y.mean()) ** 2).sum() / ((y - y.mean()) ** 2).sum())


aud = []
for v in ["ownership3", "care_type3", "diagnosis_type_grp", "age_group", "sex", "insurance_fund_grp", "admission_year"]:
    aud.append({"predictor": v, **{pv: eta2(d[pv], d[v]) for pv in [c for c in ["los_t", "claims_t", "cost_t", "oop"] if c in d.columns]}})
aud.append({"predictor": "ownership3 x care_type3", **{pv: eta2(d[pv], mirror) for pv in ["los_t", "claims_t", "cost_t", "oop"]
                                                       if pv in d.columns}})
AUD = pd.DataFrame(aud)
print(AUD.round(3).to_string(index=False))
AUD.to_csv(OUT_DIR / "P4_circularity_audit.csv", index=False)
hi = [(r["predictor"], c) for _, r in AUD.iterrows() for c in AUD.columns[1:] if r[c] >= ETA_FLAG]
print(("FLAG: predictor(s) that largely DETERMINE a profile-defining variable (eta2 >= 0.5): " + "; ".join(f"{a} -> {b}" for a, b in hi) +
       ".\n  Not data leakage (the variables are not inputs to the model) but the membership model is then a description of the "
       "tariff schedule, not an explanation of utilisation behaviour.") if hi else "No predictor determines a profile variable (eta2 < 0.5).")

# ----------------------------------------------------------------------------
# D. MEMBERSHIP MODEL (ridge-penalised multinomial logit + cluster bootstrap)
# ----------------------------------------------------------------------------
banner("[D] DETERMINANTS OF PROFILE MEMBERSHIP (reference = largest profile)")
REF = int(d["profile"].value_counts().idxmax())
ORDER = [REF] + sorted(k for k in range(1, K + 1) if k != REF)
y_all = d["profile"].map({c: i for i, c in enumerate(ORDER)}).astype(int).values
sizes = d["profile"].value_counts().sort_index()
print(f"Reference profile: {REF} (n={int(sizes[REF])}). Profile sizes: { {int(k): int(v) for k, v in sizes.items()} }")
groups = pd.factorize(d["cl"])[0]


def pool(data, covs, m=MIN_POOL_N):
    data = data.copy()
    for v in covs:
        vc = data[v].value_counts()
        rare = vc[vc < m].index
        if len(rare) and len(vc) - len(rare) >= 1:
            data.loc[data[v].isin(rare), v] = "Other (pooled)"
    return data


def design(data, covs):
    X = pd.DataFrame({"const": 1.0}, index=data.index)
    refs, cols_of = {}, {}
    for v in covs:
        vc = data[v].value_counts()
        if len(vc) < 2:
            continue
        vc = vc.sort_index().sort_values(ascending=False, kind="stable")
        refs[v], cols_of[v] = vc.index[0], []
        for l_ in sorted(vc.index):
            if l_ == refs[v]:
                continue
            c = f"{v}: {l_}"
            X[c] = (data[v] == l_).astype(float)
            cols_of[v].append(c)
    keep, r = ["const"], 1
    for c in X.columns[1:]:
        r2 = np.linalg.matrix_rank(X[keep + [c]].values)
        if r2 > r:
            keep.append(c)
            r = r2
    dropped = [c for c in X.columns if c not in keep]
    return X[keep], refs, {v: [c for c in cs if c in keep] for v, cs in cols_of.items()}, dropped


def design_report(data, X, covs, y, label):
    pcols = X.shape[1] - 1
    min_n = int(np.bincount(y, minlength=K).min())
    Xs = X.drop(columns="const")
    Xs = (Xs - Xs.mean()) / Xs.std().replace(0, 1)
    cond = float(np.linalg.cond(Xs.values))
    empty = small = 0
    for v in covs:
        if v in data.columns:
            ct = pd.crosstab(data[v], y)
            empty += int((ct.values == 0).sum())
            small += int((ct.values < 5).sum())
    epc = min_n / max(pcols, 1)
    print(f"  {label}: columns={pcols}, smallest profile n={min_n}, events/column={epc:.1f}, empty level x profile cells={empty}, "
          f"cells<5={small}, condition no. (standardised)={cond:.1f}")
    return {"model": label, "columns": pcols, "min_profile_n": min_n, "events_per_column": epc, "empty_cells": empty,
            "cells_lt5": small, "condition_number_std": cond}


def fit_pen(Xv, y, C):
    return LogisticRegression(C=C, max_iter=5000).fit(Xv, y)


def diffs(m):
    cf = np.zeros((K, m.coef_.shape[1]))
    cf[m.classes_] = m.coef_
    return cf[1:] - cf[0]                    # (K-1, p): profile j vs reference (class 0)


def ll_in(m, Xv, y):
    P = np.zeros((len(y), K))
    P[:, m.classes_] = m.predict_proba(Xv)
    return -log_loss(y, np.clip(P, 1e-9, 1), labels=list(range(K)), normalize=False)


def cv_ll(Xv, y, C, nfold=5, grp=None):
    grp = groups if grp is None else grp
    P = np.zeros((len(y), K))
    for tr, te in GroupKFold(nfold).split(Xv, y, grp):
        if Xv.shape[1] == 0:
            P[te] = np.bincount(y[tr], minlength=K) / len(tr)
        else:
            m = fit_pen(Xv[tr], y[tr], C)
            P[np.ix_(te, m.classes_)] = m.predict_proba(Xv[te])
    return log_loss(y, np.clip(P, 1e-9, 1), labels=list(range(K)))


PRIMD = pool(d, sum([c for _, c in BLOCKS], []))
dr_rows, blk_rows = [], []
Xp, refs_p, cols_p, dropped_p = design(PRIMD, PRIM_COVS)
print("\nDesign diagnostics (events/column < 10 or empty cells = use the penalised estimator):")
for bname, covs in BLOCKS:
    Xb, _, _, _ = design(PRIMD, covs)
    dr_rows.append(design_report(PRIMD, Xb, covs, y_all, bname))
pd.DataFrame(dr_rows).to_csv(OUT_DIR / "P4_membership_design_diagnostics.csv", index=False)
if dropped_p:
    print(f"Collinear columns dropped from the primary design: {dropped_p}")
Xv = Xp.drop(columns="const").values
pnames = [c for c in Xp.columns if c != "const"]

cvs = {C: cv_ll(Xv, y_all, C) for C in C_GRID}
C_STAR = min(cvs, key=cvs.get)
print("\nGrouped 5-fold CV log-loss by ridge penalty C (smaller C = stronger shrinkage): "
      + ", ".join(f"C={c}: {v:.4f}" for c, v in cvs.items()) + f"\n -> C* = {C_STAR}")
print("(C* is chosen on the same folds that are then used to report pseudo-R2: the pseudo-R2 values are slightly optimistic; "
      "the gap between blocks, not the absolute level, is the finding.)")

null_ll = cv_ll(np.zeros((n, 0)), y_all, 1.0)
for bname, covs in BLOCKS + EXTRA_BLOCKS:
    Xb, _, _, _ = design(PRIMD, covs)
    ll_b = cv_ll(Xb.drop(columns="const").values, y_all, C_STAR)
    blk_rows.append({"model": bname, "columns": Xb.shape[1] - 1, "CV_logloss": ll_b, "CV_logloss_null": null_ll,
                     "CV_pseudoR2": 1 - ll_b / null_ll})
T_D1 = pd.DataFrame(blk_rows)
print("\nAndersen blocks and single-block models (out-of-sample, cluster-grouped CV):")
print(T_D1.round(4).to_string(index=False))
T_D1.to_csv(OUT_DIR / "P4_membership_andersen_blocks.csv", index=False)
print("CV pseudo-R2 near 0 = the characteristics barely predict profile membership out-of-sample. If ownership/care type alone "
      "reach almost the full pseudo-R2, the model restates the tariff schedule (see circularity audit).")

m_star = fit_pen(Xv, y_all, C_STAR)
B0 = diffs(m_star)
grp_idx = [np.where(groups == k)[0] for k in range(groups.max() + 1)]
rng = np.random.default_rng(SEED)
boots = []
for _ in range(N_BOOT_OR):
    idx = np.concatenate([grp_idx[k] for k in rng.integers(0, len(grp_idx), len(grp_idx))])
    if len(np.unique(y_all[idx])) < K:
        continue
    try:
        boots.append(diffs(fit_pen(Xv[idx], y_all[idx], C_STAR)))
    except Exception:
        continue
boots = np.array(boots)
print(f"\nValid cluster-bootstrap replicates: {len(boots)}/{N_BOOT_OR}")
tabs = []
for j in range(K - 1):
    for i, term in enumerate(pnames):
        b = boots[:, j, i] if len(boots) else np.array([np.nan])
        lo, hi_ = np.percentile(b, [2.5, 97.5])
        v = term.split(": ")[0]
        sparse = bool(abs(B0[j, i]) > SPARSE_LOGOR)
        tabs.append({"profile": ORDER[j + 1], "vs_reference_profile": REF, "term": term, "reference_level": refs_p.get(v, ""),
                     "OR_penalised": np.exp(B0[j, i]), "ci_low": np.exp(lo), "ci_high": np.exp(hi_),
                     "ci_excludes_1": bool(lo > 0 or hi_ < 0), "sparse_cell_direction_only": sparse,
                     "reading": ("direction only (sparse cells)" if sparse else "usable magnitude"),
                     "boot_sign_p": min(1.0, 2 * min((b > 0).mean(), (b < 0).mean())) if len(boots) else np.nan})
T_D2 = pd.DataFrame(tabs)
T_D2.to_csv(OUT_DIR / "P4_membership_primary_odds_ratios.csv", index=False)
print(f"Penalised odds ratios (profile vs reference profile {REF}) saved; {int(T_D2.ci_excludes_1.sum())} of {len(T_D2)} "
      "have a bootstrap CI excluding 1. Ridge shrinks ORs toward 1 (conservative); read them as direction and relative size. "
      f"Terms with |log OR| > {SPARSE_LOGOR:g} come from near-empty cells and are marked 'direction only'.")
print(T_D2[T_D2.ci_excludes_1].round(3).to_string(index=False) if T_D2.ci_excludes_1.any() else "  none")

# ---- C4: robustness of the sex / age terms (point estimates; bootstrap CIs are for the primary model only)
print("\nROBUSTNESS OF THE DEMOGRAPHIC TERMS (penalised OR vs reference profile; point estimates):")


def pen_or(data, covs, mask=None, label=""):
    dd = data if mask is None else data[mask]
    Xd, refs_d, _, _ = design(dd, covs)
    yy = dd["profile"].map({c: i for i, c in enumerate(ORDER)}).astype(int).values
    if len(np.unique(yy)) < K:
        return None
    Bd = diffs(fit_pen(Xd.drop(columns="const").values, yy, C_STAR))
    names = [c for c in Xd.columns if c != "const"]
    return pd.DataFrame({(ORDER[j + 1], t): np.exp(Bd[j, i]) for j in range(K - 1) for i, t in enumerate(names)},
                        index=[label]).T


fits = [pen_or(PRIMD, PRIM_COVS, None, "M3 all"),
        pen_or(PRIMD, PRIM_COVS, ~PRIMD["global_pkg"].astype(bool).values, "M3 excl. global pkg"),
        pen_or(PRIMD, BLOCKS[3][1], None, "M4 + specialty")]
ROB = pd.concat([f for f in fits if f is not None], axis=1)
ROB.index = pd.MultiIndex.from_tuples(ROB.index, names=["profile", "term"])
sel = ROB[[t.startswith(("sex", "age_group")) for _, t in ROB.index]]
print(sel.round(2).to_string())
ROB.to_csv(OUT_DIR / "P4_membership_robustness_OR.csv")
if "sex: Female" in ROB.index.get_level_values("term") and "M4 + specialty" in ROB.columns:
    sx = ROB.xs("sex: Female", level="term")
    att = 1 - np.log(sx["M4 + specialty"]).abs() / np.log(sx["M3 all"]).abs()
    att_pkg = 1 - np.log(sx["M3 excl. global pkg"]).abs() / np.log(sx["M3 all"]).abs()
    big = sx["M3 all"].sub(1).abs().idxmax()
    print(f"SEX TERM (profile {big}, largest association): OR {sx.loc[big, 'M3 all']:.2f}; without the global package "
          f"{sx.loc[big, 'M3 excl. global pkg']:.2f} (log-OR reduced by {att_pkg[big]:.0%}); with specialty added "
          f"{sx.loc[big, 'M4 + specialty']:.2f} (log-OR reduced by {att[big]:.0%}).")
    print("  " + ("Attenuated by specialty: the sex term partly reflects service line (obstetrics/gynaecology/urology are sex-linked). "
                  if att[big] > 0.20 else "Not materially attenuated by specialty. ") +
          "Interpret as an association with service mix/billing, never as a sex effect on resource use.")

ll_full = ll_in(m_star, Xv, y_all)
cv_full = cv_ll(Xv, y_all, C_STAR)
jr = []
rng_p = np.random.default_rng(SEED + 1)
for v in PRIM_COVS:
    if v not in cols_p or not cols_p[v]:
        continue
    idx_v = [pnames.index(c) for c in cols_p[v]]
    keep_i = [i for i in range(len(pnames)) if i not in idx_v]
    Xr = Xv[:, keep_i]
    m_r = fit_pen(Xr, y_all, C_STAR)
    ll_r = ll_in(m_r, Xr, y_all)
    obs = ll_full - ll_r
    d_cv = cv_ll(Xr, y_all, C_STAR) - cv_full             # > 0: dropping v hurts out-of-sample
    cnt = 0
    for _ in range(N_PERM):
        Xq = Xv.copy()
        Xq[:, idx_v] = Xv[rng_p.permutation(n)][:, idx_v]
        if ll_in(fit_pen(Xq, y_all, C_STAR), Xq, y_all) - ll_r >= obs:
            cnt += 1
    jr.append({"variable": v, "n_columns": len(idx_v), "in_sample_loglik_gain": obs, "CV_logloss_increase_if_dropped": d_cv,
               "permutation_p": (cnt + 1) / (N_PERM + 1)})
T_D3 = pd.DataFrame(jr)
order_p = T_D3["permutation_p"].rank(method="first").astype(int)
m_t = len(T_D3)
holm = np.empty(m_t)
srt = T_D3["permutation_p"].sort_values()
run = 0.0
for rank_i, (ix, pv) in enumerate(srt.items()):
    run = max(run, min(1.0, (m_t - rank_i) * pv))
    holm[T_D3.index.get_loc(ix)] = run
T_D3["p_holm"] = holm
T_D3["SUPPORTED"] = (T_D3["p_holm"] < 0.05) & (T_D3["CV_logloss_increase_if_dropped"] >= MIN_CV_GAIN)
print("\nPredictor-level tests in the primary model (exploratory):")
print(T_D3.round(4).to_string(index=False))
T_D3.to_csv(OUT_DIR / "P4_membership_joint_tests.csv", index=False)
print(f"SUPPORTED = Holm-adjusted permutation p < 0.05 AND dropping the predictor worsens cross-validated log-loss by >= "
      f"{MIN_CV_GAIN}. Ownership and care type are partly nested: read them together; both are billing categories.")

# ----------------------------------------------------------------------------
# E. CLASSIFICATION-UNCERTAINTY CHECK (all episodes vs confidently assigned episodes, OUT-OF-BAG stability)
# ----------------------------------------------------------------------------
banner("[E] CLASSIFICATION-UNCERTAINTY CHECK: membership model on confidently assigned episodes only")
try:
    conf = (d[STAB] >= 0.8).values
    print(f"Confidently assigned episodes (out-of-bag stability >= 0.8): {int(conf.sum())} of {n} ({conf.mean():.1%})")
    if len(np.unique(y_all[conf])) == K and conf.sum() > 100:
        h = diffs(fit_pen(Xv, y_all, C_STAR)).ravel()
        s_ = diffs(fit_pen(Xv[conf], y_all[conf], C_STAR)).ravel()
        big = (np.abs(h) > 0.1) & (np.abs(s_) > 0.1)
        corr = float(np.corrcoef(h, s_)[0, 1])
        agree = float((np.sign(h[big]) == np.sign(s_[big])).mean()) if big.any() else np.nan
        print(f"Correlation of coefficients (all vs confident subset): {corr:.3f}; sign agreement (|coef|>0.1): {agree:.1%}")
        print("High agreement = assignment uncertainty does not change the determinant pattern. NOTE: the confident subset is "
              "selected on stability of the same partition, so this check is necessarily favourable; it is not an external validation.")
        pd.DataFrame({"term": pnames * (K - 1), "profile": np.repeat(ORDER[1:], len(pnames)),
                      "logOR_all": h, "logOR_confident": s_}).to_csv(OUT_DIR / "P4_membership_confident_vs_all.csv", index=False)
    else:
        print("  too few confidently assigned episodes in some profile - check skipped.")
except Exception as e:
    print("  check failed:", str(e)[:100])

banner("REPORTING NOTES")
print("""1. Profiles are EMPIRICAL (distance-based clustering); no threshold, weight, value or efficiency score was imposed.
2. Cohort = financing-reconciled episodes, because recorded cost-sharing in unreconciled private episodes is a recording
   artefact (Part 1 evidence table). The D-clinic bundle (100% unreconciled) is reported separately. State that the profiles
   describe RECONCILED episodes only.
3. Report the rule-based K table WITH held-out ARI, per-profile bootstrap Jaccard (incl. 2.5th percentile), silhouette, out-of-bag
   and held-out ARI, Ward agreement, the null-structure test and the STRUCTURE VERDICT. K is a chosen resolution (K=2/3 coarser).
   Call the partition an exploratory typology of a continuum. The Gaussian mixture is a sensitivity analysis only.
4. High stability/ARI values show REPRODUCIBILITY, not validity: the profile variables are strongly tariff-determined, so the
   partition is reproducible partly because billing packages are point masses. Report the tariff-like table, S8 and S9.
5. A profile containing a large global-package share is reported as "mixed: global-package and short-stay episodes".
6. Cost-sharing is largely tariff/ownership-determined (Objective 3): ownership/care-type associations describe the benefit
   schedule, not patient behaviour. The circularity audit and single-block models document this.
7. Membership model = ASSOCIATIONS in cross-sectional claims. Report penalised ORs with bootstrap CIs, CV pseudo-R2 (slightly
   optimistic: C chosen on the same folds), Holm-adjusted permutation tests and the SUPPORTED column. Odds ratios marked
   'direction only' come from sparse cells. Age, fund and year are reported as NOT supported unless SUPPORTED is TRUE.
   The sex term must be read together with the robustness table (service-mix confounding).
8. k-means on log1p of discrete variables is distance-based (no density to degenerate); Ward, S7 (de-quantised) and the mixture
   sensitivity guard against tie artefacts.
9. Never describe any profile as inefficient, wasteful or high value.""")
banner("PART 2 DONE")
print(f"All outputs in:\n{OUT_DIR}")

Loaded 650 episodes; K=4; method=k-means (standardised profile variables), rule-based K, cluster-bootstrap Jaccard; cohort: financing-reconciled (flag_financing_gap_v3_requested == 0); profile variables=['los_t', 'claims_t', 'cost_t', 'oop']; Part-1 structure verdict: WEAK-TO-MODERATE (profiles reproducible but overlapping); profile(s) [3] unreliable in some resamples
Leakage check passed: LOS, claim lines, service groups, cost, cost-sharing and the tariff-like flags are NOT predictors.

[A] ASSIGNMENT QUALITY
           n  mean_silhouette  pct_negative_silhouette  mean_assignment_stability_oob  pct_stable_ge80_oob
profile                                                                                                   
1        295            0.459                    0.000                          0.980               95.254
2        145            0.357                    2.759                          0.978               94.483
3         79            0.204                   10.127  